# codex/competition-hardening Reproducible Pipeline

Pinned commit: `a658723ef81f5c7c70fd15f720c2e440f50f4694`  
Experiment 0 label: **REPRODUCTION ONLY — NOT A COMPARATIVE SCORE**

This notebook embeds the complete branch implementation. It materializes a run-scoped package and never imports the repository checkout. External embedding weights are disabled.

In [ ]:
from __future__ import annotations
import base64, hashlib, json, os, platform, shutil, subprocess, sys, tempfile, time
from pathlib import Path

BRANCH_NAME = 'codex/competition-hardening'
BRANCH_SHA = 'a658723ef81f5c7c70fd15f720c2e440f50f4694'
SEED = int(os.environ.get('BER_SEED', '2026'))
DATA_ROOT = Path(os.environ.get('BER_DATA_ROOT', 'dataset')).expanduser().resolve()
RUN_ROOT = Path(os.environ.get('BER_RUN_ROOT', f'artifacts/notebook-{BRANCH_SHA[:8]}')).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get('BER_OUTPUT_ROOT', str(RUN_ROOT / 'output'))).expanduser().resolve()
VALIDATOR_PATH = Path(os.environ.get('BER_VALIDATOR_PATH', 'utils/validate_submission.py')).expanduser().resolve()
RUN_STAGES = {s.strip() for s in os.environ.get('BER_RUN_STAGES', 'inspect').split(',') if s.strip()}
RUN_TEST = os.environ.get('BER_RUN_TEST', '0').lower() in {'1','true','yes'}
INSTALL_DEPS = os.environ.get('BER_INSTALL_DEPS', '0').lower() in {'1','true','yes'}
PACKAGE_ROOT = RUN_ROOT / '_embedded_package' / BRANCH_SHA
SOURCE_ROOT = PACKAGE_ROOT / 'code' / 'business_entity_resolution'
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
print({'branch': BRANCH_NAME, 'sha': BRANCH_SHA, 'data_root': str(DATA_ROOT), 'run_root': str(RUN_ROOT), 'run_stages': sorted(RUN_STAGES), 'run_test': RUN_TEST})


In [ ]:
def materialize(relative_path: str, payload_b64: str, expected_sha256: str) -> Path:
    payload = base64.b64decode(payload_b64.encode('ascii'))
    actual = hashlib.sha256(payload).hexdigest()
    if actual != expected_sha256:
        raise RuntimeError(f'embedded source hash mismatch for {relative_path}: {actual}')
    destination = PACKAGE_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(payload)
    return destination


## Embedded source: `code/business_entity_resolution/README.md`

~~~~text
# Amazon ML 2026 Business Entity Resolution

This package implements a reproducible pipeline for linking every Source 1 entity to zero, one, or multiple Source 2/Source 3 records. It uses only the supplied challenge data and performs no external business lookup, geocoding, registry access, or internet augmentation.

## Robustness guarantees

- **Crash-safe resume.** Every long stage (prepare, candidates, embeddings, features) processes work in shards and commits each shard atomically (temp file + `os.replace`) with a config/input fingerprint. Re-running the same command skips completed shards and resumes from the last committed one — no full restart. Use `--force` to redo a stage.
- **Hardware-adaptive resources.** `resources.py` detects available RAM, CPU count, and CUDA/VRAM at run time and derives chunk/batch sizes from them. Nothing is hardcoded.
- **Bounded pair processing.** Candidate queries, pair features, scoring, and output writing are sharded. Target blocker indexes remain source-sized and must be capacity-tested before a full run.
- **Continuous logging.** Each stage writes to stdout and to `artifacts/<run_id>/logs/<stage>.log`, flushed per line, including per-shard progress, ETA, peak memory, and LightGBM training metrics.
- **Format guarantee.** Before writing, outputs pass an in-process gate (one row per test S1, no dupes, no whitespace, valid prefixes, matches ⊆ candidates). `write-output` can regenerate formatting from cached scores/candidates without re-running the model.

## Architecture

```text
Raw TSV
  → strict audit and schema validation
  → raw-preserving normalization (US / India / France legal suffixes)
  → leakage-safe S1 folds
  → complementary candidate generation (7-way union, resumable)
  → optional experimental embedding features (disabled by default)
  → candidate-only pair features (39 features)
  → deterministic / SGD / optional LightGBM scorer
  → macro-F0.5 threshold tuning on complete held-out S1 folds
  → format-gated matching_results.tsv + candidate_pairs.tsv
  → internal preflight + official validator
```

The final candidate table is the exact table passed to feature extraction and scoring. It is also the source for `candidate_pairs.tsv`.

## Models and licenses

- LightGBM (`lightgbm==4.6.0`), MIT licensed.
- The optional BGE-M3 integration is experimental and disabled in every supplied profile. Do not enable it unless the organizers explicitly confirm that pretrained representations satisfy the supplied-data-only rule.
- `country` is treated strictly as an open-set string label; it is never hardcoded.


## Directory layout

```text
src/business_entity_resolution/  Production package and CLI
tests/                           Unit and integration tests
configs/                         Portable local/remote JSON configurations
notebooks/                       Thin exploratory clients of the package
requirements.txt                 Core runtime dependencies
requirements-dev.txt             Test and notebook dependencies
requirements-remote.txt          Optional full-scale LightGBM backend
```

Generated artifacts are written to configured `artifacts/` and `output/` roots.

## Installation

Provision a fresh machine with the helper script (detects GPU, picks the right torch wheel):

```bash
bash setup.sh                      # compliant text-feature pipeline
ENABLE_EMBED=1 bash setup.sh       # experimental; requires organizer approval
```

Or install manually from `code/business_entity_resolution/`:

```bash
python -m pip install -e .
python -m pip install -r requirements-dev.txt
python -m pip install -r requirements-remote.txt   # LightGBM backend
```

## Configuration

Precedence:

```text
CLI override > BER_* environment variable > JSON config > default
```

Environment variables:

```text
BER_DATA_ROOT
BER_ARTIFACT_ROOT
BER_OUTPUT_ROOT
BER_RUN_ID
BER_DEVICE
BER_THREADS
```

`resources` and `embeddings` config blocks control device mode (`auto`/`cpu`/`gpu`),
the RAM/VRAM fractional budgets, and whether BGE-M3 features are enabled
(`true`/`false`; supplied profiles use `false`). `n_shards` controls checkpoint granularity.

## Input contract

```text
dataset/
├── train/
│   ├── train_source1.tsv
│   ├── train_source2.tsv
│   ├── train_source3.tsv
│   └── train_ground_truth.tsv
└── test/
    ├── test_source1.tsv
    ├── test_source2.tsv
    └── test_source3.tsv
```

All files are UTF-8 and tab-separated. Blank strings are preserved. Country is an arbitrary open-set string; no fixed `{US, India}` enumeration exists.

## Mini end-to-end cycle (run this first)

`make-mini` samples 15 realistic Source 1 entities (6 US / 6 India / 3 France,
including singletons, single matches, multi-matches, and both-source matches)
plus decoys, straight from the real files, into `dataset/mini/` with the exact
real schema. Use it to verify the whole cycle end-to-end before the full run.

```bash
python run_pipeline.py --profile mini --skip-embeddings \
  --validator /path/to/student_resource/utils/validate_submission.py
```

The runner tunes the decision threshold on fold 0, reports an unbiased estimate
on a separately trained fold-1 model, then trains the final model on all rows.

Any of these commands can be re-run after a crash and will resume.

## Remote full-scale workflow

Use the same source code on a machine with 16–32 vCPU, 64–128 GB RAM, and fast temporary storage:

```bash
export BER_DATA_ROOT=/mounted/challenge/dataset
export BER_ARTIFACT_ROOT=/mounted/work/artifacts
export BER_OUTPUT_ROOT=/mounted/work/output
export BER_THREADS=16

python -m business_entity_resolution audit --config configs/remote_full.json
python -m business_entity_resolution prepare --config configs/remote_full.json --split both
python -m business_entity_resolution make-splits --config configs/remote_full.json --folds 10
python -m business_entity_resolution generate-candidates --config configs/remote_full.json --split train
python -m business_entity_resolution build-features --config configs/remote_full.json --split train
python -m business_entity_resolution train --config configs/remote_full.json --exclude-folds 0,1 --output-name lightgbm-validation-fold0
python -m business_entity_resolution score --config configs/remote_full.json --split train --model-path /mounted/work/artifacts/remote-full/models/lightgbm-validation-fold0.txt --validation-fold 0 --output-name validation-fold0.parquet
python -m business_entity_resolution tune-decision --config configs/remote_full.json --score-file validation-fold0.parquet --validation-fold 0

# Independent evaluation: train without fold 1, then score/evaluate fold 1.
python -m business_entity_resolution train --config configs/remote_full.json --validation-fold 1 --output-name lightgbm-validation-fold1
python -m business_entity_resolution score --config configs/remote_full.json --split train --model-path /mounted/work/artifacts/remote-full/models/lightgbm-validation-fold1.txt --validation-fold 1 --output-name evaluation-fold1.parquet
python -m business_entity_resolution evaluate --config configs/remote_full.json --score-file evaluation-fold1.parquet --validation-fold 1
```

After selecting and freezing a validated model and threshold:

```bash
python -m business_entity_resolution train --config configs/remote_full.json --all-training-data --output-name lightgbm-final
python -m business_entity_resolution prepare --config configs/remote_full.json --split test
python -m business_entity_resolution generate-candidates --config configs/remote_full.json --split test
python -m business_entity_resolution build-features --config configs/remote_full.json --split test
python -m business_entity_resolution infer --config configs/remote_full.json \
    --model-path /mounted/work/artifacts/remote-full/models/lightgbm-final.txt
python -m business_entity_resolution preflight --config configs/remote_full.json \
    --official-validator /mounted/challenge/utils/validate_submission.py --check-ids
python -m business_entity_resolution package --config configs/remote_full.json --team-name YOUR_TEAM \
    --documentation /mounted/challenge/Documentation_template.md
```

Keep embedding features disabled for competition runs unless the supplied-data-only and model-license interpretation is confirmed in writing.

Complete the supplied documentation template before packaging. The package command refuses to create a submission archive without it.

## Outputs

```text
output/
├── matching_results.tsv
└── candidate_pairs.tsv
```

Both files contain exactly one row per test S1, including blank rows. Internal preflight enforces target existence, duplicate rules, candidate containment, exact columns, and empty-list handling before the official validator runs.

## Reproducibility and compliance

- Seed defaults to 2026.
- Folds and negative sampling use stable cryptographic hashes.
- Candidate ordering and output lists are deterministic.
- Stage artifacts use explicit schemas and manifests.
- Raw challenge files are never modified.
- Supplied challenge data only in the default and recommended workflows; no external address/entity lookup.
- LightGBM is optional and MIT licensed; any use must be recorded in the run manifest.
- A final model is not selected until held-out experiments run on remote compute.

~~~~

In [ ]:
materialize('code/business_entity_resolution/README.md', 'IyBBbWF6b24gTUwgMjAyNiBCdXNpbmVzcyBFbnRpdHkgUmVzb2x1dGlvbgoKVGhpcyBwYWNrYWdlIGltcGxlbWVudHMgYSByZXByb2R1Y2libGUgcGlwZWxpbmUgZm9yIGxpbmtpbmcgZXZlcnkgU291cmNlIDEgZW50aXR5IHRvIHplcm8sIG9uZSwgb3IgbXVsdGlwbGUgU291cmNlIDIvU291cmNlIDMgcmVjb3Jkcy4gSXQgdXNlcyBvbmx5IHRoZSBzdXBwbGllZCBjaGFsbGVuZ2UgZGF0YSBhbmQgcGVyZm9ybXMgbm8gZXh0ZXJuYWwgYnVzaW5lc3MgbG9va3VwLCBnZW9jb2RpbmcsIHJlZ2lzdHJ5IGFjY2Vzcywgb3IgaW50ZXJuZXQgYXVnbWVudGF0aW9uLgoKIyMgUm9idXN0bmVzcyBndWFyYW50ZWVzCgotICoqQ3Jhc2gtc2FmZSByZXN1bWUuKiogRXZlcnkgbG9uZyBzdGFnZSAocHJlcGFyZSwgY2FuZGlkYXRlcywgZW1iZWRkaW5ncywgZmVhdHVyZXMpIHByb2Nlc3NlcyB3b3JrIGluIHNoYXJkcyBhbmQgY29tbWl0cyBlYWNoIHNoYXJkIGF0b21pY2FsbHkgKHRlbXAgZmlsZSArIGBvcy5yZXBsYWNlYCkgd2l0aCBhIGNvbmZpZy9pbnB1dCBmaW5nZXJwcmludC4gUmUtcnVubmluZyB0aGUgc2FtZSBjb21tYW5kIHNraXBzIGNvbXBsZXRlZCBzaGFyZHMgYW5kIHJlc3VtZXMgZnJvbSB0aGUgbGFzdCBjb21taXR0ZWQgb25lIOKAlCBubyBmdWxsIHJlc3RhcnQuIFVzZSBgLS1mb3JjZWAgdG8gcmVkbyBhIHN0YWdlLgotICoqSGFyZHdhcmUtYWRhcHRpdmUgcmVzb3VyY2VzLioqIGByZXNvdXJjZXMucHlgIGRldGVjdHMgYXZhaWxhYmxlIFJBTSwgQ1BVIGNvdW50LCBhbmQgQ1VEQS9WUkFNIGF0IHJ1biB0aW1lIGFuZCBkZXJpdmVzIGNodW5rL2JhdGNoIHNpemVzIGZyb20gdGhlbS4gTm90aGluZyBpcyBoYXJkY29kZWQuCi0gKipCb3VuZGVkIHBhaXIgcHJvY2Vzc2luZy4qKiBDYW5kaWRhdGUgcXVlcmllcywgcGFpciBmZWF0dXJlcywgc2NvcmluZywgYW5kIG91dHB1dCB3cml0aW5nIGFyZSBzaGFyZGVkLiBUYXJnZXQgYmxvY2tlciBpbmRleGVzIHJlbWFpbiBzb3VyY2Utc2l6ZWQgYW5kIG11c3QgYmUgY2FwYWNpdHktdGVzdGVkIGJlZm9yZSBhIGZ1bGwgcnVuLgotICoqQ29udGludW91cyBsb2dnaW5nLioqIEVhY2ggc3RhZ2Ugd3JpdGVzIHRvIHN0ZG91dCBhbmQgdG8gYGFydGlmYWN0cy88cnVuX2lkPi9sb2dzLzxzdGFnZT4ubG9nYCwgZmx1c2hlZCBwZXIgbGluZSwgaW5jbHVkaW5nIHBlci1zaGFyZCBwcm9ncmVzcywgRVRBLCBwZWFrIG1lbW9yeSwgYW5kIExpZ2h0R0JNIHRyYWluaW5nIG1ldHJpY3MuCi0gKipGb3JtYXQgZ3VhcmFudGVlLioqIEJlZm9yZSB3cml0aW5nLCBvdXRwdXRzIHBhc3MgYW4gaW4tcHJvY2VzcyBnYXRlIChvbmUgcm93IHBlciB0ZXN0IFMxLCBubyBkdXBlcywgbm8gd2hpdGVzcGFjZSwgdmFsaWQgcHJlZml4ZXMsIG1hdGNoZXMg4oqGIGNhbmRpZGF0ZXMpLiBgd3JpdGUtb3V0cHV0YCBjYW4gcmVnZW5lcmF0ZSBmb3JtYXR0aW5nIGZyb20gY2FjaGVkIHNjb3Jlcy9jYW5kaWRhdGVzIHdpdGhvdXQgcmUtcnVubmluZyB0aGUgbW9kZWwuCgojIyBBcmNoaXRlY3R1cmUKCmBgYHRleHQKUmF3IFRTVgogIOKGkiBzdHJpY3QgYXVkaXQgYW5kIHNjaGVtYSB2YWxpZGF0aW9uCiAg4oaSIHJhdy1wcmVzZXJ2aW5nIG5vcm1hbGl6YXRpb24gKFVTIC8gSW5kaWEgLyBGcmFuY2UgbGVnYWwgc3VmZml4ZXMpCiAg4oaSIGxlYWthZ2Utc2FmZSBTMSBmb2xkcwogIOKGkiBjb21wbGVtZW50YXJ5IGNhbmRpZGF0ZSBnZW5lcmF0aW9uICg3LXdheSB1bmlvbiwgcmVzdW1hYmxlKQogIOKGkiBvcHRpb25hbCBleHBlcmltZW50YWwgZW1iZWRkaW5nIGZlYXR1cmVzIChkaXNhYmxlZCBieSBkZWZhdWx0KQogIOKGkiBjYW5kaWRhdGUtb25seSBwYWlyIGZlYXR1cmVzICgzOSBmZWF0dXJlcykKICDihpIgZGV0ZXJtaW5pc3RpYyAvIFNHRCAvIG9wdGlvbmFsIExpZ2h0R0JNIHNjb3JlcgogIOKGkiBtYWNyby1GMC41IHRocmVzaG9sZCB0dW5pbmcgb24gY29tcGxldGUgaGVsZC1vdXQgUzEgZm9sZHMKICDihpIgZm9ybWF0LWdhdGVkIG1hdGNoaW5nX3Jlc3VsdHMudHN2ICsgY2FuZGlkYXRlX3BhaXJzLnRzdgogIOKGkiBpbnRlcm5hbCBwcmVmbGlnaHQgKyBvZmZpY2lhbCB2YWxpZGF0b3IKYGBgCgpUaGUgZmluYWwgY2FuZGlkYXRlIHRhYmxlIGlzIHRoZSBleGFjdCB0YWJsZSBwYXNzZWQgdG8gZmVhdHVyZSBleHRyYWN0aW9uIGFuZCBzY29yaW5nLiBJdCBpcyBhbHNvIHRoZSBzb3VyY2UgZm9yIGBjYW5kaWRhdGVfcGFpcnMudHN2YC4KCiMjIE1vZGVscyBhbmQgbGljZW5zZXMKCi0gTGlnaHRHQk0gKGBsaWdodGdibT09NC42LjBgKSwgTUlUIGxpY2Vuc2VkLgotIFRoZSBvcHRpb25hbCBCR0UtTTMgaW50ZWdyYXRpb24gaXMgZXhwZXJpbWVudGFsIGFuZCBkaXNhYmxlZCBpbiBldmVyeSBzdXBwbGllZCBwcm9maWxlLiBEbyBub3QgZW5hYmxlIGl0IHVubGVzcyB0aGUgb3JnYW5pemVycyBleHBsaWNpdGx5IGNvbmZpcm0gdGhhdCBwcmV0cmFpbmVkIHJlcHJlc2VudGF0aW9ucyBzYXRpc2Z5IHRoZSBzdXBwbGllZC1kYXRhLW9ubHkgcnVsZS4KLSBgY291bnRyeWAgaXMgdHJlYXRlZCBzdHJpY3RseSBhcyBhbiBvcGVuLXNldCBzdHJpbmcgbGFiZWw7IGl0IGlzIG5ldmVyIGhhcmRjb2RlZC4KCgojIyBEaXJlY3RvcnkgbGF5b3V0CgpgYGB0ZXh0CnNyYy9idXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi8gIFByb2R1Y3Rpb24gcGFja2FnZSBhbmQgQ0xJCnRlc3RzLyAgICAgICAgICAgICAgICAgICAgICAgICAgIFVuaXQgYW5kIGludGVncmF0aW9uIHRlc3RzCmNvbmZpZ3MvICAgICAgICAgICAgICAgICAgICAgICAgIFBvcnRhYmxlIGxvY2FsL3JlbW90ZSBKU09OIGNvbmZpZ3VyYXRpb25zCm5vdGVib29rcy8gICAgICAgICAgICAgICAgICAgICAgIFRoaW4gZXhwbG9yYXRvcnkgY2xpZW50cyBvZiB0aGUgcGFja2FnZQpyZXF1aXJlbWVudHMudHh0ICAgICAgICAgICAgICAgICBDb3JlIHJ1bnRpbWUgZGVwZW5kZW5jaWVzCnJlcXVpcmVtZW50cy1kZXYudHh0ICAgICAgICAgICAgIFRlc3QgYW5kIG5vdGVib29rIGRlcGVuZGVuY2llcwpyZXF1aXJlbWVudHMtcmVtb3RlLnR4dCAgICAgICAgICBPcHRpb25hbCBmdWxsLXNjYWxlIExpZ2h0R0JNIGJhY2tlbmQKYGBgCgpHZW5lcmF0ZWQgYXJ0aWZhY3RzIGFyZSB3cml0dGVuIHRvIGNvbmZpZ3VyZWQgYGFydGlmYWN0cy9gIGFuZCBgb3V0cHV0L2Agcm9vdHMuCgojIyBJbnN0YWxsYXRpb24KClByb3Zpc2lvbiBhIGZyZXNoIG1hY2hpbmUgd2l0aCB0aGUgaGVscGVyIHNjcmlwdCAoZGV0ZWN0cyBHUFUsIHBpY2tzIHRoZSByaWdodCB0b3JjaCB3aGVlbCk6CgpgYGBiYXNoCmJhc2ggc2V0dXAuc2ggICAgICAgICAgICAgICAgICAgICAgIyBjb21wbGlhbnQgdGV4dC1mZWF0dXJlIHBpcGVsaW5lCkVOQUJMRV9FTUJFRD0xIGJhc2ggc2V0dXAuc2ggICAgICAgIyBleHBlcmltZW50YWw7IHJlcXVpcmVzIG9yZ2FuaXplciBhcHByb3ZhbApgYGAKCk9yIGluc3RhbGwgbWFudWFsbHkgZnJvbSBgY29kZS9idXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi9gOgoKYGBgYmFzaApweXRob24gLW0gcGlwIGluc3RhbGwgLWUgLgpweXRob24gLW0gcGlwIGluc3RhbGwgLXIgcmVxdWlyZW1lbnRzLWRldi50eHQKcHl0aG9uIC1tIHBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy1yZW1vdGUudHh0ICAgIyBMaWdodEdCTSBiYWNrZW5kCmBgYAoKIyMgQ29uZmlndXJhdGlvbgoKUHJlY2VkZW5jZToKCmBgYHRleHQKQ0xJIG92ZXJyaWRlID4gQkVSXyogZW52aXJvbm1lbnQgdmFyaWFibGUgPiBKU09OIGNvbmZpZyA+IGRlZmF1bHQKYGBgCgpFbnZpcm9ubWVudCB2YXJpYWJsZXM6CgpgYGB0ZXh0CkJFUl9EQVRBX1JPT1QKQkVSX0FSVElGQUNUX1JPT1QKQkVSX09VVFBVVF9ST09UCkJFUl9SVU5fSUQKQkVSX0RFVklDRQpCRVJfVEhSRUFEUwpgYGAKCmByZXNvdXJjZXNgIGFuZCBgZW1iZWRkaW5nc2AgY29uZmlnIGJsb2NrcyBjb250cm9sIGRldmljZSBtb2RlIChgYXV0b2AvYGNwdWAvYGdwdWApLAp0aGUgUkFNL1ZSQU0gZnJhY3Rpb25hbCBidWRnZXRzLCBhbmQgd2hldGhlciBCR0UtTTMgZmVhdHVyZXMgYXJlIGVuYWJsZWQKKGB0cnVlYC9gZmFsc2VgOyBzdXBwbGllZCBwcm9maWxlcyB1c2UgYGZhbHNlYCkuIGBuX3NoYXJkc2AgY29udHJvbHMgY2hlY2twb2ludCBncmFudWxhcml0eS4KCiMjIElucHV0IGNvbnRyYWN0CgpgYGB0ZXh0CmRhdGFzZXQvCuKUnOKUgOKUgCB0cmFpbi8K4pSCICAg4pSc4pSA4pSAIHRyYWluX3NvdXJjZTEudHN2CuKUgiAgIOKUnOKUgOKUgCB0cmFpbl9zb3VyY2UyLnRzdgrilIIgICDilJzilIDilIAgdHJhaW5fc291cmNlMy50c3YK4pSCICAg4pSU4pSA4pSAIHRyYWluX2dyb3VuZF90cnV0aC50c3YK4pSU4pSA4pSAIHRlc3QvCiAgICDilJzilIDilIAgdGVzdF9zb3VyY2UxLnRzdgogICAg4pSc4pSA4pSAIHRlc3Rfc291cmNlMi50c3YKICAgIOKUlOKUgOKUgCB0ZXN0X3NvdXJjZTMudHN2CmBgYAoKQWxsIGZpbGVzIGFyZSBVVEYtOCBhbmQgdGFiLXNlcGFyYXRlZC4gQmxhbmsgc3RyaW5ncyBhcmUgcHJlc2VydmVkLiBDb3VudHJ5IGlzIGFuIGFyYml0cmFyeSBvcGVuLXNldCBzdHJpbmc7IG5vIGZpeGVkIGB7VVMsIEluZGlhfWAgZW51bWVyYXRpb24gZXhpc3RzLgoKIyMgTWluaSBlbmQtdG8tZW5kIGN5Y2xlIChydW4gdGhpcyBmaXJzdCkKCmBtYWtlLW1pbmlgIHNhbXBsZXMgMTUgcmVhbGlzdGljIFNvdXJjZSAxIGVudGl0aWVzICg2IFVTIC8gNiBJbmRpYSAvIDMgRnJhbmNlLAppbmNsdWRpbmcgc2luZ2xldG9ucywgc2luZ2xlIG1hdGNoZXMsIG11bHRpLW1hdGNoZXMsIGFuZCBib3RoLXNvdXJjZSBtYXRjaGVzKQpwbHVzIGRlY295cywgc3RyYWlnaHQgZnJvbSB0aGUgcmVhbCBmaWxlcywgaW50byBgZGF0YXNldC9taW5pL2Agd2l0aCB0aGUgZXhhY3QKcmVhbCBzY2hlbWEuIFVzZSBpdCB0byB2ZXJpZnkgdGhlIHdob2xlIGN5Y2xlIGVuZC10by1lbmQgYmVmb3JlIHRoZSBmdWxsIHJ1bi4KCmBgYGJhc2gKcHl0aG9uIHJ1bl9waXBlbGluZS5weSAtLXByb2ZpbGUgbWluaSAtLXNraXAtZW1iZWRkaW5ncyBcCiAgLS12YWxpZGF0b3IgL3BhdGgvdG8vc3R1ZGVudF9yZXNvdXJjZS91dGlscy92YWxpZGF0ZV9zdWJtaXNzaW9uLnB5CmBgYAoKVGhlIHJ1bm5lciB0dW5lcyB0aGUgZGVjaXNpb24gdGhyZXNob2xkIG9uIGZvbGQgMCwgcmVwb3J0cyBhbiB1bmJpYXNlZCBlc3RpbWF0ZQpvbiBhIHNlcGFyYXRlbHkgdHJhaW5lZCBmb2xkLTEgbW9kZWwsIHRoZW4gdHJhaW5zIHRoZSBmaW5hbCBtb2RlbCBvbiBhbGwgcm93cy4KCkFueSBvZiB0aGVzZSBjb21tYW5kcyBjYW4gYmUgcmUtcnVuIGFmdGVyIGEgY3Jhc2ggYW5kIHdpbGwgcmVzdW1lLgoKIyMgUmVtb3RlIGZ1bGwtc2NhbGUgd29ya2Zsb3cKClVzZSB0aGUgc2FtZSBzb3VyY2UgY29kZSBvbiBhIG1hY2hpbmUgd2l0aCAxNuKAkzMyIHZDUFUsIDY04oCTMTI4IEdCIFJBTSwgYW5kIGZhc3QgdGVtcG9yYXJ5IHN0b3JhZ2U6CgpgYGBiYXNoCmV4cG9ydCBCRVJfREFUQV9ST09UPS9tb3VudGVkL2NoYWxsZW5nZS9kYXRhc2V0CmV4cG9ydCBCRVJfQVJUSUZBQ1RfUk9PVD0vbW91bnRlZC93b3JrL2FydGlmYWN0cwpleHBvcnQgQkVSX09VVFBVVF9ST09UPS9tb3VudGVkL3dvcmsvb3V0cHV0CmV4cG9ydCBCRVJfVEhSRUFEUz0xNgoKcHl0aG9uIC1tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uIGF1ZGl0IC0tY29uZmlnIGNvbmZpZ3MvcmVtb3RlX2Z1bGwuanNvbgpweXRob24gLW0gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24gcHJlcGFyZSAtLWNvbmZpZyBjb25maWdzL3JlbW90ZV9mdWxsLmpzb24gLS1zcGxpdCBib3RoCnB5dGhvbiAtbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbiBtYWtlLXNwbGl0cyAtLWNvbmZpZyBjb25maWdzL3JlbW90ZV9mdWxsLmpzb24gLS1mb2xkcyAxMApweXRob24gLW0gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24gZ2VuZXJhdGUtY2FuZGlkYXRlcyAtLWNvbmZpZyBjb25maWdzL3JlbW90ZV9mdWxsLmpzb24gLS1zcGxpdCB0cmFpbgpweXRob24gLW0gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24gYnVpbGQtZmVhdHVyZXMgLS1jb25maWcgY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIC0tc3BsaXQgdHJhaW4KcHl0aG9uIC1tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uIHRyYWluIC0tY29uZmlnIGNvbmZpZ3MvcmVtb3RlX2Z1bGwuanNvbiAtLWV4Y2x1ZGUtZm9sZHMgMCwxIC0tb3V0cHV0LW5hbWUgbGlnaHRnYm0tdmFsaWRhdGlvbi1mb2xkMApweXRob24gLW0gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24gc2NvcmUgLS1jb25maWcgY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIC0tc3BsaXQgdHJhaW4gLS1tb2RlbC1wYXRoIC9tb3VudGVkL3dvcmsvYXJ0aWZhY3RzL3JlbW90ZS1mdWxsL21vZGVscy9saWdodGdibS12YWxpZGF0aW9uLWZvbGQwLnR4dCAtLXZhbGlkYXRpb24tZm9sZCAwIC0tb3V0cHV0LW5hbWUgdmFsaWRhdGlvbi1mb2xkMC5wYXJxdWV0CnB5dGhvbiAtbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbiB0dW5lLWRlY2lzaW9uIC0tY29uZmlnIGNvbmZpZ3MvcmVtb3RlX2Z1bGwuanNvbiAtLXNjb3JlLWZpbGUgdmFsaWRhdGlvbi1mb2xkMC5wYXJxdWV0IC0tdmFsaWRhdGlvbi1mb2xkIDAKCiMgSW5kZXBlbmRlbnQgZXZhbHVhdGlvbjogdHJhaW4gd2l0aG91dCBmb2xkIDEsIHRoZW4gc2NvcmUvZXZhbHVhdGUgZm9sZCAxLgpweXRob24gLW0gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24gdHJhaW4gLS1jb25maWcgY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIC0tdmFsaWRhdGlvbi1mb2xkIDEgLS1vdXRwdXQtbmFtZSBsaWdodGdibS12YWxpZGF0aW9uLWZvbGQxCnB5dGhvbiAtbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbiBzY29yZSAtLWNvbmZpZyBjb25maWdzL3JlbW90ZV9mdWxsLmpzb24gLS1zcGxpdCB0cmFpbiAtLW1vZGVsLXBhdGggL21vdW50ZWQvd29yay9hcnRpZmFjdHMvcmVtb3RlLWZ1bGwvbW9kZWxzL2xpZ2h0Z2JtLXZhbGlkYXRpb24tZm9sZDEudHh0IC0tdmFsaWRhdGlvbi1mb2xkIDEgLS1vdXRwdXQtbmFtZSBldmFsdWF0aW9uLWZvbGQxLnBhcnF1ZXQKcHl0aG9uIC1tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uIGV2YWx1YXRlIC0tY29uZmlnIGNvbmZpZ3MvcmVtb3RlX2Z1bGwuanNvbiAtLXNjb3JlLWZpbGUgZXZhbHVhdGlvbi1mb2xkMS5wYXJxdWV0IC0tdmFsaWRhdGlvbi1mb2xkIDEKYGBgCgpBZnRlciBzZWxlY3RpbmcgYW5kIGZyZWV6aW5nIGEgdmFsaWRhdGVkIG1vZGVsIGFuZCB0aHJlc2hvbGQ6CgpgYGBiYXNoCnB5dGhvbiAtbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbiB0cmFpbiAtLWNvbmZpZyBjb25maWdzL3JlbW90ZV9mdWxsLmpzb24gLS1hbGwtdHJhaW5pbmctZGF0YSAtLW91dHB1dC1uYW1lIGxpZ2h0Z2JtLWZpbmFsCnB5dGhvbiAtbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbiBwcmVwYXJlIC0tY29uZmlnIGNvbmZpZ3MvcmVtb3RlX2Z1bGwuanNvbiAtLXNwbGl0IHRlc3QKcHl0aG9uIC1tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uIGdlbmVyYXRlLWNhbmRpZGF0ZXMgLS1jb25maWcgY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIC0tc3BsaXQgdGVzdApweXRob24gLW0gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24gYnVpbGQtZmVhdHVyZXMgLS1jb25maWcgY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIC0tc3BsaXQgdGVzdApweXRob24gLW0gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24gaW5mZXIgLS1jb25maWcgY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIFwKICAgIC0tbW9kZWwtcGF0aCAvbW91bnRlZC93b3JrL2FydGlmYWN0cy9yZW1vdGUtZnVsbC9tb2RlbHMvbGlnaHRnYm0tZmluYWwudHh0CnB5dGhvbiAtbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbiBwcmVmbGlnaHQgLS1jb25maWcgY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIFwKICAgIC0tb2ZmaWNpYWwtdmFsaWRhdG9yIC9tb3VudGVkL2NoYWxsZW5nZS91dGlscy92YWxpZGF0ZV9zdWJtaXNzaW9uLnB5IC0tY2hlY2staWRzCnB5dGhvbiAtbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbiBwYWNrYWdlIC0tY29uZmlnIGNvbmZpZ3MvcmVtb3RlX2Z1bGwuanNvbiAtLXRlYW0tbmFtZSBZT1VSX1RFQU0gXAogICAgLS1kb2N1bWVudGF0aW9uIC9tb3VudGVkL2NoYWxsZW5nZS9Eb2N1bWVudGF0aW9uX3RlbXBsYXRlLm1kCmBgYAoKS2VlcCBlbWJlZGRpbmcgZmVhdHVyZXMgZGlzYWJsZWQgZm9yIGNvbXBldGl0aW9uIHJ1bnMgdW5sZXNzIHRoZSBzdXBwbGllZC1kYXRhLW9ubHkgYW5kIG1vZGVsLWxpY2Vuc2UgaW50ZXJwcmV0YXRpb24gaXMgY29uZmlybWVkIGluIHdyaXRpbmcuCgpDb21wbGV0ZSB0aGUgc3VwcGxpZWQgZG9jdW1lbnRhdGlvbiB0ZW1wbGF0ZSBiZWZvcmUgcGFja2FnaW5nLiBUaGUgcGFja2FnZSBjb21tYW5kIHJlZnVzZXMgdG8gY3JlYXRlIGEgc3VibWlzc2lvbiBhcmNoaXZlIHdpdGhvdXQgaXQuCgojIyBPdXRwdXRzCgpgYGB0ZXh0Cm91dHB1dC8K4pSc4pSA4pSAIG1hdGNoaW5nX3Jlc3VsdHMudHN2CuKUlOKUgOKUgCBjYW5kaWRhdGVfcGFpcnMudHN2CmBgYAoKQm90aCBmaWxlcyBjb250YWluIGV4YWN0bHkgb25lIHJvdyBwZXIgdGVzdCBTMSwgaW5jbHVkaW5nIGJsYW5rIHJvd3MuIEludGVybmFsIHByZWZsaWdodCBlbmZvcmNlcyB0YXJnZXQgZXhpc3RlbmNlLCBkdXBsaWNhdGUgcnVsZXMsIGNhbmRpZGF0ZSBjb250YWlubWVudCwgZXhhY3QgY29sdW1ucywgYW5kIGVtcHR5LWxpc3QgaGFuZGxpbmcgYmVmb3JlIHRoZSBvZmZpY2lhbCB2YWxpZGF0b3IgcnVucy4KCiMjIFJlcHJvZHVjaWJpbGl0eSBhbmQgY29tcGxpYW5jZQoKLSBTZWVkIGRlZmF1bHRzIHRvIDIwMjYuCi0gRm9sZHMgYW5kIG5lZ2F0aXZlIHNhbXBsaW5nIHVzZSBzdGFibGUgY3J5cHRvZ3JhcGhpYyBoYXNoZXMuCi0gQ2FuZGlkYXRlIG9yZGVyaW5nIGFuZCBvdXRwdXQgbGlzdHMgYXJlIGRldGVybWluaXN0aWMuCi0gU3RhZ2UgYXJ0aWZhY3RzIHVzZSBleHBsaWNpdCBzY2hlbWFzIGFuZCBtYW5pZmVzdHMuCi0gUmF3IGNoYWxsZW5nZSBmaWxlcyBhcmUgbmV2ZXIgbW9kaWZpZWQuCi0gU3VwcGxpZWQgY2hhbGxlbmdlIGRhdGEgb25seSBpbiB0aGUgZGVmYXVsdCBhbmQgcmVjb21tZW5kZWQgd29ya2Zsb3dzOyBubyBleHRlcm5hbCBhZGRyZXNzL2VudGl0eSBsb29rdXAuCi0gTGlnaHRHQk0gaXMgb3B0aW9uYWwgYW5kIE1JVCBsaWNlbnNlZDsgYW55IHVzZSBtdXN0IGJlIHJlY29yZGVkIGluIHRoZSBydW4gbWFuaWZlc3QuCi0gQSBmaW5hbCBtb2RlbCBpcyBub3Qgc2VsZWN0ZWQgdW50aWwgaGVsZC1vdXQgZXhwZXJpbWVudHMgcnVuIG9uIHJlbW90ZSBjb21wdXRlLgo=', 'dc1198aace37b4f450d9a9d42341298a6f538f43c422f93e09834bc860760dbd')

## Embedded source: `code/business_entity_resolution/THIRD_PARTY_LICENSES.md`

~~~~text
# Runtime license record

The competition model backend recommended by the supplied remote configuration
is LightGBM, licensed under MIT. Its trained artifact is a tabular tree ensemble
and is far below the 8-billion-parameter ceiling.

Core runtime dependencies must be recorded in the final methodology package:

| Dependency | Role | License family |
|---|---|---|
| LightGBM | Recommended final classifier | MIT |
| NumPy | Numeric arrays | BSD-3-Clause |
| pandas | Tabular processing | BSD-3-Clause |
| SciPy | Numeric utilities | BSD-3-Clause |
| scikit-learn | Baseline SGD and TF-IDF | BSD-3-Clause |
| PyArrow | Parquet I/O | Apache-2.0 |
| RapidFuzz | String similarities | MIT |

The optional pretrained embedding integration is disabled in all supplied
profiles. Do not enable it for a competition run without organizer approval.

~~~~

In [ ]:
materialize('code/business_entity_resolution/THIRD_PARTY_LICENSES.md', 'IyBSdW50aW1lIGxpY2Vuc2UgcmVjb3JkCgpUaGUgY29tcGV0aXRpb24gbW9kZWwgYmFja2VuZCByZWNvbW1lbmRlZCBieSB0aGUgc3VwcGxpZWQgcmVtb3RlIGNvbmZpZ3VyYXRpb24KaXMgTGlnaHRHQk0sIGxpY2Vuc2VkIHVuZGVyIE1JVC4gSXRzIHRyYWluZWQgYXJ0aWZhY3QgaXMgYSB0YWJ1bGFyIHRyZWUgZW5zZW1ibGUKYW5kIGlzIGZhciBiZWxvdyB0aGUgOC1iaWxsaW9uLXBhcmFtZXRlciBjZWlsaW5nLgoKQ29yZSBydW50aW1lIGRlcGVuZGVuY2llcyBtdXN0IGJlIHJlY29yZGVkIGluIHRoZSBmaW5hbCBtZXRob2RvbG9neSBwYWNrYWdlOgoKfCBEZXBlbmRlbmN5IHwgUm9sZSB8IExpY2Vuc2UgZmFtaWx5IHwKfC0tLXwtLS18LS0tfAp8IExpZ2h0R0JNIHwgUmVjb21tZW5kZWQgZmluYWwgY2xhc3NpZmllciB8IE1JVCB8CnwgTnVtUHkgfCBOdW1lcmljIGFycmF5cyB8IEJTRC0zLUNsYXVzZSB8CnwgcGFuZGFzIHwgVGFidWxhciBwcm9jZXNzaW5nIHwgQlNELTMtQ2xhdXNlIHwKfCBTY2lQeSB8IE51bWVyaWMgdXRpbGl0aWVzIHwgQlNELTMtQ2xhdXNlIHwKfCBzY2lraXQtbGVhcm4gfCBCYXNlbGluZSBTR0QgYW5kIFRGLUlERiB8IEJTRC0zLUNsYXVzZSB8CnwgUHlBcnJvdyB8IFBhcnF1ZXQgSS9PIHwgQXBhY2hlLTIuMCB8CnwgUmFwaWRGdXp6IHwgU3RyaW5nIHNpbWlsYXJpdGllcyB8IE1JVCB8CgpUaGUgb3B0aW9uYWwgcHJldHJhaW5lZCBlbWJlZGRpbmcgaW50ZWdyYXRpb24gaXMgZGlzYWJsZWQgaW4gYWxsIHN1cHBsaWVkCnByb2ZpbGVzLiBEbyBub3QgZW5hYmxlIGl0IGZvciBhIGNvbXBldGl0aW9uIHJ1biB3aXRob3V0IG9yZ2FuaXplciBhcHByb3ZhbC4K', '45db09ffe6f8f0bdc6e0f3ff2a2303eae47a270496271790883b012429c976c6')

## Embedded source: `code/business_entity_resolution/configs/base.json`

~~~~text
{
  "data_root": "dataset",
  "artifact_root": "artifacts",
  "output_root": "output",
  "run_id": "default",
  "seed": 2026,
  "batch_size": 100000,
  "threads": 4,
  "device": "cpu",
  "max_rows": null,
  "n_shards": 32,
  "model": {
    "kind": "sgd",
    "threshold": 0.8,
    "source_thresholds": {},
    "params": {}
  },
  "training": {
    "max_negatives_per_entity": 50
  },
  "blocking": {
    "country_primary": true,
    "country_fallback": true,
    "rare_token_max_df": 500,
    "token_posting_cap": 250,
    "tfidf_top_k": 20,
    "tfidf_min_score": 0.15,
    "tfidf_target_batch_size": 50000,
    "per_source_cap": 100,
    "batch_size": 1000
  },
  "resources": {
    "mode": "auto",
    "ram_fraction": 0.65,
    "vram_fraction": 0.75,
    "enable_embeddings": "auto"
  },
  "embeddings": {
    "model": "BAAI/bge-m3",
    "enable": "false",
    "batch_size": 64
  },
  "logging": {
    "echo": true
  }
}

~~~~

In [ ]:
materialize('code/business_entity_resolution/configs/base.json', 'ewogICJkYXRhX3Jvb3QiOiAiZGF0YXNldCIsCiAgImFydGlmYWN0X3Jvb3QiOiAiYXJ0aWZhY3RzIiwKICAib3V0cHV0X3Jvb3QiOiAib3V0cHV0IiwKICAicnVuX2lkIjogImRlZmF1bHQiLAogICJzZWVkIjogMjAyNiwKICAiYmF0Y2hfc2l6ZSI6IDEwMDAwMCwKICAidGhyZWFkcyI6IDQsCiAgImRldmljZSI6ICJjcHUiLAogICJtYXhfcm93cyI6IG51bGwsCiAgIm5fc2hhcmRzIjogMzIsCiAgIm1vZGVsIjogewogICAgImtpbmQiOiAic2dkIiwKICAgICJ0aHJlc2hvbGQiOiAwLjgsCiAgICAic291cmNlX3RocmVzaG9sZHMiOiB7fSwKICAgICJwYXJhbXMiOiB7fQogIH0sCiAgInRyYWluaW5nIjogewogICAgIm1heF9uZWdhdGl2ZXNfcGVyX2VudGl0eSI6IDUwCiAgfSwKICAiYmxvY2tpbmciOiB7CiAgICAiY291bnRyeV9wcmltYXJ5IjogdHJ1ZSwKICAgICJjb3VudHJ5X2ZhbGxiYWNrIjogdHJ1ZSwKICAgICJyYXJlX3Rva2VuX21heF9kZiI6IDUwMCwKICAgICJ0b2tlbl9wb3N0aW5nX2NhcCI6IDI1MCwKICAgICJ0ZmlkZl90b3BfayI6IDIwLAogICAgInRmaWRmX21pbl9zY29yZSI6IDAuMTUsCiAgICAidGZpZGZfdGFyZ2V0X2JhdGNoX3NpemUiOiA1MDAwMCwKICAgICJwZXJfc291cmNlX2NhcCI6IDEwMCwKICAgICJiYXRjaF9zaXplIjogMTAwMAogIH0sCiAgInJlc291cmNlcyI6IHsKICAgICJtb2RlIjogImF1dG8iLAogICAgInJhbV9mcmFjdGlvbiI6IDAuNjUsCiAgICAidnJhbV9mcmFjdGlvbiI6IDAuNzUsCiAgICAiZW5hYmxlX2VtYmVkZGluZ3MiOiAiYXV0byIKICB9LAogICJlbWJlZGRpbmdzIjogewogICAgIm1vZGVsIjogIkJBQUkvYmdlLW0zIiwKICAgICJlbmFibGUiOiAiZmFsc2UiLAogICAgImJhdGNoX3NpemUiOiA2NAogIH0sCiAgImxvZ2dpbmciOiB7CiAgICAiZWNobyI6IHRydWUKICB9Cn0K', 'f5c0404c41e0a2d19a2cdd76dc6f537d020c19a4d43c896876a51f5dca616181')

## Embedded source: `code/business_entity_resolution/configs/local_smoke.json`

~~~~text
{
  "data_root": "dataset",
  "artifact_root": "artifacts/smoke",
  "output_root": "output/smoke",
  "run_id": "local-smoke",
  "seed": 2026,
  "batch_size": 5000,
  "threads": 2,
  "device": "cpu",
  "max_rows": 10000,
  "n_shards": 4,
  "model": {
    "kind": "sgd",
    "threshold": 0.75,
    "source_thresholds": {},
    "params": {"alpha": 0.0001, "max_iter": 500}
  },
  "training": {
    "max_negatives_per_entity": 20
  },
  "blocking": {
    "country_primary": true,
    "country_fallback": true,
    "rare_token_max_df": 50,
    "token_posting_cap": 50,
    "tfidf_top_k": 5,
    "tfidf_min_score": 0.1,
    "tfidf_target_batch_size": 10000,
    "per_source_cap": 20,
    "batch_size": 250
  },
  "resources": {
    "mode": "auto",
    "ram_fraction": 0.5,
    "vram_fraction": 0.75,
    "enable_embeddings": "false"
  },
  "embeddings": {
    "model": "BAAI/bge-m3",
    "enable": "false",
    "batch_size": 8
  },
  "logging": {
    "echo": true
  }
}

~~~~

In [ ]:
materialize('code/business_entity_resolution/configs/local_smoke.json', 'ewogICJkYXRhX3Jvb3QiOiAiZGF0YXNldCIsCiAgImFydGlmYWN0X3Jvb3QiOiAiYXJ0aWZhY3RzL3Ntb2tlIiwKICAib3V0cHV0X3Jvb3QiOiAib3V0cHV0L3Ntb2tlIiwKICAicnVuX2lkIjogImxvY2FsLXNtb2tlIiwKICAic2VlZCI6IDIwMjYsCiAgImJhdGNoX3NpemUiOiA1MDAwLAogICJ0aHJlYWRzIjogMiwKICAiZGV2aWNlIjogImNwdSIsCiAgIm1heF9yb3dzIjogMTAwMDAsCiAgIm5fc2hhcmRzIjogNCwKICAibW9kZWwiOiB7CiAgICAia2luZCI6ICJzZ2QiLAogICAgInRocmVzaG9sZCI6IDAuNzUsCiAgICAic291cmNlX3RocmVzaG9sZHMiOiB7fSwKICAgICJwYXJhbXMiOiB7ImFscGhhIjogMC4wMDAxLCAibWF4X2l0ZXIiOiA1MDB9CiAgfSwKICAidHJhaW5pbmciOiB7CiAgICAibWF4X25lZ2F0aXZlc19wZXJfZW50aXR5IjogMjAKICB9LAogICJibG9ja2luZyI6IHsKICAgICJjb3VudHJ5X3ByaW1hcnkiOiB0cnVlLAogICAgImNvdW50cnlfZmFsbGJhY2siOiB0cnVlLAogICAgInJhcmVfdG9rZW5fbWF4X2RmIjogNTAsCiAgICAidG9rZW5fcG9zdGluZ19jYXAiOiA1MCwKICAgICJ0ZmlkZl90b3BfayI6IDUsCiAgICAidGZpZGZfbWluX3Njb3JlIjogMC4xLAogICAgInRmaWRmX3RhcmdldF9iYXRjaF9zaXplIjogMTAwMDAsCiAgICAicGVyX3NvdXJjZV9jYXAiOiAyMCwKICAgICJiYXRjaF9zaXplIjogMjUwCiAgfSwKICAicmVzb3VyY2VzIjogewogICAgIm1vZGUiOiAiYXV0byIsCiAgICAicmFtX2ZyYWN0aW9uIjogMC41LAogICAgInZyYW1fZnJhY3Rpb24iOiAwLjc1LAogICAgImVuYWJsZV9lbWJlZGRpbmdzIjogImZhbHNlIgogIH0sCiAgImVtYmVkZGluZ3MiOiB7CiAgICAibW9kZWwiOiAiQkFBSS9iZ2UtbTMiLAogICAgImVuYWJsZSI6ICJmYWxzZSIsCiAgICAiYmF0Y2hfc2l6ZSI6IDgKICB9LAogICJsb2dnaW5nIjogewogICAgImVjaG8iOiB0cnVlCiAgfQp9Cg==', 'e1334683ac088988f28cea66831e4f10401b4102b31a233cccb2970755545570')

## Embedded source: `code/business_entity_resolution/configs/mini.json`

~~~~text
{
  "data_root": "dataset/mini",
  "artifact_root": "artifacts/mini",
  "output_root": "output/mini",
  "run_id": "mini",
  "seed": 2026,
  "batch_size": 5000,
  "threads": 2,
  "device": "cpu",
  "max_rows": null,
  "n_shards": 4,
  "model": {
    "kind": "lightgbm",
    "threshold": 0.8,
    "source_thresholds": {},
    "params": {
      "learning_rate": 0.05,
      "n_estimators": 300,
      "num_leaves": 31,
      "min_child_samples": 5
    }
  },
  "training": {
    "max_negatives_per_entity": 20
  },
  "blocking": {
    "country_primary": true,
    "country_fallback": true,
    "rare_token_max_df": 50,
    "token_posting_cap": 50,
    "tfidf_top_k": 10,
    "tfidf_min_score": 0.1,
    "tfidf_target_batch_size": 10000,
    "per_source_cap": 25,
    "batch_size": 250
  },
  "resources": {
    "mode": "auto",
    "ram_fraction": 0.5,
    "vram_fraction": 0.75,
    "enable_embeddings": "auto"
  },
  "embeddings": {
    "model": "BAAI/bge-m3",
    "enable": "false",
    "batch_size": 8
  },
  "logging": {
    "echo": true
  }
}

~~~~

In [ ]:
materialize('code/business_entity_resolution/configs/mini.json', 'ewogICJkYXRhX3Jvb3QiOiAiZGF0YXNldC9taW5pIiwKICAiYXJ0aWZhY3Rfcm9vdCI6ICJhcnRpZmFjdHMvbWluaSIsCiAgIm91dHB1dF9yb290IjogIm91dHB1dC9taW5pIiwKICAicnVuX2lkIjogIm1pbmkiLAogICJzZWVkIjogMjAyNiwKICAiYmF0Y2hfc2l6ZSI6IDUwMDAsCiAgInRocmVhZHMiOiAyLAogICJkZXZpY2UiOiAiY3B1IiwKICAibWF4X3Jvd3MiOiBudWxsLAogICJuX3NoYXJkcyI6IDQsCiAgIm1vZGVsIjogewogICAgImtpbmQiOiAibGlnaHRnYm0iLAogICAgInRocmVzaG9sZCI6IDAuOCwKICAgICJzb3VyY2VfdGhyZXNob2xkcyI6IHt9LAogICAgInBhcmFtcyI6IHsKICAgICAgImxlYXJuaW5nX3JhdGUiOiAwLjA1LAogICAgICAibl9lc3RpbWF0b3JzIjogMzAwLAogICAgICAibnVtX2xlYXZlcyI6IDMxLAogICAgICAibWluX2NoaWxkX3NhbXBsZXMiOiA1CiAgICB9CiAgfSwKICAidHJhaW5pbmciOiB7CiAgICAibWF4X25lZ2F0aXZlc19wZXJfZW50aXR5IjogMjAKICB9LAogICJibG9ja2luZyI6IHsKICAgICJjb3VudHJ5X3ByaW1hcnkiOiB0cnVlLAogICAgImNvdW50cnlfZmFsbGJhY2siOiB0cnVlLAogICAgInJhcmVfdG9rZW5fbWF4X2RmIjogNTAsCiAgICAidG9rZW5fcG9zdGluZ19jYXAiOiA1MCwKICAgICJ0ZmlkZl90b3BfayI6IDEwLAogICAgInRmaWRmX21pbl9zY29yZSI6IDAuMSwKICAgICJ0ZmlkZl90YXJnZXRfYmF0Y2hfc2l6ZSI6IDEwMDAwLAogICAgInBlcl9zb3VyY2VfY2FwIjogMjUsCiAgICAiYmF0Y2hfc2l6ZSI6IDI1MAogIH0sCiAgInJlc291cmNlcyI6IHsKICAgICJtb2RlIjogImF1dG8iLAogICAgInJhbV9mcmFjdGlvbiI6IDAuNSwKICAgICJ2cmFtX2ZyYWN0aW9uIjogMC43NSwKICAgICJlbmFibGVfZW1iZWRkaW5ncyI6ICJhdXRvIgogIH0sCiAgImVtYmVkZGluZ3MiOiB7CiAgICAibW9kZWwiOiAiQkFBSS9iZ2UtbTMiLAogICAgImVuYWJsZSI6ICJmYWxzZSIsCiAgICAiYmF0Y2hfc2l6ZSI6IDgKICB9LAogICJsb2dnaW5nIjogewogICAgImVjaG8iOiB0cnVlCiAgfQp9Cg==', '9e1b9f702667851e58e1baa5790c483dd9543ec24f8c544868ee079d825b0d86')

## Embedded source: `code/business_entity_resolution/configs/remote_full.json`

~~~~text
{
  "data_root": "dataset",
  "artifact_root": "artifacts/full",
  "output_root": "output",
  "run_id": "remote-full",
  "seed": 2026,
  "batch_size": 100000,
  "threads": 16,
  "device": "cpu",
  "max_rows": null,
  "n_shards": 64,
  "model": {
    "kind": "lightgbm",
    "threshold": 0.8,
    "source_thresholds": {},
    "params": {
      "learning_rate": 0.05,
      "n_estimators": 2000,
      "num_leaves": 63,
      "min_child_samples": 100,
      "subsample": 0.8,
      "colsample_bytree": 0.8
    }
  },
  "training": {
    "max_negatives_per_entity": 50
  },
  "blocking": {
    "country_primary": true,
    "country_fallback": true,
    "rare_token_max_df": 500,
    "token_posting_cap": 250,
    "tfidf_top_k": 20,
    "tfidf_min_score": 0.15,
    "tfidf_target_batch_size": 50000,
    "per_source_cap": 100,
    "batch_size": 1000
  },
  "resources": {
    "mode": "auto",
    "ram_fraction": 0.65,
    "vram_fraction": 0.75,
    "enable_embeddings": "auto"
  },
  "embeddings": {
    "model": "BAAI/bge-m3",
    "enable": "false",
    "batch_size": 64
  },
  "logging": {
    "echo": true
  }
}

~~~~

In [ ]:
materialize('code/business_entity_resolution/configs/remote_full.json', 'ewogICJkYXRhX3Jvb3QiOiAiZGF0YXNldCIsCiAgImFydGlmYWN0X3Jvb3QiOiAiYXJ0aWZhY3RzL2Z1bGwiLAogICJvdXRwdXRfcm9vdCI6ICJvdXRwdXQiLAogICJydW5faWQiOiAicmVtb3RlLWZ1bGwiLAogICJzZWVkIjogMjAyNiwKICAiYmF0Y2hfc2l6ZSI6IDEwMDAwMCwKICAidGhyZWFkcyI6IDE2LAogICJkZXZpY2UiOiAiY3B1IiwKICAibWF4X3Jvd3MiOiBudWxsLAogICJuX3NoYXJkcyI6IDY0LAogICJtb2RlbCI6IHsKICAgICJraW5kIjogImxpZ2h0Z2JtIiwKICAgICJ0aHJlc2hvbGQiOiAwLjgsCiAgICAic291cmNlX3RocmVzaG9sZHMiOiB7fSwKICAgICJwYXJhbXMiOiB7CiAgICAgICJsZWFybmluZ19yYXRlIjogMC4wNSwKICAgICAgIm5fZXN0aW1hdG9ycyI6IDIwMDAsCiAgICAgICJudW1fbGVhdmVzIjogNjMsCiAgICAgICJtaW5fY2hpbGRfc2FtcGxlcyI6IDEwMCwKICAgICAgInN1YnNhbXBsZSI6IDAuOCwKICAgICAgImNvbHNhbXBsZV9ieXRyZWUiOiAwLjgKICAgIH0KICB9LAogICJ0cmFpbmluZyI6IHsKICAgICJtYXhfbmVnYXRpdmVzX3Blcl9lbnRpdHkiOiA1MAogIH0sCiAgImJsb2NraW5nIjogewogICAgImNvdW50cnlfcHJpbWFyeSI6IHRydWUsCiAgICAiY291bnRyeV9mYWxsYmFjayI6IHRydWUsCiAgICAicmFyZV90b2tlbl9tYXhfZGYiOiA1MDAsCiAgICAidG9rZW5fcG9zdGluZ19jYXAiOiAyNTAsCiAgICAidGZpZGZfdG9wX2siOiAyMCwKICAgICJ0ZmlkZl9taW5fc2NvcmUiOiAwLjE1LAogICAgInRmaWRmX3RhcmdldF9iYXRjaF9zaXplIjogNTAwMDAsCiAgICAicGVyX3NvdXJjZV9jYXAiOiAxMDAsCiAgICAiYmF0Y2hfc2l6ZSI6IDEwMDAKICB9LAogICJyZXNvdXJjZXMiOiB7CiAgICAibW9kZSI6ICJhdXRvIiwKICAgICJyYW1fZnJhY3Rpb24iOiAwLjY1LAogICAgInZyYW1fZnJhY3Rpb24iOiAwLjc1LAogICAgImVuYWJsZV9lbWJlZGRpbmdzIjogImF1dG8iCiAgfSwKICAiZW1iZWRkaW5ncyI6IHsKICAgICJtb2RlbCI6ICJCQUFJL2JnZS1tMyIsCiAgICAiZW5hYmxlIjogImZhbHNlIiwKICAgICJiYXRjaF9zaXplIjogNjQKICB9LAogICJsb2dnaW5nIjogewogICAgImVjaG8iOiB0cnVlCiAgfQp9Cg==', 'ba2827dfb1ef16b08f9b226c9f817d37df4e732c595253c0f4ade8762cc38ca6')

## Embedded source: `code/business_entity_resolution/pyproject.toml`

~~~~text
[build-system]
requires = ["setuptools>=75", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "amazon-ml-business-entity-resolution"
version = "0.1.0"
description = "Compute-portable business entity resolution pipeline for Amazon ML 2026"
requires-python = ">=3.10"
dependencies = [
  "numpy==2.1.3",
  "pandas==2.2.3",
  "scipy==1.15.2",
  "scikit-learn==1.6.1",
  "joblib==1.4.2",
  "rapidfuzz==3.12.2",
  "pyarrow==18.0.0",
]

[project.scripts]
ber = "business_entity_resolution.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-ra"
markers = ["slow: tests that may scan substantial challenge data"]


~~~~

In [ ]:
materialize('code/business_entity_resolution/pyproject.toml', 'W2J1aWxkLXN5c3RlbV0KcmVxdWlyZXMgPSBbInNldHVwdG9vbHM+PTc1IiwgIndoZWVsIl0KYnVpbGQtYmFja2VuZCA9ICJzZXR1cHRvb2xzLmJ1aWxkX21ldGEiCgpbcHJvamVjdF0KbmFtZSA9ICJhbWF6b24tbWwtYnVzaW5lc3MtZW50aXR5LXJlc29sdXRpb24iCnZlcnNpb24gPSAiMC4xLjAiCmRlc2NyaXB0aW9uID0gIkNvbXB1dGUtcG9ydGFibGUgYnVzaW5lc3MgZW50aXR5IHJlc29sdXRpb24gcGlwZWxpbmUgZm9yIEFtYXpvbiBNTCAyMDI2IgpyZXF1aXJlcy1weXRob24gPSAiPj0zLjEwIgpkZXBlbmRlbmNpZXMgPSBbCiAgIm51bXB5PT0yLjEuMyIsCiAgInBhbmRhcz09Mi4yLjMiLAogICJzY2lweT09MS4xNS4yIiwKICAic2Npa2l0LWxlYXJuPT0xLjYuMSIsCiAgImpvYmxpYj09MS40LjIiLAogICJyYXBpZGZ1eno9PTMuMTIuMiIsCiAgInB5YXJyb3c9PTE4LjAuMCIsCl0KCltwcm9qZWN0LnNjcmlwdHNdCmJlciA9ICJidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5jbGk6bWFpbiIKClt0b29sLnNldHVwdG9vbHMucGFja2FnZXMuZmluZF0Kd2hlcmUgPSBbInNyYyJdCgpbdG9vbC5weXRlc3QuaW5pX29wdGlvbnNdCnRlc3RwYXRocyA9IFsidGVzdHMiXQphZGRvcHRzID0gIi1yYSIKbWFya2VycyA9IFsic2xvdzogdGVzdHMgdGhhdCBtYXkgc2NhbiBzdWJzdGFudGlhbCBjaGFsbGVuZ2UgZGF0YSJdCgo=', '6f9bc6c00dc3c490ddedd82e05fbf2b7d611221da8d793531378a4baa41da91a')

## Embedded source: `code/business_entity_resolution/requirements-embeddings.txt`

~~~~text
# Experimental only. Keep disabled for competition runs unless organizers
# explicitly approve pretrained representations under the supplied-data rule.
transformers==4.46.3
FlagEmbedding==1.3.3

~~~~

In [ ]:
materialize('code/business_entity_resolution/requirements-embeddings.txt', 'IyBFeHBlcmltZW50YWwgb25seS4gS2VlcCBkaXNhYmxlZCBmb3IgY29tcGV0aXRpb24gcnVucyB1bmxlc3Mgb3JnYW5pemVycwojIGV4cGxpY2l0bHkgYXBwcm92ZSBwcmV0cmFpbmVkIHJlcHJlc2VudGF0aW9ucyB1bmRlciB0aGUgc3VwcGxpZWQtZGF0YSBydWxlLgp0cmFuc2Zvcm1lcnM9PTQuNDYuMwpGbGFnRW1iZWRkaW5nPT0xLjMuMwo=', '74742ed024654a5d9318d6ecbe82dd01b4152835796f6e68da64d5f7f1d7842f')

## Embedded source: `code/business_entity_resolution/requirements-remote.txt`

~~~~text
# Optional full-scale training backend. The core pipeline and smoke tests do not require it.
lightgbm==4.6.0

~~~~

In [ ]:
materialize('code/business_entity_resolution/requirements-remote.txt', 'IyBPcHRpb25hbCBmdWxsLXNjYWxlIHRyYWluaW5nIGJhY2tlbmQuIFRoZSBjb3JlIHBpcGVsaW5lIGFuZCBzbW9rZSB0ZXN0cyBkbyBub3QgcmVxdWlyZSBpdC4KbGlnaHRnYm09PTQuNi4wCg==', '77f9a1de8c7fc618f3ad5fb5731f922c0518189111d0fd0bdf086116eb11dfc7')

## Embedded source: `code/business_entity_resolution/requirements.txt`

~~~~text
numpy==2.1.3
pandas==2.2.3
scipy==1.15.2
scikit-learn==1.6.1
joblib==1.4.2
rapidfuzz==3.12.2
pyarrow==18.0.0
psutil==6.1.0

~~~~

In [ ]:
materialize('code/business_entity_resolution/requirements.txt', 'bnVtcHk9PTIuMS4zCnBhbmRhcz09Mi4yLjMKc2NpcHk9PTEuMTUuMgpzY2lraXQtbGVhcm49PTEuNi4xCmpvYmxpYj09MS40LjIKcmFwaWRmdXp6PT0zLjEyLjIKcHlhcnJvdz09MTguMC4wCnBzdXRpbD09Ni4xLjAK', 'b0f54f1f482faf0fb2931c4e4da99153e20ef48eeab14d9f043ed37345952093')

## Embedded source: `code/business_entity_resolution/run_pipeline.py`

~~~~python
#!/usr/bin/env python3
"""Single entry point to run the entire Business Entity Resolution pipeline.

You run this one file; it runs every stage in order and resumes automatically
if anything stops. Choose the profile with a flag:

    python run_pipeline.py --profile mini     # 15 realistic records
    python run_pipeline.py --profile full     # the real dataset

Environment (all optional; sensible defaults shown):

    BER_DATA_ROOT      dataset location        (mini: dataset/mini, full: dataset)
    BER_ARTIFACT_ROOT  intermediate artifacts   (default: artifacts/<profile>)
    BER_OUTPUT_ROOT    final TSV output dir     (default: output/<profile>)
    BER_THREADS        worker threads           (default: all CPUs - 1)

Everything is idempotent: re-running skips completed work. Per-stage logs go to
<artifact_root>/<run_id>/logs/. The final files are
output/<profile>/matching_results.tsv and candidate_pairs.tsv.

Optionally validate against the official validator:

    python run_pipeline.py --profile mini --validator /path/to/validate_submission.py
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

# Make the package importable without an editable install.
PACKAGE_ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from business_entity_resolution.cli import main as cli_main  # noqa: E402

PROFILES = {
    "mini": {
        "config": "configs/mini.json",
        "data_root": "dataset/mini",
        "artifact_root": "artifacts/mini",
        "output_root": "output/mini",
        "folds": 5,
        "model": "lightgbm",
    },
    "full": {
        "config": "configs/remote_full.json",
        "data_root": "dataset",
        "artifact_root": "artifacts/full",
        "output_root": "output/full",
        "folds": 10,
        "model": "lightgbm",
    },
}


def _sh(args: list[str]) -> int:
    print(f"\n$ {' '.join(args)}", flush=True)
    return int(cli_main(args))


def run_profile(profile: str, validator: str | None, skip_embeddings: bool) -> int:
    spec = PROFILES[profile]
    config = str((PACKAGE_ROOT / spec["config"]).resolve())

    # Resolve the data root. Mini lives in the package; full may live elsewhere
    # (student_resource, a mounted volume, etc.), so auto-detect it when the
    # expected local folder is absent.
    if profile == "full" and "BER_DATA_ROOT" not in os.environ:
        local = (PACKAGE_ROOT / spec["data_root"]).resolve()
        os.environ["BER_DATA_ROOT"] = str(_find_real_dataset() or local)
    else:
        os.environ.setdefault("BER_DATA_ROOT", str((PACKAGE_ROOT / spec["data_root"]).resolve()))
    os.environ.setdefault("BER_ARTIFACT_ROOT", str((PACKAGE_ROOT / spec["artifact_root"]).resolve()))
    os.environ.setdefault("BER_OUTPUT_ROOT", str((PACKAGE_ROOT / spec["output_root"]).resolve()))
    if "BER_THREADS" not in os.environ:
        os.environ["BER_THREADS"] = str(max(1, (os.cpu_count() or 2) - 1))

    print("=" * 72)
    print(f"Business Entity Resolution — profile={profile}")
    print(f"  data_root     = {os.environ['BER_DATA_ROOT']}")
    print(f"  artifact_root = {os.environ['BER_ARTIFACT_ROOT']}")
    print(f"  output_root   = {os.environ['BER_OUTPUT_ROOT']}")
    print(f"  threads       = {os.environ['BER_THREADS']}")
    print("=" * 72)

    started = time.time()
    config_payload = json.loads(Path(config).read_text(encoding="utf-8"))
    run_id = str(config_payload.get("run_id", "default"))
    model_dir = Path(os.environ["BER_ARTIFACT_ROOT"]) / run_id / "models"
    model_suffix = {"lightgbm": ".txt", "sgd": ".joblib", "deterministic": ".json"}[spec["model"]]
    validation0_model = model_dir / f"{spec['model']}-validation-fold0{model_suffix}"
    validation1_model = model_dir / f"{spec['model']}-validation-fold1{model_suffix}"
    final_model = model_dir / f"{spec['model']}-final{model_suffix}"
    steps = [
        ["prepare", "--config", config, "--split", "both"],
        ["make-splits", "--config", config, "--folds", str(spec["folds"])],
        ["generate-candidates", "--config", config, "--split", "train"],
        ["generate-candidates", "--config", config, "--split", "test"],
    ]
    if not skip_embeddings:
        steps.append(["embed", "--config", config, "--split", "both"])
    steps += [
        ["build-features", "--config", config, "--split", "train"],
        ["build-features", "--config", config, "--split", "test"],
        ["train", "--config", config, "--model", spec["model"], "--exclude-folds", "0,1", "--output-name", f"{spec['model']}-validation-fold0"],
        ["score", "--config", config, "--split", "train", "--model", spec["model"], "--model-path", str(validation0_model), "--validation-fold", "0", "--output-name", "validation-fold0.parquet"],
        ["tune-decision", "--config", config, "--score-file", "validation-fold0.parquet", "--validation-fold", "0"],
        ["train", "--config", config, "--model", spec["model"], "--validation-fold", "1", "--output-name", f"{spec['model']}-validation-fold1"],
        ["score", "--config", config, "--split", "train", "--model", spec["model"], "--model-path", str(validation1_model), "--validation-fold", "1", "--output-name", "evaluation-fold1.parquet"],
        ["evaluate", "--config", config, "--score-file", "evaluation-fold1.parquet", "--validation-fold", "1"],
        ["train", "--config", config, "--model", spec["model"], "--all-training-data", "--output-name", f"{spec['model']}-final"],
        ["infer", "--config", config, "--model", spec["model"], "--model-path", str(final_model)],
    ]
    if validator:
        steps.append(["preflight", "--config", config, "--official-validator", validator, "--check-ids"])

    for index, step in enumerate(steps, start=1):
        print(f"\n### Step {index}/{len(steps)}: {step[0]}")
        code = _sh(step)
        if code != 0:
            print(f"\nStage '{step[0]}' exited with code {code}. Stopping.")
            print("Fix the issue and re-run the same command — completed work is reused.")
            return code

    elapsed = time.time() - started
    print("\n" + "=" * 72)
    print(f"Pipeline complete in {elapsed/60:.1f} min.")
    print(f"Outputs: {os.environ['BER_OUTPUT_ROOT']}/matching_results.tsv")
    print(f"         {os.environ['BER_OUTPUT_ROOT']}/candidate_pairs.tsv")
    print("=" * 72)
    return 0


def _find_real_dataset() -> Path | None:
    """Locate the real challenge dataset (train/ and test/ TSVs)."""
    candidates = [
        Path(os.environ["BER_SOURCE_DATA_ROOT"]) if os.environ.get("BER_SOURCE_DATA_ROOT") else None,
        PACKAGE_ROOT / "dataset",
        PACKAGE_ROOT.parent.parent / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset",
        PACKAGE_ROOT.parent.parent / "student_resource" / "dataset",
        PACKAGE_ROOT.parent.parent.parent / "student_resource" / "dataset",
        PACKAGE_ROOT.parent.parent.parent / "dataset",
        Path.cwd() / "dataset",
    ]
    for candidate in candidates:
        if candidate and (candidate / "train" / "train_source1.tsv").is_file():
            return candidate.resolve()
    return None


def main() -> int:
    parser = argparse.ArgumentParser(description="Run the full BER pipeline end-to-end.")
    parser.add_argument("--profile", choices=sorted(PROFILES), required=True,
                        help="mini = 15 realistic records; full = the real dataset")
    parser.add_argument("--validator", default=None,
                        help="Path to utils/validate_submission.py to run at the end.")
    parser.add_argument("--skip-embeddings", action="store_true",
                        help="Skip the BGE-M3 embedding stage (faster, no torch needed).")
    args = parser.parse_args()

    if args.profile == "mini" and not (PACKAGE_ROOT / "dataset/mini/train/train_source1.tsv").exists():
        # The mini set is sampled from the real files; build it first from the
        # REAL dataset (not dataset/mini, which does not exist yet).
        real_root = _find_real_dataset()
        if real_root is None:
            print("Could not find the real dataset. Set BER_SOURCE_DATA_ROOT to the "
                  "folder containing train/ and test/ TSVs.", file=sys.stderr)
            return 2
        print(f"Mini dataset not found; building 15 records from {real_root} ...")
        mini_config = str((PACKAGE_ROOT / PROFILES["mini"]["config"]).resolve())
        code = _sh(["make-mini", "--config", mini_config,
                    "--mini-root", str((PACKAGE_ROOT / "dataset/mini").resolve()),
                    "--source-root", str(real_root), "--count", "15"])
        if code != 0:
            return code

    return run_profile(args.profile, args.validator, args.skip_embeddings)


if __name__ == "__main__":
    raise SystemExit(main())

~~~~

In [ ]:
materialize('code/business_entity_resolution/run_pipeline.py', 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTaW5nbGUgZW50cnkgcG9pbnQgdG8gcnVuIHRoZSBlbnRpcmUgQnVzaW5lc3MgRW50aXR5IFJlc29sdXRpb24gcGlwZWxpbmUuCgpZb3UgcnVuIHRoaXMgb25lIGZpbGU7IGl0IHJ1bnMgZXZlcnkgc3RhZ2UgaW4gb3JkZXIgYW5kIHJlc3VtZXMgYXV0b21hdGljYWxseQppZiBhbnl0aGluZyBzdG9wcy4gQ2hvb3NlIHRoZSBwcm9maWxlIHdpdGggYSBmbGFnOgoKICAgIHB5dGhvbiBydW5fcGlwZWxpbmUucHkgLS1wcm9maWxlIG1pbmkgICAgICMgMTUgcmVhbGlzdGljIHJlY29yZHMKICAgIHB5dGhvbiBydW5fcGlwZWxpbmUucHkgLS1wcm9maWxlIGZ1bGwgICAgICMgdGhlIHJlYWwgZGF0YXNldAoKRW52aXJvbm1lbnQgKGFsbCBvcHRpb25hbDsgc2Vuc2libGUgZGVmYXVsdHMgc2hvd24pOgoKICAgIEJFUl9EQVRBX1JPT1QgICAgICBkYXRhc2V0IGxvY2F0aW9uICAgICAgICAobWluaTogZGF0YXNldC9taW5pLCBmdWxsOiBkYXRhc2V0KQogICAgQkVSX0FSVElGQUNUX1JPT1QgIGludGVybWVkaWF0ZSBhcnRpZmFjdHMgICAoZGVmYXVsdDogYXJ0aWZhY3RzLzxwcm9maWxlPikKICAgIEJFUl9PVVRQVVRfUk9PVCAgICBmaW5hbCBUU1Ygb3V0cHV0IGRpciAgICAgKGRlZmF1bHQ6IG91dHB1dC88cHJvZmlsZT4pCiAgICBCRVJfVEhSRUFEUyAgICAgICAgd29ya2VyIHRocmVhZHMgICAgICAgICAgIChkZWZhdWx0OiBhbGwgQ1BVcyAtIDEpCgpFdmVyeXRoaW5nIGlzIGlkZW1wb3RlbnQ6IHJlLXJ1bm5pbmcgc2tpcHMgY29tcGxldGVkIHdvcmsuIFBlci1zdGFnZSBsb2dzIGdvIHRvCjxhcnRpZmFjdF9yb290Pi88cnVuX2lkPi9sb2dzLy4gVGhlIGZpbmFsIGZpbGVzIGFyZQpvdXRwdXQvPHByb2ZpbGU+L21hdGNoaW5nX3Jlc3VsdHMudHN2IGFuZCBjYW5kaWRhdGVfcGFpcnMudHN2LgoKT3B0aW9uYWxseSB2YWxpZGF0ZSBhZ2FpbnN0IHRoZSBvZmZpY2lhbCB2YWxpZGF0b3I6CgogICAgcHl0aG9uIHJ1bl9waXBlbGluZS5weSAtLXByb2ZpbGUgbWluaSAtLXZhbGlkYXRvciAvcGF0aC90by92YWxpZGF0ZV9zdWJtaXNzaW9uLnB5CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKIyBNYWtlIHRoZSBwYWNrYWdlIGltcG9ydGFibGUgd2l0aG91dCBhbiBlZGl0YWJsZSBpbnN0YWxsLgpQQUNLQUdFX1JPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUEFDS0FHRV9ST09UIC8gInNyYyIpKQoKZnJvbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5jbGkgaW1wb3J0IG1haW4gYXMgY2xpX21haW4gICMgbm9xYTogRTQwMgoKUFJPRklMRVMgPSB7CiAgICAibWluaSI6IHsKICAgICAgICAiY29uZmlnIjogImNvbmZpZ3MvbWluaS5qc29uIiwKICAgICAgICAiZGF0YV9yb290IjogImRhdGFzZXQvbWluaSIsCiAgICAgICAgImFydGlmYWN0X3Jvb3QiOiAiYXJ0aWZhY3RzL21pbmkiLAogICAgICAgICJvdXRwdXRfcm9vdCI6ICJvdXRwdXQvbWluaSIsCiAgICAgICAgImZvbGRzIjogNSwKICAgICAgICAibW9kZWwiOiAibGlnaHRnYm0iLAogICAgfSwKICAgICJmdWxsIjogewogICAgICAgICJjb25maWciOiAiY29uZmlncy9yZW1vdGVfZnVsbC5qc29uIiwKICAgICAgICAiZGF0YV9yb290IjogImRhdGFzZXQiLAogICAgICAgICJhcnRpZmFjdF9yb290IjogImFydGlmYWN0cy9mdWxsIiwKICAgICAgICAib3V0cHV0X3Jvb3QiOiAib3V0cHV0L2Z1bGwiLAogICAgICAgICJmb2xkcyI6IDEwLAogICAgICAgICJtb2RlbCI6ICJsaWdodGdibSIsCiAgICB9LAp9CgoKZGVmIF9zaChhcmdzOiBsaXN0W3N0cl0pIC0+IGludDoKICAgIHByaW50KGYiXG4kIHsnICcuam9pbihhcmdzKX0iLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIGludChjbGlfbWFpbihhcmdzKSkKCgpkZWYgcnVuX3Byb2ZpbGUocHJvZmlsZTogc3RyLCB2YWxpZGF0b3I6IHN0ciB8IE5vbmUsIHNraXBfZW1iZWRkaW5nczogYm9vbCkgLT4gaW50OgogICAgc3BlYyA9IFBST0ZJTEVTW3Byb2ZpbGVdCiAgICBjb25maWcgPSBzdHIoKFBBQ0tBR0VfUk9PVCAvIHNwZWNbImNvbmZpZyJdKS5yZXNvbHZlKCkpCgogICAgIyBSZXNvbHZlIHRoZSBkYXRhIHJvb3QuIE1pbmkgbGl2ZXMgaW4gdGhlIHBhY2thZ2U7IGZ1bGwgbWF5IGxpdmUgZWxzZXdoZXJlCiAgICAjIChzdHVkZW50X3Jlc291cmNlLCBhIG1vdW50ZWQgdm9sdW1lLCBldGMuKSwgc28gYXV0by1kZXRlY3QgaXQgd2hlbiB0aGUKICAgICMgZXhwZWN0ZWQgbG9jYWwgZm9sZGVyIGlzIGFic2VudC4KICAgIGlmIHByb2ZpbGUgPT0gImZ1bGwiIGFuZCAiQkVSX0RBVEFfUk9PVCIgbm90IGluIG9zLmVudmlyb246CiAgICAgICAgbG9jYWwgPSAoUEFDS0FHRV9ST09UIC8gc3BlY1siZGF0YV9yb290Il0pLnJlc29sdmUoKQogICAgICAgIG9zLmVudmlyb25bIkJFUl9EQVRBX1JPT1QiXSA9IHN0cihfZmluZF9yZWFsX2RhdGFzZXQoKSBvciBsb2NhbCkKICAgIGVsc2U6CiAgICAgICAgb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJCRVJfREFUQV9ST09UIiwgc3RyKChQQUNLQUdFX1JPT1QgLyBzcGVjWyJkYXRhX3Jvb3QiXSkucmVzb2x2ZSgpKSkKICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgiQkVSX0FSVElGQUNUX1JPT1QiLCBzdHIoKFBBQ0tBR0VfUk9PVCAvIHNwZWNbImFydGlmYWN0X3Jvb3QiXSkucmVzb2x2ZSgpKSkKICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgiQkVSX09VVFBVVF9ST09UIiwgc3RyKChQQUNLQUdFX1JPT1QgLyBzcGVjWyJvdXRwdXRfcm9vdCJdKS5yZXNvbHZlKCkpKQogICAgaWYgIkJFUl9USFJFQURTIiBub3QgaW4gb3MuZW52aXJvbjoKICAgICAgICBvcy5lbnZpcm9uWyJCRVJfVEhSRUFEUyJdID0gc3RyKG1heCgxLCAob3MuY3B1X2NvdW50KCkgb3IgMikgLSAxKSkKCiAgICBwcmludCgiPSIgKiA3MikKICAgIHByaW50KGYiQnVzaW5lc3MgRW50aXR5IFJlc29sdXRpb24g4oCUIHByb2ZpbGU9e3Byb2ZpbGV9IikKICAgIHByaW50KGYiICBkYXRhX3Jvb3QgICAgID0ge29zLmVudmlyb25bJ0JFUl9EQVRBX1JPT1QnXX0iKQogICAgcHJpbnQoZiIgIGFydGlmYWN0X3Jvb3QgPSB7b3MuZW52aXJvblsnQkVSX0FSVElGQUNUX1JPT1QnXX0iKQogICAgcHJpbnQoZiIgIG91dHB1dF9yb290ICAgPSB7b3MuZW52aXJvblsnQkVSX09VVFBVVF9ST09UJ119IikKICAgIHByaW50KGYiICB0aHJlYWRzICAgICAgID0ge29zLmVudmlyb25bJ0JFUl9USFJFQURTJ119IikKICAgIHByaW50KCI9IiAqIDcyKQoKICAgIHN0YXJ0ZWQgPSB0aW1lLnRpbWUoKQogICAgY29uZmlnX3BheWxvYWQgPSBqc29uLmxvYWRzKFBhdGgoY29uZmlnKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBydW5faWQgPSBzdHIoY29uZmlnX3BheWxvYWQuZ2V0KCJydW5faWQiLCAiZGVmYXVsdCIpKQogICAgbW9kZWxfZGlyID0gUGF0aChvcy5lbnZpcm9uWyJCRVJfQVJUSUZBQ1RfUk9PVCJdKSAvIHJ1bl9pZCAvICJtb2RlbHMiCiAgICBtb2RlbF9zdWZmaXggPSB7ImxpZ2h0Z2JtIjogIi50eHQiLCAic2dkIjogIi5qb2JsaWIiLCAiZGV0ZXJtaW5pc3RpYyI6ICIuanNvbiJ9W3NwZWNbIm1vZGVsIl1dCiAgICB2YWxpZGF0aW9uMF9tb2RlbCA9IG1vZGVsX2RpciAvIGYie3NwZWNbJ21vZGVsJ119LXZhbGlkYXRpb24tZm9sZDB7bW9kZWxfc3VmZml4fSIKICAgIHZhbGlkYXRpb24xX21vZGVsID0gbW9kZWxfZGlyIC8gZiJ7c3BlY1snbW9kZWwnXX0tdmFsaWRhdGlvbi1mb2xkMXttb2RlbF9zdWZmaXh9IgogICAgZmluYWxfbW9kZWwgPSBtb2RlbF9kaXIgLyBmIntzcGVjWydtb2RlbCddfS1maW5hbHttb2RlbF9zdWZmaXh9IgogICAgc3RlcHMgPSBbCiAgICAgICAgWyJwcmVwYXJlIiwgIi0tY29uZmlnIiwgY29uZmlnLCAiLS1zcGxpdCIsICJib3RoIl0sCiAgICAgICAgWyJtYWtlLXNwbGl0cyIsICItLWNvbmZpZyIsIGNvbmZpZywgIi0tZm9sZHMiLCBzdHIoc3BlY1siZm9sZHMiXSldLAogICAgICAgIFsiZ2VuZXJhdGUtY2FuZGlkYXRlcyIsICItLWNvbmZpZyIsIGNvbmZpZywgIi0tc3BsaXQiLCAidHJhaW4iXSwKICAgICAgICBbImdlbmVyYXRlLWNhbmRpZGF0ZXMiLCAiLS1jb25maWciLCBjb25maWcsICItLXNwbGl0IiwgInRlc3QiXSwKICAgIF0KICAgIGlmIG5vdCBza2lwX2VtYmVkZGluZ3M6CiAgICAgICAgc3RlcHMuYXBwZW5kKFsiZW1iZWQiLCAiLS1jb25maWciLCBjb25maWcsICItLXNwbGl0IiwgImJvdGgiXSkKICAgIHN0ZXBzICs9IFsKICAgICAgICBbImJ1aWxkLWZlYXR1cmVzIiwgIi0tY29uZmlnIiwgY29uZmlnLCAiLS1zcGxpdCIsICJ0cmFpbiJdLAogICAgICAgIFsiYnVpbGQtZmVhdHVyZXMiLCAiLS1jb25maWciLCBjb25maWcsICItLXNwbGl0IiwgInRlc3QiXSwKICAgICAgICBbInRyYWluIiwgIi0tY29uZmlnIiwgY29uZmlnLCAiLS1tb2RlbCIsIHNwZWNbIm1vZGVsIl0sICItLWV4Y2x1ZGUtZm9sZHMiLCAiMCwxIiwgIi0tb3V0cHV0LW5hbWUiLCBmIntzcGVjWydtb2RlbCddfS12YWxpZGF0aW9uLWZvbGQwIl0sCiAgICAgICAgWyJzY29yZSIsICItLWNvbmZpZyIsIGNvbmZpZywgIi0tc3BsaXQiLCAidHJhaW4iLCAiLS1tb2RlbCIsIHNwZWNbIm1vZGVsIl0sICItLW1vZGVsLXBhdGgiLCBzdHIodmFsaWRhdGlvbjBfbW9kZWwpLCAiLS12YWxpZGF0aW9uLWZvbGQiLCAiMCIsICItLW91dHB1dC1uYW1lIiwgInZhbGlkYXRpb24tZm9sZDAucGFycXVldCJdLAogICAgICAgIFsidHVuZS1kZWNpc2lvbiIsICItLWNvbmZpZyIsIGNvbmZpZywgIi0tc2NvcmUtZmlsZSIsICJ2YWxpZGF0aW9uLWZvbGQwLnBhcnF1ZXQiLCAiLS12YWxpZGF0aW9uLWZvbGQiLCAiMCJdLAogICAgICAgIFsidHJhaW4iLCAiLS1jb25maWciLCBjb25maWcsICItLW1vZGVsIiwgc3BlY1sibW9kZWwiXSwgIi0tdmFsaWRhdGlvbi1mb2xkIiwgIjEiLCAiLS1vdXRwdXQtbmFtZSIsIGYie3NwZWNbJ21vZGVsJ119LXZhbGlkYXRpb24tZm9sZDEiXSwKICAgICAgICBbInNjb3JlIiwgIi0tY29uZmlnIiwgY29uZmlnLCAiLS1zcGxpdCIsICJ0cmFpbiIsICItLW1vZGVsIiwgc3BlY1sibW9kZWwiXSwgIi0tbW9kZWwtcGF0aCIsIHN0cih2YWxpZGF0aW9uMV9tb2RlbCksICItLXZhbGlkYXRpb24tZm9sZCIsICIxIiwgIi0tb3V0cHV0LW5hbWUiLCAiZXZhbHVhdGlvbi1mb2xkMS5wYXJxdWV0Il0sCiAgICAgICAgWyJldmFsdWF0ZSIsICItLWNvbmZpZyIsIGNvbmZpZywgIi0tc2NvcmUtZmlsZSIsICJldmFsdWF0aW9uLWZvbGQxLnBhcnF1ZXQiLCAiLS12YWxpZGF0aW9uLWZvbGQiLCAiMSJdLAogICAgICAgIFsidHJhaW4iLCAiLS1jb25maWciLCBjb25maWcsICItLW1vZGVsIiwgc3BlY1sibW9kZWwiXSwgIi0tYWxsLXRyYWluaW5nLWRhdGEiLCAiLS1vdXRwdXQtbmFtZSIsIGYie3NwZWNbJ21vZGVsJ119LWZpbmFsIl0sCiAgICAgICAgWyJpbmZlciIsICItLWNvbmZpZyIsIGNvbmZpZywgIi0tbW9kZWwiLCBzcGVjWyJtb2RlbCJdLCAiLS1tb2RlbC1wYXRoIiwgc3RyKGZpbmFsX21vZGVsKV0sCiAgICBdCiAgICBpZiB2YWxpZGF0b3I6CiAgICAgICAgc3RlcHMuYXBwZW5kKFsicHJlZmxpZ2h0IiwgIi0tY29uZmlnIiwgY29uZmlnLCAiLS1vZmZpY2lhbC12YWxpZGF0b3IiLCB2YWxpZGF0b3IsICItLWNoZWNrLWlkcyJdKQoKICAgIGZvciBpbmRleCwgc3RlcCBpbiBlbnVtZXJhdGUoc3RlcHMsIHN0YXJ0PTEpOgogICAgICAgIHByaW50KGYiXG4jIyMgU3RlcCB7aW5kZXh9L3tsZW4oc3RlcHMpfToge3N0ZXBbMF19IikKICAgICAgICBjb2RlID0gX3NoKHN0ZXApCiAgICAgICAgaWYgY29kZSAhPSAwOgogICAgICAgICAgICBwcmludChmIlxuU3RhZ2UgJ3tzdGVwWzBdfScgZXhpdGVkIHdpdGggY29kZSB7Y29kZX0uIFN0b3BwaW5nLiIpCiAgICAgICAgICAgIHByaW50KCJGaXggdGhlIGlzc3VlIGFuZCByZS1ydW4gdGhlIHNhbWUgY29tbWFuZCDigJQgY29tcGxldGVkIHdvcmsgaXMgcmV1c2VkLiIpCiAgICAgICAgICAgIHJldHVybiBjb2RlCgogICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gc3RhcnRlZAogICAgcHJpbnQoIlxuIiArICI9IiAqIDcyKQogICAgcHJpbnQoZiJQaXBlbGluZSBjb21wbGV0ZSBpbiB7ZWxhcHNlZC82MDouMWZ9IG1pbi4iKQogICAgcHJpbnQoZiJPdXRwdXRzOiB7b3MuZW52aXJvblsnQkVSX09VVFBVVF9ST09UJ119L21hdGNoaW5nX3Jlc3VsdHMudHN2IikKICAgIHByaW50KGYiICAgICAgICAge29zLmVudmlyb25bJ0JFUl9PVVRQVVRfUk9PVCddfS9jYW5kaWRhdGVfcGFpcnMudHN2IikKICAgIHByaW50KCI9IiAqIDcyKQogICAgcmV0dXJuIDAKCgpkZWYgX2ZpbmRfcmVhbF9kYXRhc2V0KCkgLT4gUGF0aCB8IE5vbmU6CiAgICAiIiJMb2NhdGUgdGhlIHJlYWwgY2hhbGxlbmdlIGRhdGFzZXQgKHRyYWluLyBhbmQgdGVzdC8gVFNWcykuIiIiCiAgICBjYW5kaWRhdGVzID0gWwogICAgICAgIFBhdGgob3MuZW52aXJvblsiQkVSX1NPVVJDRV9EQVRBX1JPT1QiXSkgaWYgb3MuZW52aXJvbi5nZXQoIkJFUl9TT1VSQ0VfREFUQV9ST09UIikgZWxzZSBOb25lLAogICAgICAgIFBBQ0tBR0VfUk9PVCAvICJkYXRhc2V0IiwKICAgICAgICBQQUNLQUdFX1JPT1QucGFyZW50LnBhcmVudCAvICI2YWIxMGViM2IyM2JhX3N0dWRlbnRfcmVzb3VyY2UiIC8gInN0dWRlbnRfcmVzb3VyY2UiIC8gImRhdGFzZXQiLAogICAgICAgIFBBQ0tBR0VfUk9PVC5wYXJlbnQucGFyZW50IC8gInN0dWRlbnRfcmVzb3VyY2UiIC8gImRhdGFzZXQiLAogICAgICAgIFBBQ0tBR0VfUk9PVC5wYXJlbnQucGFyZW50LnBhcmVudCAvICJzdHVkZW50X3Jlc291cmNlIiAvICJkYXRhc2V0IiwKICAgICAgICBQQUNLQUdFX1JPT1QucGFyZW50LnBhcmVudC5wYXJlbnQgLyAiZGF0YXNldCIsCiAgICAgICAgUGF0aC5jd2QoKSAvICJkYXRhc2V0IiwKICAgIF0KICAgIGZvciBjYW5kaWRhdGUgaW4gY2FuZGlkYXRlczoKICAgICAgICBpZiBjYW5kaWRhdGUgYW5kIChjYW5kaWRhdGUgLyAidHJhaW4iIC8gInRyYWluX3NvdXJjZTEudHN2IikuaXNfZmlsZSgpOgogICAgICAgICAgICByZXR1cm4gY2FuZGlkYXRlLnJlc29sdmUoKQogICAgcmV0dXJuIE5vbmUKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJSdW4gdGhlIGZ1bGwgQkVSIHBpcGVsaW5lIGVuZC10by1lbmQuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcHJvZmlsZSIsIGNob2ljZXM9c29ydGVkKFBST0ZJTEVTKSwgcmVxdWlyZWQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0ibWluaSA9IDE1IHJlYWxpc3RpYyByZWNvcmRzOyBmdWxsID0gdGhlIHJlYWwgZGF0YXNldCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXZhbGlkYXRvciIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iUGF0aCB0byB1dGlscy92YWxpZGF0ZV9zdWJtaXNzaW9uLnB5IHRvIHJ1biBhdCB0aGUgZW5kLiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNraXAtZW1iZWRkaW5ncyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IlNraXAgdGhlIEJHRS1NMyBlbWJlZGRpbmcgc3RhZ2UgKGZhc3Rlciwgbm8gdG9yY2ggbmVlZGVkKS4iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBpZiBhcmdzLnByb2ZpbGUgPT0gIm1pbmkiIGFuZCBub3QgKFBBQ0tBR0VfUk9PVCAvICJkYXRhc2V0L21pbmkvdHJhaW4vdHJhaW5fc291cmNlMS50c3YiKS5leGlzdHMoKToKICAgICAgICAjIFRoZSBtaW5pIHNldCBpcyBzYW1wbGVkIGZyb20gdGhlIHJlYWwgZmlsZXM7IGJ1aWxkIGl0IGZpcnN0IGZyb20gdGhlCiAgICAgICAgIyBSRUFMIGRhdGFzZXQgKG5vdCBkYXRhc2V0L21pbmksIHdoaWNoIGRvZXMgbm90IGV4aXN0IHlldCkuCiAgICAgICAgcmVhbF9yb290ID0gX2ZpbmRfcmVhbF9kYXRhc2V0KCkKICAgICAgICBpZiByZWFsX3Jvb3QgaXMgTm9uZToKICAgICAgICAgICAgcHJpbnQoIkNvdWxkIG5vdCBmaW5kIHRoZSByZWFsIGRhdGFzZXQuIFNldCBCRVJfU09VUkNFX0RBVEFfUk9PVCB0byB0aGUgIgogICAgICAgICAgICAgICAgICAiZm9sZGVyIGNvbnRhaW5pbmcgdHJhaW4vIGFuZCB0ZXN0LyBUU1ZzLiIsIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICAgICAgcmV0dXJuIDIKICAgICAgICBwcmludChmIk1pbmkgZGF0YXNldCBub3QgZm91bmQ7IGJ1aWxkaW5nIDE1IHJlY29yZHMgZnJvbSB7cmVhbF9yb290fSAuLi4iKQogICAgICAgIG1pbmlfY29uZmlnID0gc3RyKChQQUNLQUdFX1JPT1QgLyBQUk9GSUxFU1sibWluaSJdWyJjb25maWciXSkucmVzb2x2ZSgpKQogICAgICAgIGNvZGUgPSBfc2goWyJtYWtlLW1pbmkiLCAiLS1jb25maWciLCBtaW5pX2NvbmZpZywKICAgICAgICAgICAgICAgICAgICAiLS1taW5pLXJvb3QiLCBzdHIoKFBBQ0tBR0VfUk9PVCAvICJkYXRhc2V0L21pbmkiKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICAgICAgICItLXNvdXJjZS1yb290Iiwgc3RyKHJlYWxfcm9vdCksICItLWNvdW50IiwgIjE1Il0pCiAgICAgICAgaWYgY29kZSAhPSAwOgogICAgICAgICAgICByZXR1cm4gY29kZQoKICAgIHJldHVybiBydW5fcHJvZmlsZShhcmdzLnByb2ZpbGUsIGFyZ3MudmFsaWRhdG9yLCBhcmdzLnNraXBfZW1iZWRkaW5ncykKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcmFpc2UgU3lzdGVtRXhpdChtYWluKCkpCg==', 'f145e61f5ad0baca8aaba18680698d4ecbdb8d4278f542c78111375eee84e8a6')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/__init__.py`

~~~~python
"""Amazon ML 2026 business entity resolution pipeline."""

__version__ = "0.1.0"


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/__init__.py', 'IiIiQW1hem9uIE1MIDIwMjYgYnVzaW5lc3MgZW50aXR5IHJlc29sdXRpb24gcGlwZWxpbmUuIiIiCgpfX3ZlcnNpb25fXyA9ICIwLjEuMCIKCg==', '4f8c899ddf314d9cc236a94c622a6b761421a648cd33b1518b382e0db4c1e7aa')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/__main__.py`

~~~~python
from .cli import main

raise SystemExit(main())


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/__main__.py', 'ZnJvbSAuY2xpIGltcG9ydCBtYWluCgpyYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkKCg==', '76fa916e93e0ae1a6c1bb51dcc0bcf289c014e9f760a47165f8a84401ccf41d7')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/artifacts.py`

~~~~python
"""Artifact persistence, fingerprints, and manifests."""

from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
import platform
import sys
import time
from typing import Any
import pandas as pd


def sha256_file(path: str | Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def config_fingerprint(config: dict[str, Any]) -> str:
    payload = json.dumps(config, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def write_frame(frame: pd.DataFrame, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temporary, index=False, engine="pyarrow")
    os.replace(temporary, path)
    return path


def read_frame(path: str | Path, columns: list[str] | None = None) -> pd.DataFrame:
    return pd.read_parquet(Path(path), columns=columns, engine="pyarrow")


def write_manifest(path: str | Path, *, stage: str, config: dict[str, Any], rows: int, schema: list[str], inputs: list[str] | None = None, metrics: dict[str, Any] | None = None, started_at: float | None = None) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    manifest = {
        "stage": stage,
        "config_fingerprint": config_fingerprint(config),
        "config": config,
        "rows": int(rows),
        "schema": schema,
        "inputs": inputs or [],
        "metrics": metrics or {},
        "python": sys.version,
        "platform": platform.platform(),
        "started_at_epoch": started_at,
        "completed_at_epoch": time.time(),
        "complete": True,
    }
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True, default=str), encoding="utf-8")
    os.replace(temporary, path)
    return path


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/artifacts.py', 'IiIiQXJ0aWZhY3QgcGVyc2lzdGVuY2UsIGZpbmdlcnByaW50cywgYW5kIG1hbmlmZXN0cy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQppbXBvcnQgcGFuZGFzIGFzIHBkCgoKZGVmIHNoYTI1Nl9maWxlKHBhdGg6IHN0ciB8IFBhdGgsIGJsb2NrX3NpemU6IGludCA9IDEwMjQgKiAxMDI0KSAtPiBzdHI6CiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigicmIiKSBhcyBoYW5kbGU6CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBoYW5kbGUucmVhZChibG9ja19zaXplKSwgYiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShibG9jaykKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgY29uZmlnX2ZpbmdlcnByaW50KGNvbmZpZzogZGljdFtzdHIsIEFueV0pIC0+IHN0cjoKICAgIHBheWxvYWQgPSBqc29uLmR1bXBzKGNvbmZpZywgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSkuZW5jb2RlKCJ1dGYtOCIpCiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYocGF5bG9hZCkuaGV4ZGlnZXN0KCkKCgpkZWYgd3JpdGVfZnJhbWUoZnJhbWU6IHBkLkRhdGFGcmFtZSwgcGF0aDogc3RyIHwgUGF0aCkgLT4gUGF0aDoKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KHBhdGguc3VmZml4ICsgIi50bXAiKQogICAgZnJhbWUudG9fcGFycXVldCh0ZW1wb3JhcnksIGluZGV4PUZhbHNlLCBlbmdpbmU9InB5YXJyb3ciKQogICAgb3MucmVwbGFjZSh0ZW1wb3JhcnksIHBhdGgpCiAgICByZXR1cm4gcGF0aAoKCmRlZiByZWFkX2ZyYW1lKHBhdGg6IHN0ciB8IFBhdGgsIGNvbHVtbnM6IGxpc3Rbc3RyXSB8IE5vbmUgPSBOb25lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICByZXR1cm4gcGQucmVhZF9wYXJxdWV0KFBhdGgocGF0aCksIGNvbHVtbnM9Y29sdW1ucywgZW5naW5lPSJweWFycm93IikKCgpkZWYgd3JpdGVfbWFuaWZlc3QocGF0aDogc3RyIHwgUGF0aCwgKiwgc3RhZ2U6IHN0ciwgY29uZmlnOiBkaWN0W3N0ciwgQW55XSwgcm93czogaW50LCBzY2hlbWE6IGxpc3Rbc3RyXSwgaW5wdXRzOiBsaXN0W3N0cl0gfCBOb25lID0gTm9uZSwgbWV0cmljczogZGljdFtzdHIsIEFueV0gfCBOb25lID0gTm9uZSwgc3RhcnRlZF9hdDogZmxvYXQgfCBOb25lID0gTm9uZSkgLT4gUGF0aDoKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBtYW5pZmVzdCA9IHsKICAgICAgICAic3RhZ2UiOiBzdGFnZSwKICAgICAgICAiY29uZmlnX2ZpbmdlcnByaW50IjogY29uZmlnX2ZpbmdlcnByaW50KGNvbmZpZyksCiAgICAgICAgImNvbmZpZyI6IGNvbmZpZywKICAgICAgICAicm93cyI6IGludChyb3dzKSwKICAgICAgICAic2NoZW1hIjogc2NoZW1hLAogICAgICAgICJpbnB1dHMiOiBpbnB1dHMgb3IgW10sCiAgICAgICAgIm1ldHJpY3MiOiBtZXRyaWNzIG9yIHt9LAogICAgICAgICJweXRob24iOiBzeXMudmVyc2lvbiwKICAgICAgICAicGxhdGZvcm0iOiBwbGF0Zm9ybS5wbGF0Zm9ybSgpLAogICAgICAgICJzdGFydGVkX2F0X2Vwb2NoIjogc3RhcnRlZF9hdCwKICAgICAgICAiY29tcGxldGVkX2F0X2Vwb2NoIjogdGltZS50aW1lKCksCiAgICAgICAgImNvbXBsZXRlIjogVHJ1ZSwKICAgIH0KICAgIHRlbXBvcmFyeSA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAiLnRtcCIpCiAgICB0ZW1wb3Jhcnkud3JpdGVfdGV4dChqc29uLmR1bXBzKG1hbmlmZXN0LCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUsIGRlZmF1bHQ9c3RyKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIG9zLnJlcGxhY2UodGVtcG9yYXJ5LCBwYXRoKQogICAgcmV0dXJuIHBhdGgKCg==', '2c5a65e0886c9580999218e331a475c7b2f661e7d4f7eeca9e519bd22b9eaf76')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/audit.py`

~~~~python
"""Bounded, chunked source-data audit."""

from __future__ import annotations

from collections import Counter
from pathlib import Path
from typing import Any

from .data import iter_tsv, source_path, ground_truth_path
from .schemas import SOURCE_COLUMNS, GROUND_TRUTH_COLUMNS
from .labels import parse_match_ids


def audit_source(path: str | Path, source: int, batch_size: int = 100_000, max_rows: int | None = None) -> dict[str, Any]:
    prefix = f"S{source}-"
    rows = 0
    blanks = Counter()
    countries = Counter()
    ids: set[str] = set()
    duplicate_ids = bad_ids = 0
    for chunk in iter_tsv(path, SOURCE_COLUMNS, batch_size, max_rows):
        rows += len(chunk)
        countries.update(chunk["country"])
        for column in SOURCE_COLUMNS:
            blanks[column] += int(chunk[column].eq("").sum())
        for entity_id in chunk["entity_id"]:
            bad_ids += int(not (entity_id.startswith(prefix) and entity_id[len(prefix):].isdigit()))
            duplicate_ids += int(entity_id in ids)
            ids.add(entity_id)
    return {
        "path": str(path), "rows": rows, "columns": list(SOURCE_COLUMNS),
        "blank_counts": dict(blanks), "countries": dict(countries),
        "duplicate_entity_ids": duplicate_ids, "bad_entity_ids": bad_ids,
    }


def audit_ground_truth(path: str | Path, batch_size: int = 100_000, max_rows: int | None = None) -> dict[str, Any]:
    rows = positives = singletons = duplicate_s1 = malformed = 0
    ids: set[str] = set()
    cardinality = Counter()
    for chunk in iter_tsv(path, GROUND_TRUTH_COLUMNS, batch_size, max_rows):
        for row in chunk.itertuples(index=False):
            rows += 1
            duplicate_s1 += int(row.source1_entity_id in ids)
            ids.add(row.source1_entity_id)
            try:
                matches = parse_match_ids(row.matched_entity_ids)
            except ValueError:
                malformed += 1
                continue
            cardinality[len(matches)] += 1
            positives += len(matches)
            singletons += int(not matches)
    return {
        "path": str(path), "rows": rows, "columns": list(GROUND_TRUTH_COLUMNS),
        "positive_links": positives, "singletons": singletons,
        "singleton_rate": singletons / rows if rows else 0.0,
        "cardinality": dict(sorted(cardinality.items())),
        "duplicate_source1_ids": duplicate_s1, "malformed_rows": malformed,
    }


def audit_dataset(data_root: str | Path, batch_size: int = 100_000, max_rows: int | None = None) -> dict[str, Any]:
    result: dict[str, Any] = {"sources": {}}
    for split in ("train", "test"):
        for source in (1, 2, 3):
            key = f"{split}_source{source}"
            result["sources"][key] = audit_source(source_path(data_root, split, source), source, batch_size, max_rows)
    result["ground_truth"] = audit_ground_truth(ground_truth_path(data_root), batch_size, max_rows)
    return result


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/audit.py', 'IiIiQm91bmRlZCwgY2h1bmtlZCBzb3VyY2UtZGF0YSBhdWRpdC4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gLmRhdGEgaW1wb3J0IGl0ZXJfdHN2LCBzb3VyY2VfcGF0aCwgZ3JvdW5kX3RydXRoX3BhdGgKZnJvbSAuc2NoZW1hcyBpbXBvcnQgU09VUkNFX0NPTFVNTlMsIEdST1VORF9UUlVUSF9DT0xVTU5TCmZyb20gLmxhYmVscyBpbXBvcnQgcGFyc2VfbWF0Y2hfaWRzCgoKZGVmIGF1ZGl0X3NvdXJjZShwYXRoOiBzdHIgfCBQYXRoLCBzb3VyY2U6IGludCwgYmF0Y2hfc2l6ZTogaW50ID0gMTAwXzAwMCwgbWF4X3Jvd3M6IGludCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHByZWZpeCA9IGYiU3tzb3VyY2V9LSIKICAgIHJvd3MgPSAwCiAgICBibGFua3MgPSBDb3VudGVyKCkKICAgIGNvdW50cmllcyA9IENvdW50ZXIoKQogICAgaWRzOiBzZXRbc3RyXSA9IHNldCgpCiAgICBkdXBsaWNhdGVfaWRzID0gYmFkX2lkcyA9IDAKICAgIGZvciBjaHVuayBpbiBpdGVyX3RzdihwYXRoLCBTT1VSQ0VfQ09MVU1OUywgYmF0Y2hfc2l6ZSwgbWF4X3Jvd3MpOgogICAgICAgIHJvd3MgKz0gbGVuKGNodW5rKQogICAgICAgIGNvdW50cmllcy51cGRhdGUoY2h1bmtbImNvdW50cnkiXSkKICAgICAgICBmb3IgY29sdW1uIGluIFNPVVJDRV9DT0xVTU5TOgogICAgICAgICAgICBibGFua3NbY29sdW1uXSArPSBpbnQoY2h1bmtbY29sdW1uXS5lcSgiIikuc3VtKCkpCiAgICAgICAgZm9yIGVudGl0eV9pZCBpbiBjaHVua1siZW50aXR5X2lkIl06CiAgICAgICAgICAgIGJhZF9pZHMgKz0gaW50KG5vdCAoZW50aXR5X2lkLnN0YXJ0c3dpdGgocHJlZml4KSBhbmQgZW50aXR5X2lkW2xlbihwcmVmaXgpOl0uaXNkaWdpdCgpKSkKICAgICAgICAgICAgZHVwbGljYXRlX2lkcyArPSBpbnQoZW50aXR5X2lkIGluIGlkcykKICAgICAgICAgICAgaWRzLmFkZChlbnRpdHlfaWQpCiAgICByZXR1cm4gewogICAgICAgICJwYXRoIjogc3RyKHBhdGgpLCAicm93cyI6IHJvd3MsICJjb2x1bW5zIjogbGlzdChTT1VSQ0VfQ09MVU1OUyksCiAgICAgICAgImJsYW5rX2NvdW50cyI6IGRpY3QoYmxhbmtzKSwgImNvdW50cmllcyI6IGRpY3QoY291bnRyaWVzKSwKICAgICAgICAiZHVwbGljYXRlX2VudGl0eV9pZHMiOiBkdXBsaWNhdGVfaWRzLCAiYmFkX2VudGl0eV9pZHMiOiBiYWRfaWRzLAogICAgfQoKCmRlZiBhdWRpdF9ncm91bmRfdHJ1dGgocGF0aDogc3RyIHwgUGF0aCwgYmF0Y2hfc2l6ZTogaW50ID0gMTAwXzAwMCwgbWF4X3Jvd3M6IGludCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHJvd3MgPSBwb3NpdGl2ZXMgPSBzaW5nbGV0b25zID0gZHVwbGljYXRlX3MxID0gbWFsZm9ybWVkID0gMAogICAgaWRzOiBzZXRbc3RyXSA9IHNldCgpCiAgICBjYXJkaW5hbGl0eSA9IENvdW50ZXIoKQogICAgZm9yIGNodW5rIGluIGl0ZXJfdHN2KHBhdGgsIEdST1VORF9UUlVUSF9DT0xVTU5TLCBiYXRjaF9zaXplLCBtYXhfcm93cyk6CiAgICAgICAgZm9yIHJvdyBpbiBjaHVuay5pdGVydHVwbGVzKGluZGV4PUZhbHNlKToKICAgICAgICAgICAgcm93cyArPSAxCiAgICAgICAgICAgIGR1cGxpY2F0ZV9zMSArPSBpbnQocm93LnNvdXJjZTFfZW50aXR5X2lkIGluIGlkcykKICAgICAgICAgICAgaWRzLmFkZChyb3cuc291cmNlMV9lbnRpdHlfaWQpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG1hdGNoZXMgPSBwYXJzZV9tYXRjaF9pZHMocm93Lm1hdGNoZWRfZW50aXR5X2lkcykKICAgICAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgICAgICBtYWxmb3JtZWQgKz0gMQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgY2FyZGluYWxpdHlbbGVuKG1hdGNoZXMpXSArPSAxCiAgICAgICAgICAgIHBvc2l0aXZlcyArPSBsZW4obWF0Y2hlcykKICAgICAgICAgICAgc2luZ2xldG9ucyArPSBpbnQobm90IG1hdGNoZXMpCiAgICByZXR1cm4gewogICAgICAgICJwYXRoIjogc3RyKHBhdGgpLCAicm93cyI6IHJvd3MsICJjb2x1bW5zIjogbGlzdChHUk9VTkRfVFJVVEhfQ09MVU1OUyksCiAgICAgICAgInBvc2l0aXZlX2xpbmtzIjogcG9zaXRpdmVzLCAic2luZ2xldG9ucyI6IHNpbmdsZXRvbnMsCiAgICAgICAgInNpbmdsZXRvbl9yYXRlIjogc2luZ2xldG9ucyAvIHJvd3MgaWYgcm93cyBlbHNlIDAuMCwKICAgICAgICAiY2FyZGluYWxpdHkiOiBkaWN0KHNvcnRlZChjYXJkaW5hbGl0eS5pdGVtcygpKSksCiAgICAgICAgImR1cGxpY2F0ZV9zb3VyY2UxX2lkcyI6IGR1cGxpY2F0ZV9zMSwgIm1hbGZvcm1lZF9yb3dzIjogbWFsZm9ybWVkLAogICAgfQoKCmRlZiBhdWRpdF9kYXRhc2V0KGRhdGFfcm9vdDogc3RyIHwgUGF0aCwgYmF0Y2hfc2l6ZTogaW50ID0gMTAwXzAwMCwgbWF4X3Jvd3M6IGludCB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHJlc3VsdDogZGljdFtzdHIsIEFueV0gPSB7InNvdXJjZXMiOiB7fX0KICAgIGZvciBzcGxpdCBpbiAoInRyYWluIiwgInRlc3QiKToKICAgICAgICBmb3Igc291cmNlIGluICgxLCAyLCAzKToKICAgICAgICAgICAga2V5ID0gZiJ7c3BsaXR9X3NvdXJjZXtzb3VyY2V9IgogICAgICAgICAgICByZXN1bHRbInNvdXJjZXMiXVtrZXldID0gYXVkaXRfc291cmNlKHNvdXJjZV9wYXRoKGRhdGFfcm9vdCwgc3BsaXQsIHNvdXJjZSksIHNvdXJjZSwgYmF0Y2hfc2l6ZSwgbWF4X3Jvd3MpCiAgICByZXN1bHRbImdyb3VuZF90cnV0aCJdID0gYXVkaXRfZ3JvdW5kX3RydXRoKGdyb3VuZF90cnV0aF9wYXRoKGRhdGFfcm9vdCksIGJhdGNoX3NpemUsIG1heF9yb3dzKQogICAgcmV0dXJuIHJlc3VsdAoK', 'a24328e56d3bd8d2e9dc557d98402e63e3ebc8d70c2d21fbba28db8bdc694124')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/blocking/__init__.py`

~~~~python
"""Candidate generation strategies."""

from .base import CandidateReason, finalize_candidates
from .exact import ExactBlocker
from .tokens import RareTokenBlocker
from .tfidf import TfidfTopKBlocker
from .union import CandidateGenerator

__all__ = ["CandidateReason", "ExactBlocker", "RareTokenBlocker", "TfidfTopKBlocker", "CandidateGenerator", "finalize_candidates"]


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/blocking/__init__.py', 'IiIiQ2FuZGlkYXRlIGdlbmVyYXRpb24gc3RyYXRlZ2llcy4iIiIKCmZyb20gLmJhc2UgaW1wb3J0IENhbmRpZGF0ZVJlYXNvbiwgZmluYWxpemVfY2FuZGlkYXRlcwpmcm9tIC5leGFjdCBpbXBvcnQgRXhhY3RCbG9ja2VyCmZyb20gLnRva2VucyBpbXBvcnQgUmFyZVRva2VuQmxvY2tlcgpmcm9tIC50ZmlkZiBpbXBvcnQgVGZpZGZUb3BLQmxvY2tlcgpmcm9tIC51bmlvbiBpbXBvcnQgQ2FuZGlkYXRlR2VuZXJhdG9yCgpfX2FsbF9fID0gWyJDYW5kaWRhdGVSZWFzb24iLCAiRXhhY3RCbG9ja2VyIiwgIlJhcmVUb2tlbkJsb2NrZXIiLCAiVGZpZGZUb3BLQmxvY2tlciIsICJDYW5kaWRhdGVHZW5lcmF0b3IiLCAiZmluYWxpemVfY2FuZGlkYXRlcyJdCgo=', 'd36dbf3a5bac0ab841b0c148fc674118eaef4f8959f6baaf3d633404760ae988')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/blocking/base.py`

~~~~python
"""Shared blocking primitives."""

from __future__ import annotations

from enum import IntFlag
import pandas as pd


class CandidateReason(IntFlag):
    EXACT_NAME = 1
    EXACT_COMPACT = 2
    EXACT_CORE = 4
    RARE_NAME_TOKEN = 8
    RARE_ADDRESS_TOKEN = 16
    NUMERIC_TOKEN = 32
    TFIDF_NAME = 64
    TFIDF_ADDRESS = 128
    COUNTRY_FALLBACK = 256
    EXACT_ACCENT_FOLDED = 512


REASON_NAMES = {reason.value: reason.name.lower() for reason in CandidateReason}


def reason_text(mask: int) -> str:
    return "|".join(name for value, name in REASON_NAMES.items() if mask & value)


def empty_candidates() -> pd.DataFrame:
    return pd.DataFrame(columns=[
        "source1_entity_id", "candidate_entity_id", "candidate_source",
        "reason_bits", "reason_mask", "retrieval_score", "retrieval_rank",
    ])


def finalize_candidates(frames: list[pd.DataFrame], per_source_cap: int | None = None) -> pd.DataFrame:
    nonempty = [frame for frame in frames if frame is not None and not frame.empty]
    if not nonempty:
        return empty_candidates()
    combined = pd.concat(nonempty, ignore_index=True, sort=False)
    combined["reason_bits"] = combined["reason_bits"].astype(int)
    combined["retrieval_score"] = combined["retrieval_score"].astype(float)
    combined["retrieval_rank"] = combined["retrieval_rank"].astype(int)
    grouped = combined.groupby(["source1_entity_id", "candidate_entity_id", "candidate_source"], as_index=False).agg(
        reason_bits=("reason_bits", lambda values: int(pd.Series(values).astype(int).map(int).aggregate(lambda x: 0 if len(x) == 0 else __import__('functools').reduce(lambda a, b: a | b, x)))),
        retrieval_score=("retrieval_score", "max"),
        retrieval_rank=("retrieval_rank", "min"),
    )
    grouped["reason_mask"] = grouped["reason_bits"].map(reason_text)
    exact_mask = int(
        CandidateReason.EXACT_NAME
        | CandidateReason.EXACT_COMPACT
        | CandidateReason.EXACT_CORE
        | CandidateReason.EXACT_ACCENT_FOLDED
    )
    grouped["_exact_priority"] = grouped["reason_bits"].map(lambda value: int(bool(int(value) & exact_mask)))
    grouped["_reason_count"] = grouped["reason_bits"].map(lambda value: int(value).bit_count())
    grouped = grouped.sort_values(
        ["source1_entity_id", "candidate_source", "_exact_priority", "_reason_count", "retrieval_rank", "retrieval_score", "candidate_entity_id"],
        ascending=[True, True, False, False, True, False, True],
        kind="mergesort",
    )
    if per_source_cap is not None:
        grouped = grouped[grouped.groupby(["source1_entity_id", "candidate_source"]).cumcount() < per_source_cap]
    return grouped.drop(columns=["_exact_priority", "_reason_count"]).reset_index(drop=True)

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/blocking/base.py', 'IiIiU2hhcmVkIGJsb2NraW5nIHByaW1pdGl2ZXMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGVudW0gaW1wb3J0IEludEZsYWcKaW1wb3J0IHBhbmRhcyBhcyBwZAoKCmNsYXNzIENhbmRpZGF0ZVJlYXNvbihJbnRGbGFnKToKICAgIEVYQUNUX05BTUUgPSAxCiAgICBFWEFDVF9DT01QQUNUID0gMgogICAgRVhBQ1RfQ09SRSA9IDQKICAgIFJBUkVfTkFNRV9UT0tFTiA9IDgKICAgIFJBUkVfQUREUkVTU19UT0tFTiA9IDE2CiAgICBOVU1FUklDX1RPS0VOID0gMzIKICAgIFRGSURGX05BTUUgPSA2NAogICAgVEZJREZfQUREUkVTUyA9IDEyOAogICAgQ09VTlRSWV9GQUxMQkFDSyA9IDI1NgogICAgRVhBQ1RfQUNDRU5UX0ZPTERFRCA9IDUxMgoKClJFQVNPTl9OQU1FUyA9IHtyZWFzb24udmFsdWU6IHJlYXNvbi5uYW1lLmxvd2VyKCkgZm9yIHJlYXNvbiBpbiBDYW5kaWRhdGVSZWFzb259CgoKZGVmIHJlYXNvbl90ZXh0KG1hc2s6IGludCkgLT4gc3RyOgogICAgcmV0dXJuICJ8Ii5qb2luKG5hbWUgZm9yIHZhbHVlLCBuYW1lIGluIFJFQVNPTl9OQU1FUy5pdGVtcygpIGlmIG1hc2sgJiB2YWx1ZSkKCgpkZWYgZW1wdHlfY2FuZGlkYXRlcygpIC0+IHBkLkRhdGFGcmFtZToKICAgIHJldHVybiBwZC5EYXRhRnJhbWUoY29sdW1ucz1bCiAgICAgICAgInNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiLCAiY2FuZGlkYXRlX3NvdXJjZSIsCiAgICAgICAgInJlYXNvbl9iaXRzIiwgInJlYXNvbl9tYXNrIiwgInJldHJpZXZhbF9zY29yZSIsICJyZXRyaWV2YWxfcmFuayIsCiAgICBdKQoKCmRlZiBmaW5hbGl6ZV9jYW5kaWRhdGVzKGZyYW1lczogbGlzdFtwZC5EYXRhRnJhbWVdLCBwZXJfc291cmNlX2NhcDogaW50IHwgTm9uZSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgIG5vbmVtcHR5ID0gW2ZyYW1lIGZvciBmcmFtZSBpbiBmcmFtZXMgaWYgZnJhbWUgaXMgbm90IE5vbmUgYW5kIG5vdCBmcmFtZS5lbXB0eV0KICAgIGlmIG5vdCBub25lbXB0eToKICAgICAgICByZXR1cm4gZW1wdHlfY2FuZGlkYXRlcygpCiAgICBjb21iaW5lZCA9IHBkLmNvbmNhdChub25lbXB0eSwgaWdub3JlX2luZGV4PVRydWUsIHNvcnQ9RmFsc2UpCiAgICBjb21iaW5lZFsicmVhc29uX2JpdHMiXSA9IGNvbWJpbmVkWyJyZWFzb25fYml0cyJdLmFzdHlwZShpbnQpCiAgICBjb21iaW5lZFsicmV0cmlldmFsX3Njb3JlIl0gPSBjb21iaW5lZFsicmV0cmlldmFsX3Njb3JlIl0uYXN0eXBlKGZsb2F0KQogICAgY29tYmluZWRbInJldHJpZXZhbF9yYW5rIl0gPSBjb21iaW5lZFsicmV0cmlldmFsX3JhbmsiXS5hc3R5cGUoaW50KQogICAgZ3JvdXBlZCA9IGNvbWJpbmVkLmdyb3VwYnkoWyJzb3VyY2UxX2VudGl0eV9pZCIsICJjYW5kaWRhdGVfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9zb3VyY2UiXSwgYXNfaW5kZXg9RmFsc2UpLmFnZygKICAgICAgICByZWFzb25fYml0cz0oInJlYXNvbl9iaXRzIiwgbGFtYmRhIHZhbHVlczogaW50KHBkLlNlcmllcyh2YWx1ZXMpLmFzdHlwZShpbnQpLm1hcChpbnQpLmFnZ3JlZ2F0ZShsYW1iZGEgeDogMCBpZiBsZW4oeCkgPT0gMCBlbHNlIF9faW1wb3J0X18oJ2Z1bmN0b29scycpLnJlZHVjZShsYW1iZGEgYSwgYjogYSB8IGIsIHgpKSkpLAogICAgICAgIHJldHJpZXZhbF9zY29yZT0oInJldHJpZXZhbF9zY29yZSIsICJtYXgiKSwKICAgICAgICByZXRyaWV2YWxfcmFuaz0oInJldHJpZXZhbF9yYW5rIiwgIm1pbiIpLAogICAgKQogICAgZ3JvdXBlZFsicmVhc29uX21hc2siXSA9IGdyb3VwZWRbInJlYXNvbl9iaXRzIl0ubWFwKHJlYXNvbl90ZXh0KQogICAgZXhhY3RfbWFzayA9IGludCgKICAgICAgICBDYW5kaWRhdGVSZWFzb24uRVhBQ1RfTkFNRQogICAgICAgIHwgQ2FuZGlkYXRlUmVhc29uLkVYQUNUX0NPTVBBQ1QKICAgICAgICB8IENhbmRpZGF0ZVJlYXNvbi5FWEFDVF9DT1JFCiAgICAgICAgfCBDYW5kaWRhdGVSZWFzb24uRVhBQ1RfQUNDRU5UX0ZPTERFRAogICAgKQogICAgZ3JvdXBlZFsiX2V4YWN0X3ByaW9yaXR5Il0gPSBncm91cGVkWyJyZWFzb25fYml0cyJdLm1hcChsYW1iZGEgdmFsdWU6IGludChib29sKGludCh2YWx1ZSkgJiBleGFjdF9tYXNrKSkpCiAgICBncm91cGVkWyJfcmVhc29uX2NvdW50Il0gPSBncm91cGVkWyJyZWFzb25fYml0cyJdLm1hcChsYW1iZGEgdmFsdWU6IGludCh2YWx1ZSkuYml0X2NvdW50KCkpCiAgICBncm91cGVkID0gZ3JvdXBlZC5zb3J0X3ZhbHVlcygKICAgICAgICBbInNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9zb3VyY2UiLCAiX2V4YWN0X3ByaW9yaXR5IiwgIl9yZWFzb25fY291bnQiLCAicmV0cmlldmFsX3JhbmsiLCAicmV0cmlldmFsX3Njb3JlIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiXSwKICAgICAgICBhc2NlbmRpbmc9W1RydWUsIFRydWUsIEZhbHNlLCBGYWxzZSwgVHJ1ZSwgRmFsc2UsIFRydWVdLAogICAgICAgIGtpbmQ9Im1lcmdlc29ydCIsCiAgICApCiAgICBpZiBwZXJfc291cmNlX2NhcCBpcyBub3QgTm9uZToKICAgICAgICBncm91cGVkID0gZ3JvdXBlZFtncm91cGVkLmdyb3VwYnkoWyJzb3VyY2UxX2VudGl0eV9pZCIsICJjYW5kaWRhdGVfc291cmNlIl0pLmN1bWNvdW50KCkgPCBwZXJfc291cmNlX2NhcF0KICAgIHJldHVybiBncm91cGVkLmRyb3AoY29sdW1ucz1bIl9leGFjdF9wcmlvcml0eSIsICJfcmVhc29uX2NvdW50Il0pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkK', 'ade39457e4f7005cbe90f9a9949c355de192435ca79e3ef9bb17226470ac554b')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/blocking/exact.py`

~~~~python
"""Exact normalized-field blocking."""

from __future__ import annotations

from collections import defaultdict
import pandas as pd

from .base import CandidateReason, empty_candidates


def _tokens(value: object) -> tuple[str, ...]:
    if value is None:
        return ()
    if isinstance(value, str):
        return tuple(value.split()) if value else ()
    return tuple(value)


def _jaccard(left: object, right: object) -> float:
    a, b = set(_tokens(left)), set(_tokens(right))
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b) if a | b else 0.0


class ExactBlocker:
    def __init__(
        self,
        field: str,
        reason: CandidateReason,
        country_primary: bool = True,
        country_fallback: bool = True,
        posting_cap: int = 250,
    ) -> None:
        self.field = field
        self.reason = reason
        self.country_primary = country_primary
        self.country_fallback = country_fallback
        self.posting_cap = posting_cap
        self._country_index: dict[tuple[str, str], list[int]] = defaultdict(list)
        self._global_index: dict[str, list[int]] = defaultdict(list)
        self._targets: pd.DataFrame | None = None

    def fit(self, targets: pd.DataFrame) -> "ExactBlocker":
        required = {"entity_id", "country_norm", self.field}
        if not required.issubset(targets.columns):
            raise ValueError(f"exact blocker missing columns: {sorted(required - set(targets.columns))}")
        self._targets = targets.reset_index(drop=True)
        for index, row in self._targets.iterrows():
            value = str(row[self.field])
            if not value:
                continue
            self._country_index[(str(row["country_norm"]), value)].append(index)
            self._global_index[value].append(index)
        return self

    def transform(self, queries: pd.DataFrame) -> pd.DataFrame:
        if self._targets is None:
            raise RuntimeError("exact blocker must be fit before transform")
        rows: list[dict[str, object]] = []
        for query in queries.itertuples(index=False):
            value = str(getattr(query, self.field))
            country = str(getattr(query, "country_norm"))
            indices = self._country_index.get((country, value), []) if self.country_primary else self._global_index.get(value, [])
            fallback = False
            if not indices and self.country_fallback:
                indices = self._global_index.get(value, [])
                fallback = bool(indices)
            ranked = []
            for index in indices:
                target = self._targets.iloc[index]
                score = _jaccard(getattr(query, "address_tokens", ()), target.get("address_tokens", ()))
                ranked.append((score, str(target["entity_id"])))
            ranked.sort(key=lambda item: (-item[0], item[1]))
            for rank, (score, target_id) in enumerate(ranked[: self.posting_cap], start=1):
                bits = int(self.reason) | (int(CandidateReason.COUNTRY_FALLBACK) if fallback else 0)
                rows.append({
                    "source1_entity_id": str(getattr(query, "entity_id")),
                    "candidate_entity_id": target_id,
                    "candidate_source": target_id[:2],
                    "reason_bits": bits,
                    "retrieval_score": 1.0 + score * 0.01,
                    "retrieval_rank": rank,
                })
        return pd.DataFrame(rows) if rows else empty_candidates()

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/blocking/exact.py', 'IiIiRXhhY3Qgbm9ybWFsaXplZC1maWVsZCBibG9ja2luZy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gLmJhc2UgaW1wb3J0IENhbmRpZGF0ZVJlYXNvbiwgZW1wdHlfY2FuZGlkYXRlcwoKCmRlZiBfdG9rZW5zKHZhbHVlOiBvYmplY3QpIC0+IHR1cGxlW3N0ciwgLi4uXToKICAgIGlmIHZhbHVlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuICgpCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBzdHIpOgogICAgICAgIHJldHVybiB0dXBsZSh2YWx1ZS5zcGxpdCgpKSBpZiB2YWx1ZSBlbHNlICgpCiAgICByZXR1cm4gdHVwbGUodmFsdWUpCgoKZGVmIF9qYWNjYXJkKGxlZnQ6IG9iamVjdCwgcmlnaHQ6IG9iamVjdCkgLT4gZmxvYXQ6CiAgICBhLCBiID0gc2V0KF90b2tlbnMobGVmdCkpLCBzZXQoX3Rva2VucyhyaWdodCkpCiAgICBpZiBub3QgYSBhbmQgbm90IGI6CiAgICAgICAgcmV0dXJuIDEuMAogICAgcmV0dXJuIGxlbihhICYgYikgLyBsZW4oYSB8IGIpIGlmIGEgfCBiIGVsc2UgMC4wCgoKY2xhc3MgRXhhY3RCbG9ja2VyOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgZmllbGQ6IHN0ciwKICAgICAgICByZWFzb246IENhbmRpZGF0ZVJlYXNvbiwKICAgICAgICBjb3VudHJ5X3ByaW1hcnk6IGJvb2wgPSBUcnVlLAogICAgICAgIGNvdW50cnlfZmFsbGJhY2s6IGJvb2wgPSBUcnVlLAogICAgICAgIHBvc3RpbmdfY2FwOiBpbnQgPSAyNTAsCiAgICApIC0+IE5vbmU6CiAgICAgICAgc2VsZi5maWVsZCA9IGZpZWxkCiAgICAgICAgc2VsZi5yZWFzb24gPSByZWFzb24KICAgICAgICBzZWxmLmNvdW50cnlfcHJpbWFyeSA9IGNvdW50cnlfcHJpbWFyeQogICAgICAgIHNlbGYuY291bnRyeV9mYWxsYmFjayA9IGNvdW50cnlfZmFsbGJhY2sKICAgICAgICBzZWxmLnBvc3RpbmdfY2FwID0gcG9zdGluZ19jYXAKICAgICAgICBzZWxmLl9jb3VudHJ5X2luZGV4OiBkaWN0W3R1cGxlW3N0ciwgc3RyXSwgbGlzdFtpbnRdXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICAgICAgc2VsZi5fZ2xvYmFsX2luZGV4OiBkaWN0W3N0ciwgbGlzdFtpbnRdXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICAgICAgc2VsZi5fdGFyZ2V0czogcGQuRGF0YUZyYW1lIHwgTm9uZSA9IE5vbmUKCiAgICBkZWYgZml0KHNlbGYsIHRhcmdldHM6IHBkLkRhdGFGcmFtZSkgLT4gIkV4YWN0QmxvY2tlciI6CiAgICAgICAgcmVxdWlyZWQgPSB7ImVudGl0eV9pZCIsICJjb3VudHJ5X25vcm0iLCBzZWxmLmZpZWxkfQogICAgICAgIGlmIG5vdCByZXF1aXJlZC5pc3N1YnNldCh0YXJnZXRzLmNvbHVtbnMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXhhY3QgYmxvY2tlciBtaXNzaW5nIGNvbHVtbnM6IHtzb3J0ZWQocmVxdWlyZWQgLSBzZXQodGFyZ2V0cy5jb2x1bW5zKSl9IikKICAgICAgICBzZWxmLl90YXJnZXRzID0gdGFyZ2V0cy5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICAgICAgZm9yIGluZGV4LCByb3cgaW4gc2VsZi5fdGFyZ2V0cy5pdGVycm93cygpOgogICAgICAgICAgICB2YWx1ZSA9IHN0cihyb3dbc2VsZi5maWVsZF0pCiAgICAgICAgICAgIGlmIG5vdCB2YWx1ZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNlbGYuX2NvdW50cnlfaW5kZXhbKHN0cihyb3dbImNvdW50cnlfbm9ybSJdKSwgdmFsdWUpXS5hcHBlbmQoaW5kZXgpCiAgICAgICAgICAgIHNlbGYuX2dsb2JhbF9pbmRleFt2YWx1ZV0uYXBwZW5kKGluZGV4KQogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIHRyYW5zZm9ybShzZWxmLCBxdWVyaWVzOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICBpZiBzZWxmLl90YXJnZXRzIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiZXhhY3QgYmxvY2tlciBtdXN0IGJlIGZpdCBiZWZvcmUgdHJhbnNmb3JtIikKICAgICAgICByb3dzOiBsaXN0W2RpY3Rbc3RyLCBvYmplY3RdXSA9IFtdCiAgICAgICAgZm9yIHF1ZXJ5IGluIHF1ZXJpZXMuaXRlcnR1cGxlcyhpbmRleD1GYWxzZSk6CiAgICAgICAgICAgIHZhbHVlID0gc3RyKGdldGF0dHIocXVlcnksIHNlbGYuZmllbGQpKQogICAgICAgICAgICBjb3VudHJ5ID0gc3RyKGdldGF0dHIocXVlcnksICJjb3VudHJ5X25vcm0iKSkKICAgICAgICAgICAgaW5kaWNlcyA9IHNlbGYuX2NvdW50cnlfaW5kZXguZ2V0KChjb3VudHJ5LCB2YWx1ZSksIFtdKSBpZiBzZWxmLmNvdW50cnlfcHJpbWFyeSBlbHNlIHNlbGYuX2dsb2JhbF9pbmRleC5nZXQodmFsdWUsIFtdKQogICAgICAgICAgICBmYWxsYmFjayA9IEZhbHNlCiAgICAgICAgICAgIGlmIG5vdCBpbmRpY2VzIGFuZCBzZWxmLmNvdW50cnlfZmFsbGJhY2s6CiAgICAgICAgICAgICAgICBpbmRpY2VzID0gc2VsZi5fZ2xvYmFsX2luZGV4LmdldCh2YWx1ZSwgW10pCiAgICAgICAgICAgICAgICBmYWxsYmFjayA9IGJvb2woaW5kaWNlcykKICAgICAgICAgICAgcmFua2VkID0gW10KICAgICAgICAgICAgZm9yIGluZGV4IGluIGluZGljZXM6CiAgICAgICAgICAgICAgICB0YXJnZXQgPSBzZWxmLl90YXJnZXRzLmlsb2NbaW5kZXhdCiAgICAgICAgICAgICAgICBzY29yZSA9IF9qYWNjYXJkKGdldGF0dHIocXVlcnksICJhZGRyZXNzX3Rva2VucyIsICgpKSwgdGFyZ2V0LmdldCgiYWRkcmVzc190b2tlbnMiLCAoKSkpCiAgICAgICAgICAgICAgICByYW5rZWQuYXBwZW5kKChzY29yZSwgc3RyKHRhcmdldFsiZW50aXR5X2lkIl0pKSkKICAgICAgICAgICAgcmFua2VkLnNvcnQoa2V5PWxhbWJkYSBpdGVtOiAoLWl0ZW1bMF0sIGl0ZW1bMV0pKQogICAgICAgICAgICBmb3IgcmFuaywgKHNjb3JlLCB0YXJnZXRfaWQpIGluIGVudW1lcmF0ZShyYW5rZWRbOiBzZWxmLnBvc3RpbmdfY2FwXSwgc3RhcnQ9MSk6CiAgICAgICAgICAgICAgICBiaXRzID0gaW50KHNlbGYucmVhc29uKSB8IChpbnQoQ2FuZGlkYXRlUmVhc29uLkNPVU5UUllfRkFMTEJBQ0spIGlmIGZhbGxiYWNrIGVsc2UgMCkKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAic291cmNlMV9lbnRpdHlfaWQiOiBzdHIoZ2V0YXR0cihxdWVyeSwgImVudGl0eV9pZCIpKSwKICAgICAgICAgICAgICAgICAgICAiY2FuZGlkYXRlX2VudGl0eV9pZCI6IHRhcmdldF9pZCwKICAgICAgICAgICAgICAgICAgICAiY2FuZGlkYXRlX3NvdXJjZSI6IHRhcmdldF9pZFs6Ml0sCiAgICAgICAgICAgICAgICAgICAgInJlYXNvbl9iaXRzIjogYml0cywKICAgICAgICAgICAgICAgICAgICAicmV0cmlldmFsX3Njb3JlIjogMS4wICsgc2NvcmUgKiAwLjAxLAogICAgICAgICAgICAgICAgICAgICJyZXRyaWV2YWxfcmFuayI6IHJhbmssCiAgICAgICAgICAgICAgICB9KQogICAgICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykgaWYgcm93cyBlbHNlIGVtcHR5X2NhbmRpZGF0ZXMoKQo=', '351ae3bd3e8cbcdcf47ef854329e6a35c2381a25da478bbc08309fd82d4141b0')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/blocking/tfidf.py`

~~~~python
"""Batched sparse character-TF-IDF top-K retrieval."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.extmath import safe_sparse_dot

from .base import CandidateReason, empty_candidates


class TfidfTopKBlocker:
    def __init__(
        self,
        field: str = "name_canonical",
        top_k: int = 20,
        min_score: float = 0.15,
        batch_size: int = 1000,
        target_batch_size: int = 50_000,
        country_primary: bool = True,
        reason: CandidateReason = CandidateReason.TFIDF_NAME,
    ) -> None:
        self.field, self.top_k, self.min_score = field, top_k, min_score
        self.batch_size = batch_size
        self.target_batch_size = target_batch_size
        self.country_primary, self.reason = country_primary, reason
        self.vectorizers: dict[str, TfidfVectorizer] = {}
        self.target_matrices: dict[str, object] = {}
        self.target_ids: dict[str, np.ndarray] = {}

    def fit(self, targets: pd.DataFrame) -> "TfidfTopKBlocker":
        groups = targets.groupby("country_norm", sort=True) if self.country_primary else [("__all__", targets)]
        for country, group in groups:
            docs = group[self.field].astype(str).tolist()
            if not docs or not any(docs):
                continue
            vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), dtype=np.float32, min_df=1, norm="l2")
            matrix = vectorizer.fit_transform(docs).tocsr()
            key = str(country)
            self.vectorizers[key] = vectorizer
            self.target_matrices[key] = matrix
            self.target_ids[key] = group["entity_id"].astype(str).to_numpy()
        return self

    def transform(self, queries: pd.DataFrame) -> pd.DataFrame:
        rows: list[dict[str, object]] = []
        key_series = queries["country_norm"].astype(str) if self.country_primary else pd.Series("__all__", index=queries.index)
        for key, group in queries.groupby(key_series, sort=True):
            key = str(key)
            if key not in self.vectorizers:
                continue
            vectorizer, target_matrix, target_ids = self.vectorizers[key], self.target_matrices[key], self.target_ids[key]
            for start in range(0, len(group), self.batch_size):
                batch = group.iloc[start : start + self.batch_size]
                query_matrix = vectorizer.transform(batch[self.field].astype(str)).tocsr()
                best_indices = [np.empty(0, dtype=np.int64) for _ in range(len(batch))]
                best_scores = [np.empty(0, dtype=np.float32) for _ in range(len(batch))]
                # Multiply against bounded target blocks. Computing query x all
                # targets before top-K could create an enormous sparse matrix
                # for common character n-grams.
                for target_start in range(0, target_matrix.shape[0], self.target_batch_size):
                    target_stop = min(target_matrix.shape[0], target_start + self.target_batch_size)
                    similarities = safe_sparse_dot(
                        query_matrix,
                        target_matrix[target_start:target_stop].T,
                        dense_output=False,
                    ).tocsr()
                    for local_row in range(len(batch)):
                        begin, end = similarities.indptr[local_row], similarities.indptr[local_row + 1]
                        indices = similarities.indices[begin:end].astype(np.int64, copy=False) + target_start
                        scores = similarities.data[begin:end].astype(np.float32, copy=False)
                        valid = scores >= self.min_score
                        if not np.any(valid):
                            continue
                        indices = np.concatenate((best_indices[local_row], indices[valid]))
                        scores = np.concatenate((best_scores[local_row], scores[valid]))
                        if len(scores) > self.top_k:
                            chosen = np.argpartition(scores, -self.top_k)[-self.top_k:]
                            indices, scores = indices[chosen], scores[chosen]
                        best_indices[local_row], best_scores[local_row] = indices, scores
                for local_row, query_id in enumerate(batch["entity_id"].astype(str)):
                    indices, scores = best_indices[local_row], best_scores[local_row]
                    order = np.lexsort((target_ids[indices], -scores)) if len(scores) else []
                    for rank, pos in enumerate(order, start=1):
                        target_id = str(target_ids[indices[pos]])
                        rows.append({
                            "source1_entity_id": query_id,
                            "candidate_entity_id": target_id,
                            "candidate_source": target_id[:2],
                            "reason_bits": int(self.reason),
                            "retrieval_score": float(scores[pos]),
                            "retrieval_rank": rank,
                        })
        return pd.DataFrame(rows) if rows else empty_candidates()

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/blocking/tfidf.py', 'IiIiQmF0Y2hlZCBzcGFyc2UgY2hhcmFjdGVyLVRGLUlERiB0b3AtSyByZXRyaWV2YWwuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNrbGVhcm4uZmVhdHVyZV9leHRyYWN0aW9uLnRleHQgaW1wb3J0IFRmaWRmVmVjdG9yaXplcgpmcm9tIHNrbGVhcm4udXRpbHMuZXh0bWF0aCBpbXBvcnQgc2FmZV9zcGFyc2VfZG90Cgpmcm9tIC5iYXNlIGltcG9ydCBDYW5kaWRhdGVSZWFzb24sIGVtcHR5X2NhbmRpZGF0ZXMKCgpjbGFzcyBUZmlkZlRvcEtCbG9ja2VyOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgZmllbGQ6IHN0ciA9ICJuYW1lX2Nhbm9uaWNhbCIsCiAgICAgICAgdG9wX2s6IGludCA9IDIwLAogICAgICAgIG1pbl9zY29yZTogZmxvYXQgPSAwLjE1LAogICAgICAgIGJhdGNoX3NpemU6IGludCA9IDEwMDAsCiAgICAgICAgdGFyZ2V0X2JhdGNoX3NpemU6IGludCA9IDUwXzAwMCwKICAgICAgICBjb3VudHJ5X3ByaW1hcnk6IGJvb2wgPSBUcnVlLAogICAgICAgIHJlYXNvbjogQ2FuZGlkYXRlUmVhc29uID0gQ2FuZGlkYXRlUmVhc29uLlRGSURGX05BTUUsCiAgICApIC0+IE5vbmU6CiAgICAgICAgc2VsZi5maWVsZCwgc2VsZi50b3Bfaywgc2VsZi5taW5fc2NvcmUgPSBmaWVsZCwgdG9wX2ssIG1pbl9zY29yZQogICAgICAgIHNlbGYuYmF0Y2hfc2l6ZSA9IGJhdGNoX3NpemUKICAgICAgICBzZWxmLnRhcmdldF9iYXRjaF9zaXplID0gdGFyZ2V0X2JhdGNoX3NpemUKICAgICAgICBzZWxmLmNvdW50cnlfcHJpbWFyeSwgc2VsZi5yZWFzb24gPSBjb3VudHJ5X3ByaW1hcnksIHJlYXNvbgogICAgICAgIHNlbGYudmVjdG9yaXplcnM6IGRpY3Rbc3RyLCBUZmlkZlZlY3Rvcml6ZXJdID0ge30KICAgICAgICBzZWxmLnRhcmdldF9tYXRyaWNlczogZGljdFtzdHIsIG9iamVjdF0gPSB7fQogICAgICAgIHNlbGYudGFyZ2V0X2lkczogZGljdFtzdHIsIG5wLm5kYXJyYXldID0ge30KCiAgICBkZWYgZml0KHNlbGYsIHRhcmdldHM6IHBkLkRhdGFGcmFtZSkgLT4gIlRmaWRmVG9wS0Jsb2NrZXIiOgogICAgICAgIGdyb3VwcyA9IHRhcmdldHMuZ3JvdXBieSgiY291bnRyeV9ub3JtIiwgc29ydD1UcnVlKSBpZiBzZWxmLmNvdW50cnlfcHJpbWFyeSBlbHNlIFsoIl9fYWxsX18iLCB0YXJnZXRzKV0KICAgICAgICBmb3IgY291bnRyeSwgZ3JvdXAgaW4gZ3JvdXBzOgogICAgICAgICAgICBkb2NzID0gZ3JvdXBbc2VsZi5maWVsZF0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgICAgICAgICAgaWYgbm90IGRvY3Mgb3Igbm90IGFueShkb2NzKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHZlY3Rvcml6ZXIgPSBUZmlkZlZlY3Rvcml6ZXIoYW5hbHl6ZXI9ImNoYXJfd2IiLCBuZ3JhbV9yYW5nZT0oMywgNSksIGR0eXBlPW5wLmZsb2F0MzIsIG1pbl9kZj0xLCBub3JtPSJsMiIpCiAgICAgICAgICAgIG1hdHJpeCA9IHZlY3Rvcml6ZXIuZml0X3RyYW5zZm9ybShkb2NzKS50b2NzcigpCiAgICAgICAgICAgIGtleSA9IHN0cihjb3VudHJ5KQogICAgICAgICAgICBzZWxmLnZlY3Rvcml6ZXJzW2tleV0gPSB2ZWN0b3JpemVyCiAgICAgICAgICAgIHNlbGYudGFyZ2V0X21hdHJpY2VzW2tleV0gPSBtYXRyaXgKICAgICAgICAgICAgc2VsZi50YXJnZXRfaWRzW2tleV0gPSBncm91cFsiZW50aXR5X2lkIl0uYXN0eXBlKHN0cikudG9fbnVtcHkoKQogICAgICAgIHJldHVybiBzZWxmCgogICAgZGVmIHRyYW5zZm9ybShzZWxmLCBxdWVyaWVzOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICByb3dzOiBsaXN0W2RpY3Rbc3RyLCBvYmplY3RdXSA9IFtdCiAgICAgICAga2V5X3NlcmllcyA9IHF1ZXJpZXNbImNvdW50cnlfbm9ybSJdLmFzdHlwZShzdHIpIGlmIHNlbGYuY291bnRyeV9wcmltYXJ5IGVsc2UgcGQuU2VyaWVzKCJfX2FsbF9fIiwgaW5kZXg9cXVlcmllcy5pbmRleCkKICAgICAgICBmb3Iga2V5LCBncm91cCBpbiBxdWVyaWVzLmdyb3VwYnkoa2V5X3Nlcmllcywgc29ydD1UcnVlKToKICAgICAgICAgICAga2V5ID0gc3RyKGtleSkKICAgICAgICAgICAgaWYga2V5IG5vdCBpbiBzZWxmLnZlY3Rvcml6ZXJzOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdmVjdG9yaXplciwgdGFyZ2V0X21hdHJpeCwgdGFyZ2V0X2lkcyA9IHNlbGYudmVjdG9yaXplcnNba2V5XSwgc2VsZi50YXJnZXRfbWF0cmljZXNba2V5XSwgc2VsZi50YXJnZXRfaWRzW2tleV0KICAgICAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihncm91cCksIHNlbGYuYmF0Y2hfc2l6ZSk6CiAgICAgICAgICAgICAgICBiYXRjaCA9IGdyb3VwLmlsb2Nbc3RhcnQgOiBzdGFydCArIHNlbGYuYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgICAgIHF1ZXJ5X21hdHJpeCA9IHZlY3Rvcml6ZXIudHJhbnNmb3JtKGJhdGNoW3NlbGYuZmllbGRdLmFzdHlwZShzdHIpKS50b2NzcigpCiAgICAgICAgICAgICAgICBiZXN0X2luZGljZXMgPSBbbnAuZW1wdHkoMCwgZHR5cGU9bnAuaW50NjQpIGZvciBfIGluIHJhbmdlKGxlbihiYXRjaCkpXQogICAgICAgICAgICAgICAgYmVzdF9zY29yZXMgPSBbbnAuZW1wdHkoMCwgZHR5cGU9bnAuZmxvYXQzMikgZm9yIF8gaW4gcmFuZ2UobGVuKGJhdGNoKSldCiAgICAgICAgICAgICAgICAjIE11bHRpcGx5IGFnYWluc3QgYm91bmRlZCB0YXJnZXQgYmxvY2tzLiBDb21wdXRpbmcgcXVlcnkgeCBhbGwKICAgICAgICAgICAgICAgICMgdGFyZ2V0cyBiZWZvcmUgdG9wLUsgY291bGQgY3JlYXRlIGFuIGVub3Jtb3VzIHNwYXJzZSBtYXRyaXgKICAgICAgICAgICAgICAgICMgZm9yIGNvbW1vbiBjaGFyYWN0ZXIgbi1ncmFtcy4KICAgICAgICAgICAgICAgIGZvciB0YXJnZXRfc3RhcnQgaW4gcmFuZ2UoMCwgdGFyZ2V0X21hdHJpeC5zaGFwZVswXSwgc2VsZi50YXJnZXRfYmF0Y2hfc2l6ZSk6CiAgICAgICAgICAgICAgICAgICAgdGFyZ2V0X3N0b3AgPSBtaW4odGFyZ2V0X21hdHJpeC5zaGFwZVswXSwgdGFyZ2V0X3N0YXJ0ICsgc2VsZi50YXJnZXRfYmF0Y2hfc2l6ZSkKICAgICAgICAgICAgICAgICAgICBzaW1pbGFyaXRpZXMgPSBzYWZlX3NwYXJzZV9kb3QoCiAgICAgICAgICAgICAgICAgICAgICAgIHF1ZXJ5X21hdHJpeCwKICAgICAgICAgICAgICAgICAgICAgICAgdGFyZ2V0X21hdHJpeFt0YXJnZXRfc3RhcnQ6dGFyZ2V0X3N0b3BdLlQsCiAgICAgICAgICAgICAgICAgICAgICAgIGRlbnNlX291dHB1dD1GYWxzZSwKICAgICAgICAgICAgICAgICAgICApLnRvY3NyKCkKICAgICAgICAgICAgICAgICAgICBmb3IgbG9jYWxfcm93IGluIHJhbmdlKGxlbihiYXRjaCkpOgogICAgICAgICAgICAgICAgICAgICAgICBiZWdpbiwgZW5kID0gc2ltaWxhcml0aWVzLmluZHB0cltsb2NhbF9yb3ddLCBzaW1pbGFyaXRpZXMuaW5kcHRyW2xvY2FsX3JvdyArIDFdCiAgICAgICAgICAgICAgICAgICAgICAgIGluZGljZXMgPSBzaW1pbGFyaXRpZXMuaW5kaWNlc1tiZWdpbjplbmRdLmFzdHlwZShucC5pbnQ2NCwgY29weT1GYWxzZSkgKyB0YXJnZXRfc3RhcnQKICAgICAgICAgICAgICAgICAgICAgICAgc2NvcmVzID0gc2ltaWxhcml0aWVzLmRhdGFbYmVnaW46ZW5kXS5hc3R5cGUobnAuZmxvYXQzMiwgY29weT1GYWxzZSkKICAgICAgICAgICAgICAgICAgICAgICAgdmFsaWQgPSBzY29yZXMgPj0gc2VsZi5taW5fc2NvcmUKICAgICAgICAgICAgICAgICAgICAgICAgaWYgbm90IG5wLmFueSh2YWxpZCk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgICAgICBpbmRpY2VzID0gbnAuY29uY2F0ZW5hdGUoKGJlc3RfaW5kaWNlc1tsb2NhbF9yb3ddLCBpbmRpY2VzW3ZhbGlkXSkpCiAgICAgICAgICAgICAgICAgICAgICAgIHNjb3JlcyA9IG5wLmNvbmNhdGVuYXRlKChiZXN0X3Njb3Jlc1tsb2NhbF9yb3ddLCBzY29yZXNbdmFsaWRdKSkKICAgICAgICAgICAgICAgICAgICAgICAgaWYgbGVuKHNjb3JlcykgPiBzZWxmLnRvcF9rOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgY2hvc2VuID0gbnAuYXJncGFydGl0aW9uKHNjb3JlcywgLXNlbGYudG9wX2spWy1zZWxmLnRvcF9rOl0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIGluZGljZXMsIHNjb3JlcyA9IGluZGljZXNbY2hvc2VuXSwgc2NvcmVzW2Nob3Nlbl0KICAgICAgICAgICAgICAgICAgICAgICAgYmVzdF9pbmRpY2VzW2xvY2FsX3Jvd10sIGJlc3Rfc2NvcmVzW2xvY2FsX3Jvd10gPSBpbmRpY2VzLCBzY29yZXMKICAgICAgICAgICAgICAgIGZvciBsb2NhbF9yb3csIHF1ZXJ5X2lkIGluIGVudW1lcmF0ZShiYXRjaFsiZW50aXR5X2lkIl0uYXN0eXBlKHN0cikpOgogICAgICAgICAgICAgICAgICAgIGluZGljZXMsIHNjb3JlcyA9IGJlc3RfaW5kaWNlc1tsb2NhbF9yb3ddLCBiZXN0X3Njb3Jlc1tsb2NhbF9yb3ddCiAgICAgICAgICAgICAgICAgICAgb3JkZXIgPSBucC5sZXhzb3J0KCh0YXJnZXRfaWRzW2luZGljZXNdLCAtc2NvcmVzKSkgaWYgbGVuKHNjb3JlcykgZWxzZSBbXQogICAgICAgICAgICAgICAgICAgIGZvciByYW5rLCBwb3MgaW4gZW51bWVyYXRlKG9yZGVyLCBzdGFydD0xKToKICAgICAgICAgICAgICAgICAgICAgICAgdGFyZ2V0X2lkID0gc3RyKHRhcmdldF9pZHNbaW5kaWNlc1twb3NdXSkKICAgICAgICAgICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAgICAgICAgICAgICAgICAgInNvdXJjZTFfZW50aXR5X2lkIjogcXVlcnlfaWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiY2FuZGlkYXRlX2VudGl0eV9pZCI6IHRhcmdldF9pZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJjYW5kaWRhdGVfc291cmNlIjogdGFyZ2V0X2lkWzoyXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJyZWFzb25fYml0cyI6IGludChzZWxmLnJlYXNvbiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAicmV0cmlldmFsX3Njb3JlIjogZmxvYXQoc2NvcmVzW3Bvc10pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgInJldHJpZXZhbF9yYW5rIjogcmFuaywKICAgICAgICAgICAgICAgICAgICAgICAgfSkKICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpIGlmIHJvd3MgZWxzZSBlbXB0eV9jYW5kaWRhdGVzKCkK', 'b56e0c79910cee715455dfef47977e015aec8e364cb3078ec2b25b95a5cff1c7')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/blocking/tokens.py`

~~~~python
"""Rare-token and numeric-token inverted-index blocking."""

from __future__ import annotations

from collections import Counter, defaultdict
import math
import pandas as pd

from .base import CandidateReason, empty_candidates


def _tokens(value: object) -> tuple[str, ...]:
    if value is None:
        return ()
    if isinstance(value, str):
        return tuple(value.split()) if value else ()
    return tuple(value)


class RareTokenBlocker:
    def __init__(
        self,
        field: str,
        reason: CandidateReason,
        max_document_frequency: int = 500,
        posting_cap: int = 250,
        country_primary: bool = True,
        country_fallback: bool = True,
    ) -> None:
        self.field = field
        self.reason = reason
        self.max_df = max_document_frequency
        self.posting_cap = posting_cap
        self.country_primary = country_primary
        self.country_fallback = country_fallback
        self._postings: dict[tuple[str, str], list[str]] = defaultdict(list)
        self._global: dict[str, list[str]] = defaultdict(list)
        self._df: Counter[tuple[str, str]] = Counter()
        self._global_df: Counter[str] = Counter()

    def fit(self, targets: pd.DataFrame) -> "RareTokenBlocker":
        for row in targets.itertuples(index=False):
            country, target_id = str(getattr(row, "country_norm")), str(getattr(row, "entity_id"))
            for token in set(_tokens(getattr(row, self.field))):
                self._df[(country, token)] += 1
                self._global_df[token] += 1
                self._postings[(country, token)].append(target_id)
                self._global[token].append(target_id)
        return self

    def transform(self, queries: pd.DataFrame) -> pd.DataFrame:
        rows: list[dict[str, object]] = []
        for query in queries.itertuples(index=False):
            country = str(getattr(query, "country_norm"))
            scores: dict[str, float] = defaultdict(float)
            fallback_targets: set[str] = set()
            for token in set(_tokens(getattr(query, self.field))):
                df = self._df.get((country, token), 0)
                postings = self._postings.get((country, token), []) if self.country_primary else self._global.get(token, [])
                used_fallback = False
                if not postings and self.country_fallback:
                    postings, df, used_fallback = self._global.get(token, []), self._global_df.get(token, 0), True
                if not postings or df > self.max_df:
                    continue
                weight = 1.0 / math.log2(df + 2.0)
                # Score all postings that passed max_df, then cap the ranked
                # union. Slicing before aggregation made recall depend on raw
                # target row order whenever max_df exceeded posting_cap.
                for target_id in postings:
                    scores[target_id] += weight
                    if used_fallback:
                        fallback_targets.add(target_id)
            ranked = sorted(scores.items(), key=lambda item: (-item[1], item[0]))[: self.posting_cap]
            for rank, (target_id, score) in enumerate(ranked, start=1):
                bits = int(self.reason) | (
                    int(CandidateReason.COUNTRY_FALLBACK) if target_id in fallback_targets else 0
                )
                rows.append({
                    "source1_entity_id": str(getattr(query, "entity_id")),
                    "candidate_entity_id": target_id,
                    "candidate_source": target_id[:2],
                    "reason_bits": bits,
                    "retrieval_score": float(score),
                    "retrieval_rank": rank,
                })
        return pd.DataFrame(rows) if rows else empty_candidates()

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/blocking/tokens.py', 'IiIiUmFyZS10b2tlbiBhbmQgbnVtZXJpYy10b2tlbiBpbnZlcnRlZC1pbmRleCBibG9ja2luZy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIsIGRlZmF1bHRkaWN0CmltcG9ydCBtYXRoCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gLmJhc2UgaW1wb3J0IENhbmRpZGF0ZVJlYXNvbiwgZW1wdHlfY2FuZGlkYXRlcwoKCmRlZiBfdG9rZW5zKHZhbHVlOiBvYmplY3QpIC0+IHR1cGxlW3N0ciwgLi4uXToKICAgIGlmIHZhbHVlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuICgpCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBzdHIpOgogICAgICAgIHJldHVybiB0dXBsZSh2YWx1ZS5zcGxpdCgpKSBpZiB2YWx1ZSBlbHNlICgpCiAgICByZXR1cm4gdHVwbGUodmFsdWUpCgoKY2xhc3MgUmFyZVRva2VuQmxvY2tlcjoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGZpZWxkOiBzdHIsCiAgICAgICAgcmVhc29uOiBDYW5kaWRhdGVSZWFzb24sCiAgICAgICAgbWF4X2RvY3VtZW50X2ZyZXF1ZW5jeTogaW50ID0gNTAwLAogICAgICAgIHBvc3RpbmdfY2FwOiBpbnQgPSAyNTAsCiAgICAgICAgY291bnRyeV9wcmltYXJ5OiBib29sID0gVHJ1ZSwKICAgICAgICBjb3VudHJ5X2ZhbGxiYWNrOiBib29sID0gVHJ1ZSwKICAgICkgLT4gTm9uZToKICAgICAgICBzZWxmLmZpZWxkID0gZmllbGQKICAgICAgICBzZWxmLnJlYXNvbiA9IHJlYXNvbgogICAgICAgIHNlbGYubWF4X2RmID0gbWF4X2RvY3VtZW50X2ZyZXF1ZW5jeQogICAgICAgIHNlbGYucG9zdGluZ19jYXAgPSBwb3N0aW5nX2NhcAogICAgICAgIHNlbGYuY291bnRyeV9wcmltYXJ5ID0gY291bnRyeV9wcmltYXJ5CiAgICAgICAgc2VsZi5jb3VudHJ5X2ZhbGxiYWNrID0gY291bnRyeV9mYWxsYmFjawogICAgICAgIHNlbGYuX3Bvc3RpbmdzOiBkaWN0W3R1cGxlW3N0ciwgc3RyXSwgbGlzdFtzdHJdXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICAgICAgc2VsZi5fZ2xvYmFsOiBkaWN0W3N0ciwgbGlzdFtzdHJdXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICAgICAgc2VsZi5fZGY6IENvdW50ZXJbdHVwbGVbc3RyLCBzdHJdXSA9IENvdW50ZXIoKQogICAgICAgIHNlbGYuX2dsb2JhbF9kZjogQ291bnRlcltzdHJdID0gQ291bnRlcigpCgogICAgZGVmIGZpdChzZWxmLCB0YXJnZXRzOiBwZC5EYXRhRnJhbWUpIC0+ICJSYXJlVG9rZW5CbG9ja2VyIjoKICAgICAgICBmb3Igcm93IGluIHRhcmdldHMuaXRlcnR1cGxlcyhpbmRleD1GYWxzZSk6CiAgICAgICAgICAgIGNvdW50cnksIHRhcmdldF9pZCA9IHN0cihnZXRhdHRyKHJvdywgImNvdW50cnlfbm9ybSIpKSwgc3RyKGdldGF0dHIocm93LCAiZW50aXR5X2lkIikpCiAgICAgICAgICAgIGZvciB0b2tlbiBpbiBzZXQoX3Rva2VucyhnZXRhdHRyKHJvdywgc2VsZi5maWVsZCkpKToKICAgICAgICAgICAgICAgIHNlbGYuX2RmWyhjb3VudHJ5LCB0b2tlbildICs9IDEKICAgICAgICAgICAgICAgIHNlbGYuX2dsb2JhbF9kZlt0b2tlbl0gKz0gMQogICAgICAgICAgICAgICAgc2VsZi5fcG9zdGluZ3NbKGNvdW50cnksIHRva2VuKV0uYXBwZW5kKHRhcmdldF9pZCkKICAgICAgICAgICAgICAgIHNlbGYuX2dsb2JhbFt0b2tlbl0uYXBwZW5kKHRhcmdldF9pZCkKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiB0cmFuc2Zvcm0oc2VsZiwgcXVlcmllczogcGQuRGF0YUZyYW1lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgcm93czogbGlzdFtkaWN0W3N0ciwgb2JqZWN0XV0gPSBbXQogICAgICAgIGZvciBxdWVyeSBpbiBxdWVyaWVzLml0ZXJ0dXBsZXMoaW5kZXg9RmFsc2UpOgogICAgICAgICAgICBjb3VudHJ5ID0gc3RyKGdldGF0dHIocXVlcnksICJjb3VudHJ5X25vcm0iKSkKICAgICAgICAgICAgc2NvcmVzOiBkaWN0W3N0ciwgZmxvYXRdID0gZGVmYXVsdGRpY3QoZmxvYXQpCiAgICAgICAgICAgIGZhbGxiYWNrX3RhcmdldHM6IHNldFtzdHJdID0gc2V0KCkKICAgICAgICAgICAgZm9yIHRva2VuIGluIHNldChfdG9rZW5zKGdldGF0dHIocXVlcnksIHNlbGYuZmllbGQpKSk6CiAgICAgICAgICAgICAgICBkZiA9IHNlbGYuX2RmLmdldCgoY291bnRyeSwgdG9rZW4pLCAwKQogICAgICAgICAgICAgICAgcG9zdGluZ3MgPSBzZWxmLl9wb3N0aW5ncy5nZXQoKGNvdW50cnksIHRva2VuKSwgW10pIGlmIHNlbGYuY291bnRyeV9wcmltYXJ5IGVsc2Ugc2VsZi5fZ2xvYmFsLmdldCh0b2tlbiwgW10pCiAgICAgICAgICAgICAgICB1c2VkX2ZhbGxiYWNrID0gRmFsc2UKICAgICAgICAgICAgICAgIGlmIG5vdCBwb3N0aW5ncyBhbmQgc2VsZi5jb3VudHJ5X2ZhbGxiYWNrOgogICAgICAgICAgICAgICAgICAgIHBvc3RpbmdzLCBkZiwgdXNlZF9mYWxsYmFjayA9IHNlbGYuX2dsb2JhbC5nZXQodG9rZW4sIFtdKSwgc2VsZi5fZ2xvYmFsX2RmLmdldCh0b2tlbiwgMCksIFRydWUKICAgICAgICAgICAgICAgIGlmIG5vdCBwb3N0aW5ncyBvciBkZiA+IHNlbGYubWF4X2RmOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICB3ZWlnaHQgPSAxLjAgLyBtYXRoLmxvZzIoZGYgKyAyLjApCiAgICAgICAgICAgICAgICAjIFNjb3JlIGFsbCBwb3N0aW5ncyB0aGF0IHBhc3NlZCBtYXhfZGYsIHRoZW4gY2FwIHRoZSByYW5rZWQKICAgICAgICAgICAgICAgICMgdW5pb24uIFNsaWNpbmcgYmVmb3JlIGFnZ3JlZ2F0aW9uIG1hZGUgcmVjYWxsIGRlcGVuZCBvbiByYXcKICAgICAgICAgICAgICAgICMgdGFyZ2V0IHJvdyBvcmRlciB3aGVuZXZlciBtYXhfZGYgZXhjZWVkZWQgcG9zdGluZ19jYXAuCiAgICAgICAgICAgICAgICBmb3IgdGFyZ2V0X2lkIGluIHBvc3RpbmdzOgogICAgICAgICAgICAgICAgICAgIHNjb3Jlc1t0YXJnZXRfaWRdICs9IHdlaWdodAogICAgICAgICAgICAgICAgICAgIGlmIHVzZWRfZmFsbGJhY2s6CiAgICAgICAgICAgICAgICAgICAgICAgIGZhbGxiYWNrX3RhcmdldHMuYWRkKHRhcmdldF9pZCkKICAgICAgICAgICAgcmFua2VkID0gc29ydGVkKHNjb3Jlcy5pdGVtcygpLCBrZXk9bGFtYmRhIGl0ZW06ICgtaXRlbVsxXSwgaXRlbVswXSkpWzogc2VsZi5wb3N0aW5nX2NhcF0KICAgICAgICAgICAgZm9yIHJhbmssICh0YXJnZXRfaWQsIHNjb3JlKSBpbiBlbnVtZXJhdGUocmFua2VkLCBzdGFydD0xKToKICAgICAgICAgICAgICAgIGJpdHMgPSBpbnQoc2VsZi5yZWFzb24pIHwgKAogICAgICAgICAgICAgICAgICAgIGludChDYW5kaWRhdGVSZWFzb24uQ09VTlRSWV9GQUxMQkFDSykgaWYgdGFyZ2V0X2lkIGluIGZhbGxiYWNrX3RhcmdldHMgZWxzZSAwCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICAgICAgICAgInNvdXJjZTFfZW50aXR5X2lkIjogc3RyKGdldGF0dHIocXVlcnksICJlbnRpdHlfaWQiKSksCiAgICAgICAgICAgICAgICAgICAgImNhbmRpZGF0ZV9lbnRpdHlfaWQiOiB0YXJnZXRfaWQsCiAgICAgICAgICAgICAgICAgICAgImNhbmRpZGF0ZV9zb3VyY2UiOiB0YXJnZXRfaWRbOjJdLAogICAgICAgICAgICAgICAgICAgICJyZWFzb25fYml0cyI6IGJpdHMsCiAgICAgICAgICAgICAgICAgICAgInJldHJpZXZhbF9zY29yZSI6IGZsb2F0KHNjb3JlKSwKICAgICAgICAgICAgICAgICAgICAicmV0cmlldmFsX3JhbmsiOiByYW5rLAogICAgICAgICAgICAgICAgfSkKICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpIGlmIHJvd3MgZWxzZSBlbXB0eV9jYW5kaWRhdGVzKCkK', '9f908901fe768215457daad1a7fc675518a280cebea74bc3f45adb3d730d0eaa')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/blocking/union.py`

~~~~python
"""Configured union of complementary blockers."""

from __future__ import annotations

import pandas as pd

from .base import CandidateReason, finalize_candidates
from .exact import ExactBlocker
from .tokens import RareTokenBlocker
from .tfidf import TfidfTopKBlocker


class CandidateGenerator:
    def __init__(self, config: dict[str, object] | None = None, include_tfidf: bool = True) -> None:
        self.config = config or {}
        self.include_tfidf = include_tfidf
        self.blockers: list[object] = []

    def fit(self, targets: pd.DataFrame) -> "CandidateGenerator":
        country_primary = bool(self.config.get("country_primary", True))
        country_fallback = bool(self.config.get("country_fallback", True))
        posting_cap = int(self.config.get("token_posting_cap", 250))
        max_df = int(self.config.get("rare_token_max_df", 500))
        self.blockers = [
            ExactBlocker("name_canonical", CandidateReason.EXACT_NAME, country_primary, country_fallback, posting_cap),
            ExactBlocker("name_compact", CandidateReason.EXACT_COMPACT, country_primary, country_fallback, posting_cap),
            ExactBlocker("name_core", CandidateReason.EXACT_CORE, country_primary, country_fallback, posting_cap),
            ExactBlocker("name_accent_folded", CandidateReason.EXACT_ACCENT_FOLDED, country_primary, country_fallback, posting_cap),
            RareTokenBlocker("name_tokens", CandidateReason.RARE_NAME_TOKEN, max_df, posting_cap, country_primary, country_fallback),
            RareTokenBlocker("address_tokens", CandidateReason.RARE_ADDRESS_TOKEN, max_df, posting_cap, country_primary, country_fallback),
            RareTokenBlocker("numeric_tokens", CandidateReason.NUMERIC_TOKEN, max_df, posting_cap, country_primary, country_fallback),
        ]
        if self.include_tfidf:
            self.blockers.append(TfidfTopKBlocker(
                top_k=int(self.config.get("tfidf_top_k", 20)),
                min_score=float(self.config.get("tfidf_min_score", 0.15)),
                batch_size=int(self.config.get("batch_size", 1000)),
                target_batch_size=int(self.config.get("tfidf_target_batch_size", 50_000)),
                country_primary=country_primary,
            ))
        for blocker in self.blockers:
            blocker.fit(targets)
        return self

    def transform(self, source1: pd.DataFrame) -> pd.DataFrame:
        frames = [blocker.transform(source1) for blocker in self.blockers]
        return finalize_candidates(frames, per_source_cap=int(self.config.get("per_source_cap", 100)))

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/blocking/union.py', 'IiIiQ29uZmlndXJlZCB1bmlvbiBvZiBjb21wbGVtZW50YXJ5IGJsb2NrZXJzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuYmFzZSBpbXBvcnQgQ2FuZGlkYXRlUmVhc29uLCBmaW5hbGl6ZV9jYW5kaWRhdGVzCmZyb20gLmV4YWN0IGltcG9ydCBFeGFjdEJsb2NrZXIKZnJvbSAudG9rZW5zIGltcG9ydCBSYXJlVG9rZW5CbG9ja2VyCmZyb20gLnRmaWRmIGltcG9ydCBUZmlkZlRvcEtCbG9ja2VyCgoKY2xhc3MgQ2FuZGlkYXRlR2VuZXJhdG9yOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmZpZzogZGljdFtzdHIsIG9iamVjdF0gfCBOb25lID0gTm9uZSwgaW5jbHVkZV90ZmlkZjogYm9vbCA9IFRydWUpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcgb3Ige30KICAgICAgICBzZWxmLmluY2x1ZGVfdGZpZGYgPSBpbmNsdWRlX3RmaWRmCiAgICAgICAgc2VsZi5ibG9ja2VyczogbGlzdFtvYmplY3RdID0gW10KCiAgICBkZWYgZml0KHNlbGYsIHRhcmdldHM6IHBkLkRhdGFGcmFtZSkgLT4gIkNhbmRpZGF0ZUdlbmVyYXRvciI6CiAgICAgICAgY291bnRyeV9wcmltYXJ5ID0gYm9vbChzZWxmLmNvbmZpZy5nZXQoImNvdW50cnlfcHJpbWFyeSIsIFRydWUpKQogICAgICAgIGNvdW50cnlfZmFsbGJhY2sgPSBib29sKHNlbGYuY29uZmlnLmdldCgiY291bnRyeV9mYWxsYmFjayIsIFRydWUpKQogICAgICAgIHBvc3RpbmdfY2FwID0gaW50KHNlbGYuY29uZmlnLmdldCgidG9rZW5fcG9zdGluZ19jYXAiLCAyNTApKQogICAgICAgIG1heF9kZiA9IGludChzZWxmLmNvbmZpZy5nZXQoInJhcmVfdG9rZW5fbWF4X2RmIiwgNTAwKSkKICAgICAgICBzZWxmLmJsb2NrZXJzID0gWwogICAgICAgICAgICBFeGFjdEJsb2NrZXIoIm5hbWVfY2Fub25pY2FsIiwgQ2FuZGlkYXRlUmVhc29uLkVYQUNUX05BTUUsIGNvdW50cnlfcHJpbWFyeSwgY291bnRyeV9mYWxsYmFjaywgcG9zdGluZ19jYXApLAogICAgICAgICAgICBFeGFjdEJsb2NrZXIoIm5hbWVfY29tcGFjdCIsIENhbmRpZGF0ZVJlYXNvbi5FWEFDVF9DT01QQUNULCBjb3VudHJ5X3ByaW1hcnksIGNvdW50cnlfZmFsbGJhY2ssIHBvc3RpbmdfY2FwKSwKICAgICAgICAgICAgRXhhY3RCbG9ja2VyKCJuYW1lX2NvcmUiLCBDYW5kaWRhdGVSZWFzb24uRVhBQ1RfQ09SRSwgY291bnRyeV9wcmltYXJ5LCBjb3VudHJ5X2ZhbGxiYWNrLCBwb3N0aW5nX2NhcCksCiAgICAgICAgICAgIEV4YWN0QmxvY2tlcigibmFtZV9hY2NlbnRfZm9sZGVkIiwgQ2FuZGlkYXRlUmVhc29uLkVYQUNUX0FDQ0VOVF9GT0xERUQsIGNvdW50cnlfcHJpbWFyeSwgY291bnRyeV9mYWxsYmFjaywgcG9zdGluZ19jYXApLAogICAgICAgICAgICBSYXJlVG9rZW5CbG9ja2VyKCJuYW1lX3Rva2VucyIsIENhbmRpZGF0ZVJlYXNvbi5SQVJFX05BTUVfVE9LRU4sIG1heF9kZiwgcG9zdGluZ19jYXAsIGNvdW50cnlfcHJpbWFyeSwgY291bnRyeV9mYWxsYmFjayksCiAgICAgICAgICAgIFJhcmVUb2tlbkJsb2NrZXIoImFkZHJlc3NfdG9rZW5zIiwgQ2FuZGlkYXRlUmVhc29uLlJBUkVfQUREUkVTU19UT0tFTiwgbWF4X2RmLCBwb3N0aW5nX2NhcCwgY291bnRyeV9wcmltYXJ5LCBjb3VudHJ5X2ZhbGxiYWNrKSwKICAgICAgICAgICAgUmFyZVRva2VuQmxvY2tlcigibnVtZXJpY190b2tlbnMiLCBDYW5kaWRhdGVSZWFzb24uTlVNRVJJQ19UT0tFTiwgbWF4X2RmLCBwb3N0aW5nX2NhcCwgY291bnRyeV9wcmltYXJ5LCBjb3VudHJ5X2ZhbGxiYWNrKSwKICAgICAgICBdCiAgICAgICAgaWYgc2VsZi5pbmNsdWRlX3RmaWRmOgogICAgICAgICAgICBzZWxmLmJsb2NrZXJzLmFwcGVuZChUZmlkZlRvcEtCbG9ja2VyKAogICAgICAgICAgICAgICAgdG9wX2s9aW50KHNlbGYuY29uZmlnLmdldCgidGZpZGZfdG9wX2siLCAyMCkpLAogICAgICAgICAgICAgICAgbWluX3Njb3JlPWZsb2F0KHNlbGYuY29uZmlnLmdldCgidGZpZGZfbWluX3Njb3JlIiwgMC4xNSkpLAogICAgICAgICAgICAgICAgYmF0Y2hfc2l6ZT1pbnQoc2VsZi5jb25maWcuZ2V0KCJiYXRjaF9zaXplIiwgMTAwMCkpLAogICAgICAgICAgICAgICAgdGFyZ2V0X2JhdGNoX3NpemU9aW50KHNlbGYuY29uZmlnLmdldCgidGZpZGZfdGFyZ2V0X2JhdGNoX3NpemUiLCA1MF8wMDApKSwKICAgICAgICAgICAgICAgIGNvdW50cnlfcHJpbWFyeT1jb3VudHJ5X3ByaW1hcnksCiAgICAgICAgICAgICkpCiAgICAgICAgZm9yIGJsb2NrZXIgaW4gc2VsZi5ibG9ja2VyczoKICAgICAgICAgICAgYmxvY2tlci5maXQodGFyZ2V0cykKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiB0cmFuc2Zvcm0oc2VsZiwgc291cmNlMTogcGQuRGF0YUZyYW1lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgZnJhbWVzID0gW2Jsb2NrZXIudHJhbnNmb3JtKHNvdXJjZTEpIGZvciBibG9ja2VyIGluIHNlbGYuYmxvY2tlcnNdCiAgICAgICAgcmV0dXJuIGZpbmFsaXplX2NhbmRpZGF0ZXMoZnJhbWVzLCBwZXJfc291cmNlX2NhcD1pbnQoc2VsZi5jb25maWcuZ2V0KCJwZXJfc291cmNlX2NhcCIsIDEwMCkpKQo=', '5c5371a5e900440e33c8d8a225af965c5d31c1613d0fe88670dd1c2893a59e16')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/checkpoint.py`

~~~~python
"""Crash-safe, idempotent shard checkpointing for long pipeline stages.

A stage processes deterministic shards (by S1-ID range or input part index).
Each shard is written atomically to ``parts/part-XXXXX.parquet`` together with a
``parts/part-XXXXX.manifest.json`` carrying the config fingerprint, an input
fingerprint, the shard key, and the row count. Re-running a stage skips every
shard whose manifest matches the current fingerprint(s), so a crash or kill only
costs the in-flight shard. A top-level ``manifest.json`` with ``complete=true``
marks the stage finished.
"""

from __future__ import annotations

import json
import hashlib
import os
from pathlib import Path
from typing import Any, Iterable, Iterator

import pandas as pd

from .artifacts import config_fingerprint, read_frame, write_frame


def shard_for_id(entity_id: str, n_shards: int) -> int:
    """Deterministically map an entity ID to a shard in ``[0, n_shards)``.

    Uses the numeric suffix when present (stable, order-independent) and falls
    back to a hash of the raw string otherwise.
    """
    if n_shards <= 1:
        return 0
    digits = entity_id.split("-", 1)[-1]
    if digits.isdigit():
        return int(digits) % n_shards
    digest = hashlib.blake2b(str(entity_id).encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "big") % n_shards


def shard_ids(ids: Iterable[str], n_shards: int) -> dict[int, list[str]]:
    buckets: dict[int, list[str]] = {index: [] for index in range(n_shards)}
    for entity_id in ids:
        buckets[shard_for_id(str(entity_id), n_shards)].append(str(entity_id))
    return buckets


def input_fingerprint(paths: Iterable[str | Path]) -> str:
    """Fingerprint a set of input files by path, size, and mtime."""
    entries = []
    for path in sorted(Path(p) for p in paths):
        try:
            stat = path.stat()
            entries.append((str(path), int(stat.st_size), int(stat.st_mtime)))
        except OSError:
            entries.append((str(path), -1, -1))
    return config_fingerprint({"inputs": entries})


def stage_fingerprint(stage_dir: str | Path) -> str:
    """Return the fingerprint recorded in a stage manifest, or ``""`` if absent.

    Readers use this so they can attach to whatever fingerprint the writer
    actually used (which may depend on options like embeddings being enabled),
    without having to recompute every variant.
    """
    path = Path(stage_dir) / "manifest.json"
    if not path.is_file():
        return ""
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return ""
    return str(data.get("fingerprint", ""))


class ShardStore:
    """Manage resumable, fingerprinted shards for one stage directory."""

    def __init__(self, stage_dir: str | Path, fingerprint: str) -> None:
        self.stage_dir = Path(stage_dir)
        self.parts_dir = self.stage_dir / "parts"
        self.fingerprint = fingerprint
        self.parts_dir.mkdir(parents=True, exist_ok=True)

    # -- paths -----------------------------------------------------------------
    def part_path(self, key: int) -> Path:
        return self.parts_dir / f"part-{int(key):05d}.parquet"

    def manifest_path(self, key: int) -> Path:
        return self.parts_dir / f"part-{int(key):05d}.manifest.json"

    def stage_manifest_path(self) -> Path:
        return self.stage_dir / "manifest.json"

    # -- introspection ---------------------------------------------------------
    def completed_keys(self) -> set[int]:
        done: set[int] = set()
        for manifest_path in self.parts_dir.glob("part-*.manifest.json"):
            try:
                data = json.loads(manifest_path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                continue
            if data.get("complete") and data.get("fingerprint") == self.fingerprint and data.get("key") is not None:
                if self.part_path(int(data["key"])).is_file():
                    done.add(int(data["key"]))
        return done

    def pending_keys(self, all_keys: Iterable[int]) -> list[int]:
        done = self.completed_keys()
        return sorted(int(key) for key in all_keys if int(key) not in done)

    def is_complete(self) -> bool:
        path = self.stage_manifest_path()
        if not path.is_file():
            return False
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            return False
        return bool(data.get("complete")) and data.get("fingerprint") == self.fingerprint

    # -- mutation --------------------------------------------------------------
    def commit(self, key: int, frame: pd.DataFrame, *, rows: int | None = None, extra: dict[str, Any] | None = None) -> Path:
        part = write_frame(frame, self.part_path(key))
        manifest = {
            "key": int(key),
            "fingerprint": self.fingerprint,
            "rows": int(len(frame) if rows is None else rows),
            "schema": list(frame.columns),
            "complete": True,
        }
        if extra:
            manifest.update(extra)
        temporary = self.manifest_path(key).with_suffix(".json.tmp")
        temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True, default=str), encoding="utf-8")
        os.replace(temporary, self.manifest_path(key))
        return part

    def mark_complete(self, rows: int, schema: list[str], extra: dict[str, Any] | None = None) -> Path:
        manifest: dict[str, Any] = {
            "fingerprint": self.fingerprint,
            "rows": int(rows),
            "schema": list(schema),
            "parts": len(list(self.parts_dir.glob("part-*.parquet"))),
            "complete": True,
        }
        if extra:
            manifest.update(extra)
        path = self.stage_manifest_path()
        temporary = path.with_suffix(".json.tmp")
        temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True, default=str), encoding="utf-8")
        os.replace(temporary, path)
        return path

    def read_all(self, columns: list[str] | None = None) -> pd.DataFrame:
        paths = sorted(self.parts_dir.glob("part-*.parquet"))
        if not paths:
            return pd.DataFrame(columns=columns) if columns else pd.DataFrame()
        return pd.concat([read_frame(path, columns=columns) for path in paths], ignore_index=True)

    def iter_parts(self) -> Iterator[Path]:
        yield from sorted(self.parts_dir.glob("part-*.parquet"))

    def total_rows(self) -> int:
        total = 0
        for manifest_path in self.parts_dir.glob("part-*.manifest.json"):
            try:
                data = json.loads(manifest_path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                continue
            if data.get("fingerprint") == self.fingerprint:
                total += int(data.get("rows", 0))
        return total

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/checkpoint.py', 'IiIiQ3Jhc2gtc2FmZSwgaWRlbXBvdGVudCBzaGFyZCBjaGVja3BvaW50aW5nIGZvciBsb25nIHBpcGVsaW5lIHN0YWdlcy4KCkEgc3RhZ2UgcHJvY2Vzc2VzIGRldGVybWluaXN0aWMgc2hhcmRzIChieSBTMS1JRCByYW5nZSBvciBpbnB1dCBwYXJ0IGluZGV4KS4KRWFjaCBzaGFyZCBpcyB3cml0dGVuIGF0b21pY2FsbHkgdG8gYGBwYXJ0cy9wYXJ0LVhYWFhYLnBhcnF1ZXRgYCB0b2dldGhlciB3aXRoIGEKYGBwYXJ0cy9wYXJ0LVhYWFhYLm1hbmlmZXN0Lmpzb25gYCBjYXJyeWluZyB0aGUgY29uZmlnIGZpbmdlcnByaW50LCBhbiBpbnB1dApmaW5nZXJwcmludCwgdGhlIHNoYXJkIGtleSwgYW5kIHRoZSByb3cgY291bnQuIFJlLXJ1bm5pbmcgYSBzdGFnZSBza2lwcyBldmVyeQpzaGFyZCB3aG9zZSBtYW5pZmVzdCBtYXRjaGVzIHRoZSBjdXJyZW50IGZpbmdlcnByaW50KHMpLCBzbyBhIGNyYXNoIG9yIGtpbGwgb25seQpjb3N0cyB0aGUgaW4tZmxpZ2h0IHNoYXJkLiBBIHRvcC1sZXZlbCBgYG1hbmlmZXN0Lmpzb25gYCB3aXRoIGBgY29tcGxldGU9dHJ1ZWBgCm1hcmtzIHRoZSBzdGFnZSBmaW5pc2hlZC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgaGFzaGxpYgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIEl0ZXJhYmxlLCBJdGVyYXRvcgoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuYXJ0aWZhY3RzIGltcG9ydCBjb25maWdfZmluZ2VycHJpbnQsIHJlYWRfZnJhbWUsIHdyaXRlX2ZyYW1lCgoKZGVmIHNoYXJkX2Zvcl9pZChlbnRpdHlfaWQ6IHN0ciwgbl9zaGFyZHM6IGludCkgLT4gaW50OgogICAgIiIiRGV0ZXJtaW5pc3RpY2FsbHkgbWFwIGFuIGVudGl0eSBJRCB0byBhIHNoYXJkIGluIGBgWzAsIG5fc2hhcmRzKWBgLgoKICAgIFVzZXMgdGhlIG51bWVyaWMgc3VmZml4IHdoZW4gcHJlc2VudCAoc3RhYmxlLCBvcmRlci1pbmRlcGVuZGVudCkgYW5kIGZhbGxzCiAgICBiYWNrIHRvIGEgaGFzaCBvZiB0aGUgcmF3IHN0cmluZyBvdGhlcndpc2UuCiAgICAiIiIKICAgIGlmIG5fc2hhcmRzIDw9IDE6CiAgICAgICAgcmV0dXJuIDAKICAgIGRpZ2l0cyA9IGVudGl0eV9pZC5zcGxpdCgiLSIsIDEpWy0xXQogICAgaWYgZGlnaXRzLmlzZGlnaXQoKToKICAgICAgICByZXR1cm4gaW50KGRpZ2l0cykgJSBuX3NoYXJkcwogICAgZGlnZXN0ID0gaGFzaGxpYi5ibGFrZTJiKHN0cihlbnRpdHlfaWQpLmVuY29kZSgidXRmLTgiKSwgZGlnZXN0X3NpemU9OCkuZGlnZXN0KCkKICAgIHJldHVybiBpbnQuZnJvbV9ieXRlcyhkaWdlc3QsICJiaWciKSAlIG5fc2hhcmRzCgoKZGVmIHNoYXJkX2lkcyhpZHM6IEl0ZXJhYmxlW3N0cl0sIG5fc2hhcmRzOiBpbnQpIC0+IGRpY3RbaW50LCBsaXN0W3N0cl1dOgogICAgYnVja2V0czogZGljdFtpbnQsIGxpc3Rbc3RyXV0gPSB7aW5kZXg6IFtdIGZvciBpbmRleCBpbiByYW5nZShuX3NoYXJkcyl9CiAgICBmb3IgZW50aXR5X2lkIGluIGlkczoKICAgICAgICBidWNrZXRzW3NoYXJkX2Zvcl9pZChzdHIoZW50aXR5X2lkKSwgbl9zaGFyZHMpXS5hcHBlbmQoc3RyKGVudGl0eV9pZCkpCiAgICByZXR1cm4gYnVja2V0cwoKCmRlZiBpbnB1dF9maW5nZXJwcmludChwYXRoczogSXRlcmFibGVbc3RyIHwgUGF0aF0pIC0+IHN0cjoKICAgICIiIkZpbmdlcnByaW50IGEgc2V0IG9mIGlucHV0IGZpbGVzIGJ5IHBhdGgsIHNpemUsIGFuZCBtdGltZS4iIiIKICAgIGVudHJpZXMgPSBbXQogICAgZm9yIHBhdGggaW4gc29ydGVkKFBhdGgocCkgZm9yIHAgaW4gcGF0aHMpOgogICAgICAgIHRyeToKICAgICAgICAgICAgc3RhdCA9IHBhdGguc3RhdCgpCiAgICAgICAgICAgIGVudHJpZXMuYXBwZW5kKChzdHIocGF0aCksIGludChzdGF0LnN0X3NpemUpLCBpbnQoc3RhdC5zdF9tdGltZSkpKQogICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICBlbnRyaWVzLmFwcGVuZCgoc3RyKHBhdGgpLCAtMSwgLTEpKQogICAgcmV0dXJuIGNvbmZpZ19maW5nZXJwcmludCh7ImlucHV0cyI6IGVudHJpZXN9KQoKCmRlZiBzdGFnZV9maW5nZXJwcmludChzdGFnZV9kaXI6IHN0ciB8IFBhdGgpIC0+IHN0cjoKICAgICIiIlJldHVybiB0aGUgZmluZ2VycHJpbnQgcmVjb3JkZWQgaW4gYSBzdGFnZSBtYW5pZmVzdCwgb3IgYGAiImBgIGlmIGFic2VudC4KCiAgICBSZWFkZXJzIHVzZSB0aGlzIHNvIHRoZXkgY2FuIGF0dGFjaCB0byB3aGF0ZXZlciBmaW5nZXJwcmludCB0aGUgd3JpdGVyCiAgICBhY3R1YWxseSB1c2VkICh3aGljaCBtYXkgZGVwZW5kIG9uIG9wdGlvbnMgbGlrZSBlbWJlZGRpbmdzIGJlaW5nIGVuYWJsZWQpLAogICAgd2l0aG91dCBoYXZpbmcgdG8gcmVjb21wdXRlIGV2ZXJ5IHZhcmlhbnQuCiAgICAiIiIKICAgIHBhdGggPSBQYXRoKHN0YWdlX2RpcikgLyAibWFuaWZlc3QuanNvbiIKICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICByZXR1cm4gIiIKICAgIHRyeToKICAgICAgICBkYXRhID0ganNvbi5sb2FkcyhwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIGV4Y2VwdCAoT1NFcnJvciwganNvbi5KU09ORGVjb2RlRXJyb3IpOgogICAgICAgIHJldHVybiAiIgogICAgcmV0dXJuIHN0cihkYXRhLmdldCgiZmluZ2VycHJpbnQiLCAiIikpCgoKY2xhc3MgU2hhcmRTdG9yZToKICAgICIiIk1hbmFnZSByZXN1bWFibGUsIGZpbmdlcnByaW50ZWQgc2hhcmRzIGZvciBvbmUgc3RhZ2UgZGlyZWN0b3J5LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzdGFnZV9kaXI6IHN0ciB8IFBhdGgsIGZpbmdlcnByaW50OiBzdHIpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5zdGFnZV9kaXIgPSBQYXRoKHN0YWdlX2RpcikKICAgICAgICBzZWxmLnBhcnRzX2RpciA9IHNlbGYuc3RhZ2VfZGlyIC8gInBhcnRzIgogICAgICAgIHNlbGYuZmluZ2VycHJpbnQgPSBmaW5nZXJwcmludAogICAgICAgIHNlbGYucGFydHNfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICAjIC0tIHBhdGhzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBkZWYgcGFydF9wYXRoKHNlbGYsIGtleTogaW50KSAtPiBQYXRoOgogICAgICAgIHJldHVybiBzZWxmLnBhcnRzX2RpciAvIGYicGFydC17aW50KGtleSk6MDVkfS5wYXJxdWV0IgoKICAgIGRlZiBtYW5pZmVzdF9wYXRoKHNlbGYsIGtleTogaW50KSAtPiBQYXRoOgogICAgICAgIHJldHVybiBzZWxmLnBhcnRzX2RpciAvIGYicGFydC17aW50KGtleSk6MDVkfS5tYW5pZmVzdC5qc29uIgoKICAgIGRlZiBzdGFnZV9tYW5pZmVzdF9wYXRoKHNlbGYpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYuc3RhZ2VfZGlyIC8gIm1hbmlmZXN0Lmpzb24iCgogICAgIyAtLSBpbnRyb3NwZWN0aW9uIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIGNvbXBsZXRlZF9rZXlzKHNlbGYpIC0+IHNldFtpbnRdOgogICAgICAgIGRvbmU6IHNldFtpbnRdID0gc2V0KCkKICAgICAgICBmb3IgbWFuaWZlc3RfcGF0aCBpbiBzZWxmLnBhcnRzX2Rpci5nbG9iKCJwYXJ0LSoubWFuaWZlc3QuanNvbiIpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBkYXRhID0ganNvbi5sb2FkcyhtYW5pZmVzdF9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICAgICAgZXhjZXB0IChPU0Vycm9yLCBqc29uLkpTT05EZWNvZGVFcnJvcik6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBkYXRhLmdldCgiY29tcGxldGUiKSBhbmQgZGF0YS5nZXQoImZpbmdlcnByaW50IikgPT0gc2VsZi5maW5nZXJwcmludCBhbmQgZGF0YS5nZXQoImtleSIpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgaWYgc2VsZi5wYXJ0X3BhdGgoaW50KGRhdGFbImtleSJdKSkuaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgICAgIGRvbmUuYWRkKGludChkYXRhWyJrZXkiXSkpCiAgICAgICAgcmV0dXJuIGRvbmUKCiAgICBkZWYgcGVuZGluZ19rZXlzKHNlbGYsIGFsbF9rZXlzOiBJdGVyYWJsZVtpbnRdKSAtPiBsaXN0W2ludF06CiAgICAgICAgZG9uZSA9IHNlbGYuY29tcGxldGVkX2tleXMoKQogICAgICAgIHJldHVybiBzb3J0ZWQoaW50KGtleSkgZm9yIGtleSBpbiBhbGxfa2V5cyBpZiBpbnQoa2V5KSBub3QgaW4gZG9uZSkKCiAgICBkZWYgaXNfY29tcGxldGUoc2VsZikgLT4gYm9vbDoKICAgICAgICBwYXRoID0gc2VsZi5zdGFnZV9tYW5pZmVzdF9wYXRoKCkKICAgICAgICBpZiBub3QgcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHRyeToKICAgICAgICAgICAgZGF0YSA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgZXhjZXB0IChPU0Vycm9yLCBqc29uLkpTT05EZWNvZGVFcnJvcik6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHJldHVybiBib29sKGRhdGEuZ2V0KCJjb21wbGV0ZSIpKSBhbmQgZGF0YS5nZXQoImZpbmdlcnByaW50IikgPT0gc2VsZi5maW5nZXJwcmludAoKICAgICMgLS0gbXV0YXRpb24gLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGRlZiBjb21taXQoc2VsZiwga2V5OiBpbnQsIGZyYW1lOiBwZC5EYXRhRnJhbWUsICosIHJvd3M6IGludCB8IE5vbmUgPSBOb25lLCBleHRyYTogZGljdFtzdHIsIEFueV0gfCBOb25lID0gTm9uZSkgLT4gUGF0aDoKICAgICAgICBwYXJ0ID0gd3JpdGVfZnJhbWUoZnJhbWUsIHNlbGYucGFydF9wYXRoKGtleSkpCiAgICAgICAgbWFuaWZlc3QgPSB7CiAgICAgICAgICAgICJrZXkiOiBpbnQoa2V5KSwKICAgICAgICAgICAgImZpbmdlcnByaW50Ijogc2VsZi5maW5nZXJwcmludCwKICAgICAgICAgICAgInJvd3MiOiBpbnQobGVuKGZyYW1lKSBpZiByb3dzIGlzIE5vbmUgZWxzZSByb3dzKSwKICAgICAgICAgICAgInNjaGVtYSI6IGxpc3QoZnJhbWUuY29sdW1ucyksCiAgICAgICAgICAgICJjb21wbGV0ZSI6IFRydWUsCiAgICAgICAgfQogICAgICAgIGlmIGV4dHJhOgogICAgICAgICAgICBtYW5pZmVzdC51cGRhdGUoZXh0cmEpCiAgICAgICAgdGVtcG9yYXJ5ID0gc2VsZi5tYW5pZmVzdF9wYXRoKGtleSkud2l0aF9zdWZmaXgoIi5qc29uLnRtcCIpCiAgICAgICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyhtYW5pZmVzdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgb3MucmVwbGFjZSh0ZW1wb3JhcnksIHNlbGYubWFuaWZlc3RfcGF0aChrZXkpKQogICAgICAgIHJldHVybiBwYXJ0CgogICAgZGVmIG1hcmtfY29tcGxldGUoc2VsZiwgcm93czogaW50LCBzY2hlbWE6IGxpc3Rbc3RyXSwgZXh0cmE6IGRpY3Rbc3RyLCBBbnldIHwgTm9uZSA9IE5vbmUpIC0+IFBhdGg6CiAgICAgICAgbWFuaWZlc3Q6IGRpY3Rbc3RyLCBBbnldID0gewogICAgICAgICAgICAiZmluZ2VycHJpbnQiOiBzZWxmLmZpbmdlcnByaW50LAogICAgICAgICAgICAicm93cyI6IGludChyb3dzKSwKICAgICAgICAgICAgInNjaGVtYSI6IGxpc3Qoc2NoZW1hKSwKICAgICAgICAgICAgInBhcnRzIjogbGVuKGxpc3Qoc2VsZi5wYXJ0c19kaXIuZ2xvYigicGFydC0qLnBhcnF1ZXQiKSkpLAogICAgICAgICAgICAiY29tcGxldGUiOiBUcnVlLAogICAgICAgIH0KICAgICAgICBpZiBleHRyYToKICAgICAgICAgICAgbWFuaWZlc3QudXBkYXRlKGV4dHJhKQogICAgICAgIHBhdGggPSBzZWxmLnN0YWdlX21hbmlmZXN0X3BhdGgoKQogICAgICAgIHRlbXBvcmFyeSA9IHBhdGgud2l0aF9zdWZmaXgoIi5qc29uLnRtcCIpCiAgICAgICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyhtYW5pZmVzdCwgaW5kZW50PTIsIHNvcnRfa2V5cz1UcnVlLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgb3MucmVwbGFjZSh0ZW1wb3JhcnksIHBhdGgpCiAgICAgICAgcmV0dXJuIHBhdGgKCiAgICBkZWYgcmVhZF9hbGwoc2VsZiwgY29sdW1uczogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICBwYXRocyA9IHNvcnRlZChzZWxmLnBhcnRzX2Rpci5nbG9iKCJwYXJ0LSoucGFycXVldCIpKQogICAgICAgIGlmIG5vdCBwYXRoczoKICAgICAgICAgICAgcmV0dXJuIHBkLkRhdGFGcmFtZShjb2x1bW5zPWNvbHVtbnMpIGlmIGNvbHVtbnMgZWxzZSBwZC5EYXRhRnJhbWUoKQogICAgICAgIHJldHVybiBwZC5jb25jYXQoW3JlYWRfZnJhbWUocGF0aCwgY29sdW1ucz1jb2x1bW5zKSBmb3IgcGF0aCBpbiBwYXRoc10sIGlnbm9yZV9pbmRleD1UcnVlKQoKICAgIGRlZiBpdGVyX3BhcnRzKHNlbGYpIC0+IEl0ZXJhdG9yW1BhdGhdOgogICAgICAgIHlpZWxkIGZyb20gc29ydGVkKHNlbGYucGFydHNfZGlyLmdsb2IoInBhcnQtKi5wYXJxdWV0IikpCgogICAgZGVmIHRvdGFsX3Jvd3Moc2VsZikgLT4gaW50OgogICAgICAgIHRvdGFsID0gMAogICAgICAgIGZvciBtYW5pZmVzdF9wYXRoIGluIHNlbGYucGFydHNfZGlyLmdsb2IoInBhcnQtKi5tYW5pZmVzdC5qc29uIik6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGRhdGEgPSBqc29uLmxvYWRzKG1hbmlmZXN0X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgICAgICBleGNlcHQgKE9TRXJyb3IsIGpzb24uSlNPTkRlY29kZUVycm9yKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIGRhdGEuZ2V0KCJmaW5nZXJwcmludCIpID09IHNlbGYuZmluZ2VycHJpbnQ6CiAgICAgICAgICAgICAgICB0b3RhbCArPSBpbnQoZGF0YS5nZXQoInJvd3MiLCAwKSkKICAgICAgICByZXR1cm4gdG90YWwK', 'e9b916c832b56a7a35eedbfb3d5fb12a7b11d8f9da48bbd5e94f874b537b7a10')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/cli.py`

~~~~python
"""Command-line interface for local smoke work and remote full-scale stages."""

from __future__ import annotations

import argparse
import gc
import json
import os
from pathlib import Path
import shutil
import sys
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from .artifacts import read_frame, write_frame, write_manifest
from .audit import audit_dataset
from .blocking import CandidateGenerator
from .checkpoint import ShardStore, input_fingerprint, shard_for_id, stage_fingerprint
from .config import ProjectConfig
from .data import iter_tsv, load_ground_truth, load_ground_truth_for_ids, source_path
from .decisions import apply_thresholds, score_quantile_thresholds, sweep_thresholds
from .embeddings import embed_split, load_embedding_vectors
from .error_analysis import analyze_errors
from .features import FEATURE_COLUMNS, build_pair_features, feature_matrix
from .labels import ground_truth_sets, label_candidates
from .logging import stage_logger
from .metrics import candidate_metrics_from_frames, evaluate_entity_sets, sets_from_long
from .mini import build_mini
from .models import DeterministicScorer, SGDPairModel
from .models.lightgbm_model import LightGBMPairModel
from .normalize import normalize_records
from .pipeline import run_smoke
from .preflight import preflight_outputs, run_official_validator
from .resources import run_with_oom_backoff
from .sampling import sample_candidate_negatives
from .schemas import SOURCE_COLUMNS
from .splits import assign_s1_folds, select_pair_fold
from .submission import write_submission_outputs, write_submission_outputs_sharded


def _json_print(value: object) -> None:
    print(json.dumps(value, indent=2, sort_keys=True, default=str))


def _force(args: argparse.Namespace) -> bool:
    return bool(getattr(args, "force", False))


def _load_config(args: argparse.Namespace) -> ProjectConfig:
    overrides = {"max_rows": getattr(args, "max_rows", None)}
    return ProjectConfig.load(args.config, overrides)


def _model_kind(config: ProjectConfig, args: argparse.Namespace) -> str:
    """Use an explicit CLI model when supplied, otherwise honor the config."""
    return str(getattr(args, "model", None) or config.model.get("kind", "sgd"))


def _model_params(config: ProjectConfig, kind: str) -> dict[str, object]:
    """Return only parameters intended for the selected model backend."""
    by_kind = config.model.get("params_by_kind", {})
    if isinstance(by_kind, dict) and kind in by_kind:
        return dict(by_kind[kind])
    if str(config.model.get("kind", "sgd")) == kind:
        return dict(config.model.get("params", {}))
    return {}


def _validation_entity_ids(config: ProjectConfig, fold: int) -> list[str]:
    folds = read_frame(config.artifact_dir("splits") / "s1_folds.parquet")
    selected = folds.loc[folds["fold"].eq(int(fold)), "source1_entity_id"].astype(str)
    if selected.empty:
        raise ValueError(f"validation fold {fold} contains no Source 1 entities")
    return selected.tolist()


def _score_path(config: ProjectConfig, value: str) -> Path:
    path = Path(value)
    return path if path.is_absolute() else config.artifact_dir("scores") / path


def _decision_threshold(config: ProjectConfig, explicit: float | None) -> float:
    if explicit is not None:
        return float(explicit)
    selected = config.artifact_dir("decisions") / "selected_threshold.json"
    if selected.is_file():
        payload = json.loads(selected.read_text(encoding="utf-8"))
        return float(payload["threshold"])
    return float(config.model.get("threshold", 0.8))


def _assert_candidate_feature_alignment(candidates: pd.DataFrame, features: pd.DataFrame) -> None:
    """Ensure candidate_pairs is exactly the set passed to the matcher."""
    columns = ["source1_entity_id", "candidate_entity_id"]
    for label, frame in (("candidates", candidates), ("features", features)):
        if frame.duplicated(columns).any():
            raise ValueError(f"{label} contain duplicate pair keys")
    candidate_keys = pd.MultiIndex.from_frame(candidates[columns])
    feature_keys = pd.MultiIndex.from_frame(features[columns])
    missing_features = candidate_keys.difference(feature_keys)
    extra_features = feature_keys.difference(candidate_keys)
    if len(missing_features) or len(extra_features):
        raise ValueError(
            "candidate/feature pair mismatch: "
            f"missing_features={len(missing_features)}, extra_features={len(extra_features)}; "
            "rebuild features for the current candidate artifact"
        )


def _split_input_fingerprint(config: ProjectConfig, split: str) -> str:
    return input_fingerprint([source_path(config.data_root, split, source) for source in (1, 2, 3)])


def _candidate_fingerprint(config: ProjectConfig, split: str, include_tfidf: bool = True) -> str:
    return (
        _split_input_fingerprint(config, split)
        + ":blocking=v3:"
        + json.dumps(config.blocking, sort_keys=True)
        + f":tfidf={include_tfidf}"
    )


def _feature_fingerprint(
    config: ProjectConfig,
    split: str,
    embeddings_used: bool = False,
    include_tfidf: bool = True,
) -> str:
    return (
        _candidate_fingerprint(config, split, include_tfidf=include_tfidf)
        + f":features=v3:embeddings={embeddings_used}"
    )


def _prepared_dir(config: ProjectConfig, split: str, source: int) -> Path:
    return config.artifact_dir("prepared") / f"{split}_source{source}"


def _read_prepared(
    config: ProjectConfig,
    split: str,
    source: int,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    """Read all prepared shards for a split/source (stored under parts/)."""
    directory = _prepared_dir(config, split, source)
    store = _prepared_shard_store(config, split, source)
    frame = store.read_all(columns=columns)
    if frame.empty and not store.is_complete():
        raise FileNotFoundError(f"prepared data not found: {directory}; run prepare first")
    return frame


def _prepared_shard_store(config: ProjectConfig, split: str, source: int) -> ShardStore:
    directory = _prepared_dir(config, split, source)
    fingerprint = (
        input_fingerprint([source_path(config.data_root, split, source)])
        + f":prepare=v2:max_rows={config.max_rows}:shards={config.n_shards}"
    )
    return ShardStore(directory, fingerprint)


def _read_prepared_part(
    config: ProjectConfig,
    split: str,
    source: int,
    key: int,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    directory = _prepared_dir(config, split, source)
    path = directory / "parts" / f"part-{int(key):05d}.parquet"
    return read_frame(path, columns=columns)

def _training_truth(config: ProjectConfig, source1_ids=None) -> pd.DataFrame:
    if source1_ids is not None:
        return load_ground_truth_for_ids(config.data_root, source1_ids, config.batch_size)
    return load_ground_truth(config.data_root)


def command_audit(args: argparse.Namespace) -> int:
    config = _load_config(args)
    started = time.time()
    report = audit_dataset(config.data_root, config.batch_size, config.max_rows)
    out = config.artifact_dir("audit") / "audit.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")
    write_manifest(out.with_name("manifest.json"), stage="audit", config=config.as_dict(), rows=sum(v["rows"] for v in report["sources"].values()), schema=[], metrics=report, started_at=started)
    _json_print({"audit": str(out), "bounded": config.max_rows is not None})
    return 0


def command_prepare(args: argparse.Namespace) -> int:
    """Normalize raw TSVs into sharded parquet, resumable at chunk granularity.

    Each raw ``iter_tsv`` chunk is normalized and written as an interim part
    under ``parts/`` (bounded to one chunk of memory). On completion the interim
    parts are consolidated into S1-ID-range shards. If the process dies, already
    committed chunks are skipped on the next run.
    """
    config = _load_config(args)
    plan = config.resource_plan()
    splits = (args.split,) if args.split in {"train", "test"} else ("train", "test")
    force = _force(args)
    for split in splits:
        for source in (1, 2, 3):
            path = source_path(config.data_root, split, source)
            store = _prepared_shard_store(config, split, source)
            logger = stage_logger(config, f"prepare-{split}-source{source}")
            logger.stage_start("prepare", split=split, source=source, shards=config.n_shards)
            logger.resource_plan(plan)
            chunk_fingerprint = store.parts_dir / "input.fingerprint"
            recorded_fingerprint = chunk_fingerprint.read_text(encoding="utf-8") if chunk_fingerprint.is_file() else ""
            if force or recorded_fingerprint != store.fingerprint:
                # Prepared chunks are only reusable for the exact same raw
                # input fingerprint. Keeping old chunks after max_rows/input
                # changes previously mixed stale records into a new run.
                shutil.rmtree(store.parts_dir, ignore_errors=True)
                store.parts_dir.mkdir(parents=True, exist_ok=True)
                store.stage_manifest_path().unlink(missing_ok=True)
                chunk_fingerprint.write_text(store.fingerprint, encoding="utf-8")
            if store.is_complete():
                logger.info("stage already complete; skipping", rows=store.total_rows())
                logger.stage_end("prepare", split=split, source=source, skipped=True)
                continue

            # Interim, resumable chunk files (independent of final shard layout).
            chunk_dir = store.parts_dir / "chunks"
            chunk_dir.mkdir(parents=True, exist_ok=True)
            total_rows = 0
            written_chunks = 0
            for sequence, chunk in enumerate(iter_tsv(path, SOURCE_COLUMNS, config.batch_size, config.max_rows), start=1):
                chunk_path = chunk_dir / f"chunk-{sequence:07d}.parquet"
                if not chunk_path.is_file():
                    normalized = normalize_records(chunk)
                    write_frame(normalized, chunk_path)
                total_rows += len(chunk)
                written_chunks = sequence
                logger.progress("prepare", min(total_rows, config.max_rows or total_rows), config.max_rows or total_rows)
            logger.info("raw chunks written", chunks=written_chunks, rows_seen=total_rows)

            # Partition interim chunks into small on-disk fragments first, then
            # build each final shard once. The old buffer flush overwrote an
            # earlier shard when that shard appeared in a later raw chunk.
            fragment_dir = store.parts_dir / "fragments"
            shutil.rmtree(fragment_dir, ignore_errors=True)
            fragment_dir.mkdir(parents=True, exist_ok=True)
            for stale in store.parts_dir.glob("part-*.parquet"):
                stale.unlink()
            for stale in store.parts_dir.glob("part-*.manifest.json"):
                stale.unlink()
            for chunk_path in sorted(chunk_dir.glob("chunk-*.parquet")):
                frame = read_frame(chunk_path)
                keys = frame["entity_id"].map(lambda value: shard_for_id(value, config.n_shards))
                for key, group in frame.groupby(keys, sort=False):
                    key_dir = fragment_dir / f"shard-{int(key):05d}"
                    key_dir.mkdir(parents=True, exist_ok=True)
                    write_frame(group, key_dir / chunk_path.name)

            empty = normalize_records(pd.DataFrame(columns=SOURCE_COLUMNS)).iloc[0:0]
            for key in range(config.n_shards):
                fragments = sorted((fragment_dir / f"shard-{key:05d}").glob("chunk-*.parquet"))
                shard = pd.concat([read_frame(item) for item in fragments], ignore_index=True) if fragments else empty
                store.commit(key, shard)
                logger.progress("consolidate", key + 1, config.n_shards)
            shutil.rmtree(fragment_dir, ignore_errors=True)
            rows = store.total_rows()
            schema = list(empty.columns)
            store.mark_complete(rows, schema, extra={"split": split, "source": source})
            logger.info("prepared", split=split, source=source, rows=rows)
            logger.stage_end("prepare", split=split, source=source, rows=rows)
    return 0


def command_make_splits(args: argparse.Namespace) -> int:
    config = _load_config(args)
    s1 = _read_prepared(config, "train", 1)
    gt = _training_truth(config, s1["entity_id"])
    folds = assign_s1_folds(s1, gt, n_folds=args.folds, seed=config.seed)
    out = write_frame(folds, config.artifact_dir("splits") / "s1_folds.parquet")
    write_manifest(out.with_name("manifest.json"), stage="make-splits", config=config.as_dict(), rows=len(folds), schema=list(folds.columns))
    print(out)
    return 0


def command_generate_candidates(args: argparse.Namespace) -> int:
    """Generate the exact last-stage candidate set, sharded and resumable.

    Blockers are fit once per split against all Source 2/3 targets, then Source 1
    is transformed shard-by-shard so memory stays bounded. Each shard is
    committed atomically, so a crash resumes without refitting prior shards.
    """
    config = _load_config(args)
    plan = config.resource_plan()
    logger = stage_logger(config, f"candidates-{args.split}")
    fingerprint = _candidate_fingerprint(config, args.split, include_tfidf=not args.no_tfidf)
    store = ShardStore(config.artifact_dir("candidates") / args.split, fingerprint)
    schema = ["source1_entity_id", "candidate_entity_id", "candidate_source", "reason_bits", "reason_mask", "retrieval_score", "retrieval_rank"]
    if _force(args):
        shutil.rmtree(store.stage_dir, ignore_errors=True)
        store = ShardStore(config.artifact_dir("candidates") / args.split, fingerprint)
        shutil.rmtree(config.artifact_dir("candidate-components") / args.split, ignore_errors=True)
        store.stage_manifest_path().unlink(missing_ok=True)
    logger.stage_start("candidates", split=args.split, shards=config.n_shards)
    logger.resource_plan(plan)

    if not store.is_complete():
        blocker_columns = [
            "entity_id", "country_norm", "name_canonical", "name_compact", "name_core",
            "name_accent_folded", "name_tokens", "address_tokens", "numeric_tokens",
        ]
        component_root = config.artifact_dir("candidate-components") / args.split
        component_stores: list[ShardStore] = []
        for source in (2, 3):
            component_fingerprint = fingerprint + f":target_source={source}"
            component_store = ShardStore(component_root / f"source{source}", component_fingerprint)
            component_stores.append(component_store)
            if component_store.is_complete():
                continue
            target = _read_prepared(config, args.split, source, columns=blocker_columns)
            generator = CandidateGenerator(config.blocking, include_tfidf=not args.no_tfidf).fit(target)
            for key in component_store.pending_keys(range(config.n_shards)):
                shard_s1 = _read_prepared_part(config, args.split, 1, key, columns=blocker_columns)
                candidates = generator.transform(shard_s1)
                component_store.commit(key, candidates)
                logger.progress(
                    f"candidates-source{source}",
                    key + 1,
                    config.n_shards,
                    extra={"shard": key, "rows": len(candidates)},
                )
            component_store.mark_complete(
                component_store.total_rows(), schema, extra={"split": args.split, "source": source}
            )
            del generator, target
            gc.collect()

        for key in store.pending_keys(range(config.n_shards)):
            frames = [read_frame(component.part_path(key)) for component in component_stores]
            candidates = pd.concat(frames, ignore_index=True)
            if not candidates.empty:
                candidates = candidates.sort_values(
                    ["source1_entity_id", "candidate_source", "retrieval_score", "candidate_entity_id"],
                    ascending=[True, True, False, True],
                    kind="mergesort",
                ).reset_index(drop=True)
            store.commit(key, candidates)
        rows = store.total_rows()
        store.mark_complete(rows, schema, extra={"split": args.split})
        shutil.rmtree(component_root, ignore_errors=True)
    else:
        logger.info("stage already complete; skipping", rows=store.total_rows())

    metrics: dict[str, object] = {}
    if args.split == "train":
        source1_ids = _read_prepared(config, args.split, 1, columns=["entity_id"])["entity_id"]
        truth = ground_truth_sets(_training_truth(config, source1_ids))
        universe_size = sum(
            _prepared_shard_store(config, "train", source).total_rows() for source in (2, 3)
        )
        metrics = candidate_metrics_from_frames(
            truth,
            (read_frame(path) for path in store.iter_parts()),
            universe_size,
        )
        for name, value in metrics.items():
            logger.metric(f"candidate_{name}", value)
    # Keep the ShardStore manifest intact; overwriting it with a generic stage
    # manifest removes the fingerprint required for safe resume checks.
    store.mark_complete(store.total_rows(), schema, extra={"split": args.split, "metrics": metrics})
    logger.stage_end("candidates", split=args.split, rows=store.total_rows())
    if metrics:
        _json_print(metrics)
    print(store.stage_manifest_path())
    return 0


def command_make_mini(args: argparse.Namespace) -> int:
    """Sample a realistic mini dataset from the real files (schema-identical)."""
    config = _load_config(args)
    source_root = Path(args.source_root) if args.source_root else config.data_root
    mini_root = Path(args.mini_root)
    if not mini_root.is_absolute():
        mini_root = (Path.cwd() / mini_root).resolve()
    summary = build_mini(source_root, mini_root, count=args.count, seed=config.seed)
    _json_print({"mini_root": str(mini_root), **summary})
    return 0


def command_embed(args: argparse.Namespace) -> int:
    """Compute and cache BGE-M3 name/address vectors (resumable, device-aware)."""
    config = _load_config(args)
    splits = (args.split,) if args.split in {"train", "test"} else ("train", "test")
    for split in splits:
        logger = stage_logger(config, f"embed-{split}")
        logger.stage_start("embed", split=split)
        summary = embed_split(config, split, logger)
        for name, value in summary.items():
            logger.metric(f"embed_{name}", value)
        logger.stage_end("embed", split=split, **{k: str(v) for k, v in summary.items()})
    return 0


def command_build_features(args: argparse.Namespace) -> int:
    """Build pair features shard-by-shard, resumable and OOM-adaptive.

    Reads the committed candidate shards, joins each against prepared S1/target
    rows, and commits a feature parquet per candidate shard. Optional BGE-M3
    cosine features are added when embeddings are enabled.
    """
    config = _load_config(args)
    plan = config.resource_plan()
    logger = stage_logger(config, f"features-{args.split}")
    candidate_store = ShardStore(
        config.artifact_dir("candidates") / args.split,
        _candidate_fingerprint(config, args.split, include_tfidf=not getattr(args, "no_tfidf", False)),
    )
    feature_record_columns = [
        "entity_id", "business_name_raw", "business_address_raw", "country_norm",
        "name_canonical", "name_compact", "name_core", "name_token_sorted", "name_accent_folded",
        "address_canonical", "address_compact", "name_tokens", "address_tokens",
        "numeric_tokens", "postal_like_tokens", "missing_address",
    ]
    targets = pd.concat([
        _read_prepared(config, args.split, 2, columns=feature_record_columns),
        _read_prepared(config, args.split, 3, columns=feature_record_columns),
    ], ignore_index=True)
    targets = targets.set_index("entity_id", drop=False, verify_integrity=True)
    if args.split == "train":
        source1_ids = _read_prepared(config, args.split, 1, columns=["entity_id"])["entity_id"]
        truth = ground_truth_sets(_training_truth(config, source1_ids))
    else:
        truth = {}

    embed_names, embed_addrs = (None, None) if getattr(args, "no_embeddings", False) else load_embedding_vectors(config, args.split)
    embeddings_used = embed_names is not None

    fingerprint = _feature_fingerprint(
        config,
        args.split,
        embeddings_used,
        include_tfidf=not getattr(args, "no_tfidf", False),
    )
    store = ShardStore(config.artifact_dir("features") / args.split, fingerprint)
    if _force(args):
        for stale in store.parts_dir.glob("part-*"):
            stale.unlink()
        store.stage_manifest_path().unlink(missing_ok=True)
    logger.stage_start("features", split=args.split, shards=config.n_shards, embeddings=embeddings_used)
    logger.resource_plan(plan)

    if not store.is_complete():
        part_paths = list(candidate_store.iter_parts())
        for index, part_path in enumerate(part_paths, start=1):
            key = int(part_path.stem.split("-")[1])
            if key in store.completed_keys():
                continue

            def _build(chunk_size: int, _path=part_path, _key=key) -> pd.DataFrame:
                candidates = read_frame(_path)
                shard_source1 = _read_prepared_part(
                    config, args.split, 1, _key, columns=feature_record_columns
                )
                if args.split == "train":
                    candidates = label_candidates(candidates, truth)
                pieces = []
                for start in range(0, len(candidates), max(1, chunk_size)):
                    candidate_chunk = candidates.iloc[start : start + chunk_size]
                    target_ids = pd.Index(candidate_chunk["candidate_entity_id"].astype(str).unique())
                    candidate_targets = targets.loc[target_ids]
                    pieces.append(build_pair_features(
                        candidate_chunk,
                        shard_source1,
                        candidate_targets,
                        embed_names=embed_names,
                        embed_addrs=embed_addrs,
                    ))
                if pieces:
                    return pd.concat(pieces, ignore_index=True)
                return build_pair_features(candidates, shard_source1, targets)

            features = run_with_oom_backoff(
                _build,
                plan.chunk_pairs,
                on_retry=lambda size, exc: logger.event("oom_backoff", stage="features", new_chunk=size, error=type(exc).__name__),
            )
            store.commit(key, features)
            logger.progress("features", index, len(part_paths), extra={"shard": key, "rows": len(features)})
        rows = store.total_rows()
        first_part = next(store.iter_parts(), None)
        schema = list(read_frame(first_part).columns) if first_part is not None else []
        store.mark_complete(rows, schema, extra={"split": args.split})
    else:
        logger.info("stage already complete; skipping", rows=store.total_rows())

    logger.stage_end("features", split=args.split, rows=store.total_rows())
    print(store.stage_manifest_path())
    return 0


def _new_model(config: ProjectConfig, kind: str):
    params = _model_params(config, kind)
    if kind == "deterministic":
        return DeterministicScorer()
    if kind == "sgd":
        return SGDPairModel(seed=config.seed, **params)
    if kind == "lightgbm":
        return LightGBMPairModel(seed=config.seed, threads=config.threads, **params)
    raise ValueError(f"unknown model kind: {kind}")


def _load_model(kind: str, path: Path):
    if kind == "deterministic":
        return DeterministicScorer.load(path)
    if kind == "sgd":
        return SGDPairModel.load(path)
    if kind == "lightgbm":
        return LightGBMPairModel.load(path)
    raise ValueError(f"unknown model kind: {kind}")


def _default_model_path(config: ProjectConfig, kind: str) -> Path:
    suffixes = {"deterministic": ".json", "sgd": ".joblib", "lightgbm": ".txt"}
    path = config.artifact_dir("models") / f"{kind}{suffixes[kind]}"
    if not path.is_file():
        raise FileNotFoundError(
            f"model artifact not found: {path}. Run the train command or pass --model-path."
        )
    return path


def command_train(args: argparse.Namespace) -> int:
    config = _load_config(args)
    kind = _model_kind(config, args)
    logger = stage_logger(config, f"train-{kind}")
    logger.stage_start("train", model=kind, all_training_data=bool(args.all_training_data))
    store = _feature_store(config, "train")
    part_paths = list(store.iter_parts())
    if not part_paths:
        raise FileNotFoundError("no training features found; run build-features first")
    folds = None if args.all_training_data else read_frame(config.artifact_dir("splits") / "s1_folds.parquet")
    excluded_folds = set()
    if folds is not None:
        raw_excluded = getattr(args, "exclude_folds", None)
        excluded_folds = (
            {int(value.strip()) for value in raw_excluded.split(",") if value.strip()}
            if raw_excluded
            else {int(args.validation_fold)}
        )
        if not excluded_folds:
            raise ValueError("at least one fold must be excluded for held-out training")
        excluded_ids = set(
            folds.loc[folds["fold"].isin(excluded_folds), "source1_entity_id"].astype(str)
        )
    else:
        excluded_ids = set()
    max_negatives = int(config.training.get("max_negatives_per_entity", 50))

    def _training_part(path: Path) -> pd.DataFrame:
        frame = read_frame(path)
        if "label" not in frame:
            raise ValueError("training features are unlabeled")
        if folds is not None:
            frame = frame.loc[~frame["source1_entity_id"].astype(str).isin(excluded_ids)].copy()
        if frame.empty:
            return frame
        return sample_candidate_negatives(frame, max_negatives, config.seed)

    model = _new_model(config, kind)
    rows = positives = 0
    if kind == "sgd":
        # Two bounded passes keep SGD genuinely out-of-core while ensuring all
        # batches use one stable StandardScaler fitted on training rows only.
        for path in part_paths:
            frame = _training_part(path)
            if not frame.empty:
                model.update_scaler(feature_matrix(frame))
                rows += len(frame)
                positives += int(frame["label"].sum())
        if rows == 0:
            raise ValueError("no training pairs remain after fold selection")
        for path in part_paths:
            frame = _training_part(path)
            if frame.empty:
                continue
            X = feature_matrix(frame)
            y = frame["label"].to_numpy(dtype=np.int8)
            weights = frame["sample_weight"].to_numpy(dtype=np.float32)
            model.partial_fit_scaled(X, y, weights)
    else:
        sampled = [_training_part(path) for path in part_paths]
        sampled = [frame for frame in sampled if not frame.empty]
        if not sampled:
            raise ValueError("no training pairs remain after fold selection")
        features = pd.concat(sampled, ignore_index=True)
        rows, positives = len(features), int(features["label"].sum())
        X, y = feature_matrix(features), features["label"].to_numpy(dtype=np.int8)
        weights = features["sample_weight"].to_numpy(dtype=np.float32)
        if kind == "lightgbm":
            model.fit(X, y, sample_weight=weights, logger=logger)
        else:
            model.fit(X, y, sample_weight=weights)

    logger.info("training data ready", rows=rows, positives=positives)
    suffix = ".txt" if kind == "lightgbm" else ".joblib" if kind == "sgd" else ".json"
    output_name = getattr(args, "output_name", None) or kind
    path = config.artifact_dir("models") / f"{output_name}{suffix}"
    path.parent.mkdir(parents=True, exist_ok=True)
    model.save(path)
    write_manifest(
        path.with_suffix(path.suffix + ".manifest.json"),
        stage=f"train-{kind}",
        config=config.as_dict(),
        rows=rows,
        schema=FEATURE_COLUMNS,
        metrics={
            "positive_rows": positives,
            "validation_fold": None if args.all_training_data else args.validation_fold,
            "excluded_folds": sorted(excluded_folds),
            "all_training_data": bool(args.all_training_data),
            "model_metadata": model.metadata(),
            "max_negatives_per_entity": max_negatives,
        },
    )
    logger.stage_end("train", rows=rows, positives=positives)
    print(path)
    return 0


def _feature_store(config: ProjectConfig, split: str) -> ShardStore:
    directory = config.artifact_dir("features") / split
    return ShardStore(directory, stage_fingerprint(directory))


def _candidate_store(config: ProjectConfig, split: str) -> ShardStore:
    directory = config.artifact_dir("candidates") / split
    return ShardStore(directory, stage_fingerprint(directory))


def command_score(args: argparse.Namespace) -> int:
    config = _load_config(args)
    kind = _model_kind(config, args)
    feature_store = _feature_store(config, args.split)
    candidate_store = _candidate_store(config, args.split)
    folds = None
    if args.split == "train" and not args.all_training_data:
        folds = read_frame(config.artifact_dir("splits") / "s1_folds.parquet")
    model_path = Path(args.model_path) if args.model_path else _default_model_path(config, kind)
    model = _load_model(kind, model_path)
    output_name = getattr(args, "output_name", None) or f"{args.split}.parquet"
    if not output_name.endswith(".parquet"):
        output_name += ".parquet"
    out = config.artifact_dir("scores") / output_name
    out.parent.mkdir(parents=True, exist_ok=True)
    temporary = out.with_suffix(out.suffix + ".tmp")
    writer = None
    scored_rows = 0
    try:
        for feature_path in feature_store.iter_parts():
            key = int(feature_path.stem.split("-")[1])
            features = read_frame(feature_path)
            candidates = read_frame(candidate_store.part_path(key))
            _assert_candidate_feature_alignment(candidates, features)
            if folds is not None:
                features = select_pair_fold(features, folds, args.validation_fold, validation=True)
            if features.empty:
                continue
            scored = features[["source1_entity_id", "candidate_entity_id", "candidate_source"]].copy()
            scored["score"] = model.predict_scores(feature_matrix(features))
            if "label" in features:
                scored["label"] = features["label"]
            table = pa.Table.from_pandas(scored, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression="snappy")
            writer.write_table(table)
            scored_rows += len(scored)
    finally:
        if writer is not None:
            writer.close()
    if writer is None:
        empty_columns = ["source1_entity_id", "candidate_entity_id", "candidate_source", "score"]
        out = write_frame(pd.DataFrame(columns=empty_columns), out)
    else:
        os.replace(temporary, out)
    write_manifest(
        out.with_suffix(out.suffix + ".manifest.json"),
        stage=f"score-{kind}",
        config=config.as_dict(),
        rows=scored_rows,
        schema=list(scored.columns) if scored_rows else empty_columns,
        inputs=[str(model_path)],
        metrics={"validation_fold": None if args.all_training_data else args.validation_fold},
    )
    print(out)
    return 0


def command_tune_decision(args: argparse.Namespace) -> int:
    config = _load_config(args)
    logger = stage_logger(config, "tune-decision")
    scored = read_frame(_score_path(config, args.score_file))
    validation_ids = _validation_entity_ids(config, args.validation_fold)
    validation_set = set(validation_ids)
    scored = scored.loc[scored["source1_entity_id"].astype(str).isin(validation_set)].copy()
    truth = ground_truth_sets(_training_truth(config, validation_ids))
    thresholds = score_quantile_thresholds(scored, args.threshold_count)
    report = sweep_thresholds(scored, truth, thresholds)
    out = write_frame(report, config.artifact_dir("decisions") / "threshold_sweep.parquet")
    selected = report.iloc[0].to_dict()
    selected.update({"validation_fold": int(args.validation_fold), "score_file": str(args.score_file)})
    selected_path = config.artifact_dir("decisions") / "selected_threshold.json"
    selected_path.write_text(
        json.dumps(
            selected,
            indent=2,
            sort_keys=True,
            default=lambda value: value.item() if hasattr(value, "item") else str(value),
        ),
        encoding="utf-8",
    )
    for rank, row in report.head(10).iterrows():
        logger.metric("threshold_sweep", float(row["macro_f0_5"]), rank=int(rank), threshold=float(row["threshold"]), precision=float(row["macro_precision"]), recall=float(row["macro_recall"]))
    print(report.head(10).to_string(index=False)); print(out)
    return 0


def command_evaluate(args: argparse.Namespace) -> int:
    config = _load_config(args)
    logger = stage_logger(config, "evaluate")
    scored = read_frame(_score_path(config, args.score_file))
    validation_ids = _validation_entity_ids(config, args.validation_fold)
    validation_set = set(validation_ids)
    scored = scored.loc[scored["source1_entity_id"].astype(str).isin(validation_set)].copy()
    truth = ground_truth_sets(_training_truth(config, validation_ids))
    threshold = _decision_threshold(config, args.threshold)
    predictions = apply_thresholds(
        scored,
        truth.keys(),
        threshold,
        source_thresholds=config.model.get("source_thresholds", {}),
    )
    metrics = evaluate_entity_sets(truth, predictions)
    for name, value in metrics.items():
        logger.metric(name, value)
    _json_print(metrics)
    return 0


def command_analyze_errors(args: argparse.Namespace) -> int:
    config = _load_config(args)
    scored = read_frame(_score_path(config, args.score_file))
    candidates = _candidate_store(config, "train").read_all()
    validation_ids = _validation_entity_ids(config, args.validation_fold)
    scored_ids = set(validation_ids)
    scored = scored.loc[scored["source1_entity_id"].astype(str).isin(scored_ids)].copy()
    truth = ground_truth_sets(_training_truth(config, validation_ids))
    threshold = _decision_threshold(config, args.threshold)
    predictions = apply_thresholds(scored, truth.keys(), threshold, config.model.get("source_thresholds", {}))
    candidate_sets = sets_from_long(candidates.loc[candidates["source1_entity_id"].isin(scored_ids)], "candidate_entity_id")
    errors = analyze_errors(truth, candidate_sets, predictions, scored)
    out = write_frame(errors, config.artifact_dir("reports") / "errors.parquet")
    print(errors["error_type"].value_counts().to_string() if not errors.empty else "no errors"); print(out)
    return 0


def command_infer(args: argparse.Namespace) -> int:
    config = _load_config(args)
    kind = _model_kind(config, args)
    plan = config.resource_plan()
    logger = stage_logger(config, "infer")
    logger.stage_start("infer", model=kind)
    if not _candidate_store(config, "test").is_complete():
        command_generate_candidates(argparse.Namespace(**{**vars(args), "split": "test"}))
    if not _feature_store(config, "test").is_complete():
        command_build_features(argparse.Namespace(**{**vars(args), "split": "test"}))
    feature_store = _feature_store(config, "test")
    candidate_store = _candidate_store(config, "test")
    model_path = Path(args.model_path) if args.model_path else _default_model_path(config, kind)
    model = _load_model(kind, model_path)
    threshold = _decision_threshold(config, args.threshold)
    score_fingerprint = input_fingerprint(
        [model_path, feature_store.stage_manifest_path(), candidate_store.stage_manifest_path()]
    )
    score_store = ShardStore(config.artifact_dir("scores") / "test-shards", score_fingerprint)
    if _force(args):
        shutil.rmtree(score_store.stage_dir, ignore_errors=True)
        score_store = ShardStore(config.artifact_dir("scores") / "test-shards", score_fingerprint)

    def _output_shards():
        for feature_path in feature_store.iter_parts():
            key = int(feature_path.stem.split("-")[1])
            candidates = read_frame(candidate_store.part_path(key))
            if key in score_store.completed_keys():
                scored = read_frame(score_store.part_path(key))
                _assert_candidate_feature_alignment(candidates, scored)
            else:
                features = read_frame(feature_path)
                _assert_candidate_feature_alignment(candidates, features)
                scored = features[["source1_entity_id", "candidate_entity_id", "candidate_source"]].copy()
                scores = np.empty(len(features), dtype=np.float32)
                for start in range(0, len(features), max(1, plan.chunk_pairs)):
                    stop = min(len(features), start + max(1, plan.chunk_pairs))
                    scores[start:stop] = model.predict_scores(feature_matrix(features.iloc[start:stop]))
                scored["score"] = scores
                score_store.commit(key, scored)
            source1 = _read_prepared_part(
                config, "test", 1, key, columns=["entity_id"]
            )
            predictions = apply_thresholds(
                scored,
                source1["entity_id"],
                threshold,
                source_thresholds=config.model.get("source_thresholds", {}),
            )
            candidate_sets = sets_from_long(candidates, "candidate_entity_id")
            yield source1["entity_id"].astype(str), predictions, candidate_sets

    matching, candidate = write_submission_outputs_sharded(config.output_root, _output_shards())
    sample_part = next(score_store.iter_parts(), None)
    score_schema = list(read_frame(sample_part).columns) if sample_part is not None else []
    score_store.mark_complete(score_store.total_rows(), score_schema, extra={"model": kind})
    logger.info("wrote outputs", matching=str(matching), candidate=str(candidate))
    logger.stage_end("infer")
    print(matching); print(candidate)
    return 0


def command_write_output(args: argparse.Namespace) -> int:
    """Reformat outputs only from cached scores + candidates (never reruns the model)."""
    config = _load_config(args)
    logger = stage_logger(config, "write-output")
    threshold = _decision_threshold(config, args.threshold)
    if args.split == "test":
        score_dir = config.artifact_dir("scores") / "test-shards"
        score_store = ShardStore(score_dir, stage_fingerprint(score_dir))
        candidate_store = _candidate_store(config, "test")
        if not score_store.is_complete():
            raise FileNotFoundError(f"complete cached score shards not found: {score_dir}; run infer first")

        def _output_shards():
            for score_path in score_store.iter_parts():
                key = int(score_path.stem.split("-")[1])
                scored = read_frame(score_path)
                candidates = read_frame(candidate_store.part_path(key))
                _assert_candidate_feature_alignment(candidates, scored)
                source1 = _read_prepared_part(
                    config, "test", 1, key, columns=["entity_id"]
                )
                predictions = apply_thresholds(
                    scored,
                    source1["entity_id"],
                    threshold,
                    source_thresholds=config.model.get("source_thresholds", {}),
                )
                yield source1["entity_id"].astype(str), predictions, sets_from_long(candidates, "candidate_entity_id")

        matching, candidate = write_submission_outputs_sharded(config.output_root, _output_shards())
        logger.event("outputs_rewritten", matching=str(matching), candidate=str(candidate))
        print(matching); print(candidate)
        return 0

    candidates = _candidate_store(config, args.split).read_all()
    source1 = _read_prepared(config, args.split, 1)
    scored_path = config.artifact_dir("scores") / f"{args.split}.parquet"
    if not scored_path.is_file():
        raise FileNotFoundError(f"cached scores not found: {scored_path}; run score first")
    scored = read_frame(scored_path)
    _assert_candidate_feature_alignment(candidates, scored)
    predictions = apply_thresholds(
        scored,
        source1["entity_id"],
        threshold,
        source_thresholds=config.model.get("source_thresholds", {}),
    )
    candidate_sets = sets_from_long(candidates, "candidate_entity_id")
    matching, candidate = write_submission_outputs(config.output_root, source1["entity_id"], predictions, candidate_sets)
    logger.event("outputs_rewritten", matching=str(matching), candidate=str(candidate))
    print(matching); print(candidate)
    return 0


def command_preflight(args: argparse.Namespace) -> int:
    config = _load_config(args)
    matching, candidate = config.output_root / "matching_results.tsv", config.output_root / "candidate_pairs.tsv"
    test_dir = config.data_root / "test"
    errors = preflight_outputs(matching, candidate, test_dir)
    if errors:
        for error in errors: print(f"ERROR: {error}")
        return 1
    print("Internal preflight: PASS")
    if args.official_validator:
        result = run_official_validator(args.official_validator, matching, candidate, test_dir, check_ids=args.check_ids)
        print(result.stdout); print(result.stderr, file=sys.stderr)
        return result.returncode
    return 0


def command_package(args: argparse.Namespace) -> int:
    config = _load_config(args)
    documentation = Path(args.documentation).resolve()
    if not documentation.is_file():
        raise FileNotFoundError(f"completed competition documentation not found: {documentation}")
    matching = config.output_root / "matching_results.tsv"
    candidates = config.output_root / "candidate_pairs.tsv"
    errors = preflight_outputs(matching, candidates, config.data_root / "test")
    if errors:
        raise ValueError("submission preflight failed: " + "; ".join(errors[:10]))
    if args.official_validator:
        result = run_official_validator(
            args.official_validator,
            matching,
            candidates,
            config.data_root / "test",
            check_ids=True,
        )
        if result.returncode:
            raise ValueError(f"official validator failed:\n{result.stdout}\n{result.stderr}")

    package_root = config.artifact_dir("package") / f"{args.team_name}_submission"
    # A package is an immutable snapshot; never merge into stale staging files.
    shutil.rmtree(package_root, ignore_errors=True)
    (package_root / "output").mkdir(parents=True, exist_ok=True)
    shutil.copy2(matching, package_root / "output" / matching.name)
    shutil.copy2(candidates, package_root / "output" / candidates.name)
    project_root = Path(__file__).resolve().parents[2]
    shutil.copytree(project_root, package_root / "code" / "business_entity_resolution", ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
    shutil.copy2(documentation, package_root / "Documentation_template.md")
    archive = shutil.make_archive(str(package_root), "zip", package_root)
    print(archive)
    return 0


def command_smoke(args: argparse.Namespace) -> int:
    config = _load_config(args)
    report = run_smoke(config.output_root, config.seed)
    _json_print(report)
    return 0


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Amazon ML 2026 business entity resolution pipeline")
    sub = parser.add_subparsers(dest="command", required=True)

    def add_common(name: str, function):
        command = sub.add_parser(name)
        command.add_argument("--config", required=True)
        command.add_argument("--max-rows", type=int)
        command.add_argument("--force", action="store_true", help="Ignore committed checkpoints and redo the stage.")
        command.set_defaults(function=function)
        return command

    add_common("audit", command_audit)
    prepare = add_common("prepare", command_prepare); prepare.add_argument("--split", choices=["train", "test", "both"], default="both")
    splits = add_common("make-splits", command_make_splits); splits.add_argument("--folds", type=int, default=10)
    candidates = add_common("generate-candidates", command_generate_candidates); candidates.add_argument("--split", choices=["train", "test"], required=True); candidates.add_argument("--no-tfidf", action="store_true")
    embed = add_common("embed", command_embed); embed.add_argument("--split", choices=["train", "test", "both"], default="both")
    features = add_common("build-features", command_build_features); features.add_argument("--split", choices=["train", "test"], required=True); features.add_argument("--no-embeddings", action="store_true")
    train = add_common("train", command_train); train.add_argument("--model", choices=["deterministic", "sgd", "lightgbm"]); train.add_argument("--validation-fold", type=int, default=0); train.add_argument("--exclude-folds", help="Comma-separated folds excluded from training; defaults to --validation-fold."); train.add_argument("--all-training-data", action="store_true"); train.add_argument("--output-name")
    score = add_common("score", command_score); score.add_argument("--split", choices=["train", "test"], required=True); score.add_argument("--model", choices=["deterministic", "sgd", "lightgbm"]); score.add_argument("--model-path"); score.add_argument("--validation-fold", type=int, default=0); score.add_argument("--all-training-data", action="store_true"); score.add_argument("--output-name")
    tune = add_common("tune-decision", command_tune_decision); tune.add_argument("--threshold-count", type=int, default=101); tune.add_argument("--score-file", default="train.parquet"); tune.add_argument("--validation-fold", type=int, default=0)
    evaluate = add_common("evaluate", command_evaluate); evaluate.add_argument("--score-file", default="train.parquet"); evaluate.add_argument("--threshold", type=float); evaluate.add_argument("--validation-fold", type=int, default=0)
    errors = add_common("analyze-errors", command_analyze_errors); errors.add_argument("--score-file", default="train.parquet"); errors.add_argument("--threshold", type=float); errors.add_argument("--validation-fold", type=int, default=0)
    infer = add_common("infer", command_infer); infer.add_argument("--model", choices=["deterministic", "sgd", "lightgbm"]); infer.add_argument("--model-path"); infer.add_argument("--threshold", type=float); infer.add_argument("--no-tfidf", action="store_true")
    write_output = add_common("write-output", command_write_output); write_output.add_argument("--split", choices=["train", "test"], default="test"); write_output.add_argument("--threshold", type=float)
    preflight = add_common("preflight", command_preflight); preflight.add_argument("--official-validator"); preflight.add_argument("--check-ids", action="store_true")
    package = add_common("package", command_package); package.add_argument("--team-name", required=True); package.add_argument("--documentation", required=True); package.add_argument("--official-validator")
    mini = add_common("make-mini", command_make_mini); mini.add_argument("--mini-root", default="dataset/mini"); mini.add_argument("--count", type=int, default=15); mini.add_argument("--source-root", default=None)
    add_common("smoke", command_smoke)
    return parser


def main(argv: list[str] | None = None) -> int:
    args = build_parser().parse_args(argv)
    return int(args.function(args))


if __name__ == "__main__":
    raise SystemExit(main())

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/cli.py', '...')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/config.py`

~~~~python
"""Environment-neutral configuration loading."""

from __future__ import annotations

from dataclasses import dataclass, field
import json
import os
from pathlib import Path
from typing import Any


@dataclass(slots=True)
class ProjectConfig:
    data_root: Path
    artifact_root: Path
    output_root: Path
    run_id: str = "default"
    seed: int = 2026
    batch_size: int = 100_000
    threads: int = 1
    device: str = "cpu"
    max_rows: int | None = None
    n_shards: int = 32
    model: dict[str, Any] = field(default_factory=dict)
    training: dict[str, Any] = field(default_factory=dict)
    blocking: dict[str, Any] = field(default_factory=dict)
    resources: dict[str, Any] = field(default_factory=dict)
    embeddings: dict[str, Any] = field(default_factory=dict)
    logging: dict[str, Any] = field(default_factory=dict)
    config_path: Path | None = None

    @classmethod
    def load(cls, path: str | Path, overrides: dict[str, Any] | None = None) -> "ProjectConfig":
        config_path = Path(path).expanduser().resolve()
        with config_path.open(encoding="utf-8") as handle:
            raw = json.load(handle)
        env_map = {
            "data_root": os.getenv("BER_DATA_ROOT"),
            "artifact_root": os.getenv("BER_ARTIFACT_ROOT"),
            "output_root": os.getenv("BER_OUTPUT_ROOT"),
            "run_id": os.getenv("BER_RUN_ID"),
            "device": os.getenv("BER_DEVICE"),
            "threads": os.getenv("BER_THREADS"),
        }
        raw.update({k: v for k, v in env_map.items() if v not in (None, "")})
        if overrides:
            raw.update({k: v for k, v in overrides.items() if v is not None})

        def path_value(name: str, default: str) -> Path:
            value = Path(raw.get(name, default)).expanduser()
            return value if value.is_absolute() else (Path.cwd() / value).resolve()

        return cls(
            data_root=path_value("data_root", "dataset"),
            artifact_root=path_value("artifact_root", "artifacts"),
            output_root=path_value("output_root", "output"),
            run_id=str(raw.get("run_id", "default")),
            seed=int(raw.get("seed", 2026)),
            batch_size=int(raw.get("batch_size", 100_000)),
            threads=int(raw.get("threads", 1)),
            device=str(raw.get("device", "cpu")),
            max_rows=None if raw.get("max_rows") is None else int(raw["max_rows"]),
            n_shards=int(raw.get("n_shards", 32)),
            model=dict(raw.get("model", {})),
            training=dict(raw.get("training", {})),
            blocking=dict(raw.get("blocking", {})),
            resources=dict(raw.get("resources", {})),
            embeddings=dict(raw.get("embeddings", {})),
            logging=dict(raw.get("logging", {})),
            config_path=config_path,
        )

    def resource_plan(self):
        """Derive a :class:`~business_entity_resolution.resources.ResourcePlan`.

        Imported lazily so the config module stays dependency-light and the
        pipeline still imports without psutil/torch present.
        """
        from .resources import plan_resources

        resources = self.resources or {}
        return plan_resources(
            mode=str(resources.get("mode", "auto")),
            ram_fraction=float(resources.get("ram_fraction", 0.65)),
            vram_fraction=float(resources.get("vram_fraction", 0.75)),
            threads=self.threads or None,
            chunk_pairs=resources.get("chunk_pairs"),
            embed_batch_size=resources.get("embed_batch_size"),
        )

    def embeddings_enabled(self) -> bool:
        setting = str((self.embeddings or {}).get("enable", "auto")).lower()
        if setting in {"false", "0", "no", "off"}:
            return False
        if setting in {"true", "1", "yes", "on"}:
            return True
        # auto: enable only when the embedding dependencies import.
        try:
            import torch  # noqa: F401
            import transformers  # noqa: F401

            return True
        except Exception:
            return False

    def shares_artifacts_root(self) -> bool:
        try:
            self.artifact_root.relative_to(self.data_root)
            return True
        except ValueError:
            return False

    def artifact_dir(self, stage: str) -> Path:
        return self.artifact_root / self.run_id / stage

    def as_dict(self) -> dict[str, Any]:
        return {
            "data_root": str(self.data_root),
            "artifact_root": str(self.artifact_root),
            "output_root": str(self.output_root),
            "run_id": self.run_id,
            "seed": self.seed,
            "batch_size": self.batch_size,
            "threads": self.threads,
            "device": self.device,
            "max_rows": self.max_rows,
            "n_shards": self.n_shards,
            "model": self.model,
            "training": self.training,
            "blocking": self.blocking,
            "resources": self.resources,
            "embeddings": self.embeddings,
            "logging": self.logging,
        }


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/config.py', 'IiIiRW52aXJvbm1lbnQtbmV1dHJhbCBjb25maWd1cmF0aW9uIGxvYWRpbmcuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUHJvamVjdENvbmZpZzoKICAgIGRhdGFfcm9vdDogUGF0aAogICAgYXJ0aWZhY3Rfcm9vdDogUGF0aAogICAgb3V0cHV0X3Jvb3Q6IFBhdGgKICAgIHJ1bl9pZDogc3RyID0gImRlZmF1bHQiCiAgICBzZWVkOiBpbnQgPSAyMDI2CiAgICBiYXRjaF9zaXplOiBpbnQgPSAxMDBfMDAwCiAgICB0aHJlYWRzOiBpbnQgPSAxCiAgICBkZXZpY2U6IHN0ciA9ICJjcHUiCiAgICBtYXhfcm93czogaW50IHwgTm9uZSA9IE5vbmUKICAgIG5fc2hhcmRzOiBpbnQgPSAzMgogICAgbW9kZWw6IGRpY3Rbc3RyLCBBbnldID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICB0cmFpbmluZzogZGljdFtzdHIsIEFueV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIGJsb2NraW5nOiBkaWN0W3N0ciwgQW55XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgcmVzb3VyY2VzOiBkaWN0W3N0ciwgQW55XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgZW1iZWRkaW5nczogZGljdFtzdHIsIEFueV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIGxvZ2dpbmc6IGRpY3Rbc3RyLCBBbnldID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICBjb25maWdfcGF0aDogUGF0aCB8IE5vbmUgPSBOb25lCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIHBhdGg6IHN0ciB8IFBhdGgsIG92ZXJyaWRlczogZGljdFtzdHIsIEFueV0gfCBOb25lID0gTm9uZSkgLT4gIlByb2plY3RDb25maWciOgogICAgICAgIGNvbmZpZ19wYXRoID0gUGF0aChwYXRoKS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICAgICAgd2l0aCBjb25maWdfcGF0aC5vcGVuKGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICAgICAgcmF3ID0ganNvbi5sb2FkKGhhbmRsZSkKICAgICAgICBlbnZfbWFwID0gewogICAgICAgICAgICAiZGF0YV9yb290Ijogb3MuZ2V0ZW52KCJCRVJfREFUQV9ST09UIiksCiAgICAgICAgICAgICJhcnRpZmFjdF9yb290Ijogb3MuZ2V0ZW52KCJCRVJfQVJUSUZBQ1RfUk9PVCIpLAogICAgICAgICAgICAib3V0cHV0X3Jvb3QiOiBvcy5nZXRlbnYoIkJFUl9PVVRQVVRfUk9PVCIpLAogICAgICAgICAgICAicnVuX2lkIjogb3MuZ2V0ZW52KCJCRVJfUlVOX0lEIiksCiAgICAgICAgICAgICJkZXZpY2UiOiBvcy5nZXRlbnYoIkJFUl9ERVZJQ0UiKSwKICAgICAgICAgICAgInRocmVhZHMiOiBvcy5nZXRlbnYoIkJFUl9USFJFQURTIiksCiAgICAgICAgfQogICAgICAgIHJhdy51cGRhdGUoe2s6IHYgZm9yIGssIHYgaW4gZW52X21hcC5pdGVtcygpIGlmIHYgbm90IGluIChOb25lLCAiIil9KQogICAgICAgIGlmIG92ZXJyaWRlczoKICAgICAgICAgICAgcmF3LnVwZGF0ZSh7azogdiBmb3IgaywgdiBpbiBvdmVycmlkZXMuaXRlbXMoKSBpZiB2IGlzIG5vdCBOb25lfSkKCiAgICAgICAgZGVmIHBhdGhfdmFsdWUobmFtZTogc3RyLCBkZWZhdWx0OiBzdHIpIC0+IFBhdGg6CiAgICAgICAgICAgIHZhbHVlID0gUGF0aChyYXcuZ2V0KG5hbWUsIGRlZmF1bHQpKS5leHBhbmR1c2VyKCkKICAgICAgICAgICAgcmV0dXJuIHZhbHVlIGlmIHZhbHVlLmlzX2Fic29sdXRlKCkgZWxzZSAoUGF0aC5jd2QoKSAvIHZhbHVlKS5yZXNvbHZlKCkKCiAgICAgICAgcmV0dXJuIGNscygKICAgICAgICAgICAgZGF0YV9yb290PXBhdGhfdmFsdWUoImRhdGFfcm9vdCIsICJkYXRhc2V0IiksCiAgICAgICAgICAgIGFydGlmYWN0X3Jvb3Q9cGF0aF92YWx1ZSgiYXJ0aWZhY3Rfcm9vdCIsICJhcnRpZmFjdHMiKSwKICAgICAgICAgICAgb3V0cHV0X3Jvb3Q9cGF0aF92YWx1ZSgib3V0cHV0X3Jvb3QiLCAib3V0cHV0IiksCiAgICAgICAgICAgIHJ1bl9pZD1zdHIocmF3LmdldCgicnVuX2lkIiwgImRlZmF1bHQiKSksCiAgICAgICAgICAgIHNlZWQ9aW50KHJhdy5nZXQoInNlZWQiLCAyMDI2KSksCiAgICAgICAgICAgIGJhdGNoX3NpemU9aW50KHJhdy5nZXQoImJhdGNoX3NpemUiLCAxMDBfMDAwKSksCiAgICAgICAgICAgIHRocmVhZHM9aW50KHJhdy5nZXQoInRocmVhZHMiLCAxKSksCiAgICAgICAgICAgIGRldmljZT1zdHIocmF3LmdldCgiZGV2aWNlIiwgImNwdSIpKSwKICAgICAgICAgICAgbWF4X3Jvd3M9Tm9uZSBpZiByYXcuZ2V0KCJtYXhfcm93cyIpIGlzIE5vbmUgZWxzZSBpbnQocmF3WyJtYXhfcm93cyJdKSwKICAgICAgICAgICAgbl9zaGFyZHM9aW50KHJhdy5nZXQoIm5fc2hhcmRzIiwgMzIpKSwKICAgICAgICAgICAgbW9kZWw9ZGljdChyYXcuZ2V0KCJtb2RlbCIsIHt9KSksCiAgICAgICAgICAgIHRyYWluaW5nPWRpY3QocmF3LmdldCgidHJhaW5pbmciLCB7fSkpLAogICAgICAgICAgICBibG9ja2luZz1kaWN0KHJhdy5nZXQoImJsb2NraW5nIiwge30pKSwKICAgICAgICAgICAgcmVzb3VyY2VzPWRpY3QocmF3LmdldCgicmVzb3VyY2VzIiwge30pKSwKICAgICAgICAgICAgZW1iZWRkaW5ncz1kaWN0KHJhdy5nZXQoImVtYmVkZGluZ3MiLCB7fSkpLAogICAgICAgICAgICBsb2dnaW5nPWRpY3QocmF3LmdldCgibG9nZ2luZyIsIHt9KSksCiAgICAgICAgICAgIGNvbmZpZ19wYXRoPWNvbmZpZ19wYXRoLAogICAgICAgICkKCiAgICBkZWYgcmVzb3VyY2VfcGxhbihzZWxmKToKICAgICAgICAiIiJEZXJpdmUgYSA6Y2xhc3M6YH5idXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5yZXNvdXJjZXMuUmVzb3VyY2VQbGFuYC4KCiAgICAgICAgSW1wb3J0ZWQgbGF6aWx5IHNvIHRoZSBjb25maWcgbW9kdWxlIHN0YXlzIGRlcGVuZGVuY3ktbGlnaHQgYW5kIHRoZQogICAgICAgIHBpcGVsaW5lIHN0aWxsIGltcG9ydHMgd2l0aG91dCBwc3V0aWwvdG9yY2ggcHJlc2VudC4KICAgICAgICAiIiIKICAgICAgICBmcm9tIC5yZXNvdXJjZXMgaW1wb3J0IHBsYW5fcmVzb3VyY2VzCgogICAgICAgIHJlc291cmNlcyA9IHNlbGYucmVzb3VyY2VzIG9yIHt9CiAgICAgICAgcmV0dXJuIHBsYW5fcmVzb3VyY2VzKAogICAgICAgICAgICBtb2RlPXN0cihyZXNvdXJjZXMuZ2V0KCJtb2RlIiwgImF1dG8iKSksCiAgICAgICAgICAgIHJhbV9mcmFjdGlvbj1mbG9hdChyZXNvdXJjZXMuZ2V0KCJyYW1fZnJhY3Rpb24iLCAwLjY1KSksCiAgICAgICAgICAgIHZyYW1fZnJhY3Rpb249ZmxvYXQocmVzb3VyY2VzLmdldCgidnJhbV9mcmFjdGlvbiIsIDAuNzUpKSwKICAgICAgICAgICAgdGhyZWFkcz1zZWxmLnRocmVhZHMgb3IgTm9uZSwKICAgICAgICAgICAgY2h1bmtfcGFpcnM9cmVzb3VyY2VzLmdldCgiY2h1bmtfcGFpcnMiKSwKICAgICAgICAgICAgZW1iZWRfYmF0Y2hfc2l6ZT1yZXNvdXJjZXMuZ2V0KCJlbWJlZF9iYXRjaF9zaXplIiksCiAgICAgICAgKQoKICAgIGRlZiBlbWJlZGRpbmdzX2VuYWJsZWQoc2VsZikgLT4gYm9vbDoKICAgICAgICBzZXR0aW5nID0gc3RyKChzZWxmLmVtYmVkZGluZ3Mgb3Ige30pLmdldCgiZW5hYmxlIiwgImF1dG8iKSkubG93ZXIoKQogICAgICAgIGlmIHNldHRpbmcgaW4geyJmYWxzZSIsICIwIiwgIm5vIiwgIm9mZiJ9OgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBpZiBzZXR0aW5nIGluIHsidHJ1ZSIsICIxIiwgInllcyIsICJvbiJ9OgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgICMgYXV0bzogZW5hYmxlIG9ubHkgd2hlbiB0aGUgZW1iZWRkaW5nIGRlcGVuZGVuY2llcyBpbXBvcnQuCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgdG9yY2ggICMgbm9xYTogRjQwMQogICAgICAgICAgICBpbXBvcnQgdHJhbnNmb3JtZXJzICAjIG5vcWE6IEY0MDEKCiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgZGVmIHNoYXJlc19hcnRpZmFjdHNfcm9vdChzZWxmKSAtPiBib29sOgogICAgICAgIHRyeToKICAgICAgICAgICAgc2VsZi5hcnRpZmFjdF9yb290LnJlbGF0aXZlX3RvKHNlbGYuZGF0YV9yb290KQogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICBkZWYgYXJ0aWZhY3RfZGlyKHNlbGYsIHN0YWdlOiBzdHIpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYuYXJ0aWZhY3Rfcm9vdCAvIHNlbGYucnVuX2lkIC8gc3RhZ2UKCiAgICBkZWYgYXNfZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgQW55XToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiZGF0YV9yb290Ijogc3RyKHNlbGYuZGF0YV9yb290KSwKICAgICAgICAgICAgImFydGlmYWN0X3Jvb3QiOiBzdHIoc2VsZi5hcnRpZmFjdF9yb290KSwKICAgICAgICAgICAgIm91dHB1dF9yb290Ijogc3RyKHNlbGYub3V0cHV0X3Jvb3QpLAogICAgICAgICAgICAicnVuX2lkIjogc2VsZi5ydW5faWQsCiAgICAgICAgICAgICJzZWVkIjogc2VsZi5zZWVkLAogICAgICAgICAgICAiYmF0Y2hfc2l6ZSI6IHNlbGYuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgInRocmVhZHMiOiBzZWxmLnRocmVhZHMsCiAgICAgICAgICAgICJkZXZpY2UiOiBzZWxmLmRldmljZSwKICAgICAgICAgICAgIm1heF9yb3dzIjogc2VsZi5tYXhfcm93cywKICAgICAgICAgICAgIm5fc2hhcmRzIjogc2VsZi5uX3NoYXJkcywKICAgICAgICAgICAgIm1vZGVsIjogc2VsZi5tb2RlbCwKICAgICAgICAgICAgInRyYWluaW5nIjogc2VsZi50cmFpbmluZywKICAgICAgICAgICAgImJsb2NraW5nIjogc2VsZi5ibG9ja2luZywKICAgICAgICAgICAgInJlc291cmNlcyI6IHNlbGYucmVzb3VyY2VzLAogICAgICAgICAgICAiZW1iZWRkaW5ncyI6IHNlbGYuZW1iZWRkaW5ncywKICAgICAgICAgICAgImxvZ2dpbmciOiBzZWxmLmxvZ2dpbmcsCiAgICAgICAgfQoK', '422bb6957b275d5ac35a4cbf3a35a10aeadfb920e2dc9173ca5a5da0499c8333')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/data.py`

~~~~python
"""Schema-safe challenge TSV I/O."""

from __future__ import annotations

from collections.abc import Iterable, Iterator
from pathlib import Path
import pandas as pd

from .schemas import GROUND_TRUTH_COLUMNS, SOURCE_COLUMNS, SOURCE_PREFIXES, SchemaError, require_columns


def source_path(data_root: str | Path, split: str, source: int) -> Path:
    if split not in {"train", "test"}:
        raise ValueError(f"split must be train or test, got {split!r}")
    if source not in {1, 2, 3}:
        raise ValueError(f"source must be 1, 2, or 3, got {source!r}")
    return Path(data_root) / split / f"{split}_source{source}.tsv"


def ground_truth_path(data_root: str | Path) -> Path:
    return Path(data_root) / "train" / "train_ground_truth.tsv"


def read_tsv(path: str | Path, expected_columns: tuple[str, ...], nrows: int | None = None) -> pd.DataFrame:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"TSV file not found: {path}")
    frame = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8",
        nrows=nrows,
    )
    require_columns(list(frame.columns), expected_columns, str(path))
    return frame


def iter_tsv(
    path: str | Path,
    expected_columns: tuple[str, ...],
    batch_size: int = 100_000,
    max_rows: int | None = None,
) -> Iterator[pd.DataFrame]:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"TSV file not found: {path}")
    remaining = max_rows
    reader = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        encoding="utf-8",
        chunksize=batch_size,
    )
    for chunk in reader:
        require_columns(list(chunk.columns), expected_columns, str(path))
        if remaining is not None:
            chunk = chunk.iloc[:remaining].copy()
            remaining -= len(chunk)
        if not chunk.empty:
            yield chunk
        if remaining is not None and remaining <= 0:
            break


def validate_entity_frame(frame: pd.DataFrame, source: int, label: str = "entity frame") -> None:
    require_columns(list(frame.columns[:4]), SOURCE_COLUMNS, label)
    prefix = SOURCE_PREFIXES[f"source{source}"]
    if frame["entity_id"].eq("").any():
        raise SchemaError(f"{label}: blank entity_id")
    invalid = ~frame["entity_id"].str.match(rf"^{prefix}\d+$")
    if invalid.any():
        example = frame.loc[invalid, "entity_id"].iloc[0]
        raise SchemaError(f"{label}: invalid source-{source} ID {example!r}")
    if frame["entity_id"].duplicated().any():
        raise SchemaError(f"{label}: duplicate entity_id")
    for column in ("business_name", "country"):
        if frame[column].eq("").any():
            raise SchemaError(f"{label}: blank {column}")


def load_source(data_root: str | Path, split: str, source: int, nrows: int | None = None) -> pd.DataFrame:
    frame = read_tsv(source_path(data_root, split, source), SOURCE_COLUMNS, nrows=nrows)
    validate_entity_frame(frame, source, f"{split} source{source}")
    return frame


def load_ground_truth(data_root: str | Path, nrows: int | None = None) -> pd.DataFrame:
    return read_tsv(ground_truth_path(data_root), GROUND_TRUTH_COLUMNS, nrows=nrows)


def load_ground_truth_for_ids(
    data_root: str | Path,
    source1_ids: Iterable[str],
    batch_size: int = 100_000,
) -> pd.DataFrame:
    """Stream ground truth and return rows for an exact Source 1 entity set."""
    wanted = {str(value) for value in source1_ids}
    if not wanted:
        return pd.DataFrame(columns=GROUND_TRUTH_COLUMNS)
    found: list[pd.DataFrame] = []
    for chunk in iter_tsv(ground_truth_path(data_root), GROUND_TRUTH_COLUMNS, batch_size=batch_size):
        selected = chunk.loc[chunk["source1_entity_id"].isin(wanted)]
        if not selected.empty:
            found.append(selected)
            wanted.difference_update(selected["source1_entity_id"].astype(str))
        if not wanted:
            break
    if wanted:
        examples = ", ".join(sorted(wanted)[:5])
        raise SchemaError(f"ground truth is missing {len(wanted)} requested Source 1 IDs; examples: {examples}")
    return pd.concat(found, ignore_index=True)

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/data.py', 'IiIiU2NoZW1hLXNhZmUgY2hhbGxlbmdlIFRTViBJL08uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgSXRlcmFibGUsIEl0ZXJhdG9yCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIC5zY2hlbWFzIGltcG9ydCBHUk9VTkRfVFJVVEhfQ09MVU1OUywgU09VUkNFX0NPTFVNTlMsIFNPVVJDRV9QUkVGSVhFUywgU2NoZW1hRXJyb3IsIHJlcXVpcmVfY29sdW1ucwoKCmRlZiBzb3VyY2VfcGF0aChkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgsIHNwbGl0OiBzdHIsIHNvdXJjZTogaW50KSAtPiBQYXRoOgogICAgaWYgc3BsaXQgbm90IGluIHsidHJhaW4iLCAidGVzdCJ9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJzcGxpdCBtdXN0IGJlIHRyYWluIG9yIHRlc3QsIGdvdCB7c3BsaXQhcn0iKQogICAgaWYgc291cmNlIG5vdCBpbiB7MSwgMiwgM306CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInNvdXJjZSBtdXN0IGJlIDEsIDIsIG9yIDMsIGdvdCB7c291cmNlIXJ9IikKICAgIHJldHVybiBQYXRoKGRhdGFfcm9vdCkgLyBzcGxpdCAvIGYie3NwbGl0fV9zb3VyY2V7c291cmNlfS50c3YiCgoKZGVmIGdyb3VuZF90cnV0aF9wYXRoKGRhdGFfcm9vdDogc3RyIHwgUGF0aCkgLT4gUGF0aDoKICAgIHJldHVybiBQYXRoKGRhdGFfcm9vdCkgLyAidHJhaW4iIC8gInRyYWluX2dyb3VuZF90cnV0aC50c3YiCgoKZGVmIHJlYWRfdHN2KHBhdGg6IHN0ciB8IFBhdGgsIGV4cGVjdGVkX2NvbHVtbnM6IHR1cGxlW3N0ciwgLi4uXSwgbnJvd3M6IGludCB8IE5vbmUgPSBOb25lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiVFNWIGZpbGUgbm90IGZvdW5kOiB7cGF0aH0iKQogICAgZnJhbWUgPSBwZC5yZWFkX2NzdigKICAgICAgICBwYXRoLAogICAgICAgIHNlcD0iXHQiLAogICAgICAgIGR0eXBlPXN0ciwKICAgICAgICBrZWVwX2RlZmF1bHRfbmE9RmFsc2UsCiAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwKICAgICAgICBucm93cz1ucm93cywKICAgICkKICAgIHJlcXVpcmVfY29sdW1ucyhsaXN0KGZyYW1lLmNvbHVtbnMpLCBleHBlY3RlZF9jb2x1bW5zLCBzdHIocGF0aCkpCiAgICByZXR1cm4gZnJhbWUKCgpkZWYgaXRlcl90c3YoCiAgICBwYXRoOiBzdHIgfCBQYXRoLAogICAgZXhwZWN0ZWRfY29sdW1uczogdHVwbGVbc3RyLCAuLi5dLAogICAgYmF0Y2hfc2l6ZTogaW50ID0gMTAwXzAwMCwKICAgIG1heF9yb3dzOiBpbnQgfCBOb25lID0gTm9uZSwKKSAtPiBJdGVyYXRvcltwZC5EYXRhRnJhbWVdOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIlRTViBmaWxlIG5vdCBmb3VuZDoge3BhdGh9IikKICAgIHJlbWFpbmluZyA9IG1heF9yb3dzCiAgICByZWFkZXIgPSBwZC5yZWFkX2NzdigKICAgICAgICBwYXRoLAogICAgICAgIHNlcD0iXHQiLAogICAgICAgIGR0eXBlPXN0ciwKICAgICAgICBrZWVwX2RlZmF1bHRfbmE9RmFsc2UsCiAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwKICAgICAgICBjaHVua3NpemU9YmF0Y2hfc2l6ZSwKICAgICkKICAgIGZvciBjaHVuayBpbiByZWFkZXI6CiAgICAgICAgcmVxdWlyZV9jb2x1bW5zKGxpc3QoY2h1bmsuY29sdW1ucyksIGV4cGVjdGVkX2NvbHVtbnMsIHN0cihwYXRoKSkKICAgICAgICBpZiByZW1haW5pbmcgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGNodW5rID0gY2h1bmsuaWxvY1s6cmVtYWluaW5nXS5jb3B5KCkKICAgICAgICAgICAgcmVtYWluaW5nIC09IGxlbihjaHVuaykKICAgICAgICBpZiBub3QgY2h1bmsuZW1wdHk6CiAgICAgICAgICAgIHlpZWxkIGNodW5rCiAgICAgICAgaWYgcmVtYWluaW5nIGlzIG5vdCBOb25lIGFuZCByZW1haW5pbmcgPD0gMDoKICAgICAgICAgICAgYnJlYWsKCgpkZWYgdmFsaWRhdGVfZW50aXR5X2ZyYW1lKGZyYW1lOiBwZC5EYXRhRnJhbWUsIHNvdXJjZTogaW50LCBsYWJlbDogc3RyID0gImVudGl0eSBmcmFtZSIpIC0+IE5vbmU6CiAgICByZXF1aXJlX2NvbHVtbnMobGlzdChmcmFtZS5jb2x1bW5zWzo0XSksIFNPVVJDRV9DT0xVTU5TLCBsYWJlbCkKICAgIHByZWZpeCA9IFNPVVJDRV9QUkVGSVhFU1tmInNvdXJjZXtzb3VyY2V9Il0KICAgIGlmIGZyYW1lWyJlbnRpdHlfaWQiXS5lcSgiIikuYW55KCk6CiAgICAgICAgcmFpc2UgU2NoZW1hRXJyb3IoZiJ7bGFiZWx9OiBibGFuayBlbnRpdHlfaWQiKQogICAgaW52YWxpZCA9IH5mcmFtZVsiZW50aXR5X2lkIl0uc3RyLm1hdGNoKHJmIl57cHJlZml4fVxkKyQiKQogICAgaWYgaW52YWxpZC5hbnkoKToKICAgICAgICBleGFtcGxlID0gZnJhbWUubG9jW2ludmFsaWQsICJlbnRpdHlfaWQiXS5pbG9jWzBdCiAgICAgICAgcmFpc2UgU2NoZW1hRXJyb3IoZiJ7bGFiZWx9OiBpbnZhbGlkIHNvdXJjZS17c291cmNlfSBJRCB7ZXhhbXBsZSFyfSIpCiAgICBpZiBmcmFtZVsiZW50aXR5X2lkIl0uZHVwbGljYXRlZCgpLmFueSgpOgogICAgICAgIHJhaXNlIFNjaGVtYUVycm9yKGYie2xhYmVsfTogZHVwbGljYXRlIGVudGl0eV9pZCIpCiAgICBmb3IgY29sdW1uIGluICgiYnVzaW5lc3NfbmFtZSIsICJjb3VudHJ5Iik6CiAgICAgICAgaWYgZnJhbWVbY29sdW1uXS5lcSgiIikuYW55KCk6CiAgICAgICAgICAgIHJhaXNlIFNjaGVtYUVycm9yKGYie2xhYmVsfTogYmxhbmsge2NvbHVtbn0iKQoKCmRlZiBsb2FkX3NvdXJjZShkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgsIHNwbGl0OiBzdHIsIHNvdXJjZTogaW50LCBucm93czogaW50IHwgTm9uZSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgIGZyYW1lID0gcmVhZF90c3Yoc291cmNlX3BhdGgoZGF0YV9yb290LCBzcGxpdCwgc291cmNlKSwgU09VUkNFX0NPTFVNTlMsIG5yb3dzPW5yb3dzKQogICAgdmFsaWRhdGVfZW50aXR5X2ZyYW1lKGZyYW1lLCBzb3VyY2UsIGYie3NwbGl0fSBzb3VyY2V7c291cmNlfSIpCiAgICByZXR1cm4gZnJhbWUKCgpkZWYgbG9hZF9ncm91bmRfdHJ1dGgoZGF0YV9yb290OiBzdHIgfCBQYXRoLCBucm93czogaW50IHwgTm9uZSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgIHJldHVybiByZWFkX3Rzdihncm91bmRfdHJ1dGhfcGF0aChkYXRhX3Jvb3QpLCBHUk9VTkRfVFJVVEhfQ09MVU1OUywgbnJvd3M9bnJvd3MpCgoKZGVmIGxvYWRfZ3JvdW5kX3RydXRoX2Zvcl9pZHMoCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBzb3VyY2UxX2lkczogSXRlcmFibGVbc3RyXSwKICAgIGJhdGNoX3NpemU6IGludCA9IDEwMF8wMDAsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiU3RyZWFtIGdyb3VuZCB0cnV0aCBhbmQgcmV0dXJuIHJvd3MgZm9yIGFuIGV4YWN0IFNvdXJjZSAxIGVudGl0eSBzZXQuIiIiCiAgICB3YW50ZWQgPSB7c3RyKHZhbHVlKSBmb3IgdmFsdWUgaW4gc291cmNlMV9pZHN9CiAgICBpZiBub3Qgd2FudGVkOgogICAgICAgIHJldHVybiBwZC5EYXRhRnJhbWUoY29sdW1ucz1HUk9VTkRfVFJVVEhfQ09MVU1OUykKICAgIGZvdW5kOiBsaXN0W3BkLkRhdGFGcmFtZV0gPSBbXQogICAgZm9yIGNodW5rIGluIGl0ZXJfdHN2KGdyb3VuZF90cnV0aF9wYXRoKGRhdGFfcm9vdCksIEdST1VORF9UUlVUSF9DT0xVTU5TLCBiYXRjaF9zaXplPWJhdGNoX3NpemUpOgogICAgICAgIHNlbGVjdGVkID0gY2h1bmsubG9jW2NodW5rWyJzb3VyY2UxX2VudGl0eV9pZCJdLmlzaW4od2FudGVkKV0KICAgICAgICBpZiBub3Qgc2VsZWN0ZWQuZW1wdHk6CiAgICAgICAgICAgIGZvdW5kLmFwcGVuZChzZWxlY3RlZCkKICAgICAgICAgICAgd2FudGVkLmRpZmZlcmVuY2VfdXBkYXRlKHNlbGVjdGVkWyJzb3VyY2UxX2VudGl0eV9pZCJdLmFzdHlwZShzdHIpKQogICAgICAgIGlmIG5vdCB3YW50ZWQ6CiAgICAgICAgICAgIGJyZWFrCiAgICBpZiB3YW50ZWQ6CiAgICAgICAgZXhhbXBsZXMgPSAiLCAiLmpvaW4oc29ydGVkKHdhbnRlZClbOjVdKQogICAgICAgIHJhaXNlIFNjaGVtYUVycm9yKGYiZ3JvdW5kIHRydXRoIGlzIG1pc3Npbmcge2xlbih3YW50ZWQpfSByZXF1ZXN0ZWQgU291cmNlIDEgSURzOyBleGFtcGxlczoge2V4YW1wbGVzfSIpCiAgICByZXR1cm4gcGQuY29uY2F0KGZvdW5kLCBpZ25vcmVfaW5kZXg9VHJ1ZSkK', '9bf747141ea31df111fc13e1b0df99ba2f5a70f3d70948d2499fa5e1eb6f3394')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/decisions.py`

~~~~python
"""Score-to-match decision rules and threshold optimization."""

from __future__ import annotations

from collections.abc import Mapping, Iterable
import math
import pandas as pd

from .metrics import evaluate_entity_sets


def apply_thresholds(
    scored_pairs: pd.DataFrame,
    all_source1_ids: Iterable[str],
    threshold: float,
    source_thresholds: Mapping[str, float] | None = None,
) -> dict[str, frozenset[str]]:
    source_thresholds = source_thresholds or {}
    predictions: dict[str, set[str]] = {str(entity_id): set() for entity_id in all_source1_ids}
    for row in scored_pairs.itertuples(index=False):
        candidate_id = str(row.candidate_entity_id)
        source = str(getattr(row, "candidate_source", candidate_id[:2]))
        cutoff = float(source_thresholds.get(source, threshold))
        if float(row.score) >= cutoff:
            predictions.setdefault(str(row.source1_entity_id), set()).add(candidate_id)
    return {key: frozenset(value) for key, value in predictions.items()}


def sweep_thresholds(
    scored_pairs: pd.DataFrame,
    truth: Mapping[str, set[str] | frozenset[str]],
    thresholds: Iterable[float],
) -> pd.DataFrame:
    rows = []
    for threshold in sorted(set(float(value) for value in thresholds)):
        predictions = apply_thresholds(scored_pairs, truth.keys(), threshold)
        metrics = evaluate_entity_sets(truth, predictions)
        rows.append({"threshold": threshold, **metrics})
    return pd.DataFrame(rows).sort_values(
        ["macro_f0_5", "macro_precision", "singleton_accuracy", "threshold"],
        ascending=[False, False, False, False],
        kind="mergesort",
    ).reset_index(drop=True)


def score_quantile_thresholds(scored_pairs: pd.DataFrame, count: int = 101) -> list[float]:
    if scored_pairs.empty:
        return [1.0]
    quantiles = [index / (count - 1) for index in range(count)]
    observed = [float(value) for value in scored_pairs["score"].quantile(quantiles)]
    maximum = float(scored_pairs["score"].max())
    # Include the all-empty decision, which cannot be represented by an
    # observed maximum because apply_thresholds uses >=.
    observed.extend([0.0, 1.0, math.nextafter(maximum, math.inf)])
    return sorted(set(observed))


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/decisions.py', 'IiIiU2NvcmUtdG8tbWF0Y2ggZGVjaXNpb24gcnVsZXMgYW5kIHRocmVzaG9sZCBvcHRpbWl6YXRpb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgTWFwcGluZywgSXRlcmFibGUKaW1wb3J0IG1hdGgKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAubWV0cmljcyBpbXBvcnQgZXZhbHVhdGVfZW50aXR5X3NldHMKCgpkZWYgYXBwbHlfdGhyZXNob2xkcygKICAgIHNjb3JlZF9wYWlyczogcGQuRGF0YUZyYW1lLAogICAgYWxsX3NvdXJjZTFfaWRzOiBJdGVyYWJsZVtzdHJdLAogICAgdGhyZXNob2xkOiBmbG9hdCwKICAgIHNvdXJjZV90aHJlc2hvbGRzOiBNYXBwaW5nW3N0ciwgZmxvYXRdIHwgTm9uZSA9IE5vbmUsCikgLT4gZGljdFtzdHIsIGZyb3plbnNldFtzdHJdXToKICAgIHNvdXJjZV90aHJlc2hvbGRzID0gc291cmNlX3RocmVzaG9sZHMgb3Ige30KICAgIHByZWRpY3Rpb25zOiBkaWN0W3N0ciwgc2V0W3N0cl1dID0ge3N0cihlbnRpdHlfaWQpOiBzZXQoKSBmb3IgZW50aXR5X2lkIGluIGFsbF9zb3VyY2UxX2lkc30KICAgIGZvciByb3cgaW4gc2NvcmVkX3BhaXJzLml0ZXJ0dXBsZXMoaW5kZXg9RmFsc2UpOgogICAgICAgIGNhbmRpZGF0ZV9pZCA9IHN0cihyb3cuY2FuZGlkYXRlX2VudGl0eV9pZCkKICAgICAgICBzb3VyY2UgPSBzdHIoZ2V0YXR0cihyb3csICJjYW5kaWRhdGVfc291cmNlIiwgY2FuZGlkYXRlX2lkWzoyXSkpCiAgICAgICAgY3V0b2ZmID0gZmxvYXQoc291cmNlX3RocmVzaG9sZHMuZ2V0KHNvdXJjZSwgdGhyZXNob2xkKSkKICAgICAgICBpZiBmbG9hdChyb3cuc2NvcmUpID49IGN1dG9mZjoKICAgICAgICAgICAgcHJlZGljdGlvbnMuc2V0ZGVmYXVsdChzdHIocm93LnNvdXJjZTFfZW50aXR5X2lkKSwgc2V0KCkpLmFkZChjYW5kaWRhdGVfaWQpCiAgICByZXR1cm4ge2tleTogZnJvemVuc2V0KHZhbHVlKSBmb3Iga2V5LCB2YWx1ZSBpbiBwcmVkaWN0aW9ucy5pdGVtcygpfQoKCmRlZiBzd2VlcF90aHJlc2hvbGRzKAogICAgc2NvcmVkX3BhaXJzOiBwZC5EYXRhRnJhbWUsCiAgICB0cnV0aDogTWFwcGluZ1tzdHIsIHNldFtzdHJdIHwgZnJvemVuc2V0W3N0cl1dLAogICAgdGhyZXNob2xkczogSXRlcmFibGVbZmxvYXRdLAopIC0+IHBkLkRhdGFGcmFtZToKICAgIHJvd3MgPSBbXQogICAgZm9yIHRocmVzaG9sZCBpbiBzb3J0ZWQoc2V0KGZsb2F0KHZhbHVlKSBmb3IgdmFsdWUgaW4gdGhyZXNob2xkcykpOgogICAgICAgIHByZWRpY3Rpb25zID0gYXBwbHlfdGhyZXNob2xkcyhzY29yZWRfcGFpcnMsIHRydXRoLmtleXMoKSwgdGhyZXNob2xkKQogICAgICAgIG1ldHJpY3MgPSBldmFsdWF0ZV9lbnRpdHlfc2V0cyh0cnV0aCwgcHJlZGljdGlvbnMpCiAgICAgICAgcm93cy5hcHBlbmQoeyJ0aHJlc2hvbGQiOiB0aHJlc2hvbGQsICoqbWV0cmljc30pCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHJvd3MpLnNvcnRfdmFsdWVzKAogICAgICAgIFsibWFjcm9fZjBfNSIsICJtYWNyb19wcmVjaXNpb24iLCAic2luZ2xldG9uX2FjY3VyYWN5IiwgInRocmVzaG9sZCJdLAogICAgICAgIGFzY2VuZGluZz1bRmFsc2UsIEZhbHNlLCBGYWxzZSwgRmFsc2VdLAogICAgICAgIGtpbmQ9Im1lcmdlc29ydCIsCiAgICApLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgc2NvcmVfcXVhbnRpbGVfdGhyZXNob2xkcyhzY29yZWRfcGFpcnM6IHBkLkRhdGFGcmFtZSwgY291bnQ6IGludCA9IDEwMSkgLT4gbGlzdFtmbG9hdF06CiAgICBpZiBzY29yZWRfcGFpcnMuZW1wdHk6CiAgICAgICAgcmV0dXJuIFsxLjBdCiAgICBxdWFudGlsZXMgPSBbaW5kZXggLyAoY291bnQgLSAxKSBmb3IgaW5kZXggaW4gcmFuZ2UoY291bnQpXQogICAgb2JzZXJ2ZWQgPSBbZmxvYXQodmFsdWUpIGZvciB2YWx1ZSBpbiBzY29yZWRfcGFpcnNbInNjb3JlIl0ucXVhbnRpbGUocXVhbnRpbGVzKV0KICAgIG1heGltdW0gPSBmbG9hdChzY29yZWRfcGFpcnNbInNjb3JlIl0ubWF4KCkpCiAgICAjIEluY2x1ZGUgdGhlIGFsbC1lbXB0eSBkZWNpc2lvbiwgd2hpY2ggY2Fubm90IGJlIHJlcHJlc2VudGVkIGJ5IGFuCiAgICAjIG9ic2VydmVkIG1heGltdW0gYmVjYXVzZSBhcHBseV90aHJlc2hvbGRzIHVzZXMgPj0uCiAgICBvYnNlcnZlZC5leHRlbmQoWzAuMCwgMS4wLCBtYXRoLm5leHRhZnRlcihtYXhpbXVtLCBtYXRoLmluZildKQogICAgcmV0dXJuIHNvcnRlZChzZXQob2JzZXJ2ZWQpKQoK', 'a77a755384aef20a00e9862fc496c0a5ac61c75638cbc9b02ce0a73b3e6f2675')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/embeddings.py`

~~~~python
"""Optional BGE-M3 embedding features (pair-level cosine only).

The challenge permits only permissively licensed models up to 8B parameters.
``BAAI/bge-m3`` is MIT licensed and ~568M parameters. Embeddings are computed
once per entity (name and address separately), cached under ``artifacts/`` as
fp16 vectors, and then reused to produce pairwise cosine features via batched
gather + dot products. No external data or services are used.

When ``torch``/``transformers``/``FlagEmbedding`` are unavailable, every entry
point returns ``None`` and the pipeline continues without embedding features.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from .artifacts import read_frame
from .checkpoint import ShardStore, input_fingerprint, shard_for_id

DEFAULT_MODEL = "BAAI/bge-m3"
_EMPTY_SENTINEL = "__EMPTY__"


def _import_backend():
    """Return (torch, model_class) or (None, None) when unavailable."""
    try:
        import torch  # type: ignore
    except Exception:
        return None, None
    try:
        from FlagEmbedding import BGEM3FlagModel  # type: ignore

        return torch, BGEM3FlagModel
    except Exception:
        pass
    try:
        from sentence_transformers import SentenceTransformer  # type: ignore

        return torch, SentenceTransformer
    except Exception:
        return None, None


def backend_available() -> bool:
    torch, model_class = _import_backend()
    return torch is not None and model_class is not None


def _embed_dir(config: Any, split: str, source: int) -> Path:
    return config.artifact_dir("embeddings") / f"{split}_source{source}"


def load_embedding_vectors(config: Any, split: str) -> tuple[dict[str, np.ndarray] | None, dict[str, np.ndarray] | None]:
    """Return (name_vectors, address_vectors) keyed by entity_id, or (None, None).

    Vectors for all three sources are loaded from the cached embedding shards.
    Source 1 vectors are required because pair cosine compares an S1 entity
    against an S2/S3 candidate. Returns ``(None, None)`` when the Source 2/3
    caches are not complete, so feature building proceeds without them.
    """
    names: dict[str, np.ndarray] = {}
    addrs: dict[str, np.ndarray] = {}
    # Source 2 and 3 must be complete for the features to be meaningful.
    for source in (2, 3):
        store = ShardStore(_embed_dir(config, split, source), _embedding_fingerprint(config, split, source))
        if not store.is_complete():
            return None, None
    for source in (1, 2, 3):
        store = ShardStore(_embed_dir(config, split, source), _embedding_fingerprint(config, split, source))
        if not store.is_complete():
            continue
        for part in store.iter_parts():
            frame = read_frame(part)
            if frame.empty:
                continue
            name_matrix = np.vstack(frame["vec_name"].to_numpy())
            addr_matrix = np.vstack(frame["vec_addr"].to_numpy())
            for index, entity_id in enumerate(frame["entity_id"].to_numpy()):
                names[str(entity_id)] = name_matrix[index].astype(np.float32)
                addrs[str(entity_id)] = addr_matrix[index].astype(np.float32)
    return names, addrs


def _embedding_fingerprint(config: Any, split: str, source: int) -> str:
    """Fingerprint the prepared shards plus the model name for cache validity."""
    model = str((config.embeddings or {}).get("model", DEFAULT_MODEL))
    prepared = config.artifact_dir("prepared") / f"{split}_source{source}"
    return input_fingerprint([prepared]) + f":model={model}"


def _encode_texts(model: Any, backend: str, texts: list[str], batch_size: int) -> np.ndarray:
    if backend == "bge":
        output = model.encode(texts, batch_size=batch_size, max_length=256)["dense_vecs"]
        return np.asarray(output, dtype=np.float32)
    return np.asarray(model.encode(texts, batch_size=batch_size, show_progress_bar=False), dtype=np.float32)


def _load_model(config: Any, logger: Any):
    torch, model_class = _import_backend()
    if torch is None or model_class is None:
        logger.event("embeddings_unavailable", reason="backend_import_failed")
        return None, None, None
    model_name = str((config.embeddings or {}).get("model", DEFAULT_MODEL))
    plan = config.resource_plan()
    device = "cuda" if plan.device == "cuda" else "cpu"
    if model_class.__name__ == "BGEM3FlagModel":
        model = model_class(model_name, use_fp16=(device == "cuda"))
        backend = "bge"
    else:
        model = model_class(model_name, device=device)
        backend = "st"
    return model, backend, device


def embed_split(config: Any, split: str, logger: Any | None = None) -> dict[str, object]:
    """Compute and cache BGE-M3 name/address vectors for Source 2 and 3.

    Sharded by S1-ID-range style bucketing over target entity IDs and fully
    resumable. Returns a summary dict. When embeddings are disabled or the
    backend is unavailable, this is a no-op.
    """
    from .logging import NullLogger

    logger = logger if logger is not None else NullLogger()

    if not config.embeddings_enabled():
        logger.event("embeddings_skipped", split=split, reason="disabled_or_unavailable")
        return {"enabled": False, "split": split}

    model, backend, device = _load_model(config, logger)
    if model is None:
        return {"enabled": False, "split": split}

    plan = config.resource_plan()
    batch_size = int((config.embeddings or {}).get("batch_size", plan.embed_batch_size))
    total_vectors = 0
    for source in (1, 2, 3):
        store = ShardStore(_embed_dir(config, split, source), _embedding_fingerprint(config, split, source))
        if store.is_complete():
            logger.info("embeddings already complete", split=split, source=source)
            continue
        prepared_dir = config.artifact_dir("prepared") / f"{split}_source{source}"
        prepared_store = ShardStore(prepared_dir, input_fingerprint([prepared_dir]))
        frame = prepared_store.read_all(columns=["entity_id", "business_name", "business_address"])
        n_shards = config.n_shards
        keys = frame["entity_id"].map(lambda value: shard_for_id(value, n_shards))
        buckets = {int(key): group for key, group in frame.groupby(keys, sort=False)}
        pending = store.pending_keys(range(n_shards))
        source_vectors = 0
        for index, key in enumerate(pending, start=1):
            group = buckets.get(int(key), frame.iloc[0:0])
            name_texts = [str(value) if value else _EMPTY_SENTINEL for value in group["business_name"].tolist()]
            addr_texts = [str(value) if value else _EMPTY_SENTINEL for value in group["business_address"].tolist()]

            def _run(chunk_size: int, _name=name_texts, _addr=addr_texts, _group=group) -> pd.DataFrame:
                name_matrix = _encode_texts(model, backend, _name, chunk_size)
                addr_matrix = _encode_texts(model, backend, _addr, chunk_size)
                return pd.DataFrame({
                    "entity_id": _group["entity_id"].to_numpy(),
                    "vec_name": list(name_matrix.astype(np.float16)),
                    "vec_addr": list(addr_matrix.astype(np.float16)),
                })

            from .resources import run_with_oom_backoff

            vectors = run_with_oom_backoff(
                _run,
                max(1, batch_size),
                on_retry=lambda size, exc: logger.event("oom_backoff", stage="embeddings", new_batch=size, error=type(exc).__name__),
            )
            store.commit(int(key), vectors)
            source_vectors += len(vectors)
            logger.progress("embeddings", index, len(pending), extra={"split": split, "source": source, "shard": int(key), "device": device})
        store.mark_complete(source_vectors, ["entity_id", "vec_name", "vec_addr"], extra={"split": split, "source": source, "model": str((config.embeddings or {}).get("model", DEFAULT_MODEL))})
        total_vectors += source_vectors
    return {"enabled": True, "split": split, "vectors": total_vectors, "device": device}

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/embeddings.py', 'IiIiT3B0aW9uYWwgQkdFLU0zIGVtYmVkZGluZyBmZWF0dXJlcyAocGFpci1sZXZlbCBjb3NpbmUgb25seSkuCgpUaGUgY2hhbGxlbmdlIHBlcm1pdHMgb25seSBwZXJtaXNzaXZlbHkgbGljZW5zZWQgbW9kZWxzIHVwIHRvIDhCIHBhcmFtZXRlcnMuCmBgQkFBSS9iZ2UtbTNgYCBpcyBNSVQgbGljZW5zZWQgYW5kIH41NjhNIHBhcmFtZXRlcnMuIEVtYmVkZGluZ3MgYXJlIGNvbXB1dGVkCm9uY2UgcGVyIGVudGl0eSAobmFtZSBhbmQgYWRkcmVzcyBzZXBhcmF0ZWx5KSwgY2FjaGVkIHVuZGVyIGBgYXJ0aWZhY3RzL2BgIGFzCmZwMTYgdmVjdG9ycywgYW5kIHRoZW4gcmV1c2VkIHRvIHByb2R1Y2UgcGFpcndpc2UgY29zaW5lIGZlYXR1cmVzIHZpYSBiYXRjaGVkCmdhdGhlciArIGRvdCBwcm9kdWN0cy4gTm8gZXh0ZXJuYWwgZGF0YSBvciBzZXJ2aWNlcyBhcmUgdXNlZC4KCldoZW4gYGB0b3JjaGBgL2BgdHJhbnNmb3JtZXJzYGAvYGBGbGFnRW1iZWRkaW5nYGAgYXJlIHVuYXZhaWxhYmxlLCBldmVyeSBlbnRyeQpwb2ludCByZXR1cm5zIGBgTm9uZWBgIGFuZCB0aGUgcGlwZWxpbmUgY29udGludWVzIHdpdGhvdXQgZW1iZWRkaW5nIGZlYXR1cmVzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuYXJ0aWZhY3RzIGltcG9ydCByZWFkX2ZyYW1lCmZyb20gLmNoZWNrcG9pbnQgaW1wb3J0IFNoYXJkU3RvcmUsIGlucHV0X2ZpbmdlcnByaW50LCBzaGFyZF9mb3JfaWQKCkRFRkFVTFRfTU9ERUwgPSAiQkFBSS9iZ2UtbTMiCl9FTVBUWV9TRU5USU5FTCA9ICJfX0VNUFRZX18iCgoKZGVmIF9pbXBvcnRfYmFja2VuZCgpOgogICAgIiIiUmV0dXJuICh0b3JjaCwgbW9kZWxfY2xhc3MpIG9yIChOb25lLCBOb25lKSB3aGVuIHVuYXZhaWxhYmxlLiIiIgogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaCAgIyB0eXBlOiBpZ25vcmUKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUKICAgIHRyeToKICAgICAgICBmcm9tIEZsYWdFbWJlZGRpbmcgaW1wb3J0IEJHRU0zRmxhZ01vZGVsICAjIHR5cGU6IGlnbm9yZQoKICAgICAgICByZXR1cm4gdG9yY2gsIEJHRU0zRmxhZ01vZGVsCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICBmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lciAgIyB0eXBlOiBpZ25vcmUKCiAgICAgICAgcmV0dXJuIHRvcmNoLCBTZW50ZW5jZVRyYW5zZm9ybWVyCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBOb25lLCBOb25lCgoKZGVmIGJhY2tlbmRfYXZhaWxhYmxlKCkgLT4gYm9vbDoKICAgIHRvcmNoLCBtb2RlbF9jbGFzcyA9IF9pbXBvcnRfYmFja2VuZCgpCiAgICByZXR1cm4gdG9yY2ggaXMgbm90IE5vbmUgYW5kIG1vZGVsX2NsYXNzIGlzIG5vdCBOb25lCgoKZGVmIF9lbWJlZF9kaXIoY29uZmlnOiBBbnksIHNwbGl0OiBzdHIsIHNvdXJjZTogaW50KSAtPiBQYXRoOgogICAgcmV0dXJuIGNvbmZpZy5hcnRpZmFjdF9kaXIoImVtYmVkZGluZ3MiKSAvIGYie3NwbGl0fV9zb3VyY2V7c291cmNlfSIKCgpkZWYgbG9hZF9lbWJlZGRpbmdfdmVjdG9ycyhjb25maWc6IEFueSwgc3BsaXQ6IHN0cikgLT4gdHVwbGVbZGljdFtzdHIsIG5wLm5kYXJyYXldIHwgTm9uZSwgZGljdFtzdHIsIG5wLm5kYXJyYXldIHwgTm9uZV06CiAgICAiIiJSZXR1cm4gKG5hbWVfdmVjdG9ycywgYWRkcmVzc192ZWN0b3JzKSBrZXllZCBieSBlbnRpdHlfaWQsIG9yIChOb25lLCBOb25lKS4KCiAgICBWZWN0b3JzIGZvciBhbGwgdGhyZWUgc291cmNlcyBhcmUgbG9hZGVkIGZyb20gdGhlIGNhY2hlZCBlbWJlZGRpbmcgc2hhcmRzLgogICAgU291cmNlIDEgdmVjdG9ycyBhcmUgcmVxdWlyZWQgYmVjYXVzZSBwYWlyIGNvc2luZSBjb21wYXJlcyBhbiBTMSBlbnRpdHkKICAgIGFnYWluc3QgYW4gUzIvUzMgY2FuZGlkYXRlLiBSZXR1cm5zIGBgKE5vbmUsIE5vbmUpYGAgd2hlbiB0aGUgU291cmNlIDIvMwogICAgY2FjaGVzIGFyZSBub3QgY29tcGxldGUsIHNvIGZlYXR1cmUgYnVpbGRpbmcgcHJvY2VlZHMgd2l0aG91dCB0aGVtLgogICAgIiIiCiAgICBuYW1lczogZGljdFtzdHIsIG5wLm5kYXJyYXldID0ge30KICAgIGFkZHJzOiBkaWN0W3N0ciwgbnAubmRhcnJheV0gPSB7fQogICAgIyBTb3VyY2UgMiBhbmQgMyBtdXN0IGJlIGNvbXBsZXRlIGZvciB0aGUgZmVhdHVyZXMgdG8gYmUgbWVhbmluZ2Z1bC4KICAgIGZvciBzb3VyY2UgaW4gKDIsIDMpOgogICAgICAgIHN0b3JlID0gU2hhcmRTdG9yZShfZW1iZWRfZGlyKGNvbmZpZywgc3BsaXQsIHNvdXJjZSksIF9lbWJlZGRpbmdfZmluZ2VycHJpbnQoY29uZmlnLCBzcGxpdCwgc291cmNlKSkKICAgICAgICBpZiBub3Qgc3RvcmUuaXNfY29tcGxldGUoKToKICAgICAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUKICAgIGZvciBzb3VyY2UgaW4gKDEsIDIsIDMpOgogICAgICAgIHN0b3JlID0gU2hhcmRTdG9yZShfZW1iZWRfZGlyKGNvbmZpZywgc3BsaXQsIHNvdXJjZSksIF9lbWJlZGRpbmdfZmluZ2VycHJpbnQoY29uZmlnLCBzcGxpdCwgc291cmNlKSkKICAgICAgICBpZiBub3Qgc3RvcmUuaXNfY29tcGxldGUoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3IgcGFydCBpbiBzdG9yZS5pdGVyX3BhcnRzKCk6CiAgICAgICAgICAgIGZyYW1lID0gcmVhZF9mcmFtZShwYXJ0KQogICAgICAgICAgICBpZiBmcmFtZS5lbXB0eToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIG5hbWVfbWF0cml4ID0gbnAudnN0YWNrKGZyYW1lWyJ2ZWNfbmFtZSJdLnRvX251bXB5KCkpCiAgICAgICAgICAgIGFkZHJfbWF0cml4ID0gbnAudnN0YWNrKGZyYW1lWyJ2ZWNfYWRkciJdLnRvX251bXB5KCkpCiAgICAgICAgICAgIGZvciBpbmRleCwgZW50aXR5X2lkIGluIGVudW1lcmF0ZShmcmFtZVsiZW50aXR5X2lkIl0udG9fbnVtcHkoKSk6CiAgICAgICAgICAgICAgICBuYW1lc1tzdHIoZW50aXR5X2lkKV0gPSBuYW1lX21hdHJpeFtpbmRleF0uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgICAgICAgICBhZGRyc1tzdHIoZW50aXR5X2lkKV0gPSBhZGRyX21hdHJpeFtpbmRleF0uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICByZXR1cm4gbmFtZXMsIGFkZHJzCgoKZGVmIF9lbWJlZGRpbmdfZmluZ2VycHJpbnQoY29uZmlnOiBBbnksIHNwbGl0OiBzdHIsIHNvdXJjZTogaW50KSAtPiBzdHI6CiAgICAiIiJGaW5nZXJwcmludCB0aGUgcHJlcGFyZWQgc2hhcmRzIHBsdXMgdGhlIG1vZGVsIG5hbWUgZm9yIGNhY2hlIHZhbGlkaXR5LiIiIgogICAgbW9kZWwgPSBzdHIoKGNvbmZpZy5lbWJlZGRpbmdzIG9yIHt9KS5nZXQoIm1vZGVsIiwgREVGQVVMVF9NT0RFTCkpCiAgICBwcmVwYXJlZCA9IGNvbmZpZy5hcnRpZmFjdF9kaXIoInByZXBhcmVkIikgLyBmIntzcGxpdH1fc291cmNle3NvdXJjZX0iCiAgICByZXR1cm4gaW5wdXRfZmluZ2VycHJpbnQoW3ByZXBhcmVkXSkgKyBmIjptb2RlbD17bW9kZWx9IgoKCmRlZiBfZW5jb2RlX3RleHRzKG1vZGVsOiBBbnksIGJhY2tlbmQ6IHN0ciwgdGV4dHM6IGxpc3Rbc3RyXSwgYmF0Y2hfc2l6ZTogaW50KSAtPiBucC5uZGFycmF5OgogICAgaWYgYmFja2VuZCA9PSAiYmdlIjoKICAgICAgICBvdXRwdXQgPSBtb2RlbC5lbmNvZGUodGV4dHMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgbWF4X2xlbmd0aD0yNTYpWyJkZW5zZV92ZWNzIl0KICAgICAgICByZXR1cm4gbnAuYXNhcnJheShvdXRwdXQsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICByZXR1cm4gbnAuYXNhcnJheShtb2RlbC5lbmNvZGUodGV4dHMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgc2hvd19wcm9ncmVzc19iYXI9RmFsc2UpLCBkdHlwZT1ucC5mbG9hdDMyKQoKCmRlZiBfbG9hZF9tb2RlbChjb25maWc6IEFueSwgbG9nZ2VyOiBBbnkpOgogICAgdG9yY2gsIG1vZGVsX2NsYXNzID0gX2ltcG9ydF9iYWNrZW5kKCkKICAgIGlmIHRvcmNoIGlzIE5vbmUgb3IgbW9kZWxfY2xhc3MgaXMgTm9uZToKICAgICAgICBsb2dnZXIuZXZlbnQoImVtYmVkZGluZ3NfdW5hdmFpbGFibGUiLCByZWFzb249ImJhY2tlbmRfaW1wb3J0X2ZhaWxlZCIpCiAgICAgICAgcmV0dXJuIE5vbmUsIE5vbmUsIE5vbmUKICAgIG1vZGVsX25hbWUgPSBzdHIoKGNvbmZpZy5lbWJlZGRpbmdzIG9yIHt9KS5nZXQoIm1vZGVsIiwgREVGQVVMVF9NT0RFTCkpCiAgICBwbGFuID0gY29uZmlnLnJlc291cmNlX3BsYW4oKQogICAgZGV2aWNlID0gImN1ZGEiIGlmIHBsYW4uZGV2aWNlID09ICJjdWRhIiBlbHNlICJjcHUiCiAgICBpZiBtb2RlbF9jbGFzcy5fX25hbWVfXyA9PSAiQkdFTTNGbGFnTW9kZWwiOgogICAgICAgIG1vZGVsID0gbW9kZWxfY2xhc3MobW9kZWxfbmFtZSwgdXNlX2ZwMTY9KGRldmljZSA9PSAiY3VkYSIpKQogICAgICAgIGJhY2tlbmQgPSAiYmdlIgogICAgZWxzZToKICAgICAgICBtb2RlbCA9IG1vZGVsX2NsYXNzKG1vZGVsX25hbWUsIGRldmljZT1kZXZpY2UpCiAgICAgICAgYmFja2VuZCA9ICJzdCIKICAgIHJldHVybiBtb2RlbCwgYmFja2VuZCwgZGV2aWNlCgoKZGVmIGVtYmVkX3NwbGl0KGNvbmZpZzogQW55LCBzcGxpdDogc3RyLCBsb2dnZXI6IEFueSB8IE5vbmUgPSBOb25lKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgICIiIkNvbXB1dGUgYW5kIGNhY2hlIEJHRS1NMyBuYW1lL2FkZHJlc3MgdmVjdG9ycyBmb3IgU291cmNlIDIgYW5kIDMuCgogICAgU2hhcmRlZCBieSBTMS1JRC1yYW5nZSBzdHlsZSBidWNrZXRpbmcgb3ZlciB0YXJnZXQgZW50aXR5IElEcyBhbmQgZnVsbHkKICAgIHJlc3VtYWJsZS4gUmV0dXJucyBhIHN1bW1hcnkgZGljdC4gV2hlbiBlbWJlZGRpbmdzIGFyZSBkaXNhYmxlZCBvciB0aGUKICAgIGJhY2tlbmQgaXMgdW5hdmFpbGFibGUsIHRoaXMgaXMgYSBuby1vcC4KICAgICIiIgogICAgZnJvbSAubG9nZ2luZyBpbXBvcnQgTnVsbExvZ2dlcgoKICAgIGxvZ2dlciA9IGxvZ2dlciBpZiBsb2dnZXIgaXMgbm90IE5vbmUgZWxzZSBOdWxsTG9nZ2VyKCkKCiAgICBpZiBub3QgY29uZmlnLmVtYmVkZGluZ3NfZW5hYmxlZCgpOgogICAgICAgIGxvZ2dlci5ldmVudCgiZW1iZWRkaW5nc19za2lwcGVkIiwgc3BsaXQ9c3BsaXQsIHJlYXNvbj0iZGlzYWJsZWRfb3JfdW5hdmFpbGFibGUiKQogICAgICAgIHJldHVybiB7ImVuYWJsZWQiOiBGYWxzZSwgInNwbGl0Ijogc3BsaXR9CgogICAgbW9kZWwsIGJhY2tlbmQsIGRldmljZSA9IF9sb2FkX21vZGVsKGNvbmZpZywgbG9nZ2VyKQogICAgaWYgbW9kZWwgaXMgTm9uZToKICAgICAgICByZXR1cm4geyJlbmFibGVkIjogRmFsc2UsICJzcGxpdCI6IHNwbGl0fQoKICAgIHBsYW4gPSBjb25maWcucmVzb3VyY2VfcGxhbigpCiAgICBiYXRjaF9zaXplID0gaW50KChjb25maWcuZW1iZWRkaW5ncyBvciB7fSkuZ2V0KCJiYXRjaF9zaXplIiwgcGxhbi5lbWJlZF9iYXRjaF9zaXplKSkKICAgIHRvdGFsX3ZlY3RvcnMgPSAwCiAgICBmb3Igc291cmNlIGluICgxLCAyLCAzKToKICAgICAgICBzdG9yZSA9IFNoYXJkU3RvcmUoX2VtYmVkX2Rpcihjb25maWcsIHNwbGl0LCBzb3VyY2UpLCBfZW1iZWRkaW5nX2ZpbmdlcnByaW50KGNvbmZpZywgc3BsaXQsIHNvdXJjZSkpCiAgICAgICAgaWYgc3RvcmUuaXNfY29tcGxldGUoKToKICAgICAgICAgICAgbG9nZ2VyLmluZm8oImVtYmVkZGluZ3MgYWxyZWFkeSBjb21wbGV0ZSIsIHNwbGl0PXNwbGl0LCBzb3VyY2U9c291cmNlKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHByZXBhcmVkX2RpciA9IGNvbmZpZy5hcnRpZmFjdF9kaXIoInByZXBhcmVkIikgLyBmIntzcGxpdH1fc291cmNle3NvdXJjZX0iCiAgICAgICAgcHJlcGFyZWRfc3RvcmUgPSBTaGFyZFN0b3JlKHByZXBhcmVkX2RpciwgaW5wdXRfZmluZ2VycHJpbnQoW3ByZXBhcmVkX2Rpcl0pKQogICAgICAgIGZyYW1lID0gcHJlcGFyZWRfc3RvcmUucmVhZF9hbGwoY29sdW1ucz1bImVudGl0eV9pZCIsICJidXNpbmVzc19uYW1lIiwgImJ1c2luZXNzX2FkZHJlc3MiXSkKICAgICAgICBuX3NoYXJkcyA9IGNvbmZpZy5uX3NoYXJkcwogICAgICAgIGtleXMgPSBmcmFtZVsiZW50aXR5X2lkIl0ubWFwKGxhbWJkYSB2YWx1ZTogc2hhcmRfZm9yX2lkKHZhbHVlLCBuX3NoYXJkcykpCiAgICAgICAgYnVja2V0cyA9IHtpbnQoa2V5KTogZ3JvdXAgZm9yIGtleSwgZ3JvdXAgaW4gZnJhbWUuZ3JvdXBieShrZXlzLCBzb3J0PUZhbHNlKX0KICAgICAgICBwZW5kaW5nID0gc3RvcmUucGVuZGluZ19rZXlzKHJhbmdlKG5fc2hhcmRzKSkKICAgICAgICBzb3VyY2VfdmVjdG9ycyA9IDAKICAgICAgICBmb3IgaW5kZXgsIGtleSBpbiBlbnVtZXJhdGUocGVuZGluZywgc3RhcnQ9MSk6CiAgICAgICAgICAgIGdyb3VwID0gYnVja2V0cy5nZXQoaW50KGtleSksIGZyYW1lLmlsb2NbMDowXSkKICAgICAgICAgICAgbmFtZV90ZXh0cyA9IFtzdHIodmFsdWUpIGlmIHZhbHVlIGVsc2UgX0VNUFRZX1NFTlRJTkVMIGZvciB2YWx1ZSBpbiBncm91cFsiYnVzaW5lc3NfbmFtZSJdLnRvbGlzdCgpXQogICAgICAgICAgICBhZGRyX3RleHRzID0gW3N0cih2YWx1ZSkgaWYgdmFsdWUgZWxzZSBfRU1QVFlfU0VOVElORUwgZm9yIHZhbHVlIGluIGdyb3VwWyJidXNpbmVzc19hZGRyZXNzIl0udG9saXN0KCldCgogICAgICAgICAgICBkZWYgX3J1bihjaHVua19zaXplOiBpbnQsIF9uYW1lPW5hbWVfdGV4dHMsIF9hZGRyPWFkZHJfdGV4dHMsIF9ncm91cD1ncm91cCkgLT4gcGQuRGF0YUZyYW1lOgogICAgICAgICAgICAgICAgbmFtZV9tYXRyaXggPSBfZW5jb2RlX3RleHRzKG1vZGVsLCBiYWNrZW5kLCBfbmFtZSwgY2h1bmtfc2l6ZSkKICAgICAgICAgICAgICAgIGFkZHJfbWF0cml4ID0gX2VuY29kZV90ZXh0cyhtb2RlbCwgYmFja2VuZCwgX2FkZHIsIGNodW5rX3NpemUpCiAgICAgICAgICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKHsKICAgICAgICAgICAgICAgICAgICAiZW50aXR5X2lkIjogX2dyb3VwWyJlbnRpdHlfaWQiXS50b19udW1weSgpLAogICAgICAgICAgICAgICAgICAgICJ2ZWNfbmFtZSI6IGxpc3QobmFtZV9tYXRyaXguYXN0eXBlKG5wLmZsb2F0MTYpKSwKICAgICAgICAgICAgICAgICAgICAidmVjX2FkZHIiOiBsaXN0KGFkZHJfbWF0cml4LmFzdHlwZShucC5mbG9hdDE2KSksCiAgICAgICAgICAgICAgICB9KQoKICAgICAgICAgICAgZnJvbSAucmVzb3VyY2VzIGltcG9ydCBydW5fd2l0aF9vb21fYmFja29mZgoKICAgICAgICAgICAgdmVjdG9ycyA9IHJ1bl93aXRoX29vbV9iYWNrb2ZmKAogICAgICAgICAgICAgICAgX3J1biwKICAgICAgICAgICAgICAgIG1heCgxLCBiYXRjaF9zaXplKSwKICAgICAgICAgICAgICAgIG9uX3JldHJ5PWxhbWJkYSBzaXplLCBleGM6IGxvZ2dlci5ldmVudCgib29tX2JhY2tvZmYiLCBzdGFnZT0iZW1iZWRkaW5ncyIsIG5ld19iYXRjaD1zaXplLCBlcnJvcj10eXBlKGV4YykuX19uYW1lX18pLAogICAgICAgICAgICApCiAgICAgICAgICAgIHN0b3JlLmNvbW1pdChpbnQoa2V5KSwgdmVjdG9ycykKICAgICAgICAgICAgc291cmNlX3ZlY3RvcnMgKz0gbGVuKHZlY3RvcnMpCiAgICAgICAgICAgIGxvZ2dlci5wcm9ncmVzcygiZW1iZWRkaW5ncyIsIGluZGV4LCBsZW4ocGVuZGluZyksIGV4dHJhPXsic3BsaXQiOiBzcGxpdCwgInNvdXJjZSI6IHNvdXJjZSwgInNoYXJkIjogaW50KGtleSksICJkZXZpY2UiOiBkZXZpY2V9KQogICAgICAgIHN0b3JlLm1hcmtfY29tcGxldGUoc291cmNlX3ZlY3RvcnMsIFsiZW50aXR5X2lkIiwgInZlY19uYW1lIiwgInZlY19hZGRyIl0sIGV4dHJhPXsic3BsaXQiOiBzcGxpdCwgInNvdXJjZSI6IHNvdXJjZSwgIm1vZGVsIjogc3RyKChjb25maWcuZW1iZWRkaW5ncyBvciB7fSkuZ2V0KCJtb2RlbCIsIERFRkFVTFRfTU9ERUwpKX0pCiAgICAgICAgdG90YWxfdmVjdG9ycyArPSBzb3VyY2VfdmVjdG9ycwogICAgcmV0dXJuIHsiZW5hYmxlZCI6IFRydWUsICJzcGxpdCI6IHNwbGl0LCAidmVjdG9ycyI6IHRvdGFsX3ZlY3RvcnMsICJkZXZpY2UiOiBkZXZpY2V9Cg==', '2fac3c61edb6f97a700109c44464bb26f88f2faba787042c567f276765338e5d')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/error_analysis.py`

~~~~python
"""Stage-aware validation error analysis."""

from __future__ import annotations

from collections.abc import Mapping
import pandas as pd


def analyze_errors(
    truth: Mapping[str, set[str] | frozenset[str]],
    candidate_sets: Mapping[str, set[str] | frozenset[str]],
    predictions: Mapping[str, set[str] | frozenset[str]],
    scored_pairs: pd.DataFrame | None = None,
) -> pd.DataFrame:
    score_lookup: dict[tuple[str, str], float] = {}
    if scored_pairs is not None and not scored_pairs.empty:
        score_lookup = {
            (str(row.source1_entity_id), str(row.candidate_entity_id)): float(row.score)
            for row in scored_pairs.itertuples(index=False)
        }
    rows: list[dict[str, object]] = []
    for source1_id, targets in truth.items():
        target_set = set(targets)
        candidates = set(candidate_sets.get(source1_id, frozenset()))
        predicted = set(predictions.get(source1_id, frozenset()))
        for target_id in sorted(target_set - candidates):
            rows.append({"source1_entity_id": source1_id, "candidate_entity_id": target_id, "stage": "blocking", "error_type": "missed_true_pair", "score": None})
        for target_id in sorted((target_set & candidates) - predicted):
            rows.append({"source1_entity_id": source1_id, "candidate_entity_id": target_id, "stage": "model_or_decision", "error_type": "false_negative", "score": score_lookup.get((source1_id, target_id))})
        for target_id in sorted(predicted - target_set):
            rows.append({"source1_entity_id": source1_id, "candidate_entity_id": target_id, "stage": "model_or_decision", "error_type": "singleton_false_positive" if not target_set else "false_merge", "score": score_lookup.get((source1_id, target_id))})
    return pd.DataFrame(rows, columns=["source1_entity_id", "candidate_entity_id", "stage", "error_type", "score"])


def high_confidence_pairs(scored_labeled: pd.DataFrame, correct: bool, limit: int = 100) -> pd.DataFrame:
    required = {"label", "score"}
    if not required.issubset(scored_labeled.columns):
        raise ValueError("scored labeled pairs require label and score")
    subset = scored_labeled[scored_labeled["label"].eq(1 if correct else 0)]
    return subset.nlargest(limit, "score")


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/error_analysis.py', 'IiIiU3RhZ2UtYXdhcmUgdmFsaWRhdGlvbiBlcnJvciBhbmFseXNpcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBNYXBwaW5nCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpkZWYgYW5hbHl6ZV9lcnJvcnMoCiAgICB0cnV0aDogTWFwcGluZ1tzdHIsIHNldFtzdHJdIHwgZnJvemVuc2V0W3N0cl1dLAogICAgY2FuZGlkYXRlX3NldHM6IE1hcHBpbmdbc3RyLCBzZXRbc3RyXSB8IGZyb3plbnNldFtzdHJdXSwKICAgIHByZWRpY3Rpb25zOiBNYXBwaW5nW3N0ciwgc2V0W3N0cl0gfCBmcm96ZW5zZXRbc3RyXV0sCiAgICBzY29yZWRfcGFpcnM6IHBkLkRhdGFGcmFtZSB8IE5vbmUgPSBOb25lLAopIC0+IHBkLkRhdGFGcmFtZToKICAgIHNjb3JlX2xvb2t1cDogZGljdFt0dXBsZVtzdHIsIHN0cl0sIGZsb2F0XSA9IHt9CiAgICBpZiBzY29yZWRfcGFpcnMgaXMgbm90IE5vbmUgYW5kIG5vdCBzY29yZWRfcGFpcnMuZW1wdHk6CiAgICAgICAgc2NvcmVfbG9va3VwID0gewogICAgICAgICAgICAoc3RyKHJvdy5zb3VyY2UxX2VudGl0eV9pZCksIHN0cihyb3cuY2FuZGlkYXRlX2VudGl0eV9pZCkpOiBmbG9hdChyb3cuc2NvcmUpCiAgICAgICAgICAgIGZvciByb3cgaW4gc2NvcmVkX3BhaXJzLml0ZXJ0dXBsZXMoaW5kZXg9RmFsc2UpCiAgICAgICAgfQogICAgcm93czogbGlzdFtkaWN0W3N0ciwgb2JqZWN0XV0gPSBbXQogICAgZm9yIHNvdXJjZTFfaWQsIHRhcmdldHMgaW4gdHJ1dGguaXRlbXMoKToKICAgICAgICB0YXJnZXRfc2V0ID0gc2V0KHRhcmdldHMpCiAgICAgICAgY2FuZGlkYXRlcyA9IHNldChjYW5kaWRhdGVfc2V0cy5nZXQoc291cmNlMV9pZCwgZnJvemVuc2V0KCkpKQogICAgICAgIHByZWRpY3RlZCA9IHNldChwcmVkaWN0aW9ucy5nZXQoc291cmNlMV9pZCwgZnJvemVuc2V0KCkpKQogICAgICAgIGZvciB0YXJnZXRfaWQgaW4gc29ydGVkKHRhcmdldF9zZXQgLSBjYW5kaWRhdGVzKToKICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJzb3VyY2UxX2VudGl0eV9pZCI6IHNvdXJjZTFfaWQsICJjYW5kaWRhdGVfZW50aXR5X2lkIjogdGFyZ2V0X2lkLCAic3RhZ2UiOiAiYmxvY2tpbmciLCAiZXJyb3JfdHlwZSI6ICJtaXNzZWRfdHJ1ZV9wYWlyIiwgInNjb3JlIjogTm9uZX0pCiAgICAgICAgZm9yIHRhcmdldF9pZCBpbiBzb3J0ZWQoKHRhcmdldF9zZXQgJiBjYW5kaWRhdGVzKSAtIHByZWRpY3RlZCk6CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsic291cmNlMV9lbnRpdHlfaWQiOiBzb3VyY2UxX2lkLCAiY2FuZGlkYXRlX2VudGl0eV9pZCI6IHRhcmdldF9pZCwgInN0YWdlIjogIm1vZGVsX29yX2RlY2lzaW9uIiwgImVycm9yX3R5cGUiOiAiZmFsc2VfbmVnYXRpdmUiLCAic2NvcmUiOiBzY29yZV9sb29rdXAuZ2V0KChzb3VyY2UxX2lkLCB0YXJnZXRfaWQpKX0pCiAgICAgICAgZm9yIHRhcmdldF9pZCBpbiBzb3J0ZWQocHJlZGljdGVkIC0gdGFyZ2V0X3NldCk6CiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsic291cmNlMV9lbnRpdHlfaWQiOiBzb3VyY2UxX2lkLCAiY2FuZGlkYXRlX2VudGl0eV9pZCI6IHRhcmdldF9pZCwgInN0YWdlIjogIm1vZGVsX29yX2RlY2lzaW9uIiwgImVycm9yX3R5cGUiOiAic2luZ2xldG9uX2ZhbHNlX3Bvc2l0aXZlIiBpZiBub3QgdGFyZ2V0X3NldCBlbHNlICJmYWxzZV9tZXJnZSIsICJzY29yZSI6IHNjb3JlX2xvb2t1cC5nZXQoKHNvdXJjZTFfaWQsIHRhcmdldF9pZCkpfSkKICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cywgY29sdW1ucz1bInNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiLCAic3RhZ2UiLCAiZXJyb3JfdHlwZSIsICJzY29yZSJdKQoKCmRlZiBoaWdoX2NvbmZpZGVuY2VfcGFpcnMoc2NvcmVkX2xhYmVsZWQ6IHBkLkRhdGFGcmFtZSwgY29ycmVjdDogYm9vbCwgbGltaXQ6IGludCA9IDEwMCkgLT4gcGQuRGF0YUZyYW1lOgogICAgcmVxdWlyZWQgPSB7ImxhYmVsIiwgInNjb3JlIn0KICAgIGlmIG5vdCByZXF1aXJlZC5pc3N1YnNldChzY29yZWRfbGFiZWxlZC5jb2x1bW5zKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzY29yZWQgbGFiZWxlZCBwYWlycyByZXF1aXJlIGxhYmVsIGFuZCBzY29yZSIpCiAgICBzdWJzZXQgPSBzY29yZWRfbGFiZWxlZFtzY29yZWRfbGFiZWxlZFsibGFiZWwiXS5lcSgxIGlmIGNvcnJlY3QgZWxzZSAwKV0KICAgIHJldHVybiBzdWJzZXQubmxhcmdlc3QobGltaXQsICJzY29yZSIpCgo=', 'c6bde6f001882b3f04e2a02705f440fb48de02196aa2318f6aeb29dcf24dffbc')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/features.py`

~~~~python
"""Memory-conscious pairwise feature construction over candidate pairs."""

from __future__ import annotations

import math
import numpy as np
import pandas as pd
from rapidfuzz import fuzz


def _set(value: object) -> set[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return set()
    if isinstance(value, str):
        return set(value.split()) if value else set()
    return set(value)


def _jaccard(left: object, right: object) -> float:
    a, b = _set(left), _set(right)
    # Missing evidence is not positive evidence. Empty/empty used to score as
    # a perfect match, which was particularly harmful for blank addresses.
    return 0.0 if not a or not b else len(a & b) / len(a | b)


def _containment(left: object, right: object) -> float:
    a, b = _set(left), _set(right)
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))


def _ratio(left: str, right: str) -> float:
    return 0.0 if not left or not right else fuzz.ratio(left, right) / 100.0


def _partial(left: str, right: str) -> float:
    return 0.0 if not left or not right else fuzz.partial_ratio(left, right) / 100.0


FEATURE_COLUMNS = [
    "name_raw_exact", "name_canonical_exact", "name_compact_exact", "name_core_exact",
    "name_token_sorted_exact", "name_accent_folded_exact", "name_accent_folded_ratio",
    "name_ratio", "name_partial", "name_token_sort_ratio",
    "name_token_set_ratio", "name_token_jaccard", "name_token_containment", "name_length_ratio",
    "name_jaro_winkler",
    "address_canonical_exact", "address_compact_exact", "address_ratio", "address_partial",
    "address_token_jaccard", "address_token_containment", "address_length_ratio",
    "numeric_exact", "numeric_jaccard", "numeric_overlap", "numeric_conflicts",
    "postal_exact", "postal_jaccard", "country_equal", "candidate_is_s3",
    "s1_missing_address", "candidate_missing_address", "blocking_reason_count",
    "retrieval_score", "retrieval_rank_inverse", "name_address_interaction",
    "name_numeric_interaction",
    "bge_name_cosine", "bge_addr_cosine",
]

# Features that only exist when an embedding backend and cache are available.
EMBEDDING_COLUMNS = ["bge_name_cosine", "bge_addr_cosine"]


def _length_ratio(left: str, right: str) -> float:
    a, b = len(left or ""), len(right or "")
    return 0.0 if not a or not b else min(a, b) / max(a, b)


def _jaro_winkler(left: str, right: str) -> float:
    a, b = left or "", right or ""
    if a == b:
        return 1.0
    if not a or not b:
        return 0.0
    match_distance = max(len(a), len(b)) // 2 - 1
    a_matches = [False] * len(a)
    b_matches = [False] * len(b)
    matches = 0
    for i, char in enumerate(a):
        start, end = max(0, i - match_distance), min(i + match_distance + 1, len(b))
        for j in range(start, end):
            if b_matches[j] or b[j] != char:
                continue
            a_matches[i] = b_matches[j] = True
            matches += 1
            break
    if matches == 0:
        return 0.0
    transpositions = _transpositions(a, b, a_matches, b_matches)
    jaro = (matches / len(a) + matches / len(b) + (matches - transpositions) / matches) / 3.0
    prefix = 0
    for x, y in zip(a, b):
        if x != y or prefix == 4:
            break
        prefix += 1
    return jaro + prefix * 0.1 * (1 - jaro)


def _transpositions(a: str, b: str, a_matches: list[bool], b_matches: list[bool]) -> int:
    a_chars = [a[i] for i in range(len(a)) if a_matches[i]]
    b_chars = [b[j] for j in range(len(b)) if b_matches[j]]
    return sum(1 for x, y in zip(a_chars, b_chars) if x != y) // 2


def build_pair_features(
    candidates: pd.DataFrame,
    source1: pd.DataFrame,
    targets: pd.DataFrame,
    *,
    embed_names: dict[str, np.ndarray] | None = None,
    embed_addrs: dict[str, np.ndarray] | None = None,
) -> pd.DataFrame:
    required = {"source1_entity_id", "candidate_entity_id"}
    if not required.issubset(candidates.columns):
        raise ValueError(f"candidate table missing columns: {sorted(required - set(candidates.columns))}")
    if candidates.empty:
        columns = ["source1_entity_id", "candidate_entity_id", "candidate_source", *FEATURE_COLUMNS]
        for passthrough in ("label", "sample_weight", "forced_positive"):
            if passthrough in candidates:
                columns.append(passthrough)
        return pd.DataFrame(columns=columns)
    s1 = source1.add_prefix("s1_").rename(columns={"s1_entity_id": "source1_entity_id"})
    target = targets.add_prefix("c_").rename(columns={"c_entity_id": "candidate_entity_id"})
    pairs = candidates.merge(s1, on="source1_entity_id", how="left", validate="many_to_one")
    pairs = pairs.merge(target, on="candidate_entity_id", how="left", validate="many_to_one")
    if pairs["s1_name_canonical"].isna().any() or pairs["c_name_canonical"].isna().any():
        raise ValueError("candidate table references an unknown Source 1 or target ID")

    # Precompute embedding cosines vectorized (batched gather + dot) to avoid
    # per-row Python overhead; falls back to zeros when vectors are unavailable.
    bge_name = np.zeros(len(pairs), dtype=np.float32)
    bge_addr = np.zeros(len(pairs), dtype=np.float32)
    if embed_names is not None and embed_addrs is not None:
        bge_name = _pair_cosine(pairs["source1_entity_id"], pairs["candidate_entity_id"], embed_names)
        bge_addr = _pair_cosine(pairs["source1_entity_id"], pairs["candidate_entity_id"], embed_addrs)

    records: list[dict[str, object]] = []
    for position, row in enumerate(pairs.itertuples(index=False)):
        s1_name, c_name = row.s1_name_canonical, row.c_name_canonical
        s1_addr, c_addr = row.s1_address_canonical, row.c_address_canonical
        numeric_a, numeric_b = _set(row.s1_numeric_tokens), _set(row.c_numeric_tokens)
        name_ratio, address_ratio = _ratio(s1_name, c_name), _ratio(s1_addr, c_addr)
        reason_mask = str(getattr(row, "reason_mask", ""))
        records.append({
            "source1_entity_id": row.source1_entity_id,
            "candidate_entity_id": row.candidate_entity_id,
            "candidate_source": getattr(row, "candidate_source", row.candidate_entity_id[:2]),
            "name_raw_exact": float(row.s1_business_name_raw == row.c_business_name_raw),
            "name_canonical_exact": float(s1_name == c_name),
            "name_compact_exact": float(row.s1_name_compact == row.c_name_compact),
            "name_core_exact": float(row.s1_name_core == row.c_name_core),
            "name_token_sorted_exact": float(row.s1_name_token_sorted == row.c_name_token_sorted),
            "name_accent_folded_exact": float(
                bool(row.s1_name_accent_folded)
                and row.s1_name_accent_folded == row.c_name_accent_folded
            ),
            "name_accent_folded_ratio": _ratio(row.s1_name_accent_folded, row.c_name_accent_folded),
            "name_ratio": name_ratio,
            "name_partial": _partial(s1_name, c_name),
            "name_token_sort_ratio": fuzz.token_sort_ratio(s1_name, c_name) / 100.0,
            "name_token_set_ratio": fuzz.token_set_ratio(s1_name, c_name) / 100.0,
            "name_token_jaccard": _jaccard(row.s1_name_tokens, row.c_name_tokens),
            "name_token_containment": _containment(row.s1_name_tokens, row.c_name_tokens),
            "name_length_ratio": _length_ratio(s1_name, c_name),
            "name_jaro_winkler": _jaro_winkler(s1_name, c_name),
            "address_canonical_exact": float(s1_addr == c_addr and bool(s1_addr)),
            "address_compact_exact": float(row.s1_address_compact == row.c_address_compact and bool(row.s1_address_compact)),
            "address_ratio": address_ratio,
            "address_partial": _partial(s1_addr, c_addr),
            "address_token_jaccard": _jaccard(row.s1_address_tokens, row.c_address_tokens),
            "address_token_containment": _containment(row.s1_address_tokens, row.c_address_tokens),
            "address_length_ratio": _length_ratio(s1_addr, c_addr),
            "numeric_exact": float(numeric_a == numeric_b and bool(numeric_a)),
            "numeric_jaccard": _jaccard(numeric_a, numeric_b),
            "numeric_overlap": float(bool(numeric_a & numeric_b)),
            "numeric_conflicts": float(len(numeric_a ^ numeric_b)),
            "postal_exact": float(_set(row.s1_postal_like_tokens) == _set(row.c_postal_like_tokens) and bool(_set(row.s1_postal_like_tokens))),
            "postal_jaccard": _jaccard(row.s1_postal_like_tokens, row.c_postal_like_tokens),
            "country_equal": float(row.s1_country_norm == row.c_country_norm),
            "candidate_is_s3": float(str(row.candidate_entity_id).startswith("S3-")),
            "s1_missing_address": float(row.s1_missing_address),
            "candidate_missing_address": float(row.c_missing_address),
            "blocking_reason_count": float(len([part for part in reason_mask.split("|") if part])),
            "retrieval_score": float(getattr(row, "retrieval_score", 0.0)),
            "retrieval_rank_inverse": 1.0 / max(1, int(getattr(row, "retrieval_rank", 1))),
            "name_address_interaction": name_ratio * address_ratio,
            "name_numeric_interaction": name_ratio * float(bool(numeric_a & numeric_b)),
            "bge_name_cosine": float(bge_name[position]),
            "bge_addr_cosine": float(bge_addr[position]),
        })
    out = pd.DataFrame.from_records(records)
    for column in FEATURE_COLUMNS:
        out[column] = out[column].astype(np.float32)
    for passthrough in ("label", "sample_weight", "forced_positive"):
        if passthrough in candidates:
            out[passthrough] = candidates[passthrough].to_numpy()
    return out


def _pair_cosine(s1_ids: pd.Series, c_ids: pd.Series, vectors: dict[str, np.ndarray]) -> np.ndarray:
    """Vectorized cosine between aligned S1 and candidate embedding vectors."""
    if not vectors:
        return np.zeros(len(s1_ids), dtype=np.float32)
    dim = len(next(iter(vectors.values())))
    left_ids = s1_ids.astype(str).to_numpy()
    right_ids = c_ids.astype(str).to_numpy()
    left = np.zeros((len(left_ids), dim), dtype=np.float32)
    right = np.zeros((len(right_ids), dim), dtype=np.float32)
    for index, entity_id in enumerate(left_ids):
        vector = vectors.get(entity_id)
        if vector is not None:
            left[index] = vector
    for index, entity_id in enumerate(right_ids):
        vector = vectors.get(entity_id)
        if vector is not None:
            right[index] = vector
    left_norm = left / (np.linalg.norm(left, axis=1, keepdims=True) + 1e-9)
    right_norm = right / (np.linalg.norm(right, axis=1, keepdims=True) + 1e-9)
    return np.einsum("ij,ij->i", left_norm, right_norm).astype(np.float32)


def feature_matrix(frame: pd.DataFrame) -> np.ndarray:
    missing = set(FEATURE_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f"feature table missing columns: {sorted(missing)}")
    return frame[FEATURE_COLUMNS].to_numpy(dtype=np.float32, copy=False)


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/features.py', 'IiIiTWVtb3J5LWNvbnNjaW91cyBwYWlyd2lzZSBmZWF0dXJlIGNvbnN0cnVjdGlvbiBvdmVyIGNhbmRpZGF0ZSBwYWlycy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBtYXRoCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gcmFwaWRmdXp6IGltcG9ydCBmdXp6CgoKZGVmIF9zZXQodmFsdWU6IG9iamVjdCkgLT4gc2V0W3N0cl06CiAgICBpZiB2YWx1ZSBpcyBOb25lIG9yIChpc2luc3RhbmNlKHZhbHVlLCBmbG9hdCkgYW5kIG1hdGguaXNuYW4odmFsdWUpKToKICAgICAgICByZXR1cm4gc2V0KCkKICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIHN0cik6CiAgICAgICAgcmV0dXJuIHNldCh2YWx1ZS5zcGxpdCgpKSBpZiB2YWx1ZSBlbHNlIHNldCgpCiAgICByZXR1cm4gc2V0KHZhbHVlKQoKCmRlZiBfamFjY2FyZChsZWZ0OiBvYmplY3QsIHJpZ2h0OiBvYmplY3QpIC0+IGZsb2F0OgogICAgYSwgYiA9IF9zZXQobGVmdCksIF9zZXQocmlnaHQpCiAgICAjIE1pc3NpbmcgZXZpZGVuY2UgaXMgbm90IHBvc2l0aXZlIGV2aWRlbmNlLiBFbXB0eS9lbXB0eSB1c2VkIHRvIHNjb3JlIGFzCiAgICAjIGEgcGVyZmVjdCBtYXRjaCwgd2hpY2ggd2FzIHBhcnRpY3VsYXJseSBoYXJtZnVsIGZvciBibGFuayBhZGRyZXNzZXMuCiAgICByZXR1cm4gMC4wIGlmIG5vdCBhIG9yIG5vdCBiIGVsc2UgbGVuKGEgJiBiKSAvIGxlbihhIHwgYikKCgpkZWYgX2NvbnRhaW5tZW50KGxlZnQ6IG9iamVjdCwgcmlnaHQ6IG9iamVjdCkgLT4gZmxvYXQ6CiAgICBhLCBiID0gX3NldChsZWZ0KSwgX3NldChyaWdodCkKICAgIGlmIG5vdCBhIG9yIG5vdCBiOgogICAgICAgIHJldHVybiAwLjAKICAgIHJldHVybiBsZW4oYSAmIGIpIC8gbWluKGxlbihhKSwgbGVuKGIpKQoKCmRlZiBfcmF0aW8obGVmdDogc3RyLCByaWdodDogc3RyKSAtPiBmbG9hdDoKICAgIHJldHVybiAwLjAgaWYgbm90IGxlZnQgb3Igbm90IHJpZ2h0IGVsc2UgZnV6ei5yYXRpbyhsZWZ0LCByaWdodCkgLyAxMDAuMAoKCmRlZiBfcGFydGlhbChsZWZ0OiBzdHIsIHJpZ2h0OiBzdHIpIC0+IGZsb2F0OgogICAgcmV0dXJuIDAuMCBpZiBub3QgbGVmdCBvciBub3QgcmlnaHQgZWxzZSBmdXp6LnBhcnRpYWxfcmF0aW8obGVmdCwgcmlnaHQpIC8gMTAwLjAKCgpGRUFUVVJFX0NPTFVNTlMgPSBbCiAgICAibmFtZV9yYXdfZXhhY3QiLCAibmFtZV9jYW5vbmljYWxfZXhhY3QiLCAibmFtZV9jb21wYWN0X2V4YWN0IiwgIm5hbWVfY29yZV9leGFjdCIsCiAgICAibmFtZV90b2tlbl9zb3J0ZWRfZXhhY3QiLCAibmFtZV9hY2NlbnRfZm9sZGVkX2V4YWN0IiwgIm5hbWVfYWNjZW50X2ZvbGRlZF9yYXRpbyIsCiAgICAibmFtZV9yYXRpbyIsICJuYW1lX3BhcnRpYWwiLCAibmFtZV90b2tlbl9zb3J0X3JhdGlvIiwKICAgICJuYW1lX3Rva2VuX3NldF9yYXRpbyIsICJuYW1lX3Rva2VuX2phY2NhcmQiLCAibmFtZV90b2tlbl9jb250YWlubWVudCIsICJuYW1lX2xlbmd0aF9yYXRpbyIsCiAgICAibmFtZV9qYXJvX3dpbmtsZXIiLAogICAgImFkZHJlc3NfY2Fub25pY2FsX2V4YWN0IiwgImFkZHJlc3NfY29tcGFjdF9leGFjdCIsICJhZGRyZXNzX3JhdGlvIiwgImFkZHJlc3NfcGFydGlhbCIsCiAgICAiYWRkcmVzc190b2tlbl9qYWNjYXJkIiwgImFkZHJlc3NfdG9rZW5fY29udGFpbm1lbnQiLCAiYWRkcmVzc19sZW5ndGhfcmF0aW8iLAogICAgIm51bWVyaWNfZXhhY3QiLCAibnVtZXJpY19qYWNjYXJkIiwgIm51bWVyaWNfb3ZlcmxhcCIsICJudW1lcmljX2NvbmZsaWN0cyIsCiAgICAicG9zdGFsX2V4YWN0IiwgInBvc3RhbF9qYWNjYXJkIiwgImNvdW50cnlfZXF1YWwiLCAiY2FuZGlkYXRlX2lzX3MzIiwKICAgICJzMV9taXNzaW5nX2FkZHJlc3MiLCAiY2FuZGlkYXRlX21pc3NpbmdfYWRkcmVzcyIsICJibG9ja2luZ19yZWFzb25fY291bnQiLAogICAgInJldHJpZXZhbF9zY29yZSIsICJyZXRyaWV2YWxfcmFua19pbnZlcnNlIiwgIm5hbWVfYWRkcmVzc19pbnRlcmFjdGlvbiIsCiAgICAibmFtZV9udW1lcmljX2ludGVyYWN0aW9uIiwKICAgICJiZ2VfbmFtZV9jb3NpbmUiLCAiYmdlX2FkZHJfY29zaW5lIiwKXQoKIyBGZWF0dXJlcyB0aGF0IG9ubHkgZXhpc3Qgd2hlbiBhbiBlbWJlZGRpbmcgYmFja2VuZCBhbmQgY2FjaGUgYXJlIGF2YWlsYWJsZS4KRU1CRURESU5HX0NPTFVNTlMgPSBbImJnZV9uYW1lX2Nvc2luZSIsICJiZ2VfYWRkcl9jb3NpbmUiXQoKCmRlZiBfbGVuZ3RoX3JhdGlvKGxlZnQ6IHN0ciwgcmlnaHQ6IHN0cikgLT4gZmxvYXQ6CiAgICBhLCBiID0gbGVuKGxlZnQgb3IgIiIpLCBsZW4ocmlnaHQgb3IgIiIpCiAgICByZXR1cm4gMC4wIGlmIG5vdCBhIG9yIG5vdCBiIGVsc2UgbWluKGEsIGIpIC8gbWF4KGEsIGIpCgoKZGVmIF9qYXJvX3dpbmtsZXIobGVmdDogc3RyLCByaWdodDogc3RyKSAtPiBmbG9hdDoKICAgIGEsIGIgPSBsZWZ0IG9yICIiLCByaWdodCBvciAiIgogICAgaWYgYSA9PSBiOgogICAgICAgIHJldHVybiAxLjAKICAgIGlmIG5vdCBhIG9yIG5vdCBiOgogICAgICAgIHJldHVybiAwLjAKICAgIG1hdGNoX2Rpc3RhbmNlID0gbWF4KGxlbihhKSwgbGVuKGIpKSAvLyAyIC0gMQogICAgYV9tYXRjaGVzID0gW0ZhbHNlXSAqIGxlbihhKQogICAgYl9tYXRjaGVzID0gW0ZhbHNlXSAqIGxlbihiKQogICAgbWF0Y2hlcyA9IDAKICAgIGZvciBpLCBjaGFyIGluIGVudW1lcmF0ZShhKToKICAgICAgICBzdGFydCwgZW5kID0gbWF4KDAsIGkgLSBtYXRjaF9kaXN0YW5jZSksIG1pbihpICsgbWF0Y2hfZGlzdGFuY2UgKyAxLCBsZW4oYikpCiAgICAgICAgZm9yIGogaW4gcmFuZ2Uoc3RhcnQsIGVuZCk6CiAgICAgICAgICAgIGlmIGJfbWF0Y2hlc1tqXSBvciBiW2pdICE9IGNoYXI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBhX21hdGNoZXNbaV0gPSBiX21hdGNoZXNbal0gPSBUcnVlCiAgICAgICAgICAgIG1hdGNoZXMgKz0gMQogICAgICAgICAgICBicmVhawogICAgaWYgbWF0Y2hlcyA9PSAwOgogICAgICAgIHJldHVybiAwLjAKICAgIHRyYW5zcG9zaXRpb25zID0gX3RyYW5zcG9zaXRpb25zKGEsIGIsIGFfbWF0Y2hlcywgYl9tYXRjaGVzKQogICAgamFybyA9IChtYXRjaGVzIC8gbGVuKGEpICsgbWF0Y2hlcyAvIGxlbihiKSArIChtYXRjaGVzIC0gdHJhbnNwb3NpdGlvbnMpIC8gbWF0Y2hlcykgLyAzLjAKICAgIHByZWZpeCA9IDAKICAgIGZvciB4LCB5IGluIHppcChhLCBiKToKICAgICAgICBpZiB4ICE9IHkgb3IgcHJlZml4ID09IDQ6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgcHJlZml4ICs9IDEKICAgIHJldHVybiBqYXJvICsgcHJlZml4ICogMC4xICogKDEgLSBqYXJvKQoKCmRlZiBfdHJhbnNwb3NpdGlvbnMoYTogc3RyLCBiOiBzdHIsIGFfbWF0Y2hlczogbGlzdFtib29sXSwgYl9tYXRjaGVzOiBsaXN0W2Jvb2xdKSAtPiBpbnQ6CiAgICBhX2NoYXJzID0gW2FbaV0gZm9yIGkgaW4gcmFuZ2UobGVuKGEpKSBpZiBhX21hdGNoZXNbaV1dCiAgICBiX2NoYXJzID0gW2Jbal0gZm9yIGogaW4gcmFuZ2UobGVuKGIpKSBpZiBiX21hdGNoZXNbal1dCiAgICByZXR1cm4gc3VtKDEgZm9yIHgsIHkgaW4gemlwKGFfY2hhcnMsIGJfY2hhcnMpIGlmIHggIT0geSkgLy8gMgoKCmRlZiBidWlsZF9wYWlyX2ZlYXR1cmVzKAogICAgY2FuZGlkYXRlczogcGQuRGF0YUZyYW1lLAogICAgc291cmNlMTogcGQuRGF0YUZyYW1lLAogICAgdGFyZ2V0czogcGQuRGF0YUZyYW1lLAogICAgKiwKICAgIGVtYmVkX25hbWVzOiBkaWN0W3N0ciwgbnAubmRhcnJheV0gfCBOb25lID0gTm9uZSwKICAgIGVtYmVkX2FkZHJzOiBkaWN0W3N0ciwgbnAubmRhcnJheV0gfCBOb25lID0gTm9uZSwKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICByZXF1aXJlZCA9IHsic291cmNlMV9lbnRpdHlfaWQiLCAiY2FuZGlkYXRlX2VudGl0eV9pZCJ9CiAgICBpZiBub3QgcmVxdWlyZWQuaXNzdWJzZXQoY2FuZGlkYXRlcy5jb2x1bW5zKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY2FuZGlkYXRlIHRhYmxlIG1pc3NpbmcgY29sdW1uczoge3NvcnRlZChyZXF1aXJlZCAtIHNldChjYW5kaWRhdGVzLmNvbHVtbnMpKX0iKQogICAgaWYgY2FuZGlkYXRlcy5lbXB0eToKICAgICAgICBjb2x1bW5zID0gWyJzb3VyY2UxX2VudGl0eV9pZCIsICJjYW5kaWRhdGVfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9zb3VyY2UiLCAqRkVBVFVSRV9DT0xVTU5TXQogICAgICAgIGZvciBwYXNzdGhyb3VnaCBpbiAoImxhYmVsIiwgInNhbXBsZV93ZWlnaHQiLCAiZm9yY2VkX3Bvc2l0aXZlIik6CiAgICAgICAgICAgIGlmIHBhc3N0aHJvdWdoIGluIGNhbmRpZGF0ZXM6CiAgICAgICAgICAgICAgICBjb2x1bW5zLmFwcGVuZChwYXNzdGhyb3VnaCkKICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKGNvbHVtbnM9Y29sdW1ucykKICAgIHMxID0gc291cmNlMS5hZGRfcHJlZml4KCJzMV8iKS5yZW5hbWUoY29sdW1ucz17InMxX2VudGl0eV9pZCI6ICJzb3VyY2UxX2VudGl0eV9pZCJ9KQogICAgdGFyZ2V0ID0gdGFyZ2V0cy5hZGRfcHJlZml4KCJjXyIpLnJlbmFtZShjb2x1bW5zPXsiY19lbnRpdHlfaWQiOiAiY2FuZGlkYXRlX2VudGl0eV9pZCJ9KQogICAgcGFpcnMgPSBjYW5kaWRhdGVzLm1lcmdlKHMxLCBvbj0ic291cmNlMV9lbnRpdHlfaWQiLCBob3c9ImxlZnQiLCB2YWxpZGF0ZT0ibWFueV90b19vbmUiKQogICAgcGFpcnMgPSBwYWlycy5tZXJnZSh0YXJnZXQsIG9uPSJjYW5kaWRhdGVfZW50aXR5X2lkIiwgaG93PSJsZWZ0IiwgdmFsaWRhdGU9Im1hbnlfdG9fb25lIikKICAgIGlmIHBhaXJzWyJzMV9uYW1lX2Nhbm9uaWNhbCJdLmlzbmEoKS5hbnkoKSBvciBwYWlyc1siY19uYW1lX2Nhbm9uaWNhbCJdLmlzbmEoKS5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjYW5kaWRhdGUgdGFibGUgcmVmZXJlbmNlcyBhbiB1bmtub3duIFNvdXJjZSAxIG9yIHRhcmdldCBJRCIpCgogICAgIyBQcmVjb21wdXRlIGVtYmVkZGluZyBjb3NpbmVzIHZlY3Rvcml6ZWQgKGJhdGNoZWQgZ2F0aGVyICsgZG90KSB0byBhdm9pZAogICAgIyBwZXItcm93IFB5dGhvbiBvdmVyaGVhZDsgZmFsbHMgYmFjayB0byB6ZXJvcyB3aGVuIHZlY3RvcnMgYXJlIHVuYXZhaWxhYmxlLgogICAgYmdlX25hbWUgPSBucC56ZXJvcyhsZW4ocGFpcnMpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgYmdlX2FkZHIgPSBucC56ZXJvcyhsZW4ocGFpcnMpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgaWYgZW1iZWRfbmFtZXMgaXMgbm90IE5vbmUgYW5kIGVtYmVkX2FkZHJzIGlzIG5vdCBOb25lOgogICAgICAgIGJnZV9uYW1lID0gX3BhaXJfY29zaW5lKHBhaXJzWyJzb3VyY2UxX2VudGl0eV9pZCJdLCBwYWlyc1siY2FuZGlkYXRlX2VudGl0eV9pZCJdLCBlbWJlZF9uYW1lcykKICAgICAgICBiZ2VfYWRkciA9IF9wYWlyX2Nvc2luZShwYWlyc1sic291cmNlMV9lbnRpdHlfaWQiXSwgcGFpcnNbImNhbmRpZGF0ZV9lbnRpdHlfaWQiXSwgZW1iZWRfYWRkcnMpCgogICAgcmVjb3JkczogbGlzdFtkaWN0W3N0ciwgb2JqZWN0XV0gPSBbXQogICAgZm9yIHBvc2l0aW9uLCByb3cgaW4gZW51bWVyYXRlKHBhaXJzLml0ZXJ0dXBsZXMoaW5kZXg9RmFsc2UpKToKICAgICAgICBzMV9uYW1lLCBjX25hbWUgPSByb3cuczFfbmFtZV9jYW5vbmljYWwsIHJvdy5jX25hbWVfY2Fub25pY2FsCiAgICAgICAgczFfYWRkciwgY19hZGRyID0gcm93LnMxX2FkZHJlc3NfY2Fub25pY2FsLCByb3cuY19hZGRyZXNzX2Nhbm9uaWNhbAogICAgICAgIG51bWVyaWNfYSwgbnVtZXJpY19iID0gX3NldChyb3cuczFfbnVtZXJpY190b2tlbnMpLCBfc2V0KHJvdy5jX251bWVyaWNfdG9rZW5zKQogICAgICAgIG5hbWVfcmF0aW8sIGFkZHJlc3NfcmF0aW8gPSBfcmF0aW8oczFfbmFtZSwgY19uYW1lKSwgX3JhdGlvKHMxX2FkZHIsIGNfYWRkcikKICAgICAgICByZWFzb25fbWFzayA9IHN0cihnZXRhdHRyKHJvdywgInJlYXNvbl9tYXNrIiwgIiIpKQogICAgICAgIHJlY29yZHMuYXBwZW5kKHsKICAgICAgICAgICAgInNvdXJjZTFfZW50aXR5X2lkIjogcm93LnNvdXJjZTFfZW50aXR5X2lkLAogICAgICAgICAgICAiY2FuZGlkYXRlX2VudGl0eV9pZCI6IHJvdy5jYW5kaWRhdGVfZW50aXR5X2lkLAogICAgICAgICAgICAiY2FuZGlkYXRlX3NvdXJjZSI6IGdldGF0dHIocm93LCAiY2FuZGlkYXRlX3NvdXJjZSIsIHJvdy5jYW5kaWRhdGVfZW50aXR5X2lkWzoyXSksCiAgICAgICAgICAgICJuYW1lX3Jhd19leGFjdCI6IGZsb2F0KHJvdy5zMV9idXNpbmVzc19uYW1lX3JhdyA9PSByb3cuY19idXNpbmVzc19uYW1lX3JhdyksCiAgICAgICAgICAgICJuYW1lX2Nhbm9uaWNhbF9leGFjdCI6IGZsb2F0KHMxX25hbWUgPT0gY19uYW1lKSwKICAgICAgICAgICAgIm5hbWVfY29tcGFjdF9leGFjdCI6IGZsb2F0KHJvdy5zMV9uYW1lX2NvbXBhY3QgPT0gcm93LmNfbmFtZV9jb21wYWN0KSwKICAgICAgICAgICAgIm5hbWVfY29yZV9leGFjdCI6IGZsb2F0KHJvdy5zMV9uYW1lX2NvcmUgPT0gcm93LmNfbmFtZV9jb3JlKSwKICAgICAgICAgICAgIm5hbWVfdG9rZW5fc29ydGVkX2V4YWN0IjogZmxvYXQocm93LnMxX25hbWVfdG9rZW5fc29ydGVkID09IHJvdy5jX25hbWVfdG9rZW5fc29ydGVkKSwKICAgICAgICAgICAgIm5hbWVfYWNjZW50X2ZvbGRlZF9leGFjdCI6IGZsb2F0KAogICAgICAgICAgICAgICAgYm9vbChyb3cuczFfbmFtZV9hY2NlbnRfZm9sZGVkKQogICAgICAgICAgICAgICAgYW5kIHJvdy5zMV9uYW1lX2FjY2VudF9mb2xkZWQgPT0gcm93LmNfbmFtZV9hY2NlbnRfZm9sZGVkCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJuYW1lX2FjY2VudF9mb2xkZWRfcmF0aW8iOiBfcmF0aW8ocm93LnMxX25hbWVfYWNjZW50X2ZvbGRlZCwgcm93LmNfbmFtZV9hY2NlbnRfZm9sZGVkKSwKICAgICAgICAgICAgIm5hbWVfcmF0aW8iOiBuYW1lX3JhdGlvLAogICAgICAgICAgICAibmFtZV9wYXJ0aWFsIjogX3BhcnRpYWwoczFfbmFtZSwgY19uYW1lKSwKICAgICAgICAgICAgIm5hbWVfdG9rZW5fc29ydF9yYXRpbyI6IGZ1enoudG9rZW5fc29ydF9yYXRpbyhzMV9uYW1lLCBjX25hbWUpIC8gMTAwLjAsCiAgICAgICAgICAgICJuYW1lX3Rva2VuX3NldF9yYXRpbyI6IGZ1enoudG9rZW5fc2V0X3JhdGlvKHMxX25hbWUsIGNfbmFtZSkgLyAxMDAuMCwKICAgICAgICAgICAgIm5hbWVfdG9rZW5famFjY2FyZCI6IF9qYWNjYXJkKHJvdy5zMV9uYW1lX3Rva2Vucywgcm93LmNfbmFtZV90b2tlbnMpLAogICAgICAgICAgICAibmFtZV90b2tlbl9jb250YWlubWVudCI6IF9jb250YWlubWVudChyb3cuczFfbmFtZV90b2tlbnMsIHJvdy5jX25hbWVfdG9rZW5zKSwKICAgICAgICAgICAgIm5hbWVfbGVuZ3RoX3JhdGlvIjogX2xlbmd0aF9yYXRpbyhzMV9uYW1lLCBjX25hbWUpLAogICAgICAgICAgICAibmFtZV9qYXJvX3dpbmtsZXIiOiBfamFyb193aW5rbGVyKHMxX25hbWUsIGNfbmFtZSksCiAgICAgICAgICAgICJhZGRyZXNzX2Nhbm9uaWNhbF9leGFjdCI6IGZsb2F0KHMxX2FkZHIgPT0gY19hZGRyIGFuZCBib29sKHMxX2FkZHIpKSwKICAgICAgICAgICAgImFkZHJlc3NfY29tcGFjdF9leGFjdCI6IGZsb2F0KHJvdy5zMV9hZGRyZXNzX2NvbXBhY3QgPT0gcm93LmNfYWRkcmVzc19jb21wYWN0IGFuZCBib29sKHJvdy5zMV9hZGRyZXNzX2NvbXBhY3QpKSwKICAgICAgICAgICAgImFkZHJlc3NfcmF0aW8iOiBhZGRyZXNzX3JhdGlvLAogICAgICAgICAgICAiYWRkcmVzc19wYXJ0aWFsIjogX3BhcnRpYWwoczFfYWRkciwgY19hZGRyKSwKICAgICAgICAgICAgImFkZHJlc3NfdG9rZW5famFjY2FyZCI6IF9qYWNjYXJkKHJvdy5zMV9hZGRyZXNzX3Rva2Vucywgcm93LmNfYWRkcmVzc190b2tlbnMpLAogICAgICAgICAgICAiYWRkcmVzc190b2tlbl9jb250YWlubWVudCI6IF9jb250YWlubWVudChyb3cuczFfYWRkcmVzc190b2tlbnMsIHJvdy5jX2FkZHJlc3NfdG9rZW5zKSwKICAgICAgICAgICAgImFkZHJlc3NfbGVuZ3RoX3JhdGlvIjogX2xlbmd0aF9yYXRpbyhzMV9hZGRyLCBjX2FkZHIpLAogICAgICAgICAgICAibnVtZXJpY19leGFjdCI6IGZsb2F0KG51bWVyaWNfYSA9PSBudW1lcmljX2IgYW5kIGJvb2wobnVtZXJpY19hKSksCiAgICAgICAgICAgICJudW1lcmljX2phY2NhcmQiOiBfamFjY2FyZChudW1lcmljX2EsIG51bWVyaWNfYiksCiAgICAgICAgICAgICJudW1lcmljX292ZXJsYXAiOiBmbG9hdChib29sKG51bWVyaWNfYSAmIG51bWVyaWNfYikpLAogICAgICAgICAgICAibnVtZXJpY19jb25mbGljdHMiOiBmbG9hdChsZW4obnVtZXJpY19hIF4gbnVtZXJpY19iKSksCiAgICAgICAgICAgICJwb3N0YWxfZXhhY3QiOiBmbG9hdChfc2V0KHJvdy5zMV9wb3N0YWxfbGlrZV90b2tlbnMpID09IF9zZXQocm93LmNfcG9zdGFsX2xpa2VfdG9rZW5zKSBhbmQgYm9vbChfc2V0KHJvdy5zMV9wb3N0YWxfbGlrZV90b2tlbnMpKSksCiAgICAgICAgICAgICJwb3N0YWxfamFjY2FyZCI6IF9qYWNjYXJkKHJvdy5zMV9wb3N0YWxfbGlrZV90b2tlbnMsIHJvdy5jX3Bvc3RhbF9saWtlX3Rva2VucyksCiAgICAgICAgICAgICJjb3VudHJ5X2VxdWFsIjogZmxvYXQocm93LnMxX2NvdW50cnlfbm9ybSA9PSByb3cuY19jb3VudHJ5X25vcm0pLAogICAgICAgICAgICAiY2FuZGlkYXRlX2lzX3MzIjogZmxvYXQoc3RyKHJvdy5jYW5kaWRhdGVfZW50aXR5X2lkKS5zdGFydHN3aXRoKCJTMy0iKSksCiAgICAgICAgICAgICJzMV9taXNzaW5nX2FkZHJlc3MiOiBmbG9hdChyb3cuczFfbWlzc2luZ19hZGRyZXNzKSwKICAgICAgICAgICAgImNhbmRpZGF0ZV9taXNzaW5nX2FkZHJlc3MiOiBmbG9hdChyb3cuY19taXNzaW5nX2FkZHJlc3MpLAogICAgICAgICAgICAiYmxvY2tpbmdfcmVhc29uX2NvdW50IjogZmxvYXQobGVuKFtwYXJ0IGZvciBwYXJ0IGluIHJlYXNvbl9tYXNrLnNwbGl0KCJ8IikgaWYgcGFydF0pKSwKICAgICAgICAgICAgInJldHJpZXZhbF9zY29yZSI6IGZsb2F0KGdldGF0dHIocm93LCAicmV0cmlldmFsX3Njb3JlIiwgMC4wKSksCiAgICAgICAgICAgICJyZXRyaWV2YWxfcmFua19pbnZlcnNlIjogMS4wIC8gbWF4KDEsIGludChnZXRhdHRyKHJvdywgInJldHJpZXZhbF9yYW5rIiwgMSkpKSwKICAgICAgICAgICAgIm5hbWVfYWRkcmVzc19pbnRlcmFjdGlvbiI6IG5hbWVfcmF0aW8gKiBhZGRyZXNzX3JhdGlvLAogICAgICAgICAgICAibmFtZV9udW1lcmljX2ludGVyYWN0aW9uIjogbmFtZV9yYXRpbyAqIGZsb2F0KGJvb2wobnVtZXJpY19hICYgbnVtZXJpY19iKSksCiAgICAgICAgICAgICJiZ2VfbmFtZV9jb3NpbmUiOiBmbG9hdChiZ2VfbmFtZVtwb3NpdGlvbl0pLAogICAgICAgICAgICAiYmdlX2FkZHJfY29zaW5lIjogZmxvYXQoYmdlX2FkZHJbcG9zaXRpb25dKSwKICAgICAgICB9KQogICAgb3V0ID0gcGQuRGF0YUZyYW1lLmZyb21fcmVjb3JkcyhyZWNvcmRzKQogICAgZm9yIGNvbHVtbiBpbiBGRUFUVVJFX0NPTFVNTlM6CiAgICAgICAgb3V0W2NvbHVtbl0gPSBvdXRbY29sdW1uXS5hc3R5cGUobnAuZmxvYXQzMikKICAgIGZvciBwYXNzdGhyb3VnaCBpbiAoImxhYmVsIiwgInNhbXBsZV93ZWlnaHQiLCAiZm9yY2VkX3Bvc2l0aXZlIik6CiAgICAgICAgaWYgcGFzc3Rocm91Z2ggaW4gY2FuZGlkYXRlczoKICAgICAgICAgICAgb3V0W3Bhc3N0aHJvdWdoXSA9IGNhbmRpZGF0ZXNbcGFzc3Rocm91Z2hdLnRvX251bXB5KCkKICAgIHJldHVybiBvdXQKCgpkZWYgX3BhaXJfY29zaW5lKHMxX2lkczogcGQuU2VyaWVzLCBjX2lkczogcGQuU2VyaWVzLCB2ZWN0b3JzOiBkaWN0W3N0ciwgbnAubmRhcnJheV0pIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJWZWN0b3JpemVkIGNvc2luZSBiZXR3ZWVuIGFsaWduZWQgUzEgYW5kIGNhbmRpZGF0ZSBlbWJlZGRpbmcgdmVjdG9ycy4iIiIKICAgIGlmIG5vdCB2ZWN0b3JzOgogICAgICAgIHJldHVybiBucC56ZXJvcyhsZW4oczFfaWRzKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgIGRpbSA9IGxlbihuZXh0KGl0ZXIodmVjdG9ycy52YWx1ZXMoKSkpKQogICAgbGVmdF9pZHMgPSBzMV9pZHMuYXN0eXBlKHN0cikudG9fbnVtcHkoKQogICAgcmlnaHRfaWRzID0gY19pZHMuYXN0eXBlKHN0cikudG9fbnVtcHkoKQogICAgbGVmdCA9IG5wLnplcm9zKChsZW4obGVmdF9pZHMpLCBkaW0pLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgcmlnaHQgPSBucC56ZXJvcygobGVuKHJpZ2h0X2lkcyksIGRpbSksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBmb3IgaW5kZXgsIGVudGl0eV9pZCBpbiBlbnVtZXJhdGUobGVmdF9pZHMpOgogICAgICAgIHZlY3RvciA9IHZlY3RvcnMuZ2V0KGVudGl0eV9pZCkKICAgICAgICBpZiB2ZWN0b3IgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGxlZnRbaW5kZXhdID0gdmVjdG9yCiAgICBmb3IgaW5kZXgsIGVudGl0eV9pZCBpbiBlbnVtZXJhdGUocmlnaHRfaWRzKToKICAgICAgICB2ZWN0b3IgPSB2ZWN0b3JzLmdldChlbnRpdHlfaWQpCiAgICAgICAgaWYgdmVjdG9yIGlzIG5vdCBOb25lOgogICAgICAgICAgICByaWdodFtpbmRleF0gPSB2ZWN0b3IKICAgIGxlZnRfbm9ybSA9IGxlZnQgLyAobnAubGluYWxnLm5vcm0obGVmdCwgYXhpcz0xLCBrZWVwZGltcz1UcnVlKSArIDFlLTkpCiAgICByaWdodF9ub3JtID0gcmlnaHQgLyAobnAubGluYWxnLm5vcm0ocmlnaHQsIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkgKyAxZS05KQogICAgcmV0dXJuIG5wLmVpbnN1bSgiaWosaWotPmkiLCBsZWZ0X25vcm0sIHJpZ2h0X25vcm0pLmFzdHlwZShucC5mbG9hdDMyKQoKCmRlZiBmZWF0dXJlX21hdHJpeChmcmFtZTogcGQuRGF0YUZyYW1lKSAtPiBucC5uZGFycmF5OgogICAgbWlzc2luZyA9IHNldChGRUFUVVJFX0NPTFVNTlMpIC0gc2V0KGZyYW1lLmNvbHVtbnMpCiAgICBpZiBtaXNzaW5nOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJmZWF0dXJlIHRhYmxlIG1pc3NpbmcgY29sdW1uczoge3NvcnRlZChtaXNzaW5nKX0iKQogICAgcmV0dXJuIGZyYW1lW0ZFQVRVUkVfQ09MVU1OU10udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQzMiwgY29weT1GYWxzZSkKCg==', '19631d8ce73f9cc1ae51f83079dce2301487facb90ecf784b9b645652aee2aac')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/inference.py`

~~~~python
"""End-to-end in-memory inference orchestration for one partition/smoke run."""

from __future__ import annotations

from collections.abc import Mapping
import pandas as pd

from .blocking import CandidateGenerator
from .decisions import apply_thresholds
from .features import build_pair_features, feature_matrix
from .metrics import sets_from_long
from .normalize import normalize_records


def generate_candidates_for_sources(source1: pd.DataFrame, source2: pd.DataFrame, source3: pd.DataFrame, blocking_config: dict[str, object], include_tfidf: bool = True) -> pd.DataFrame:
    normalized_s1 = normalize_records(source1)
    frames = []
    for targets in (source2, source3):
        normalized_targets = normalize_records(targets)
        frames.append(CandidateGenerator(blocking_config, include_tfidf=include_tfidf).fit(normalized_targets).transform(normalized_s1))
    if not frames:
        return pd.DataFrame()
    candidates = pd.concat(frames, ignore_index=True)
    return candidates.sort_values(["source1_entity_id", "candidate_source", "retrieval_score", "candidate_entity_id"], ascending=[True, True, False, True], kind="mergesort").reset_index(drop=True)


def score_candidates(candidates: pd.DataFrame, source1: pd.DataFrame, targets: pd.DataFrame, model) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_table = build_pair_features(candidates, normalize_records(source1), normalize_records(targets))
    scored = feature_table[["source1_entity_id", "candidate_entity_id", "candidate_source"]].copy()
    scored["score"] = model.predict_scores(feature_matrix(feature_table))
    return scored, feature_table


def infer_partition(
    source1: pd.DataFrame,
    source2: pd.DataFrame,
    source3: pd.DataFrame,
    model,
    blocking_config: dict[str, object],
    threshold: float,
    source_thresholds: Mapping[str, float] | None = None,
    include_tfidf: bool = True,
) -> tuple[dict[str, frozenset[str]], dict[str, frozenset[str]], pd.DataFrame, pd.DataFrame]:
    candidates = generate_candidates_for_sources(source1, source2, source3, blocking_config, include_tfidf)
    targets = pd.concat([source2, source3], ignore_index=True)
    scored, features = score_candidates(candidates, source1, targets, model)
    ids = source1["entity_id"].astype(str).tolist()
    predictions = apply_thresholds(scored, ids, threshold, source_thresholds)
    candidate_sets = sets_from_long(candidates, "candidate_entity_id")
    for entity_id in ids:
        candidate_sets.setdefault(entity_id, frozenset())
    return predictions, candidate_sets, scored, features


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/inference.py', 'IiIiRW5kLXRvLWVuZCBpbi1tZW1vcnkgaW5mZXJlbmNlIG9yY2hlc3RyYXRpb24gZm9yIG9uZSBwYXJ0aXRpb24vc21va2UgcnVuLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IE1hcHBpbmcKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuYmxvY2tpbmcgaW1wb3J0IENhbmRpZGF0ZUdlbmVyYXRvcgpmcm9tIC5kZWNpc2lvbnMgaW1wb3J0IGFwcGx5X3RocmVzaG9sZHMKZnJvbSAuZmVhdHVyZXMgaW1wb3J0IGJ1aWxkX3BhaXJfZmVhdHVyZXMsIGZlYXR1cmVfbWF0cml4CmZyb20gLm1ldHJpY3MgaW1wb3J0IHNldHNfZnJvbV9sb25nCmZyb20gLm5vcm1hbGl6ZSBpbXBvcnQgbm9ybWFsaXplX3JlY29yZHMKCgpkZWYgZ2VuZXJhdGVfY2FuZGlkYXRlc19mb3Jfc291cmNlcyhzb3VyY2UxOiBwZC5EYXRhRnJhbWUsIHNvdXJjZTI6IHBkLkRhdGFGcmFtZSwgc291cmNlMzogcGQuRGF0YUZyYW1lLCBibG9ja2luZ19jb25maWc6IGRpY3Rbc3RyLCBvYmplY3RdLCBpbmNsdWRlX3RmaWRmOiBib29sID0gVHJ1ZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgbm9ybWFsaXplZF9zMSA9IG5vcm1hbGl6ZV9yZWNvcmRzKHNvdXJjZTEpCiAgICBmcmFtZXMgPSBbXQogICAgZm9yIHRhcmdldHMgaW4gKHNvdXJjZTIsIHNvdXJjZTMpOgogICAgICAgIG5vcm1hbGl6ZWRfdGFyZ2V0cyA9IG5vcm1hbGl6ZV9yZWNvcmRzKHRhcmdldHMpCiAgICAgICAgZnJhbWVzLmFwcGVuZChDYW5kaWRhdGVHZW5lcmF0b3IoYmxvY2tpbmdfY29uZmlnLCBpbmNsdWRlX3RmaWRmPWluY2x1ZGVfdGZpZGYpLmZpdChub3JtYWxpemVkX3RhcmdldHMpLnRyYW5zZm9ybShub3JtYWxpemVkX3MxKSkKICAgIGlmIG5vdCBmcmFtZXM6CiAgICAgICAgcmV0dXJuIHBkLkRhdGFGcmFtZSgpCiAgICBjYW5kaWRhdGVzID0gcGQuY29uY2F0KGZyYW1lcywgaWdub3JlX2luZGV4PVRydWUpCiAgICByZXR1cm4gY2FuZGlkYXRlcy5zb3J0X3ZhbHVlcyhbInNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9zb3VyY2UiLCAicmV0cmlldmFsX3Njb3JlIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiXSwgYXNjZW5kaW5nPVtUcnVlLCBUcnVlLCBGYWxzZSwgVHJ1ZV0sIGtpbmQ9Im1lcmdlc29ydCIpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgc2NvcmVfY2FuZGlkYXRlcyhjYW5kaWRhdGVzOiBwZC5EYXRhRnJhbWUsIHNvdXJjZTE6IHBkLkRhdGFGcmFtZSwgdGFyZ2V0czogcGQuRGF0YUZyYW1lLCBtb2RlbCkgLT4gdHVwbGVbcGQuRGF0YUZyYW1lLCBwZC5EYXRhRnJhbWVdOgogICAgZmVhdHVyZV90YWJsZSA9IGJ1aWxkX3BhaXJfZmVhdHVyZXMoY2FuZGlkYXRlcywgbm9ybWFsaXplX3JlY29yZHMoc291cmNlMSksIG5vcm1hbGl6ZV9yZWNvcmRzKHRhcmdldHMpKQogICAgc2NvcmVkID0gZmVhdHVyZV90YWJsZVtbInNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiLCAiY2FuZGlkYXRlX3NvdXJjZSJdXS5jb3B5KCkKICAgIHNjb3JlZFsic2NvcmUiXSA9IG1vZGVsLnByZWRpY3Rfc2NvcmVzKGZlYXR1cmVfbWF0cml4KGZlYXR1cmVfdGFibGUpKQogICAgcmV0dXJuIHNjb3JlZCwgZmVhdHVyZV90YWJsZQoKCmRlZiBpbmZlcl9wYXJ0aXRpb24oCiAgICBzb3VyY2UxOiBwZC5EYXRhRnJhbWUsCiAgICBzb3VyY2UyOiBwZC5EYXRhRnJhbWUsCiAgICBzb3VyY2UzOiBwZC5EYXRhRnJhbWUsCiAgICBtb2RlbCwKICAgIGJsb2NraW5nX2NvbmZpZzogZGljdFtzdHIsIG9iamVjdF0sCiAgICB0aHJlc2hvbGQ6IGZsb2F0LAogICAgc291cmNlX3RocmVzaG9sZHM6IE1hcHBpbmdbc3RyLCBmbG9hdF0gfCBOb25lID0gTm9uZSwKICAgIGluY2x1ZGVfdGZpZGY6IGJvb2wgPSBUcnVlLAopIC0+IHR1cGxlW2RpY3Rbc3RyLCBmcm96ZW5zZXRbc3RyXV0sIGRpY3Rbc3RyLCBmcm96ZW5zZXRbc3RyXV0sIHBkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lXToKICAgIGNhbmRpZGF0ZXMgPSBnZW5lcmF0ZV9jYW5kaWRhdGVzX2Zvcl9zb3VyY2VzKHNvdXJjZTEsIHNvdXJjZTIsIHNvdXJjZTMsIGJsb2NraW5nX2NvbmZpZywgaW5jbHVkZV90ZmlkZikKICAgIHRhcmdldHMgPSBwZC5jb25jYXQoW3NvdXJjZTIsIHNvdXJjZTNdLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKICAgIHNjb3JlZCwgZmVhdHVyZXMgPSBzY29yZV9jYW5kaWRhdGVzKGNhbmRpZGF0ZXMsIHNvdXJjZTEsIHRhcmdldHMsIG1vZGVsKQogICAgaWRzID0gc291cmNlMVsiZW50aXR5X2lkIl0uYXN0eXBlKHN0cikudG9saXN0KCkKICAgIHByZWRpY3Rpb25zID0gYXBwbHlfdGhyZXNob2xkcyhzY29yZWQsIGlkcywgdGhyZXNob2xkLCBzb3VyY2VfdGhyZXNob2xkcykKICAgIGNhbmRpZGF0ZV9zZXRzID0gc2V0c19mcm9tX2xvbmcoY2FuZGlkYXRlcywgImNhbmRpZGF0ZV9lbnRpdHlfaWQiKQogICAgZm9yIGVudGl0eV9pZCBpbiBpZHM6CiAgICAgICAgY2FuZGlkYXRlX3NldHMuc2V0ZGVmYXVsdChlbnRpdHlfaWQsIGZyb3plbnNldCgpKQogICAgcmV0dXJuIHByZWRpY3Rpb25zLCBjYW5kaWRhdGVfc2V0cywgc2NvcmVkLCBmZWF0dXJlcwoK', '7e4e9b93277e4f6096caefcdcfec786316a26e7b8e2e688eee93114fb6bdee0f')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/labels.py`

~~~~python
"""Ground-truth parsing and pair labeling."""

from __future__ import annotations

from collections.abc import Mapping
import pandas as pd

from .schemas import GROUND_TRUTH_COLUMNS, SchemaError, require_columns


def parse_match_ids(value: str) -> tuple[str, ...]:
    if value is None or value == "":
        return ()
    ids = tuple(value.split(","))
    if any(not item.startswith(("S2-", "S3-")) for item in ids):
        raise SchemaError(f"ground truth contains non-S2/S3 ID: {value!r}")
    if len(ids) != len(set(ids)):
        raise SchemaError(f"ground truth contains duplicate target ID: {value!r}")
    return ids


def ground_truth_sets(frame: pd.DataFrame) -> dict[str, frozenset[str]]:
    require_columns(list(frame.columns), GROUND_TRUTH_COLUMNS, "ground truth")
    if frame["source1_entity_id"].duplicated().any():
        raise SchemaError("ground truth contains duplicate source1_entity_id rows")
    if (~frame["source1_entity_id"].str.match(r"^S1-\d+$")).any():
        raise SchemaError("ground truth contains malformed S1 IDs")
    return {
        row.source1_entity_id: frozenset(parse_match_ids(row.matched_entity_ids))
        for row in frame.itertuples(index=False)
    }


def explode_ground_truth(frame: pd.DataFrame) -> pd.DataFrame:
    rows = [
        (source1_id, target_id)
        for source1_id, targets in ground_truth_sets(frame).items()
        for target_id in sorted(targets)
    ]
    return pd.DataFrame(rows, columns=["source1_entity_id", "candidate_entity_id"])


def label_candidates(candidates: pd.DataFrame, truth: Mapping[str, set[str] | frozenset[str]]) -> pd.DataFrame:
    required = {"source1_entity_id", "candidate_entity_id"}
    if not required.issubset(candidates.columns):
        raise ValueError(f"candidate table missing columns: {sorted(required - set(candidates.columns))}")
    out = candidates.copy()
    out["label"] = [
        int(candidate in truth.get(source1_id, frozenset()))
        for source1_id, candidate in zip(out["source1_entity_id"], out["candidate_entity_id"])
    ]
    return out


def force_add_training_positives(candidates: pd.DataFrame, truth: Mapping[str, set[str] | frozenset[str]]) -> pd.DataFrame:
    existing = set(zip(candidates["source1_entity_id"], candidates["candidate_entity_id"]))
    rows = []
    for source1_id, targets in truth.items():
        for target_id in targets:
            if (source1_id, target_id) not in existing:
                rows.append({
                    "source1_entity_id": source1_id,
                    "candidate_entity_id": target_id,
                    "candidate_source": target_id[:2],
                    "reason_mask": "forced_positive",
                    "retrieval_score": 1.0,
                    "retrieval_rank": 0,
                    "forced_positive": True,
                })
    base = candidates.copy()
    if "forced_positive" not in base:
        base["forced_positive"] = False
    return pd.concat([base, pd.DataFrame(rows)], ignore_index=True, sort=False) if rows else base


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/labels.py', 'IiIiR3JvdW5kLXRydXRoIHBhcnNpbmcgYW5kIHBhaXIgbGFiZWxpbmcuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgTWFwcGluZwppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIC5zY2hlbWFzIGltcG9ydCBHUk9VTkRfVFJVVEhfQ09MVU1OUywgU2NoZW1hRXJyb3IsIHJlcXVpcmVfY29sdW1ucwoKCmRlZiBwYXJzZV9tYXRjaF9pZHModmFsdWU6IHN0cikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgaWYgdmFsdWUgaXMgTm9uZSBvciB2YWx1ZSA9PSAiIjoKICAgICAgICByZXR1cm4gKCkKICAgIGlkcyA9IHR1cGxlKHZhbHVlLnNwbGl0KCIsIikpCiAgICBpZiBhbnkobm90IGl0ZW0uc3RhcnRzd2l0aCgoIlMyLSIsICJTMy0iKSkgZm9yIGl0ZW0gaW4gaWRzKToKICAgICAgICByYWlzZSBTY2hlbWFFcnJvcihmImdyb3VuZCB0cnV0aCBjb250YWlucyBub24tUzIvUzMgSUQ6IHt2YWx1ZSFyfSIpCiAgICBpZiBsZW4oaWRzKSAhPSBsZW4oc2V0KGlkcykpOgogICAgICAgIHJhaXNlIFNjaGVtYUVycm9yKGYiZ3JvdW5kIHRydXRoIGNvbnRhaW5zIGR1cGxpY2F0ZSB0YXJnZXQgSUQ6IHt2YWx1ZSFyfSIpCiAgICByZXR1cm4gaWRzCgoKZGVmIGdyb3VuZF90cnV0aF9zZXRzKGZyYW1lOiBwZC5EYXRhRnJhbWUpIC0+IGRpY3Rbc3RyLCBmcm96ZW5zZXRbc3RyXV06CiAgICByZXF1aXJlX2NvbHVtbnMobGlzdChmcmFtZS5jb2x1bW5zKSwgR1JPVU5EX1RSVVRIX0NPTFVNTlMsICJncm91bmQgdHJ1dGgiKQogICAgaWYgZnJhbWVbInNvdXJjZTFfZW50aXR5X2lkIl0uZHVwbGljYXRlZCgpLmFueSgpOgogICAgICAgIHJhaXNlIFNjaGVtYUVycm9yKCJncm91bmQgdHJ1dGggY29udGFpbnMgZHVwbGljYXRlIHNvdXJjZTFfZW50aXR5X2lkIHJvd3MiKQogICAgaWYgKH5mcmFtZVsic291cmNlMV9lbnRpdHlfaWQiXS5zdHIubWF0Y2gociJeUzEtXGQrJCIpKS5hbnkoKToKICAgICAgICByYWlzZSBTY2hlbWFFcnJvcigiZ3JvdW5kIHRydXRoIGNvbnRhaW5zIG1hbGZvcm1lZCBTMSBJRHMiKQogICAgcmV0dXJuIHsKICAgICAgICByb3cuc291cmNlMV9lbnRpdHlfaWQ6IGZyb3plbnNldChwYXJzZV9tYXRjaF9pZHMocm93Lm1hdGNoZWRfZW50aXR5X2lkcykpCiAgICAgICAgZm9yIHJvdyBpbiBmcmFtZS5pdGVydHVwbGVzKGluZGV4PUZhbHNlKQogICAgfQoKCmRlZiBleHBsb2RlX2dyb3VuZF90cnV0aChmcmFtZTogcGQuRGF0YUZyYW1lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICByb3dzID0gWwogICAgICAgIChzb3VyY2UxX2lkLCB0YXJnZXRfaWQpCiAgICAgICAgZm9yIHNvdXJjZTFfaWQsIHRhcmdldHMgaW4gZ3JvdW5kX3RydXRoX3NldHMoZnJhbWUpLml0ZW1zKCkKICAgICAgICBmb3IgdGFyZ2V0X2lkIGluIHNvcnRlZCh0YXJnZXRzKQogICAgXQogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShyb3dzLCBjb2x1bW5zPVsic291cmNlMV9lbnRpdHlfaWQiLCAiY2FuZGlkYXRlX2VudGl0eV9pZCJdKQoKCmRlZiBsYWJlbF9jYW5kaWRhdGVzKGNhbmRpZGF0ZXM6IHBkLkRhdGFGcmFtZSwgdHJ1dGg6IE1hcHBpbmdbc3RyLCBzZXRbc3RyXSB8IGZyb3plbnNldFtzdHJdXSkgLT4gcGQuRGF0YUZyYW1lOgogICAgcmVxdWlyZWQgPSB7InNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQifQogICAgaWYgbm90IHJlcXVpcmVkLmlzc3Vic2V0KGNhbmRpZGF0ZXMuY29sdW1ucyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImNhbmRpZGF0ZSB0YWJsZSBtaXNzaW5nIGNvbHVtbnM6IHtzb3J0ZWQocmVxdWlyZWQgLSBzZXQoY2FuZGlkYXRlcy5jb2x1bW5zKSl9IikKICAgIG91dCA9IGNhbmRpZGF0ZXMuY29weSgpCiAgICBvdXRbImxhYmVsIl0gPSBbCiAgICAgICAgaW50KGNhbmRpZGF0ZSBpbiB0cnV0aC5nZXQoc291cmNlMV9pZCwgZnJvemVuc2V0KCkpKQogICAgICAgIGZvciBzb3VyY2UxX2lkLCBjYW5kaWRhdGUgaW4gemlwKG91dFsic291cmNlMV9lbnRpdHlfaWQiXSwgb3V0WyJjYW5kaWRhdGVfZW50aXR5X2lkIl0pCiAgICBdCiAgICByZXR1cm4gb3V0CgoKZGVmIGZvcmNlX2FkZF90cmFpbmluZ19wb3NpdGl2ZXMoY2FuZGlkYXRlczogcGQuRGF0YUZyYW1lLCB0cnV0aDogTWFwcGluZ1tzdHIsIHNldFtzdHJdIHwgZnJvemVuc2V0W3N0cl1dKSAtPiBwZC5EYXRhRnJhbWU6CiAgICBleGlzdGluZyA9IHNldCh6aXAoY2FuZGlkYXRlc1sic291cmNlMV9lbnRpdHlfaWQiXSwgY2FuZGlkYXRlc1siY2FuZGlkYXRlX2VudGl0eV9pZCJdKSkKICAgIHJvd3MgPSBbXQogICAgZm9yIHNvdXJjZTFfaWQsIHRhcmdldHMgaW4gdHJ1dGguaXRlbXMoKToKICAgICAgICBmb3IgdGFyZ2V0X2lkIGluIHRhcmdldHM6CiAgICAgICAgICAgIGlmIChzb3VyY2UxX2lkLCB0YXJnZXRfaWQpIG5vdCBpbiBleGlzdGluZzoKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAic291cmNlMV9lbnRpdHlfaWQiOiBzb3VyY2UxX2lkLAogICAgICAgICAgICAgICAgICAgICJjYW5kaWRhdGVfZW50aXR5X2lkIjogdGFyZ2V0X2lkLAogICAgICAgICAgICAgICAgICAgICJjYW5kaWRhdGVfc291cmNlIjogdGFyZ2V0X2lkWzoyXSwKICAgICAgICAgICAgICAgICAgICAicmVhc29uX21hc2siOiAiZm9yY2VkX3Bvc2l0aXZlIiwKICAgICAgICAgICAgICAgICAgICAicmV0cmlldmFsX3Njb3JlIjogMS4wLAogICAgICAgICAgICAgICAgICAgICJyZXRyaWV2YWxfcmFuayI6IDAsCiAgICAgICAgICAgICAgICAgICAgImZvcmNlZF9wb3NpdGl2ZSI6IFRydWUsCiAgICAgICAgICAgICAgICB9KQogICAgYmFzZSA9IGNhbmRpZGF0ZXMuY29weSgpCiAgICBpZiAiZm9yY2VkX3Bvc2l0aXZlIiBub3QgaW4gYmFzZToKICAgICAgICBiYXNlWyJmb3JjZWRfcG9zaXRpdmUiXSA9IEZhbHNlCiAgICByZXR1cm4gcGQuY29uY2F0KFtiYXNlLCBwZC5EYXRhRnJhbWUocm93cyldLCBpZ25vcmVfaW5kZXg9VHJ1ZSwgc29ydD1GYWxzZSkgaWYgcm93cyBlbHNlIGJhc2UKCg==', 'f61e48606892201cf5ef70ea22c24bcf28aaa63a5574177a01d66ec2b09b1caa')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/logging.py`

~~~~python
"""Continuous, crash-resilient run logging for long stages.

Every logger writes to stdout *and* to a per-stage log file, flushed on every
record so that a crash or kill leaves the tail of the log on disk. Progress,
resource plans, per-shard rows/elapsed/ETA/peak memory, and model metrics
(training loss, AUC, threshold sweeps) all flow through here.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
import time
from typing import Any


def _peak_rss_bytes() -> int:
    try:
        import resource  # type: ignore

        return int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss) * 1024
    except Exception:
        pass
    try:
        import psutil  # type: ignore

        return int(psutil.Process().memory_info().rss)
    except Exception:
        return 0


def _format_bytes(value: int) -> str:
    size = float(value)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.2f}{unit}"
        size /= 1024
    return f"{size:.2f}TB"


class StageLogger:
    """A tiny append-only logger with stdout mirroring and flush-per-line."""

    def __init__(self, path: str | Path | None = None, name: str = "run", echo: bool = True) -> None:
        self.name = name
        self.echo = echo
        self.path = Path(path) if path is not None else None
        self.started = time.time()
        if self.path is not None:
            self.path.parent.mkdir(parents=True, exist_ok=True)

    def _emit(self, line: str) -> None:
        stamped = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {line}"
        if self.echo:
            print(stamped, flush=True)
        if self.path is not None:
            with self.path.open("a", encoding="utf-8") as handle:
                handle.write(stamped + "\n")
                handle.flush()

    def info(self, message: str, **fields: Any) -> None:
        self._emit(self._compose("INFO", message, fields))

    def event(self, event: str, **fields: Any) -> None:
        self._emit(self._compose("EVENT", event, fields))

    def metric(self, name: str, value: Any, **fields: Any) -> None:
        self._emit(self._compose("METRIC", name, {"value": value, **fields}))

    def progress(self, stage: str, done: int, total: int, extra: dict[str, Any] | None = None) -> None:
        elapsed = time.time() - self.started
        rate = done / elapsed if elapsed > 0 and done > 0 else 0.0
        eta = (total - done) / rate if rate > 0 else float("inf")
        fields: dict[str, Any] = {
            "done": done,
            "total": total,
            "elapsed_s": round(elapsed, 1),
            "eta_s": round(eta, 1) if eta != float("inf") else None,
            "rate_per_s": round(rate, 2),
            "peak_rss": _format_bytes(_peak_rss_bytes()),
        }
        if extra:
            fields.update(extra)
        self._emit(self._compose("PROGRESS", stage, fields))

    def resource_plan(self, plan: Any) -> None:
        data = plan.as_dict() if hasattr(plan, "as_dict") else dict(plan)
        if "ram_budget_bytes" in data:
            data["ram_budget"] = _format_bytes(int(data.pop("ram_budget_bytes")))
        if "vram_budget_bytes" in data:
            data["vram_budget"] = _format_bytes(int(data.pop("vram_budget_bytes")))
        self._emit(self._compose("RESOURCES", "plan", data))

    def stage_start(self, stage: str, **fields: Any) -> None:
        self.started = time.time()
        self._emit(self._compose("STAGE_START", stage, fields))

    def stage_end(self, stage: str, **fields: Any) -> None:
        fields.setdefault("elapsed_s", round(time.time() - self.started, 1))
        fields.setdefault("peak_rss", _format_bytes(_peak_rss_bytes()))
        self._emit(self._compose("STAGE_END", stage, fields))

    def _compose(self, level: str, message: str, fields: dict[str, Any]) -> str:
        suffix = " ".join(f"{key}={_stringify(value)}" for key, value in fields.items() if value is not None)
        return f"[{self.name}][{level}] {message}" + (f" | {suffix}" if suffix else "")


def _stringify(value: Any) -> str:
    if isinstance(value, float):
        return f"{value:.6g}"
    return str(value)


@dataclass(slots=True)
class NullLogger:
    """No-op logger so call sites never branch on availability."""

    name: str = "null"
    echo: bool = False

    def info(self, *_a: Any, **_k: Any) -> None: ...
    def event(self, *_a: Any, **_k: Any) -> None: ...
    def metric(self, *_a: Any, **_k: Any) -> None: ...
    def progress(self, *_a: Any, **_k: Any) -> None: ...
    def resource_plan(self, *_a: Any, **_k: Any) -> None: ...
    def stage_start(self, *_a: Any, **_k: Any) -> None: ...
    def stage_end(self, *_a: Any, **_k: Any) -> None: ...


def stage_logger(config: Any, stage: str, echo: bool = True) -> StageLogger:
    """Create a logger rooted at ``<artifact_root>/<run_id>/logs/<stage>.log``."""
    try:
        path = config.artifact_dir("logs") / f"{stage}.log"
    except Exception:
        path = None
    return StageLogger(path, name=stage, echo=echo and sys.stdout is not None)

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/logging.py', 'IiIiQ29udGludW91cywgY3Jhc2gtcmVzaWxpZW50IHJ1biBsb2dnaW5nIGZvciBsb25nIHN0YWdlcy4KCkV2ZXJ5IGxvZ2dlciB3cml0ZXMgdG8gc3Rkb3V0ICphbmQqIHRvIGEgcGVyLXN0YWdlIGxvZyBmaWxlLCBmbHVzaGVkIG9uIGV2ZXJ5CnJlY29yZCBzbyB0aGF0IGEgY3Jhc2ggb3Iga2lsbCBsZWF2ZXMgdGhlIHRhaWwgb2YgdGhlIGxvZyBvbiBkaXNrLiBQcm9ncmVzcywKcmVzb3VyY2UgcGxhbnMsIHBlci1zaGFyZCByb3dzL2VsYXBzZWQvRVRBL3BlYWsgbWVtb3J5LCBhbmQgbW9kZWwgbWV0cmljcwoodHJhaW5pbmcgbG9zcywgQVVDLCB0aHJlc2hvbGQgc3dlZXBzKSBhbGwgZmxvdyB0aHJvdWdoIGhlcmUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCgpkZWYgX3BlYWtfcnNzX2J5dGVzKCkgLT4gaW50OgogICAgdHJ5OgogICAgICAgIGltcG9ydCByZXNvdXJjZSAgIyB0eXBlOiBpZ25vcmUKCiAgICAgICAgcmV0dXJuIGludChyZXNvdXJjZS5nZXRydXNhZ2UocmVzb3VyY2UuUlVTQUdFX1NFTEYpLnJ1X21heHJzcykgKiAxMDI0CiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgcHN1dGlsICAjIHR5cGU6IGlnbm9yZQoKICAgICAgICByZXR1cm4gaW50KHBzdXRpbC5Qcm9jZXNzKCkubWVtb3J5X2luZm8oKS5yc3MpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiAwCgoKZGVmIF9mb3JtYXRfYnl0ZXModmFsdWU6IGludCkgLT4gc3RyOgogICAgc2l6ZSA9IGZsb2F0KHZhbHVlKQogICAgZm9yIHVuaXQgaW4gKCJCIiwgIktCIiwgIk1CIiwgIkdCIiwgIlRCIik6CiAgICAgICAgaWYgc2l6ZSA8IDEwMjQgb3IgdW5pdCA9PSAiVEIiOgogICAgICAgICAgICByZXR1cm4gZiJ7c2l6ZTouMmZ9e3VuaXR9IgogICAgICAgIHNpemUgLz0gMTAyNAogICAgcmV0dXJuIGYie3NpemU6LjJmfVRCIgoKCmNsYXNzIFN0YWdlTG9nZ2VyOgogICAgIiIiQSB0aW55IGFwcGVuZC1vbmx5IGxvZ2dlciB3aXRoIHN0ZG91dCBtaXJyb3JpbmcgYW5kIGZsdXNoLXBlci1saW5lLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXRoOiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUsIG5hbWU6IHN0ciA9ICJydW4iLCBlY2hvOiBib29sID0gVHJ1ZSkgLT4gTm9uZToKICAgICAgICBzZWxmLm5hbWUgPSBuYW1lCiAgICAgICAgc2VsZi5lY2hvID0gZWNobwogICAgICAgIHNlbGYucGF0aCA9IFBhdGgocGF0aCkgaWYgcGF0aCBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgICAgICBzZWxmLnN0YXJ0ZWQgPSB0aW1lLnRpbWUoKQogICAgICAgIGlmIHNlbGYucGF0aCBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGVmIF9lbWl0KHNlbGYsIGxpbmU6IHN0cikgLT4gTm9uZToKICAgICAgICBzdGFtcGVkID0gZiJbe3RpbWUuc3RyZnRpbWUoJyVZLSVtLSVkICVIOiVNOiVTJyl9XSB7bGluZX0iCiAgICAgICAgaWYgc2VsZi5lY2hvOgogICAgICAgICAgICBwcmludChzdGFtcGVkLCBmbHVzaD1UcnVlKQogICAgICAgIGlmIHNlbGYucGF0aCBpcyBub3QgTm9uZToKICAgICAgICAgICAgd2l0aCBzZWxmLnBhdGgub3BlbigiYSIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICAgICAgICAgIGhhbmRsZS53cml0ZShzdGFtcGVkICsgIlxuIikKICAgICAgICAgICAgICAgIGhhbmRsZS5mbHVzaCgpCgogICAgZGVmIGluZm8oc2VsZiwgbWVzc2FnZTogc3RyLCAqKmZpZWxkczogQW55KSAtPiBOb25lOgogICAgICAgIHNlbGYuX2VtaXQoc2VsZi5fY29tcG9zZSgiSU5GTyIsIG1lc3NhZ2UsIGZpZWxkcykpCgogICAgZGVmIGV2ZW50KHNlbGYsIGV2ZW50OiBzdHIsICoqZmllbGRzOiBBbnkpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5fZW1pdChzZWxmLl9jb21wb3NlKCJFVkVOVCIsIGV2ZW50LCBmaWVsZHMpKQoKICAgIGRlZiBtZXRyaWMoc2VsZiwgbmFtZTogc3RyLCB2YWx1ZTogQW55LCAqKmZpZWxkczogQW55KSAtPiBOb25lOgogICAgICAgIHNlbGYuX2VtaXQoc2VsZi5fY29tcG9zZSgiTUVUUklDIiwgbmFtZSwgeyJ2YWx1ZSI6IHZhbHVlLCAqKmZpZWxkc30pKQoKICAgIGRlZiBwcm9ncmVzcyhzZWxmLCBzdGFnZTogc3RyLCBkb25lOiBpbnQsIHRvdGFsOiBpbnQsIGV4dHJhOiBkaWN0W3N0ciwgQW55XSB8IE5vbmUgPSBOb25lKSAtPiBOb25lOgogICAgICAgIGVsYXBzZWQgPSB0aW1lLnRpbWUoKSAtIHNlbGYuc3RhcnRlZAogICAgICAgIHJhdGUgPSBkb25lIC8gZWxhcHNlZCBpZiBlbGFwc2VkID4gMCBhbmQgZG9uZSA+IDAgZWxzZSAwLjAKICAgICAgICBldGEgPSAodG90YWwgLSBkb25lKSAvIHJhdGUgaWYgcmF0ZSA+IDAgZWxzZSBmbG9hdCgiaW5mIikKICAgICAgICBmaWVsZHM6IGRpY3Rbc3RyLCBBbnldID0gewogICAgICAgICAgICAiZG9uZSI6IGRvbmUsCiAgICAgICAgICAgICJ0b3RhbCI6IHRvdGFsLAogICAgICAgICAgICAiZWxhcHNlZF9zIjogcm91bmQoZWxhcHNlZCwgMSksCiAgICAgICAgICAgICJldGFfcyI6IHJvdW5kKGV0YSwgMSkgaWYgZXRhICE9IGZsb2F0KCJpbmYiKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJyYXRlX3Blcl9zIjogcm91bmQocmF0ZSwgMiksCiAgICAgICAgICAgICJwZWFrX3JzcyI6IF9mb3JtYXRfYnl0ZXMoX3BlYWtfcnNzX2J5dGVzKCkpLAogICAgICAgIH0KICAgICAgICBpZiBleHRyYToKICAgICAgICAgICAgZmllbGRzLnVwZGF0ZShleHRyYSkKICAgICAgICBzZWxmLl9lbWl0KHNlbGYuX2NvbXBvc2UoIlBST0dSRVNTIiwgc3RhZ2UsIGZpZWxkcykpCgogICAgZGVmIHJlc291cmNlX3BsYW4oc2VsZiwgcGxhbjogQW55KSAtPiBOb25lOgogICAgICAgIGRhdGEgPSBwbGFuLmFzX2RpY3QoKSBpZiBoYXNhdHRyKHBsYW4sICJhc19kaWN0IikgZWxzZSBkaWN0KHBsYW4pCiAgICAgICAgaWYgInJhbV9idWRnZXRfYnl0ZXMiIGluIGRhdGE6CiAgICAgICAgICAgIGRhdGFbInJhbV9idWRnZXQiXSA9IF9mb3JtYXRfYnl0ZXMoaW50KGRhdGEucG9wKCJyYW1fYnVkZ2V0X2J5dGVzIikpKQogICAgICAgIGlmICJ2cmFtX2J1ZGdldF9ieXRlcyIgaW4gZGF0YToKICAgICAgICAgICAgZGF0YVsidnJhbV9idWRnZXQiXSA9IF9mb3JtYXRfYnl0ZXMoaW50KGRhdGEucG9wKCJ2cmFtX2J1ZGdldF9ieXRlcyIpKSkKICAgICAgICBzZWxmLl9lbWl0KHNlbGYuX2NvbXBvc2UoIlJFU09VUkNFUyIsICJwbGFuIiwgZGF0YSkpCgogICAgZGVmIHN0YWdlX3N0YXJ0KHNlbGYsIHN0YWdlOiBzdHIsICoqZmllbGRzOiBBbnkpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5zdGFydGVkID0gdGltZS50aW1lKCkKICAgICAgICBzZWxmLl9lbWl0KHNlbGYuX2NvbXBvc2UoIlNUQUdFX1NUQVJUIiwgc3RhZ2UsIGZpZWxkcykpCgogICAgZGVmIHN0YWdlX2VuZChzZWxmLCBzdGFnZTogc3RyLCAqKmZpZWxkczogQW55KSAtPiBOb25lOgogICAgICAgIGZpZWxkcy5zZXRkZWZhdWx0KCJlbGFwc2VkX3MiLCByb3VuZCh0aW1lLnRpbWUoKSAtIHNlbGYuc3RhcnRlZCwgMSkpCiAgICAgICAgZmllbGRzLnNldGRlZmF1bHQoInBlYWtfcnNzIiwgX2Zvcm1hdF9ieXRlcyhfcGVha19yc3NfYnl0ZXMoKSkpCiAgICAgICAgc2VsZi5fZW1pdChzZWxmLl9jb21wb3NlKCJTVEFHRV9FTkQiLCBzdGFnZSwgZmllbGRzKSkKCiAgICBkZWYgX2NvbXBvc2Uoc2VsZiwgbGV2ZWw6IHN0ciwgbWVzc2FnZTogc3RyLCBmaWVsZHM6IGRpY3Rbc3RyLCBBbnldKSAtPiBzdHI6CiAgICAgICAgc3VmZml4ID0gIiAiLmpvaW4oZiJ7a2V5fT17X3N0cmluZ2lmeSh2YWx1ZSl9IiBmb3Iga2V5LCB2YWx1ZSBpbiBmaWVsZHMuaXRlbXMoKSBpZiB2YWx1ZSBpcyBub3QgTm9uZSkKICAgICAgICByZXR1cm4gZiJbe3NlbGYubmFtZX1dW3tsZXZlbH1dIHttZXNzYWdlfSIgKyAoZiIgfCB7c3VmZml4fSIgaWYgc3VmZml4IGVsc2UgIiIpCgoKZGVmIF9zdHJpbmdpZnkodmFsdWU6IEFueSkgLT4gc3RyOgogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgZmxvYXQpOgogICAgICAgIHJldHVybiBmInt2YWx1ZTouNmd9IgogICAgcmV0dXJuIHN0cih2YWx1ZSkKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIE51bGxMb2dnZXI6CiAgICAiIiJOby1vcCBsb2dnZXIgc28gY2FsbCBzaXRlcyBuZXZlciBicmFuY2ggb24gYXZhaWxhYmlsaXR5LiIiIgoKICAgIG5hbWU6IHN0ciA9ICJudWxsIgogICAgZWNobzogYm9vbCA9IEZhbHNlCgogICAgZGVmIGluZm8oc2VsZiwgKl9hOiBBbnksICoqX2s6IEFueSkgLT4gTm9uZTogLi4uCiAgICBkZWYgZXZlbnQoc2VsZiwgKl9hOiBBbnksICoqX2s6IEFueSkgLT4gTm9uZTogLi4uCiAgICBkZWYgbWV0cmljKHNlbGYsICpfYTogQW55LCAqKl9rOiBBbnkpIC0+IE5vbmU6IC4uLgogICAgZGVmIHByb2dyZXNzKHNlbGYsICpfYTogQW55LCAqKl9rOiBBbnkpIC0+IE5vbmU6IC4uLgogICAgZGVmIHJlc291cmNlX3BsYW4oc2VsZiwgKl9hOiBBbnksICoqX2s6IEFueSkgLT4gTm9uZTogLi4uCiAgICBkZWYgc3RhZ2Vfc3RhcnQoc2VsZiwgKl9hOiBBbnksICoqX2s6IEFueSkgLT4gTm9uZTogLi4uCiAgICBkZWYgc3RhZ2VfZW5kKHNlbGYsICpfYTogQW55LCAqKl9rOiBBbnkpIC0+IE5vbmU6IC4uLgoKCmRlZiBzdGFnZV9sb2dnZXIoY29uZmlnOiBBbnksIHN0YWdlOiBzdHIsIGVjaG86IGJvb2wgPSBUcnVlKSAtPiBTdGFnZUxvZ2dlcjoKICAgICIiIkNyZWF0ZSBhIGxvZ2dlciByb290ZWQgYXQgYGA8YXJ0aWZhY3Rfcm9vdD4vPHJ1bl9pZD4vbG9ncy88c3RhZ2U+LmxvZ2BgLiIiIgogICAgdHJ5OgogICAgICAgIHBhdGggPSBjb25maWcuYXJ0aWZhY3RfZGlyKCJsb2dzIikgLyBmIntzdGFnZX0ubG9nIgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXRoID0gTm9uZQogICAgcmV0dXJuIFN0YWdlTG9nZ2VyKHBhdGgsIG5hbWU9c3RhZ2UsIGVjaG89ZWNobyBhbmQgc3lzLnN0ZG91dCBpcyBub3QgTm9uZSkK', '4e384d2fc1af6504bb4e6f1be5a4783b55e8bc6b4287ba5b49dd0b5ea8e489d3')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/metrics.py`

~~~~python
"""Competition-faithful entity-set and candidate metrics."""

from __future__ import annotations

from collections.abc import Iterable, Mapping
import numpy as np
import pandas as pd


def entity_scores(truth: set[str] | frozenset[str], prediction: set[str] | frozenset[str], beta: float = 0.5) -> tuple[float, float, float]:
    truth_set, prediction_set = set(truth), set(prediction)
    if not truth_set and not prediction_set:
        return 1.0, 1.0, 1.0
    if not truth_set or not prediction_set:
        return 0.0, 0.0, 0.0
    tp = len(truth_set & prediction_set)
    precision, recall = tp / len(prediction_set), tp / len(truth_set)
    beta2 = beta * beta
    denominator = beta2 * precision + recall
    return precision, recall, 0.0 if denominator == 0 else (1 + beta2) * precision * recall / denominator


def evaluate_entity_sets(
    truth: Mapping[str, set[str] | frozenset[str]],
    predictions: Mapping[str, set[str] | frozenset[str]],
    beta: float = 0.5,
) -> dict[str, float | int]:
    rows: list[tuple[float, float, float]] = []
    false_merges = missed_matches = singleton_correct = singleton_total = 0
    for source1_id, target_truth in truth.items():
        predicted = set(predictions.get(source1_id, frozenset()))
        truth_set = set(target_truth)
        rows.append(entity_scores(truth_set, predicted, beta))
        false_merges += len(predicted - truth_set)
        missed_matches += len(truth_set - predicted)
        if not truth_set:
            singleton_total += 1
            singleton_correct += int(not predicted)
    values = np.asarray(rows, dtype=float)
    return {
        "macro_precision": float(values[:, 0].mean()) if len(values) else 0.0,
        "macro_recall": float(values[:, 1].mean()) if len(values) else 0.0,
        "macro_f0_5": float(values[:, 2].mean()) if len(values) else 0.0,
        "singleton_accuracy": singleton_correct / singleton_total if singleton_total else 0.0,
        "singleton_total": singleton_total,
        "false_merges": false_merges,
        "missed_matches": missed_matches,
        "entity_count": len(rows),
    }


def candidate_metrics(
    truth: Mapping[str, set[str] | frozenset[str]],
    candidates: Mapping[str, set[str] | frozenset[str]],
    target_universe_size: int,
) -> dict[str, float | int]:
    total_true = retrieved_true = non_singletons = any_found = complete = 0
    counts: list[int] = []
    for source1_id, target_truth in truth.items():
        candidate_set, truth_set = set(candidates.get(source1_id, frozenset())), set(target_truth)
        counts.append(len(candidate_set))
        total_true += len(truth_set)
        retrieved_true += len(truth_set & candidate_set)
        if truth_set:
            non_singletons += 1
            any_found += int(bool(truth_set & candidate_set))
            complete += int(truth_set.issubset(candidate_set))
    arr = np.asarray(counts, dtype=np.int64)
    total_pairs, possible = int(arr.sum()), len(truth) * target_universe_size
    return {
        "candidate_recall": retrieved_true / total_true if total_true else 1.0,
        "any_match_entity_recall": any_found / non_singletons if non_singletons else 1.0,
        "complete_entity_recall": complete / non_singletons if non_singletons else 1.0,
        "candidate_count": total_pairs,
        "average_candidates": float(arr.mean()) if len(arr) else 0.0,
        "p50_candidates": float(np.quantile(arr, 0.5)) if len(arr) else 0.0,
        "p95_candidates": float(np.quantile(arr, 0.95)) if len(arr) else 0.0,
        "p99_candidates": float(np.quantile(arr, 0.99)) if len(arr) else 0.0,
        "max_candidates": int(arr.max()) if len(arr) else 0,
        "zero_candidate_rate": float((arr == 0).mean()) if len(arr) else 0.0,
        "reduction_ratio": 1.0 - total_pairs / possible if possible else 1.0,
    }


def candidate_metrics_from_frames(
    truth: Mapping[str, set[str] | frozenset[str]],
    candidate_frames: Iterable[pd.DataFrame],
    target_universe_size: int,
) -> dict[str, float | int]:
    """Calculate exact candidate metrics without materializing all pairs.

    Candidate shards must partition Source 1 IDs, as the production sharding
    scheme does. Entities absent from every shard are included with zero
    candidates, preserving singleton and blocking-miss behavior.
    """
    total_true = sum(len(values) for values in truth.values())
    non_singletons = sum(bool(values) for values in truth.values())
    retrieved_true = any_found = complete = total_pairs = seen_entities = 0
    counts: list[int] = []
    for frame in candidate_frames:
        if frame.empty:
            continue
        for source1_id, group in frame.groupby("source1_entity_id", sort=False):
            candidate_set = set(group["candidate_entity_id"].astype(str))
            truth_set = set(truth.get(str(source1_id), frozenset()))
            count = len(candidate_set)
            counts.append(count)
            seen_entities += 1
            total_pairs += count
            retrieved_true += len(candidate_set & truth_set)
            if truth_set:
                any_found += int(bool(candidate_set & truth_set))
                complete += int(truth_set.issubset(candidate_set))
    counts.extend([0] * max(0, len(truth) - seen_entities))
    arr = np.asarray(counts, dtype=np.int64)
    possible = len(truth) * target_universe_size
    return {
        "candidate_recall": retrieved_true / total_true if total_true else 1.0,
        "any_match_entity_recall": any_found / non_singletons if non_singletons else 1.0,
        "complete_entity_recall": complete / non_singletons if non_singletons else 1.0,
        "candidate_count": total_pairs,
        "average_candidates": float(arr.mean()) if len(arr) else 0.0,
        "p50_candidates": float(np.quantile(arr, 0.5)) if len(arr) else 0.0,
        "p95_candidates": float(np.quantile(arr, 0.95)) if len(arr) else 0.0,
        "p99_candidates": float(np.quantile(arr, 0.99)) if len(arr) else 0.0,
        "max_candidates": int(arr.max()) if len(arr) else 0,
        "zero_candidate_rate": float((arr == 0).mean()) if len(arr) else 0.0,
        "reduction_ratio": 1.0 - total_pairs / possible if possible else 1.0,
    }


def sets_from_long(frame: pd.DataFrame, id_column: str) -> dict[str, frozenset[str]]:
    return {
        source1_id: frozenset(group[id_column].astype(str))
        for source1_id, group in frame.groupby("source1_entity_id", sort=False)
    } if not frame.empty else {}

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/metrics.py', 'IiIiQ29tcGV0aXRpb24tZmFpdGhmdWwgZW50aXR5LXNldCBhbmQgY2FuZGlkYXRlIG1ldHJpY3MuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgSXRlcmFibGUsIE1hcHBpbmcKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpkZWYgZW50aXR5X3Njb3Jlcyh0cnV0aDogc2V0W3N0cl0gfCBmcm96ZW5zZXRbc3RyXSwgcHJlZGljdGlvbjogc2V0W3N0cl0gfCBmcm96ZW5zZXRbc3RyXSwgYmV0YTogZmxvYXQgPSAwLjUpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdOgogICAgdHJ1dGhfc2V0LCBwcmVkaWN0aW9uX3NldCA9IHNldCh0cnV0aCksIHNldChwcmVkaWN0aW9uKQogICAgaWYgbm90IHRydXRoX3NldCBhbmQgbm90IHByZWRpY3Rpb25fc2V0OgogICAgICAgIHJldHVybiAxLjAsIDEuMCwgMS4wCiAgICBpZiBub3QgdHJ1dGhfc2V0IG9yIG5vdCBwcmVkaWN0aW9uX3NldDoKICAgICAgICByZXR1cm4gMC4wLCAwLjAsIDAuMAogICAgdHAgPSBsZW4odHJ1dGhfc2V0ICYgcHJlZGljdGlvbl9zZXQpCiAgICBwcmVjaXNpb24sIHJlY2FsbCA9IHRwIC8gbGVuKHByZWRpY3Rpb25fc2V0KSwgdHAgLyBsZW4odHJ1dGhfc2V0KQogICAgYmV0YTIgPSBiZXRhICogYmV0YQogICAgZGVub21pbmF0b3IgPSBiZXRhMiAqIHByZWNpc2lvbiArIHJlY2FsbAogICAgcmV0dXJuIHByZWNpc2lvbiwgcmVjYWxsLCAwLjAgaWYgZGVub21pbmF0b3IgPT0gMCBlbHNlICgxICsgYmV0YTIpICogcHJlY2lzaW9uICogcmVjYWxsIC8gZGVub21pbmF0b3IKCgpkZWYgZXZhbHVhdGVfZW50aXR5X3NldHMoCiAgICB0cnV0aDogTWFwcGluZ1tzdHIsIHNldFtzdHJdIHwgZnJvemVuc2V0W3N0cl1dLAogICAgcHJlZGljdGlvbnM6IE1hcHBpbmdbc3RyLCBzZXRbc3RyXSB8IGZyb3plbnNldFtzdHJdXSwKICAgIGJldGE6IGZsb2F0ID0gMC41LAopIC0+IGRpY3Rbc3RyLCBmbG9hdCB8IGludF06CiAgICByb3dzOiBsaXN0W3R1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdXSA9IFtdCiAgICBmYWxzZV9tZXJnZXMgPSBtaXNzZWRfbWF0Y2hlcyA9IHNpbmdsZXRvbl9jb3JyZWN0ID0gc2luZ2xldG9uX3RvdGFsID0gMAogICAgZm9yIHNvdXJjZTFfaWQsIHRhcmdldF90cnV0aCBpbiB0cnV0aC5pdGVtcygpOgogICAgICAgIHByZWRpY3RlZCA9IHNldChwcmVkaWN0aW9ucy5nZXQoc291cmNlMV9pZCwgZnJvemVuc2V0KCkpKQogICAgICAgIHRydXRoX3NldCA9IHNldCh0YXJnZXRfdHJ1dGgpCiAgICAgICAgcm93cy5hcHBlbmQoZW50aXR5X3Njb3Jlcyh0cnV0aF9zZXQsIHByZWRpY3RlZCwgYmV0YSkpCiAgICAgICAgZmFsc2VfbWVyZ2VzICs9IGxlbihwcmVkaWN0ZWQgLSB0cnV0aF9zZXQpCiAgICAgICAgbWlzc2VkX21hdGNoZXMgKz0gbGVuKHRydXRoX3NldCAtIHByZWRpY3RlZCkKICAgICAgICBpZiBub3QgdHJ1dGhfc2V0OgogICAgICAgICAgICBzaW5nbGV0b25fdG90YWwgKz0gMQogICAgICAgICAgICBzaW5nbGV0b25fY29ycmVjdCArPSBpbnQobm90IHByZWRpY3RlZCkKICAgIHZhbHVlcyA9IG5wLmFzYXJyYXkocm93cywgZHR5cGU9ZmxvYXQpCiAgICByZXR1cm4gewogICAgICAgICJtYWNyb19wcmVjaXNpb24iOiBmbG9hdCh2YWx1ZXNbOiwgMF0ubWVhbigpKSBpZiBsZW4odmFsdWVzKSBlbHNlIDAuMCwKICAgICAgICAibWFjcm9fcmVjYWxsIjogZmxvYXQodmFsdWVzWzosIDFdLm1lYW4oKSkgaWYgbGVuKHZhbHVlcykgZWxzZSAwLjAsCiAgICAgICAgIm1hY3JvX2YwXzUiOiBmbG9hdCh2YWx1ZXNbOiwgMl0ubWVhbigpKSBpZiBsZW4odmFsdWVzKSBlbHNlIDAuMCwKICAgICAgICAic2luZ2xldG9uX2FjY3VyYWN5Ijogc2luZ2xldG9uX2NvcnJlY3QgLyBzaW5nbGV0b25fdG90YWwgaWYgc2luZ2xldG9uX3RvdGFsIGVsc2UgMC4wLAogICAgICAgICJzaW5nbGV0b25fdG90YWwiOiBzaW5nbGV0b25fdG90YWwsCiAgICAgICAgImZhbHNlX21lcmdlcyI6IGZhbHNlX21lcmdlcywKICAgICAgICAibWlzc2VkX21hdGNoZXMiOiBtaXNzZWRfbWF0Y2hlcywKICAgICAgICAiZW50aXR5X2NvdW50IjogbGVuKHJvd3MpLAogICAgfQoKCmRlZiBjYW5kaWRhdGVfbWV0cmljcygKICAgIHRydXRoOiBNYXBwaW5nW3N0ciwgc2V0W3N0cl0gfCBmcm96ZW5zZXRbc3RyXV0sCiAgICBjYW5kaWRhdGVzOiBNYXBwaW5nW3N0ciwgc2V0W3N0cl0gfCBmcm96ZW5zZXRbc3RyXV0sCiAgICB0YXJnZXRfdW5pdmVyc2Vfc2l6ZTogaW50LAopIC0+IGRpY3Rbc3RyLCBmbG9hdCB8IGludF06CiAgICB0b3RhbF90cnVlID0gcmV0cmlldmVkX3RydWUgPSBub25fc2luZ2xldG9ucyA9IGFueV9mb3VuZCA9IGNvbXBsZXRlID0gMAogICAgY291bnRzOiBsaXN0W2ludF0gPSBbXQogICAgZm9yIHNvdXJjZTFfaWQsIHRhcmdldF90cnV0aCBpbiB0cnV0aC5pdGVtcygpOgogICAgICAgIGNhbmRpZGF0ZV9zZXQsIHRydXRoX3NldCA9IHNldChjYW5kaWRhdGVzLmdldChzb3VyY2UxX2lkLCBmcm96ZW5zZXQoKSkpLCBzZXQodGFyZ2V0X3RydXRoKQogICAgICAgIGNvdW50cy5hcHBlbmQobGVuKGNhbmRpZGF0ZV9zZXQpKQogICAgICAgIHRvdGFsX3RydWUgKz0gbGVuKHRydXRoX3NldCkKICAgICAgICByZXRyaWV2ZWRfdHJ1ZSArPSBsZW4odHJ1dGhfc2V0ICYgY2FuZGlkYXRlX3NldCkKICAgICAgICBpZiB0cnV0aF9zZXQ6CiAgICAgICAgICAgIG5vbl9zaW5nbGV0b25zICs9IDEKICAgICAgICAgICAgYW55X2ZvdW5kICs9IGludChib29sKHRydXRoX3NldCAmIGNhbmRpZGF0ZV9zZXQpKQogICAgICAgICAgICBjb21wbGV0ZSArPSBpbnQodHJ1dGhfc2V0Lmlzc3Vic2V0KGNhbmRpZGF0ZV9zZXQpKQogICAgYXJyID0gbnAuYXNhcnJheShjb3VudHMsIGR0eXBlPW5wLmludDY0KQogICAgdG90YWxfcGFpcnMsIHBvc3NpYmxlID0gaW50KGFyci5zdW0oKSksIGxlbih0cnV0aCkgKiB0YXJnZXRfdW5pdmVyc2Vfc2l6ZQogICAgcmV0dXJuIHsKICAgICAgICAiY2FuZGlkYXRlX3JlY2FsbCI6IHJldHJpZXZlZF90cnVlIC8gdG90YWxfdHJ1ZSBpZiB0b3RhbF90cnVlIGVsc2UgMS4wLAogICAgICAgICJhbnlfbWF0Y2hfZW50aXR5X3JlY2FsbCI6IGFueV9mb3VuZCAvIG5vbl9zaW5nbGV0b25zIGlmIG5vbl9zaW5nbGV0b25zIGVsc2UgMS4wLAogICAgICAgICJjb21wbGV0ZV9lbnRpdHlfcmVjYWxsIjogY29tcGxldGUgLyBub25fc2luZ2xldG9ucyBpZiBub25fc2luZ2xldG9ucyBlbHNlIDEuMCwKICAgICAgICAiY2FuZGlkYXRlX2NvdW50IjogdG90YWxfcGFpcnMsCiAgICAgICAgImF2ZXJhZ2VfY2FuZGlkYXRlcyI6IGZsb2F0KGFyci5tZWFuKCkpIGlmIGxlbihhcnIpIGVsc2UgMC4wLAogICAgICAgICJwNTBfY2FuZGlkYXRlcyI6IGZsb2F0KG5wLnF1YW50aWxlKGFyciwgMC41KSkgaWYgbGVuKGFycikgZWxzZSAwLjAsCiAgICAgICAgInA5NV9jYW5kaWRhdGVzIjogZmxvYXQobnAucXVhbnRpbGUoYXJyLCAwLjk1KSkgaWYgbGVuKGFycikgZWxzZSAwLjAsCiAgICAgICAgInA5OV9jYW5kaWRhdGVzIjogZmxvYXQobnAucXVhbnRpbGUoYXJyLCAwLjk5KSkgaWYgbGVuKGFycikgZWxzZSAwLjAsCiAgICAgICAgIm1heF9jYW5kaWRhdGVzIjogaW50KGFyci5tYXgoKSkgaWYgbGVuKGFycikgZWxzZSAwLAogICAgICAgICJ6ZXJvX2NhbmRpZGF0ZV9yYXRlIjogZmxvYXQoKGFyciA9PSAwKS5tZWFuKCkpIGlmIGxlbihhcnIpIGVsc2UgMC4wLAogICAgICAgICJyZWR1Y3Rpb25fcmF0aW8iOiAxLjAgLSB0b3RhbF9wYWlycyAvIHBvc3NpYmxlIGlmIHBvc3NpYmxlIGVsc2UgMS4wLAogICAgfQoKCmRlZiBjYW5kaWRhdGVfbWV0cmljc19mcm9tX2ZyYW1lcygKICAgIHRydXRoOiBNYXBwaW5nW3N0ciwgc2V0W3N0cl0gfCBmcm96ZW5zZXRbc3RyXV0sCiAgICBjYW5kaWRhdGVfZnJhbWVzOiBJdGVyYWJsZVtwZC5EYXRhRnJhbWVdLAogICAgdGFyZ2V0X3VuaXZlcnNlX3NpemU6IGludCwKKSAtPiBkaWN0W3N0ciwgZmxvYXQgfCBpbnRdOgogICAgIiIiQ2FsY3VsYXRlIGV4YWN0IGNhbmRpZGF0ZSBtZXRyaWNzIHdpdGhvdXQgbWF0ZXJpYWxpemluZyBhbGwgcGFpcnMuCgogICAgQ2FuZGlkYXRlIHNoYXJkcyBtdXN0IHBhcnRpdGlvbiBTb3VyY2UgMSBJRHMsIGFzIHRoZSBwcm9kdWN0aW9uIHNoYXJkaW5nCiAgICBzY2hlbWUgZG9lcy4gRW50aXRpZXMgYWJzZW50IGZyb20gZXZlcnkgc2hhcmQgYXJlIGluY2x1ZGVkIHdpdGggemVybwogICAgY2FuZGlkYXRlcywgcHJlc2VydmluZyBzaW5nbGV0b24gYW5kIGJsb2NraW5nLW1pc3MgYmVoYXZpb3IuCiAgICAiIiIKICAgIHRvdGFsX3RydWUgPSBzdW0obGVuKHZhbHVlcykgZm9yIHZhbHVlcyBpbiB0cnV0aC52YWx1ZXMoKSkKICAgIG5vbl9zaW5nbGV0b25zID0gc3VtKGJvb2wodmFsdWVzKSBmb3IgdmFsdWVzIGluIHRydXRoLnZhbHVlcygpKQogICAgcmV0cmlldmVkX3RydWUgPSBhbnlfZm91bmQgPSBjb21wbGV0ZSA9IHRvdGFsX3BhaXJzID0gc2Vlbl9lbnRpdGllcyA9IDAKICAgIGNvdW50czogbGlzdFtpbnRdID0gW10KICAgIGZvciBmcmFtZSBpbiBjYW5kaWRhdGVfZnJhbWVzOgogICAgICAgIGlmIGZyYW1lLmVtcHR5OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBzb3VyY2UxX2lkLCBncm91cCBpbiBmcmFtZS5ncm91cGJ5KCJzb3VyY2UxX2VudGl0eV9pZCIsIHNvcnQ9RmFsc2UpOgogICAgICAgICAgICBjYW5kaWRhdGVfc2V0ID0gc2V0KGdyb3VwWyJjYW5kaWRhdGVfZW50aXR5X2lkIl0uYXN0eXBlKHN0cikpCiAgICAgICAgICAgIHRydXRoX3NldCA9IHNldCh0cnV0aC5nZXQoc3RyKHNvdXJjZTFfaWQpLCBmcm96ZW5zZXQoKSkpCiAgICAgICAgICAgIGNvdW50ID0gbGVuKGNhbmRpZGF0ZV9zZXQpCiAgICAgICAgICAgIGNvdW50cy5hcHBlbmQoY291bnQpCiAgICAgICAgICAgIHNlZW5fZW50aXRpZXMgKz0gMQogICAgICAgICAgICB0b3RhbF9wYWlycyArPSBjb3VudAogICAgICAgICAgICByZXRyaWV2ZWRfdHJ1ZSArPSBsZW4oY2FuZGlkYXRlX3NldCAmIHRydXRoX3NldCkKICAgICAgICAgICAgaWYgdHJ1dGhfc2V0OgogICAgICAgICAgICAgICAgYW55X2ZvdW5kICs9IGludChib29sKGNhbmRpZGF0ZV9zZXQgJiB0cnV0aF9zZXQpKQogICAgICAgICAgICAgICAgY29tcGxldGUgKz0gaW50KHRydXRoX3NldC5pc3N1YnNldChjYW5kaWRhdGVfc2V0KSkKICAgIGNvdW50cy5leHRlbmQoWzBdICogbWF4KDAsIGxlbih0cnV0aCkgLSBzZWVuX2VudGl0aWVzKSkKICAgIGFyciA9IG5wLmFzYXJyYXkoY291bnRzLCBkdHlwZT1ucC5pbnQ2NCkKICAgIHBvc3NpYmxlID0gbGVuKHRydXRoKSAqIHRhcmdldF91bml2ZXJzZV9zaXplCiAgICByZXR1cm4gewogICAgICAgICJjYW5kaWRhdGVfcmVjYWxsIjogcmV0cmlldmVkX3RydWUgLyB0b3RhbF90cnVlIGlmIHRvdGFsX3RydWUgZWxzZSAxLjAsCiAgICAgICAgImFueV9tYXRjaF9lbnRpdHlfcmVjYWxsIjogYW55X2ZvdW5kIC8gbm9uX3NpbmdsZXRvbnMgaWYgbm9uX3NpbmdsZXRvbnMgZWxzZSAxLjAsCiAgICAgICAgImNvbXBsZXRlX2VudGl0eV9yZWNhbGwiOiBjb21wbGV0ZSAvIG5vbl9zaW5nbGV0b25zIGlmIG5vbl9zaW5nbGV0b25zIGVsc2UgMS4wLAogICAgICAgICJjYW5kaWRhdGVfY291bnQiOiB0b3RhbF9wYWlycywKICAgICAgICAiYXZlcmFnZV9jYW5kaWRhdGVzIjogZmxvYXQoYXJyLm1lYW4oKSkgaWYgbGVuKGFycikgZWxzZSAwLjAsCiAgICAgICAgInA1MF9jYW5kaWRhdGVzIjogZmxvYXQobnAucXVhbnRpbGUoYXJyLCAwLjUpKSBpZiBsZW4oYXJyKSBlbHNlIDAuMCwKICAgICAgICAicDk1X2NhbmRpZGF0ZXMiOiBmbG9hdChucC5xdWFudGlsZShhcnIsIDAuOTUpKSBpZiBsZW4oYXJyKSBlbHNlIDAuMCwKICAgICAgICAicDk5X2NhbmRpZGF0ZXMiOiBmbG9hdChucC5xdWFudGlsZShhcnIsIDAuOTkpKSBpZiBsZW4oYXJyKSBlbHNlIDAuMCwKICAgICAgICAibWF4X2NhbmRpZGF0ZXMiOiBpbnQoYXJyLm1heCgpKSBpZiBsZW4oYXJyKSBlbHNlIDAsCiAgICAgICAgInplcm9fY2FuZGlkYXRlX3JhdGUiOiBmbG9hdCgoYXJyID09IDApLm1lYW4oKSkgaWYgbGVuKGFycikgZWxzZSAwLjAsCiAgICAgICAgInJlZHVjdGlvbl9yYXRpbyI6IDEuMCAtIHRvdGFsX3BhaXJzIC8gcG9zc2libGUgaWYgcG9zc2libGUgZWxzZSAxLjAsCiAgICB9CgoKZGVmIHNldHNfZnJvbV9sb25nKGZyYW1lOiBwZC5EYXRhRnJhbWUsIGlkX2NvbHVtbjogc3RyKSAtPiBkaWN0W3N0ciwgZnJvemVuc2V0W3N0cl1dOgogICAgcmV0dXJuIHsKICAgICAgICBzb3VyY2UxX2lkOiBmcm96ZW5zZXQoZ3JvdXBbaWRfY29sdW1uXS5hc3R5cGUoc3RyKSkKICAgICAgICBmb3Igc291cmNlMV9pZCwgZ3JvdXAgaW4gZnJhbWUuZ3JvdXBieSgic291cmNlMV9lbnRpdHlfaWQiLCBzb3J0PUZhbHNlKQogICAgfSBpZiBub3QgZnJhbWUuZW1wdHkgZWxzZSB7fQo=', 'a4140f3f0f7eb6190fb1500ec455c6bc50693b1f42e9f751ad187aa1e750cc73')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/mini.py`

~~~~python
"""Build a realistic mini dataset that mirrors the real challenge distribution.

The mini set is *sampled from the real files* so it carries the same schema,
noise (abbreviations, transliterations, reordered/landmark addresses, missing
addresses), country mix, and match cardinality — only tiny. It is used to run
the full pipeline end-to-end on a laptop without touching the full corpus.

Key realism point: France appears only in the **test** files, not in training.
The mini set therefore builds:
  * ``train/``  from the real training files (US + India) with ground truth
  * ``test/``   from the real test files (US + India + France), no ground truth

Composition (default 15 Source 1 entities per split):
  * several US, India in train; US + India + France in test
  * at least one singleton, several single-match, several multi-match,
    and at least one entity matched by both Source 2 and Source 3
  * Source 2/3 pool includes true matches plus non-matching decoys

Scans are streamed and stop as soon as enough records are collected, so this is
fast regardless of the full dataset size.
"""

from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import random

import pandas as pd

from .artifacts import write_frame
from .data import ground_truth_path, iter_tsv, source_path
from .schemas import GROUND_TRUTH_COLUMNS, SOURCE_COLUMNS

TRAIN_QUOTA = {"US": 6, "India": 6}
TEST_QUOTA = {"US": 5, "India": 5, "France": 5}
S1_BUFFER_PER_COUNTRY = 60
DECOYS_PER_COUNTRY = 4


def _parse_ids(value: str) -> list[str]:
    return [item for item in str(value).split(",") if item]


def _collect_s1(chunk, quotas: dict[str, int], buffers: dict[str, list[str]], rows: dict[str, dict]) -> None:
    for row in chunk.itertuples(index=False):
        country = str(row.country)
        if country in quotas and len(buffers[country]) < S1_BUFFER_PER_COUNTRY:
            entity_id = str(row.entity_id)
            buffers[country].append(entity_id)
            rows[entity_id] = {"entity_id": entity_id, "business_name": row.business_name,
                               "business_address": row.business_address, "country": row.country}


def _buffers_ready(buffers: dict[str, list[str]], quotas: dict[str, int]) -> bool:
    return all(len(buffers.get(c, [])) >= min(quotas[c] * 3, S1_BUFFER_PER_COUNTRY) for c in quotas)


def _choose(buffers: dict[str, list[str]], quotas: dict[str, int], rng: random.Random) -> list[str]:
    chosen: list[str] = []
    for country, wanted in quotas.items():
        pool = sorted(buffers.get(country, []))
        rng.shuffle(pool)
        chosen.extend(pool[:wanted])
    return chosen


def _scan_targets(
    data_root: Path,
    split: str,
    source: int,
    needed: set[str],
    quotas: dict[str, int],
    rng: random.Random,
) -> tuple[list[dict], dict[str, list[dict]]]:
    """Return (rows for needed IDs, decoys per country) from one target source."""
    lookup: dict[str, dict] = {}
    decoys: dict[str, list[dict]] = defaultdict(list)
    remaining = set(needed)
    for chunk in iter_tsv(source_path(data_root, split, source), SOURCE_COLUMNS, batch_size=100_000):
        if remaining:
            selected = chunk[chunk["entity_id"].isin(remaining)]
            for row in selected.itertuples(index=False):
                lookup[str(row.entity_id)] = {"entity_id": str(row.entity_id), "business_name": row.business_name,
                                              "business_address": row.business_address, "country": row.country}
                remaining.discard(str(row.entity_id))
        for country in quotas:
            if len(decoys[country]) < DECOYS_PER_COUNTRY:
                pool = chunk[chunk["country"].eq(country) & ~chunk["entity_id"].isin(needed)]
                if not pool.empty:
                    take = pool.sample(min(DECOYS_PER_COUNTRY - len(decoys[country]), len(pool)), random_state=rng.randint(0, 2**31))
                    decoys[country].extend(
                        {"entity_id": str(r.entity_id), "business_name": r.business_name,
                         "business_address": r.business_address, "country": r.country}
                        for r in take.itertuples(index=False)
                    )
        if not remaining and all(len(decoys[c]) >= DECOYS_PER_COUNTRY for c in quotas):
            break
    return list(lookup.values()), decoys


def _build_split(
    data_root: Path,
    out_dir: Path,
    split: str,
    quotas: dict[str, int],
    rng: random.Random,
    with_ground_truth: bool,
) -> dict[str, int]:
    # 1) Stream Source 1 until each country buffer is full.
    buffers: dict[str, list[str]] = defaultdict(list)
    s1_rows: dict[str, dict] = {}
    for chunk in iter_tsv(source_path(data_root, split, 1), SOURCE_COLUMNS, batch_size=100_000):
        _collect_s1(chunk, quotas, buffers, s1_rows)
        if _buffers_ready(buffers, quotas):
            break

    chosen = _choose(buffers, quotas, rng)
    chosen_set = set(chosen)
    truth_map: dict[str, list[str]] = {}

    if with_ground_truth:
        pool = {e for ids in buffers.values() for e in ids}
        remaining = set(chosen) | pool
        for chunk in iter_tsv(ground_truth_path(data_root), GROUND_TRUTH_COLUMNS, batch_size=200_000):
            matches = chunk[chunk["source1_entity_id"].isin(remaining)]
            for row in matches.itertuples(index=False):
                truth_map[str(row.source1_entity_id)] = _parse_ids(row.matched_entity_ids)
                remaining.discard(str(row.source1_entity_id))
            if not remaining:
                break
        for entity_id in chosen:
            truth_map.setdefault(entity_id, [])

        # Guarantee a singleton and a both-source match when the buffer allows.
        def _profile(ids):
            singleton = any(not truth_map[i] for i in ids)
            both = any(
                any(t.startswith("S2-") for t in truth_map[i]) and any(t.startswith("S3-") for t in truth_map[i])
                for i in ids
            )
            return singleton, both

        def _swap(predicate) -> None:
            for country in quotas:
                for replacement in buffers.get(country, []):
                    if replacement in chosen_set or not predicate(truth_map[replacement]):
                        continue
                    for index in reversed(range(len(chosen))):
                        if str(s1_rows.get(chosen[index], {}).get("country")) == country:
                            chosen_set.discard(chosen[index])
                            chosen[index] = replacement
                            chosen_set.add(replacement)
                            return

        has_singleton, has_both = _profile(chosen)
        if not has_singleton:
            _swap(lambda m: not m)
        if not has_both:
            _swap(lambda m: any(t.startswith("S2-") for t in m) and any(t.startswith("S3-") for t in m))

    mini_s1 = pd.DataFrame([s1_rows[e] for e in chosen if e in s1_rows], columns=SOURCE_COLUMNS)

    # 2) Collect target rows (true matches + decoys) for both target sources.
    needed_s2 = {t for e in chosen for t in truth_map.get(e, []) if t.startswith("S2-")}
    needed_s3 = {t for e in chosen for t in truth_map.get(e, []) if t.startswith("S3-")}
    s2_rows, s2_decoys = _scan_targets(data_root, split, 2, needed_s2, quotas, rng)
    s3_rows, s3_decoys = _scan_targets(data_root, split, 3, needed_s3, quotas, rng)
    for country in quotas:
        s2_rows.extend(s2_decoys.get(country, []))
        s3_rows.extend(s3_decoys.get(country, []))
    mini_s2 = pd.DataFrame(s2_rows, columns=SOURCE_COLUMNS) if s2_rows else pd.DataFrame(columns=SOURCE_COLUMNS)
    mini_s3 = pd.DataFrame(s3_rows, columns=SOURCE_COLUMNS) if s3_rows else pd.DataFrame(columns=SOURCE_COLUMNS)

    out_dir.mkdir(parents=True, exist_ok=True)
    _write_tsv(mini_s1, out_dir / f"{split}_source1.tsv")
    _write_tsv(mini_s2, out_dir / f"{split}_source2.tsv")
    _write_tsv(mini_s3, out_dir / f"{split}_source3.tsv")

    if with_ground_truth:
        included = set(mini_s2["entity_id"]) | set(mini_s3["entity_id"])
        gt_rows = [(e, ",".join(t for t in truth_map.get(e, []) if t in included)) for e in chosen]
        mini_gt = pd.DataFrame(gt_rows, columns=GROUND_TRUTH_COLUMNS)
        _write_tsv(mini_gt, out_dir / "train_ground_truth.tsv")

    countries = mini_s1["country"].value_counts().to_dict() if not mini_s1.empty else {}
    return {"source1": len(mini_s1), "source2": len(mini_s2), "source3": len(mini_s3),
            "singletons": int(sum(1 for e in chosen if not truth_map.get(e))) if with_ground_truth else 0,
            "countries": countries}


def build_mini(
    data_root: str | Path,
    mini_root: str | Path,
    *,
    count: int = 15,
    seed: int = 2026,
) -> dict[str, object]:
    """Sample a realistic mini train/test dataset into ``mini_root``."""
    data_root = Path(data_root)
    mini_root = Path(mini_root)
    rng = random.Random(seed)

    train_summary = _build_split(data_root, mini_root / "train", "train", dict(TRAIN_QUOTA), rng, with_ground_truth=True)
    test_summary = _build_split(data_root, mini_root / "test", "test", dict(TEST_QUOTA), rng, with_ground_truth=False)
    return {"train": train_summary, "test": test_summary}


def _write_tsv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, sep="\t", index=False, encoding="utf-8", lineterminator="\n")

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/mini.py', 'IiIiQnVpbGQgYSByZWFsaXN0aWMgbWluaSBkYXRhc2V0IHRoYXQgbWlycm9ycyB0aGUgcmVhbCBjaGFsbGVuZ2UgZGlzdHJpYnV0aW9uLgoKVGhlIG1pbmkgc2V0IGlzICpzYW1wbGVkIGZyb20gdGhlIHJlYWwgZmlsZXMqIHNvIGl0IGNhcnJpZXMgdGhlIHNhbWUgc2NoZW1hLApub2lzZSAoYWJicmV2aWF0aW9ucywgdHJhbnNsaXRlcmF0aW9ucywgcmVvcmRlcmVkL2xhbmRtYXJrIGFkZHJlc3NlcywgbWlzc2luZwphZGRyZXNzZXMpLCBjb3VudHJ5IG1peCwgYW5kIG1hdGNoIGNhcmRpbmFsaXR5IOKAlCBvbmx5IHRpbnkuIEl0IGlzIHVzZWQgdG8gcnVuCnRoZSBmdWxsIHBpcGVsaW5lIGVuZC10by1lbmQgb24gYSBsYXB0b3Agd2l0aG91dCB0b3VjaGluZyB0aGUgZnVsbCBjb3JwdXMuCgpLZXkgcmVhbGlzbSBwb2ludDogRnJhbmNlIGFwcGVhcnMgb25seSBpbiB0aGUgKip0ZXN0KiogZmlsZXMsIG5vdCBpbiB0cmFpbmluZy4KVGhlIG1pbmkgc2V0IHRoZXJlZm9yZSBidWlsZHM6CiAgKiBgYHRyYWluL2BgICBmcm9tIHRoZSByZWFsIHRyYWluaW5nIGZpbGVzIChVUyArIEluZGlhKSB3aXRoIGdyb3VuZCB0cnV0aAogICogYGB0ZXN0L2BgICAgZnJvbSB0aGUgcmVhbCB0ZXN0IGZpbGVzIChVUyArIEluZGlhICsgRnJhbmNlKSwgbm8gZ3JvdW5kIHRydXRoCgpDb21wb3NpdGlvbiAoZGVmYXVsdCAxNSBTb3VyY2UgMSBlbnRpdGllcyBwZXIgc3BsaXQpOgogICogc2V2ZXJhbCBVUywgSW5kaWEgaW4gdHJhaW47IFVTICsgSW5kaWEgKyBGcmFuY2UgaW4gdGVzdAogICogYXQgbGVhc3Qgb25lIHNpbmdsZXRvbiwgc2V2ZXJhbCBzaW5nbGUtbWF0Y2gsIHNldmVyYWwgbXVsdGktbWF0Y2gsCiAgICBhbmQgYXQgbGVhc3Qgb25lIGVudGl0eSBtYXRjaGVkIGJ5IGJvdGggU291cmNlIDIgYW5kIFNvdXJjZSAzCiAgKiBTb3VyY2UgMi8zIHBvb2wgaW5jbHVkZXMgdHJ1ZSBtYXRjaGVzIHBsdXMgbm9uLW1hdGNoaW5nIGRlY295cwoKU2NhbnMgYXJlIHN0cmVhbWVkIGFuZCBzdG9wIGFzIHNvb24gYXMgZW5vdWdoIHJlY29yZHMgYXJlIGNvbGxlY3RlZCwgc28gdGhpcyBpcwpmYXN0IHJlZ2FyZGxlc3Mgb2YgdGhlIGZ1bGwgZGF0YXNldCBzaXplLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmFuZG9tCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIC5hcnRpZmFjdHMgaW1wb3J0IHdyaXRlX2ZyYW1lCmZyb20gLmRhdGEgaW1wb3J0IGdyb3VuZF90cnV0aF9wYXRoLCBpdGVyX3Rzdiwgc291cmNlX3BhdGgKZnJvbSAuc2NoZW1hcyBpbXBvcnQgR1JPVU5EX1RSVVRIX0NPTFVNTlMsIFNPVVJDRV9DT0xVTU5TCgpUUkFJTl9RVU9UQSA9IHsiVVMiOiA2LCAiSW5kaWEiOiA2fQpURVNUX1FVT1RBID0geyJVUyI6IDUsICJJbmRpYSI6IDUsICJGcmFuY2UiOiA1fQpTMV9CVUZGRVJfUEVSX0NPVU5UUlkgPSA2MApERUNPWVNfUEVSX0NPVU5UUlkgPSA0CgoKZGVmIF9wYXJzZV9pZHModmFsdWU6IHN0cikgLT4gbGlzdFtzdHJdOgogICAgcmV0dXJuIFtpdGVtIGZvciBpdGVtIGluIHN0cih2YWx1ZSkuc3BsaXQoIiwiKSBpZiBpdGVtXQoKCmRlZiBfY29sbGVjdF9zMShjaHVuaywgcXVvdGFzOiBkaWN0W3N0ciwgaW50XSwgYnVmZmVyczogZGljdFtzdHIsIGxpc3Rbc3RyXV0sIHJvd3M6IGRpY3Rbc3RyLCBkaWN0XSkgLT4gTm9uZToKICAgIGZvciByb3cgaW4gY2h1bmsuaXRlcnR1cGxlcyhpbmRleD1GYWxzZSk6CiAgICAgICAgY291bnRyeSA9IHN0cihyb3cuY291bnRyeSkKICAgICAgICBpZiBjb3VudHJ5IGluIHF1b3RhcyBhbmQgbGVuKGJ1ZmZlcnNbY291bnRyeV0pIDwgUzFfQlVGRkVSX1BFUl9DT1VOVFJZOgogICAgICAgICAgICBlbnRpdHlfaWQgPSBzdHIocm93LmVudGl0eV9pZCkKICAgICAgICAgICAgYnVmZmVyc1tjb3VudHJ5XS5hcHBlbmQoZW50aXR5X2lkKQogICAgICAgICAgICByb3dzW2VudGl0eV9pZF0gPSB7ImVudGl0eV9pZCI6IGVudGl0eV9pZCwgImJ1c2luZXNzX25hbWUiOiByb3cuYnVzaW5lc3NfbmFtZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJidXNpbmVzc19hZGRyZXNzIjogcm93LmJ1c2luZXNzX2FkZHJlc3MsICJjb3VudHJ5Ijogcm93LmNvdW50cnl9CgoKZGVmIF9idWZmZXJzX3JlYWR5KGJ1ZmZlcnM6IGRpY3Rbc3RyLCBsaXN0W3N0cl1dLCBxdW90YXM6IGRpY3Rbc3RyLCBpbnRdKSAtPiBib29sOgogICAgcmV0dXJuIGFsbChsZW4oYnVmZmVycy5nZXQoYywgW10pKSA+PSBtaW4ocXVvdGFzW2NdICogMywgUzFfQlVGRkVSX1BFUl9DT1VOVFJZKSBmb3IgYyBpbiBxdW90YXMpCgoKZGVmIF9jaG9vc2UoYnVmZmVyczogZGljdFtzdHIsIGxpc3Rbc3RyXV0sIHF1b3RhczogZGljdFtzdHIsIGludF0sIHJuZzogcmFuZG9tLlJhbmRvbSkgLT4gbGlzdFtzdHJdOgogICAgY2hvc2VuOiBsaXN0W3N0cl0gPSBbXQogICAgZm9yIGNvdW50cnksIHdhbnRlZCBpbiBxdW90YXMuaXRlbXMoKToKICAgICAgICBwb29sID0gc29ydGVkKGJ1ZmZlcnMuZ2V0KGNvdW50cnksIFtdKSkKICAgICAgICBybmcuc2h1ZmZsZShwb29sKQogICAgICAgIGNob3Nlbi5leHRlbmQocG9vbFs6d2FudGVkXSkKICAgIHJldHVybiBjaG9zZW4KCgpkZWYgX3NjYW5fdGFyZ2V0cygKICAgIGRhdGFfcm9vdDogUGF0aCwKICAgIHNwbGl0OiBzdHIsCiAgICBzb3VyY2U6IGludCwKICAgIG5lZWRlZDogc2V0W3N0cl0sCiAgICBxdW90YXM6IGRpY3Rbc3RyLCBpbnRdLAogICAgcm5nOiByYW5kb20uUmFuZG9tLAopIC0+IHR1cGxlW2xpc3RbZGljdF0sIGRpY3Rbc3RyLCBsaXN0W2RpY3RdXV06CiAgICAiIiJSZXR1cm4gKHJvd3MgZm9yIG5lZWRlZCBJRHMsIGRlY295cyBwZXIgY291bnRyeSkgZnJvbSBvbmUgdGFyZ2V0IHNvdXJjZS4iIiIKICAgIGxvb2t1cDogZGljdFtzdHIsIGRpY3RdID0ge30KICAgIGRlY295czogZGljdFtzdHIsIGxpc3RbZGljdF1dID0gZGVmYXVsdGRpY3QobGlzdCkKICAgIHJlbWFpbmluZyA9IHNldChuZWVkZWQpCiAgICBmb3IgY2h1bmsgaW4gaXRlcl90c3Yoc291cmNlX3BhdGgoZGF0YV9yb290LCBzcGxpdCwgc291cmNlKSwgU09VUkNFX0NPTFVNTlMsIGJhdGNoX3NpemU9MTAwXzAwMCk6CiAgICAgICAgaWYgcmVtYWluaW5nOgogICAgICAgICAgICBzZWxlY3RlZCA9IGNodW5rW2NodW5rWyJlbnRpdHlfaWQiXS5pc2luKHJlbWFpbmluZyldCiAgICAgICAgICAgIGZvciByb3cgaW4gc2VsZWN0ZWQuaXRlcnR1cGxlcyhpbmRleD1GYWxzZSk6CiAgICAgICAgICAgICAgICBsb29rdXBbc3RyKHJvdy5lbnRpdHlfaWQpXSA9IHsiZW50aXR5X2lkIjogc3RyKHJvdy5lbnRpdHlfaWQpLCAiYnVzaW5lc3NfbmFtZSI6IHJvdy5idXNpbmVzc19uYW1lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImJ1c2luZXNzX2FkZHJlc3MiOiByb3cuYnVzaW5lc3NfYWRkcmVzcywgImNvdW50cnkiOiByb3cuY291bnRyeX0KICAgICAgICAgICAgICAgIHJlbWFpbmluZy5kaXNjYXJkKHN0cihyb3cuZW50aXR5X2lkKSkKICAgICAgICBmb3IgY291bnRyeSBpbiBxdW90YXM6CiAgICAgICAgICAgIGlmIGxlbihkZWNveXNbY291bnRyeV0pIDwgREVDT1lTX1BFUl9DT1VOVFJZOgogICAgICAgICAgICAgICAgcG9vbCA9IGNodW5rW2NodW5rWyJjb3VudHJ5Il0uZXEoY291bnRyeSkgJiB+Y2h1bmtbImVudGl0eV9pZCJdLmlzaW4obmVlZGVkKV0KICAgICAgICAgICAgICAgIGlmIG5vdCBwb29sLmVtcHR5OgogICAgICAgICAgICAgICAgICAgIHRha2UgPSBwb29sLnNhbXBsZShtaW4oREVDT1lTX1BFUl9DT1VOVFJZIC0gbGVuKGRlY295c1tjb3VudHJ5XSksIGxlbihwb29sKSksIHJhbmRvbV9zdGF0ZT1ybmcucmFuZGludCgwLCAyKiozMSkpCiAgICAgICAgICAgICAgICAgICAgZGVjb3lzW2NvdW50cnldLmV4dGVuZCgKICAgICAgICAgICAgICAgICAgICAgICAgeyJlbnRpdHlfaWQiOiBzdHIoci5lbnRpdHlfaWQpLCAiYnVzaW5lc3NfbmFtZSI6IHIuYnVzaW5lc3NfbmFtZSwKICAgICAgICAgICAgICAgICAgICAgICAgICJidXNpbmVzc19hZGRyZXNzIjogci5idXNpbmVzc19hZGRyZXNzLCAiY291bnRyeSI6IHIuY291bnRyeX0KICAgICAgICAgICAgICAgICAgICAgICAgZm9yIHIgaW4gdGFrZS5pdGVydHVwbGVzKGluZGV4PUZhbHNlKQogICAgICAgICAgICAgICAgICAgICkKICAgICAgICBpZiBub3QgcmVtYWluaW5nIGFuZCBhbGwobGVuKGRlY295c1tjXSkgPj0gREVDT1lTX1BFUl9DT1VOVFJZIGZvciBjIGluIHF1b3Rhcyk6CiAgICAgICAgICAgIGJyZWFrCiAgICByZXR1cm4gbGlzdChsb29rdXAudmFsdWVzKCkpLCBkZWNveXMKCgpkZWYgX2J1aWxkX3NwbGl0KAogICAgZGF0YV9yb290OiBQYXRoLAogICAgb3V0X2RpcjogUGF0aCwKICAgIHNwbGl0OiBzdHIsCiAgICBxdW90YXM6IGRpY3Rbc3RyLCBpbnRdLAogICAgcm5nOiByYW5kb20uUmFuZG9tLAogICAgd2l0aF9ncm91bmRfdHJ1dGg6IGJvb2wsCikgLT4gZGljdFtzdHIsIGludF06CiAgICAjIDEpIFN0cmVhbSBTb3VyY2UgMSB1bnRpbCBlYWNoIGNvdW50cnkgYnVmZmVyIGlzIGZ1bGwuCiAgICBidWZmZXJzOiBkaWN0W3N0ciwgbGlzdFtzdHJdXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBzMV9yb3dzOiBkaWN0W3N0ciwgZGljdF0gPSB7fQogICAgZm9yIGNodW5rIGluIGl0ZXJfdHN2KHNvdXJjZV9wYXRoKGRhdGFfcm9vdCwgc3BsaXQsIDEpLCBTT1VSQ0VfQ09MVU1OUywgYmF0Y2hfc2l6ZT0xMDBfMDAwKToKICAgICAgICBfY29sbGVjdF9zMShjaHVuaywgcXVvdGFzLCBidWZmZXJzLCBzMV9yb3dzKQogICAgICAgIGlmIF9idWZmZXJzX3JlYWR5KGJ1ZmZlcnMsIHF1b3Rhcyk6CiAgICAgICAgICAgIGJyZWFrCgogICAgY2hvc2VuID0gX2Nob29zZShidWZmZXJzLCBxdW90YXMsIHJuZykKICAgIGNob3Nlbl9zZXQgPSBzZXQoY2hvc2VuKQogICAgdHJ1dGhfbWFwOiBkaWN0W3N0ciwgbGlzdFtzdHJdXSA9IHt9CgogICAgaWYgd2l0aF9ncm91bmRfdHJ1dGg6CiAgICAgICAgcG9vbCA9IHtlIGZvciBpZHMgaW4gYnVmZmVycy52YWx1ZXMoKSBmb3IgZSBpbiBpZHN9CiAgICAgICAgcmVtYWluaW5nID0gc2V0KGNob3NlbikgfCBwb29sCiAgICAgICAgZm9yIGNodW5rIGluIGl0ZXJfdHN2KGdyb3VuZF90cnV0aF9wYXRoKGRhdGFfcm9vdCksIEdST1VORF9UUlVUSF9DT0xVTU5TLCBiYXRjaF9zaXplPTIwMF8wMDApOgogICAgICAgICAgICBtYXRjaGVzID0gY2h1bmtbY2h1bmtbInNvdXJjZTFfZW50aXR5X2lkIl0uaXNpbihyZW1haW5pbmcpXQogICAgICAgICAgICBmb3Igcm93IGluIG1hdGNoZXMuaXRlcnR1cGxlcyhpbmRleD1GYWxzZSk6CiAgICAgICAgICAgICAgICB0cnV0aF9tYXBbc3RyKHJvdy5zb3VyY2UxX2VudGl0eV9pZCldID0gX3BhcnNlX2lkcyhyb3cubWF0Y2hlZF9lbnRpdHlfaWRzKQogICAgICAgICAgICAgICAgcmVtYWluaW5nLmRpc2NhcmQoc3RyKHJvdy5zb3VyY2UxX2VudGl0eV9pZCkpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGZvciBlbnRpdHlfaWQgaW4gY2hvc2VuOgogICAgICAgICAgICB0cnV0aF9tYXAuc2V0ZGVmYXVsdChlbnRpdHlfaWQsIFtdKQoKICAgICAgICAjIEd1YXJhbnRlZSBhIHNpbmdsZXRvbiBhbmQgYSBib3RoLXNvdXJjZSBtYXRjaCB3aGVuIHRoZSBidWZmZXIgYWxsb3dzLgogICAgICAgIGRlZiBfcHJvZmlsZShpZHMpOgogICAgICAgICAgICBzaW5nbGV0b24gPSBhbnkobm90IHRydXRoX21hcFtpXSBmb3IgaSBpbiBpZHMpCiAgICAgICAgICAgIGJvdGggPSBhbnkoCiAgICAgICAgICAgICAgICBhbnkodC5zdGFydHN3aXRoKCJTMi0iKSBmb3IgdCBpbiB0cnV0aF9tYXBbaV0pIGFuZCBhbnkodC5zdGFydHN3aXRoKCJTMy0iKSBmb3IgdCBpbiB0cnV0aF9tYXBbaV0pCiAgICAgICAgICAgICAgICBmb3IgaSBpbiBpZHMKICAgICAgICAgICAgKQogICAgICAgICAgICByZXR1cm4gc2luZ2xldG9uLCBib3RoCgogICAgICAgIGRlZiBfc3dhcChwcmVkaWNhdGUpIC0+IE5vbmU6CiAgICAgICAgICAgIGZvciBjb3VudHJ5IGluIHF1b3RhczoKICAgICAgICAgICAgICAgIGZvciByZXBsYWNlbWVudCBpbiBidWZmZXJzLmdldChjb3VudHJ5LCBbXSk6CiAgICAgICAgICAgICAgICAgICAgaWYgcmVwbGFjZW1lbnQgaW4gY2hvc2VuX3NldCBvciBub3QgcHJlZGljYXRlKHRydXRoX21hcFtyZXBsYWNlbWVudF0pOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIGZvciBpbmRleCBpbiByZXZlcnNlZChyYW5nZShsZW4oY2hvc2VuKSkpOgogICAgICAgICAgICAgICAgICAgICAgICBpZiBzdHIoczFfcm93cy5nZXQoY2hvc2VuW2luZGV4XSwge30pLmdldCgiY291bnRyeSIpKSA9PSBjb3VudHJ5OgogICAgICAgICAgICAgICAgICAgICAgICAgICAgY2hvc2VuX3NldC5kaXNjYXJkKGNob3NlbltpbmRleF0pCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBjaG9zZW5baW5kZXhdID0gcmVwbGFjZW1lbnQKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNob3Nlbl9zZXQuYWRkKHJlcGxhY2VtZW50KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuCgogICAgICAgIGhhc19zaW5nbGV0b24sIGhhc19ib3RoID0gX3Byb2ZpbGUoY2hvc2VuKQogICAgICAgIGlmIG5vdCBoYXNfc2luZ2xldG9uOgogICAgICAgICAgICBfc3dhcChsYW1iZGEgbTogbm90IG0pCiAgICAgICAgaWYgbm90IGhhc19ib3RoOgogICAgICAgICAgICBfc3dhcChsYW1iZGEgbTogYW55KHQuc3RhcnRzd2l0aCgiUzItIikgZm9yIHQgaW4gbSkgYW5kIGFueSh0LnN0YXJ0c3dpdGgoIlMzLSIpIGZvciB0IGluIG0pKQoKICAgIG1pbmlfczEgPSBwZC5EYXRhRnJhbWUoW3MxX3Jvd3NbZV0gZm9yIGUgaW4gY2hvc2VuIGlmIGUgaW4gczFfcm93c10sIGNvbHVtbnM9U09VUkNFX0NPTFVNTlMpCgogICAgIyAyKSBDb2xsZWN0IHRhcmdldCByb3dzICh0cnVlIG1hdGNoZXMgKyBkZWNveXMpIGZvciBib3RoIHRhcmdldCBzb3VyY2VzLgogICAgbmVlZGVkX3MyID0ge3QgZm9yIGUgaW4gY2hvc2VuIGZvciB0IGluIHRydXRoX21hcC5nZXQoZSwgW10pIGlmIHQuc3RhcnRzd2l0aCgiUzItIil9CiAgICBuZWVkZWRfczMgPSB7dCBmb3IgZSBpbiBjaG9zZW4gZm9yIHQgaW4gdHJ1dGhfbWFwLmdldChlLCBbXSkgaWYgdC5zdGFydHN3aXRoKCJTMy0iKX0KICAgIHMyX3Jvd3MsIHMyX2RlY295cyA9IF9zY2FuX3RhcmdldHMoZGF0YV9yb290LCBzcGxpdCwgMiwgbmVlZGVkX3MyLCBxdW90YXMsIHJuZykKICAgIHMzX3Jvd3MsIHMzX2RlY295cyA9IF9zY2FuX3RhcmdldHMoZGF0YV9yb290LCBzcGxpdCwgMywgbmVlZGVkX3MzLCBxdW90YXMsIHJuZykKICAgIGZvciBjb3VudHJ5IGluIHF1b3RhczoKICAgICAgICBzMl9yb3dzLmV4dGVuZChzMl9kZWNveXMuZ2V0KGNvdW50cnksIFtdKSkKICAgICAgICBzM19yb3dzLmV4dGVuZChzM19kZWNveXMuZ2V0KGNvdW50cnksIFtdKSkKICAgIG1pbmlfczIgPSBwZC5EYXRhRnJhbWUoczJfcm93cywgY29sdW1ucz1TT1VSQ0VfQ09MVU1OUykgaWYgczJfcm93cyBlbHNlIHBkLkRhdGFGcmFtZShjb2x1bW5zPVNPVVJDRV9DT0xVTU5TKQogICAgbWluaV9zMyA9IHBkLkRhdGFGcmFtZShzM19yb3dzLCBjb2x1bW5zPVNPVVJDRV9DT0xVTU5TKSBpZiBzM19yb3dzIGVsc2UgcGQuRGF0YUZyYW1lKGNvbHVtbnM9U09VUkNFX0NPTFVNTlMpCgogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBfd3JpdGVfdHN2KG1pbmlfczEsIG91dF9kaXIgLyBmIntzcGxpdH1fc291cmNlMS50c3YiKQogICAgX3dyaXRlX3RzdihtaW5pX3MyLCBvdXRfZGlyIC8gZiJ7c3BsaXR9X3NvdXJjZTIudHN2IikKICAgIF93cml0ZV90c3YobWluaV9zMywgb3V0X2RpciAvIGYie3NwbGl0fV9zb3VyY2UzLnRzdiIpCgogICAgaWYgd2l0aF9ncm91bmRfdHJ1dGg6CiAgICAgICAgaW5jbHVkZWQgPSBzZXQobWluaV9zMlsiZW50aXR5X2lkIl0pIHwgc2V0KG1pbmlfczNbImVudGl0eV9pZCJdKQogICAgICAgIGd0X3Jvd3MgPSBbKGUsICIsIi5qb2luKHQgZm9yIHQgaW4gdHJ1dGhfbWFwLmdldChlLCBbXSkgaWYgdCBpbiBpbmNsdWRlZCkpIGZvciBlIGluIGNob3Nlbl0KICAgICAgICBtaW5pX2d0ID0gcGQuRGF0YUZyYW1lKGd0X3Jvd3MsIGNvbHVtbnM9R1JPVU5EX1RSVVRIX0NPTFVNTlMpCiAgICAgICAgX3dyaXRlX3RzdihtaW5pX2d0LCBvdXRfZGlyIC8gInRyYWluX2dyb3VuZF90cnV0aC50c3YiKQoKICAgIGNvdW50cmllcyA9IG1pbmlfczFbImNvdW50cnkiXS52YWx1ZV9jb3VudHMoKS50b19kaWN0KCkgaWYgbm90IG1pbmlfczEuZW1wdHkgZWxzZSB7fQogICAgcmV0dXJuIHsic291cmNlMSI6IGxlbihtaW5pX3MxKSwgInNvdXJjZTIiOiBsZW4obWluaV9zMiksICJzb3VyY2UzIjogbGVuKG1pbmlfczMpLAogICAgICAgICAgICAic2luZ2xldG9ucyI6IGludChzdW0oMSBmb3IgZSBpbiBjaG9zZW4gaWYgbm90IHRydXRoX21hcC5nZXQoZSkpKSBpZiB3aXRoX2dyb3VuZF90cnV0aCBlbHNlIDAsCiAgICAgICAgICAgICJjb3VudHJpZXMiOiBjb3VudHJpZXN9CgoKZGVmIGJ1aWxkX21pbmkoCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBtaW5pX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICAqLAogICAgY291bnQ6IGludCA9IDE1LAogICAgc2VlZDogaW50ID0gMjAyNiwKKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgICIiIlNhbXBsZSBhIHJlYWxpc3RpYyBtaW5pIHRyYWluL3Rlc3QgZGF0YXNldCBpbnRvIGBgbWluaV9yb290YGAuIiIiCiAgICBkYXRhX3Jvb3QgPSBQYXRoKGRhdGFfcm9vdCkKICAgIG1pbmlfcm9vdCA9IFBhdGgobWluaV9yb290KQogICAgcm5nID0gcmFuZG9tLlJhbmRvbShzZWVkKQoKICAgIHRyYWluX3N1bW1hcnkgPSBfYnVpbGRfc3BsaXQoZGF0YV9yb290LCBtaW5pX3Jvb3QgLyAidHJhaW4iLCAidHJhaW4iLCBkaWN0KFRSQUlOX1FVT1RBKSwgcm5nLCB3aXRoX2dyb3VuZF90cnV0aD1UcnVlKQogICAgdGVzdF9zdW1tYXJ5ID0gX2J1aWxkX3NwbGl0KGRhdGFfcm9vdCwgbWluaV9yb290IC8gInRlc3QiLCAidGVzdCIsIGRpY3QoVEVTVF9RVU9UQSksIHJuZywgd2l0aF9ncm91bmRfdHJ1dGg9RmFsc2UpCiAgICByZXR1cm4geyJ0cmFpbiI6IHRyYWluX3N1bW1hcnksICJ0ZXN0IjogdGVzdF9zdW1tYXJ5fQoKCmRlZiBfd3JpdGVfdHN2KGZyYW1lOiBwZC5EYXRhRnJhbWUsIHBhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmcmFtZS50b19jc3YocGF0aCwgc2VwPSJcdCIsIGluZGV4PUZhbHNlLCBlbmNvZGluZz0idXRmLTgiLCBsaW5ldGVybWluYXRvcj0iXG4iKQo=', 'bd2dc5042c8452e0eae73f3d9a985cbc4debfcb8536bce331f4c4a5d0f4661f9')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/mining.py`

~~~~python
"""Hard-negative mining from out-of-fold scored candidates."""

from __future__ import annotations

import pandas as pd


def mine_hard_negatives(scored_labeled: pd.DataFrame, per_entity: int = 10, minimum_score: float = 0.0) -> pd.DataFrame:
    required = {"source1_entity_id", "candidate_entity_id", "label", "score"}
    if not required.issubset(scored_labeled.columns):
        raise ValueError(f"hard-negative table missing: {sorted(required - set(scored_labeled.columns))}")
    negatives = scored_labeled[(scored_labeled["label"] == 0) & (scored_labeled["score"] >= minimum_score)].copy()
    negatives = negatives.sort_values(
        ["source1_entity_id", "score", "candidate_entity_id"],
        ascending=[True, False, True],
        kind="mergesort",
    )
    return negatives[negatives.groupby("source1_entity_id").cumcount() < per_entity].reset_index(drop=True)


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/mining.py', 'IiIiSGFyZC1uZWdhdGl2ZSBtaW5pbmcgZnJvbSBvdXQtb2YtZm9sZCBzY29yZWQgY2FuZGlkYXRlcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBwYW5kYXMgYXMgcGQKCgpkZWYgbWluZV9oYXJkX25lZ2F0aXZlcyhzY29yZWRfbGFiZWxlZDogcGQuRGF0YUZyYW1lLCBwZXJfZW50aXR5OiBpbnQgPSAxMCwgbWluaW11bV9zY29yZTogZmxvYXQgPSAwLjApIC0+IHBkLkRhdGFGcmFtZToKICAgIHJlcXVpcmVkID0geyJzb3VyY2UxX2VudGl0eV9pZCIsICJjYW5kaWRhdGVfZW50aXR5X2lkIiwgImxhYmVsIiwgInNjb3JlIn0KICAgIGlmIG5vdCByZXF1aXJlZC5pc3N1YnNldChzY29yZWRfbGFiZWxlZC5jb2x1bW5zKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiaGFyZC1uZWdhdGl2ZSB0YWJsZSBtaXNzaW5nOiB7c29ydGVkKHJlcXVpcmVkIC0gc2V0KHNjb3JlZF9sYWJlbGVkLmNvbHVtbnMpKX0iKQogICAgbmVnYXRpdmVzID0gc2NvcmVkX2xhYmVsZWRbKHNjb3JlZF9sYWJlbGVkWyJsYWJlbCJdID09IDApICYgKHNjb3JlZF9sYWJlbGVkWyJzY29yZSJdID49IG1pbmltdW1fc2NvcmUpXS5jb3B5KCkKICAgIG5lZ2F0aXZlcyA9IG5lZ2F0aXZlcy5zb3J0X3ZhbHVlcygKICAgICAgICBbInNvdXJjZTFfZW50aXR5X2lkIiwgInNjb3JlIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiXSwKICAgICAgICBhc2NlbmRpbmc9W1RydWUsIEZhbHNlLCBUcnVlXSwKICAgICAgICBraW5kPSJtZXJnZXNvcnQiLAogICAgKQogICAgcmV0dXJuIG5lZ2F0aXZlc1tuZWdhdGl2ZXMuZ3JvdXBieSgic291cmNlMV9lbnRpdHlfaWQiKS5jdW1jb3VudCgpIDwgcGVyX2VudGl0eV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoK', '7426776c2c7b1024091147ed71f3fc1d7bc8b1cc0788ac63a2bfa254ee31e2a4')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/models/__init__.py`

~~~~python
"""Pair-scoring model implementations."""

from .deterministic import DeterministicScorer
from .sklearn_model import SGDPairModel

__all__ = ["DeterministicScorer", "SGDPairModel"]


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/models/__init__.py', 'IiIiUGFpci1zY29yaW5nIG1vZGVsIGltcGxlbWVudGF0aW9ucy4iIiIKCmZyb20gLmRldGVybWluaXN0aWMgaW1wb3J0IERldGVybWluaXN0aWNTY29yZXIKZnJvbSAuc2tsZWFybl9tb2RlbCBpbXBvcnQgU0dEUGFpck1vZGVsCgpfX2FsbF9fID0gWyJEZXRlcm1pbmlzdGljU2NvcmVyIiwgIlNHRFBhaXJNb2RlbCJdCgo=', 'c1bb37397bbc4d455b2560ebd960c00db35d610d37f236db8ce661e17bd1a513')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/models/base.py`

~~~~python
"""Pair model protocol."""

from __future__ import annotations

from typing import Protocol, Any
import numpy as np


class PairModel(Protocol):
    def fit(self, X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray | None = None, eval_data: tuple[np.ndarray, np.ndarray] | None = None) -> "PairModel": ...
    def predict_scores(self, X: np.ndarray) -> np.ndarray: ...
    def save(self, path: str) -> None: ...
    def metadata(self) -> dict[str, Any]: ...


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/models/base.py', 'IiIiUGFpciBtb2RlbCBwcm90b2NvbC4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gdHlwaW5nIGltcG9ydCBQcm90b2NvbCwgQW55CmltcG9ydCBudW1weSBhcyBucAoKCmNsYXNzIFBhaXJNb2RlbChQcm90b2NvbCk6CiAgICBkZWYgZml0KHNlbGYsIFg6IG5wLm5kYXJyYXksIHk6IG5wLm5kYXJyYXksIHNhbXBsZV93ZWlnaHQ6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSwgZXZhbF9kYXRhOiB0dXBsZVtucC5uZGFycmF5LCBucC5uZGFycmF5XSB8IE5vbmUgPSBOb25lKSAtPiAiUGFpck1vZGVsIjogLi4uCiAgICBkZWYgcHJlZGljdF9zY29yZXMoc2VsZiwgWDogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheTogLi4uCiAgICBkZWYgc2F2ZShzZWxmLCBwYXRoOiBzdHIpIC0+IE5vbmU6IC4uLgogICAgZGVmIG1ldGFkYXRhKHNlbGYpIC0+IGRpY3Rbc3RyLCBBbnldOiAuLi4KCg==', '2d34c84ae027ebdfc727d8afbd5f54e5ec86cc94729dbe936bc9ab5d310bde39')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/models/deterministic.py`

~~~~python
"""Transparent deterministic reference scorer."""

from __future__ import annotations

import json
from pathlib import Path
import numpy as np

from ..features import FEATURE_COLUMNS


def _column(frame: np.ndarray, name: str) -> np.ndarray:
    return frame[:, FEATURE_COLUMNS.index(name)]


class DeterministicScorer:
    def fit(self, X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray | None = None, eval_data=None) -> "DeterministicScorer":
        return self

    def predict_scores(self, X: np.ndarray) -> np.ndarray:
        # Reference feature by name so column insertions cannot silently break it.
        name_exact = np.maximum.reduce([
            _column(X, "name_canonical_exact"),
            _column(X, "name_compact_exact"),
            _column(X, "name_core_exact"),
        ])
        name_similarity = np.maximum(
            _column(X, "name_token_sort_ratio"),
            _column(X, "name_token_set_ratio"),
        )
        address_similarity = np.maximum.reduce([
            _column(X, "address_canonical_exact"),
            _column(X, "address_ratio"),
            _column(X, "address_token_jaccard"),
        ])
        numeric = np.maximum(_column(X, "numeric_exact"), _column(X, "numeric_jaccard"))
        score = 0.35 * name_exact + 0.30 * name_similarity + 0.25 * address_similarity + 0.10 * numeric
        return np.clip(score, 0.0, 1.0).astype(np.float32)

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(self.metadata(), indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: str | Path) -> "DeterministicScorer":
        json.loads(Path(path).read_text(encoding="utf-8"))
        return cls()

    def metadata(self) -> dict[str, object]:
        return {"kind": "deterministic", "license": "project-code"}


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/models/deterministic.py', 'IiIiVHJhbnNwYXJlbnQgZGV0ZXJtaW5pc3RpYyByZWZlcmVuY2Ugc2NvcmVyLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuLmZlYXR1cmVzIGltcG9ydCBGRUFUVVJFX0NPTFVNTlMKCgpkZWYgX2NvbHVtbihmcmFtZTogbnAubmRhcnJheSwgbmFtZTogc3RyKSAtPiBucC5uZGFycmF5OgogICAgcmV0dXJuIGZyYW1lWzosIEZFQVRVUkVfQ09MVU1OUy5pbmRleChuYW1lKV0KCgpjbGFzcyBEZXRlcm1pbmlzdGljU2NvcmVyOgogICAgZGVmIGZpdChzZWxmLCBYOiBucC5uZGFycmF5LCB5OiBucC5uZGFycmF5LCBzYW1wbGVfd2VpZ2h0OiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsIGV2YWxfZGF0YT1Ob25lKSAtPiAiRGV0ZXJtaW5pc3RpY1Njb3JlciI6CiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgcHJlZGljdF9zY29yZXMoc2VsZiwgWDogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICAjIFJlZmVyZW5jZSBmZWF0dXJlIGJ5IG5hbWUgc28gY29sdW1uIGluc2VydGlvbnMgY2Fubm90IHNpbGVudGx5IGJyZWFrIGl0LgogICAgICAgIG5hbWVfZXhhY3QgPSBucC5tYXhpbXVtLnJlZHVjZShbCiAgICAgICAgICAgIF9jb2x1bW4oWCwgIm5hbWVfY2Fub25pY2FsX2V4YWN0IiksCiAgICAgICAgICAgIF9jb2x1bW4oWCwgIm5hbWVfY29tcGFjdF9leGFjdCIpLAogICAgICAgICAgICBfY29sdW1uKFgsICJuYW1lX2NvcmVfZXhhY3QiKSwKICAgICAgICBdKQogICAgICAgIG5hbWVfc2ltaWxhcml0eSA9IG5wLm1heGltdW0oCiAgICAgICAgICAgIF9jb2x1bW4oWCwgIm5hbWVfdG9rZW5fc29ydF9yYXRpbyIpLAogICAgICAgICAgICBfY29sdW1uKFgsICJuYW1lX3Rva2VuX3NldF9yYXRpbyIpLAogICAgICAgICkKICAgICAgICBhZGRyZXNzX3NpbWlsYXJpdHkgPSBucC5tYXhpbXVtLnJlZHVjZShbCiAgICAgICAgICAgIF9jb2x1bW4oWCwgImFkZHJlc3NfY2Fub25pY2FsX2V4YWN0IiksCiAgICAgICAgICAgIF9jb2x1bW4oWCwgImFkZHJlc3NfcmF0aW8iKSwKICAgICAgICAgICAgX2NvbHVtbihYLCAiYWRkcmVzc190b2tlbl9qYWNjYXJkIiksCiAgICAgICAgXSkKICAgICAgICBudW1lcmljID0gbnAubWF4aW11bShfY29sdW1uKFgsICJudW1lcmljX2V4YWN0IiksIF9jb2x1bW4oWCwgIm51bWVyaWNfamFjY2FyZCIpKQogICAgICAgIHNjb3JlID0gMC4zNSAqIG5hbWVfZXhhY3QgKyAwLjMwICogbmFtZV9zaW1pbGFyaXR5ICsgMC4yNSAqIGFkZHJlc3Nfc2ltaWxhcml0eSArIDAuMTAgKiBudW1lcmljCiAgICAgICAgcmV0dXJuIG5wLmNsaXAoc2NvcmUsIDAuMCwgMS4wKS5hc3R5cGUobnAuZmxvYXQzMikKCiAgICBkZWYgc2F2ZShzZWxmLCBwYXRoOiBzdHIgfCBQYXRoKSAtPiBOb25lOgogICAgICAgIFBhdGgocGF0aCkud3JpdGVfdGV4dChqc29uLmR1bXBzKHNlbGYubWV0YWRhdGEoKSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGxvYWQoY2xzLCBwYXRoOiBzdHIgfCBQYXRoKSAtPiAiRGV0ZXJtaW5pc3RpY1Njb3JlciI6CiAgICAgICAganNvbi5sb2FkcyhQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICByZXR1cm4gY2xzKCkKCiAgICBkZWYgbWV0YWRhdGEoc2VsZikgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICAgICAgcmV0dXJuIHsia2luZCI6ICJkZXRlcm1pbmlzdGljIiwgImxpY2Vuc2UiOiAicHJvamVjdC1jb2RlIn0KCg==', 'f4fa9de801fd4ca1b286c19c052bbc85ecf5adf451ae8037dc8fb5fdd45035e0')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/models/lightgbm_model.py`

~~~~python
"""Optional MIT-licensed LightGBM pair-model adapter."""

from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import numpy as np


class LightGBMPairModel:
    def __init__(self, seed: int = 2026, threads: int = 1, **params: Any) -> None:
        try:
            import lightgbm as lgb
        except ImportError as exc:
            raise RuntimeError("LightGBM is optional; install requirements-remote.txt") from exc
        defaults = {
            "objective": "binary", "learning_rate": 0.05, "n_estimators": 2000,
            "num_leaves": 63, "min_child_samples": 100, "random_state": seed,
            "n_jobs": threads, "deterministic": True, "force_col_wise": True,
        }
        defaults.update(params)
        self.lgb, self.params = lgb, defaults
        self.estimator = lgb.LGBMClassifier(**defaults)
        self.booster = None

    def fit(self, X, y, sample_weight=None, eval_data=None, logger=None) -> "LightGBMPairModel":
        kwargs: dict[str, object] = {"sample_weight": sample_weight}
        callbacks: list[object] = []
        if eval_data is not None:
            kwargs.update({"eval_set": [eval_data]})
            callbacks.append(self.lgb.early_stopping(100, first_metric_only=True))
        if logger is not None:
            callbacks.append(self._logging_callback(logger))
        if callbacks:
            kwargs["callbacks"] = callbacks
        self.estimator.fit(X, y, **kwargs)
        return self

    def _logging_callback(self, logger):
        def _callback(environment) -> None:
            results = environment.evaluation_result_list or []
            for _dataset, metric, value, _higher in results:
                logger.metric(f"train_{metric}", float(value), iteration=int(environment.iteration))
        return _callback

    def predict_scores(self, X) -> np.ndarray:
        if self.booster is not None:
            return np.asarray(self.booster.predict(X), dtype=np.float32)
        return self.estimator.predict_proba(X)[:, 1].astype(np.float32)

    def save(self, path: str | Path) -> None:
        path = Path(path)
        self.estimator.booster_.save_model(str(path))
        path.with_suffix(path.suffix + ".json").write_text(json.dumps(self.metadata(), indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: str | Path) -> "LightGBMPairModel":
        path = Path(path)
        metadata = json.loads(path.with_suffix(path.suffix + ".json").read_text(encoding="utf-8"))
        obj = cls(**metadata["params"])
        # Use the stable native Booster API after loading instead of mutating
        # private sklearn-wrapper attributes that vary between LightGBM releases.
        obj.booster = obj.lgb.Booster(model_file=str(path))
        return obj

    def metadata(self) -> dict[str, object]:
        return {"kind": "lightgbm", "implementation_license": "MIT", "params": self.params}


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/models/lightgbm_model.py', 'IiIiT3B0aW9uYWwgTUlULWxpY2Vuc2VkIExpZ2h0R0JNIHBhaXItbW9kZWwgYWRhcHRlci4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CmltcG9ydCBqc29uCmltcG9ydCBudW1weSBhcyBucAoKCmNsYXNzIExpZ2h0R0JNUGFpck1vZGVsOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNlZWQ6IGludCA9IDIwMjYsIHRocmVhZHM6IGludCA9IDEsICoqcGFyYW1zOiBBbnkpIC0+IE5vbmU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBpbXBvcnQgbGlnaHRnYm0gYXMgbGdiCiAgICAgICAgZXhjZXB0IEltcG9ydEVycm9yIGFzIGV4YzoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJMaWdodEdCTSBpcyBvcHRpb25hbDsgaW5zdGFsbCByZXF1aXJlbWVudHMtcmVtb3RlLnR4dCIpIGZyb20gZXhjCiAgICAgICAgZGVmYXVsdHMgPSB7CiAgICAgICAgICAgICJvYmplY3RpdmUiOiAiYmluYXJ5IiwgImxlYXJuaW5nX3JhdGUiOiAwLjA1LCAibl9lc3RpbWF0b3JzIjogMjAwMCwKICAgICAgICAgICAgIm51bV9sZWF2ZXMiOiA2MywgIm1pbl9jaGlsZF9zYW1wbGVzIjogMTAwLCAicmFuZG9tX3N0YXRlIjogc2VlZCwKICAgICAgICAgICAgIm5fam9icyI6IHRocmVhZHMsICJkZXRlcm1pbmlzdGljIjogVHJ1ZSwgImZvcmNlX2NvbF93aXNlIjogVHJ1ZSwKICAgICAgICB9CiAgICAgICAgZGVmYXVsdHMudXBkYXRlKHBhcmFtcykKICAgICAgICBzZWxmLmxnYiwgc2VsZi5wYXJhbXMgPSBsZ2IsIGRlZmF1bHRzCiAgICAgICAgc2VsZi5lc3RpbWF0b3IgPSBsZ2IuTEdCTUNsYXNzaWZpZXIoKipkZWZhdWx0cykKICAgICAgICBzZWxmLmJvb3N0ZXIgPSBOb25lCgogICAgZGVmIGZpdChzZWxmLCBYLCB5LCBzYW1wbGVfd2VpZ2h0PU5vbmUsIGV2YWxfZGF0YT1Ob25lLCBsb2dnZXI9Tm9uZSkgLT4gIkxpZ2h0R0JNUGFpck1vZGVsIjoKICAgICAgICBrd2FyZ3M6IGRpY3Rbc3RyLCBvYmplY3RdID0geyJzYW1wbGVfd2VpZ2h0Ijogc2FtcGxlX3dlaWdodH0KICAgICAgICBjYWxsYmFja3M6IGxpc3Rbb2JqZWN0XSA9IFtdCiAgICAgICAgaWYgZXZhbF9kYXRhIGlzIG5vdCBOb25lOgogICAgICAgICAgICBrd2FyZ3MudXBkYXRlKHsiZXZhbF9zZXQiOiBbZXZhbF9kYXRhXX0pCiAgICAgICAgICAgIGNhbGxiYWNrcy5hcHBlbmQoc2VsZi5sZ2IuZWFybHlfc3RvcHBpbmcoMTAwLCBmaXJzdF9tZXRyaWNfb25seT1UcnVlKSkKICAgICAgICBpZiBsb2dnZXIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGNhbGxiYWNrcy5hcHBlbmQoc2VsZi5fbG9nZ2luZ19jYWxsYmFjayhsb2dnZXIpKQogICAgICAgIGlmIGNhbGxiYWNrczoKICAgICAgICAgICAga3dhcmdzWyJjYWxsYmFja3MiXSA9IGNhbGxiYWNrcwogICAgICAgIHNlbGYuZXN0aW1hdG9yLmZpdChYLCB5LCAqKmt3YXJncykKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiBfbG9nZ2luZ19jYWxsYmFjayhzZWxmLCBsb2dnZXIpOgogICAgICAgIGRlZiBfY2FsbGJhY2soZW52aXJvbm1lbnQpIC0+IE5vbmU6CiAgICAgICAgICAgIHJlc3VsdHMgPSBlbnZpcm9ubWVudC5ldmFsdWF0aW9uX3Jlc3VsdF9saXN0IG9yIFtdCiAgICAgICAgICAgIGZvciBfZGF0YXNldCwgbWV0cmljLCB2YWx1ZSwgX2hpZ2hlciBpbiByZXN1bHRzOgogICAgICAgICAgICAgICAgbG9nZ2VyLm1ldHJpYyhmInRyYWluX3ttZXRyaWN9IiwgZmxvYXQodmFsdWUpLCBpdGVyYXRpb249aW50KGVudmlyb25tZW50Lml0ZXJhdGlvbikpCiAgICAgICAgcmV0dXJuIF9jYWxsYmFjawoKICAgIGRlZiBwcmVkaWN0X3Njb3JlcyhzZWxmLCBYKSAtPiBucC5uZGFycmF5OgogICAgICAgIGlmIHNlbGYuYm9vc3RlciBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkoc2VsZi5ib29zdGVyLnByZWRpY3QoWCksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgcmV0dXJuIHNlbGYuZXN0aW1hdG9yLnByZWRpY3RfcHJvYmEoWClbOiwgMV0uYXN0eXBlKG5wLmZsb2F0MzIpCgogICAgZGVmIHNhdmUoc2VsZiwgcGF0aDogc3RyIHwgUGF0aCkgLT4gTm9uZToKICAgICAgICBwYXRoID0gUGF0aChwYXRoKQogICAgICAgIHNlbGYuZXN0aW1hdG9yLmJvb3N0ZXJfLnNhdmVfbW9kZWwoc3RyKHBhdGgpKQogICAgICAgIHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAiLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc2VsZi5tZXRhZGF0YSgpLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIHBhdGg6IHN0ciB8IFBhdGgpIC0+ICJMaWdodEdCTVBhaXJNb2RlbCI6CiAgICAgICAgcGF0aCA9IFBhdGgocGF0aCkKICAgICAgICBtZXRhZGF0YSA9IGpzb24ubG9hZHMocGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCArICIuanNvbiIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBvYmogPSBjbHMoKiptZXRhZGF0YVsicGFyYW1zIl0pCiAgICAgICAgIyBVc2UgdGhlIHN0YWJsZSBuYXRpdmUgQm9vc3RlciBBUEkgYWZ0ZXIgbG9hZGluZyBpbnN0ZWFkIG9mIG11dGF0aW5nCiAgICAgICAgIyBwcml2YXRlIHNrbGVhcm4td3JhcHBlciBhdHRyaWJ1dGVzIHRoYXQgdmFyeSBiZXR3ZWVuIExpZ2h0R0JNIHJlbGVhc2VzLgogICAgICAgIG9iai5ib29zdGVyID0gb2JqLmxnYi5Cb29zdGVyKG1vZGVsX2ZpbGU9c3RyKHBhdGgpKQogICAgICAgIHJldHVybiBvYmoKCiAgICBkZWYgbWV0YWRhdGEoc2VsZikgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICAgICAgcmV0dXJuIHsia2luZCI6ICJsaWdodGdibSIsICJpbXBsZW1lbnRhdGlvbl9saWNlbnNlIjogIk1JVCIsICJwYXJhbXMiOiBzZWxmLnBhcmFtc30KCg==', 'b020472c24ec64d6ddbd2af8f45f36b3faed75b30cc12221b34932ac38ea17c6')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/models/sklearn_model.py`

~~~~python
"""Incremental scikit-learn logistic pair model."""

from __future__ import annotations

from pathlib import Path
from typing import Any
import joblib
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler


class SGDPairModel:
    def __init__(self, seed: int = 2026, **params: Any) -> None:
        defaults = {"loss": "log_loss", "penalty": "l2", "alpha": 1e-4, "max_iter": 1000, "tol": 1e-4, "random_state": seed}
        defaults.update(params)
        self.estimator = SGDClassifier(**defaults)
        self.scaler = StandardScaler()
        self._scaler_fitted = False
        self.params = defaults

    def fit(self, X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray | None = None, eval_data=None, logger=None) -> "SGDPairModel":
        scaled = self.scaler.fit_transform(X)
        self._scaler_fitted = True
        self.estimator.fit(scaled, y, sample_weight=sample_weight)
        if logger is not None:
            logger.metric("train_iterations", int(getattr(self.estimator, "n_iter_", 0)))
        return self

    def partial_fit(self, X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray | None = None) -> "SGDPairModel":
        self.update_scaler(X)
        self.partial_fit_scaled(X, y, sample_weight)
        return self

    def update_scaler(self, X: np.ndarray) -> None:
        """Update scaling statistics without retaining the training batch."""
        self.scaler.partial_fit(X)
        self._scaler_fitted = True

    def partial_fit_scaled(self, X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray | None = None) -> None:
        """Train after a separate scaler pass, enabling bounded two-pass SGD."""
        if not self._scaler_fitted:
            raise RuntimeError("update_scaler must be called before partial_fit_scaled")
        scaled = self.scaler.transform(X)
        self.estimator.partial_fit(scaled, y, classes=np.array([0, 1]), sample_weight=sample_weight)

    def predict_scores(self, X: np.ndarray) -> np.ndarray:
        scaled = self.scaler.transform(X) if self._scaler_fitted else X
        return self.estimator.predict_proba(scaled)[:, 1].astype(np.float32)

    def save(self, path: str | Path) -> None:
        joblib.dump({
            "estimator": self.estimator,
            "scaler": self.scaler,
            "scaler_fitted": self._scaler_fitted,
            "params": self.params,
        }, Path(path))

    @classmethod
    def load(cls, path: str | Path) -> "SGDPairModel":
        payload = joblib.load(Path(path))
        obj = cls(**payload["params"])
        obj.estimator = payload["estimator"]
        if "scaler" in payload:
            obj.scaler = payload["scaler"]
            obj._scaler_fitted = bool(payload.get("scaler_fitted", True))
        else:
            # Backward compatibility for pre-hardening artifacts.
            obj._scaler_fitted = False
        return obj

    def metadata(self) -> dict[str, object]:
        return {
            "kind": "sgd_logistic",
            "implementation": "scikit-learn",
            "implementation_license": "BSD-3-Clause",
            "scaled": True,
            "params": self.params,
        }


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/models/sklearn_model.py', 'IiIiSW5jcmVtZW50YWwgc2Npa2l0LWxlYXJuIGxvZ2lzdGljIHBhaXIgbW9kZWwuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQppbXBvcnQgam9ibGliCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBTR0RDbGFzc2lmaWVyCmZyb20gc2tsZWFybi5wcmVwcm9jZXNzaW5nIGltcG9ydCBTdGFuZGFyZFNjYWxlcgoKCmNsYXNzIFNHRFBhaXJNb2RlbDoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzZWVkOiBpbnQgPSAyMDI2LCAqKnBhcmFtczogQW55KSAtPiBOb25lOgogICAgICAgIGRlZmF1bHRzID0geyJsb3NzIjogImxvZ19sb3NzIiwgInBlbmFsdHkiOiAibDIiLCAiYWxwaGEiOiAxZS00LCAibWF4X2l0ZXIiOiAxMDAwLCAidG9sIjogMWUtNCwgInJhbmRvbV9zdGF0ZSI6IHNlZWR9CiAgICAgICAgZGVmYXVsdHMudXBkYXRlKHBhcmFtcykKICAgICAgICBzZWxmLmVzdGltYXRvciA9IFNHRENsYXNzaWZpZXIoKipkZWZhdWx0cykKICAgICAgICBzZWxmLnNjYWxlciA9IFN0YW5kYXJkU2NhbGVyKCkKICAgICAgICBzZWxmLl9zY2FsZXJfZml0dGVkID0gRmFsc2UKICAgICAgICBzZWxmLnBhcmFtcyA9IGRlZmF1bHRzCgogICAgZGVmIGZpdChzZWxmLCBYOiBucC5uZGFycmF5LCB5OiBucC5uZGFycmF5LCBzYW1wbGVfd2VpZ2h0OiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUsIGV2YWxfZGF0YT1Ob25lLCBsb2dnZXI9Tm9uZSkgLT4gIlNHRFBhaXJNb2RlbCI6CiAgICAgICAgc2NhbGVkID0gc2VsZi5zY2FsZXIuZml0X3RyYW5zZm9ybShYKQogICAgICAgIHNlbGYuX3NjYWxlcl9maXR0ZWQgPSBUcnVlCiAgICAgICAgc2VsZi5lc3RpbWF0b3IuZml0KHNjYWxlZCwgeSwgc2FtcGxlX3dlaWdodD1zYW1wbGVfd2VpZ2h0KQogICAgICAgIGlmIGxvZ2dlciBpcyBub3QgTm9uZToKICAgICAgICAgICAgbG9nZ2VyLm1ldHJpYygidHJhaW5faXRlcmF0aW9ucyIsIGludChnZXRhdHRyKHNlbGYuZXN0aW1hdG9yLCAibl9pdGVyXyIsIDApKSkKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiBwYXJ0aWFsX2ZpdChzZWxmLCBYOiBucC5uZGFycmF5LCB5OiBucC5uZGFycmF5LCBzYW1wbGVfd2VpZ2h0OiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUpIC0+ICJTR0RQYWlyTW9kZWwiOgogICAgICAgIHNlbGYudXBkYXRlX3NjYWxlcihYKQogICAgICAgIHNlbGYucGFydGlhbF9maXRfc2NhbGVkKFgsIHksIHNhbXBsZV93ZWlnaHQpCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgdXBkYXRlX3NjYWxlcihzZWxmLCBYOiBucC5uZGFycmF5KSAtPiBOb25lOgogICAgICAgICIiIlVwZGF0ZSBzY2FsaW5nIHN0YXRpc3RpY3Mgd2l0aG91dCByZXRhaW5pbmcgdGhlIHRyYWluaW5nIGJhdGNoLiIiIgogICAgICAgIHNlbGYuc2NhbGVyLnBhcnRpYWxfZml0KFgpCiAgICAgICAgc2VsZi5fc2NhbGVyX2ZpdHRlZCA9IFRydWUKCiAgICBkZWYgcGFydGlhbF9maXRfc2NhbGVkKHNlbGYsIFg6IG5wLm5kYXJyYXksIHk6IG5wLm5kYXJyYXksIHNhbXBsZV93ZWlnaHQ6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZSkgLT4gTm9uZToKICAgICAgICAiIiJUcmFpbiBhZnRlciBhIHNlcGFyYXRlIHNjYWxlciBwYXNzLCBlbmFibGluZyBib3VuZGVkIHR3by1wYXNzIFNHRC4iIiIKICAgICAgICBpZiBub3Qgc2VsZi5fc2NhbGVyX2ZpdHRlZDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJ1cGRhdGVfc2NhbGVyIG11c3QgYmUgY2FsbGVkIGJlZm9yZSBwYXJ0aWFsX2ZpdF9zY2FsZWQiKQogICAgICAgIHNjYWxlZCA9IHNlbGYuc2NhbGVyLnRyYW5zZm9ybShYKQogICAgICAgIHNlbGYuZXN0aW1hdG9yLnBhcnRpYWxfZml0KHNjYWxlZCwgeSwgY2xhc3Nlcz1ucC5hcnJheShbMCwgMV0pLCBzYW1wbGVfd2VpZ2h0PXNhbXBsZV93ZWlnaHQpCgogICAgZGVmIHByZWRpY3Rfc2NvcmVzKHNlbGYsIFg6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgc2NhbGVkID0gc2VsZi5zY2FsZXIudHJhbnNmb3JtKFgpIGlmIHNlbGYuX3NjYWxlcl9maXR0ZWQgZWxzZSBYCiAgICAgICAgcmV0dXJuIHNlbGYuZXN0aW1hdG9yLnByZWRpY3RfcHJvYmEoc2NhbGVkKVs6LCAxXS5hc3R5cGUobnAuZmxvYXQzMikKCiAgICBkZWYgc2F2ZShzZWxmLCBwYXRoOiBzdHIgfCBQYXRoKSAtPiBOb25lOgogICAgICAgIGpvYmxpYi5kdW1wKHsKICAgICAgICAgICAgImVzdGltYXRvciI6IHNlbGYuZXN0aW1hdG9yLAogICAgICAgICAgICAic2NhbGVyIjogc2VsZi5zY2FsZXIsCiAgICAgICAgICAgICJzY2FsZXJfZml0dGVkIjogc2VsZi5fc2NhbGVyX2ZpdHRlZCwKICAgICAgICAgICAgInBhcmFtcyI6IHNlbGYucGFyYW1zLAogICAgICAgIH0sIFBhdGgocGF0aCkpCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgbG9hZChjbHMsIHBhdGg6IHN0ciB8IFBhdGgpIC0+ICJTR0RQYWlyTW9kZWwiOgogICAgICAgIHBheWxvYWQgPSBqb2JsaWIubG9hZChQYXRoKHBhdGgpKQogICAgICAgIG9iaiA9IGNscygqKnBheWxvYWRbInBhcmFtcyJdKQogICAgICAgIG9iai5lc3RpbWF0b3IgPSBwYXlsb2FkWyJlc3RpbWF0b3IiXQogICAgICAgIGlmICJzY2FsZXIiIGluIHBheWxvYWQ6CiAgICAgICAgICAgIG9iai5zY2FsZXIgPSBwYXlsb2FkWyJzY2FsZXIiXQogICAgICAgICAgICBvYmouX3NjYWxlcl9maXR0ZWQgPSBib29sKHBheWxvYWQuZ2V0KCJzY2FsZXJfZml0dGVkIiwgVHJ1ZSkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgIyBCYWNrd2FyZCBjb21wYXRpYmlsaXR5IGZvciBwcmUtaGFyZGVuaW5nIGFydGlmYWN0cy4KICAgICAgICAgICAgb2JqLl9zY2FsZXJfZml0dGVkID0gRmFsc2UKICAgICAgICByZXR1cm4gb2JqCgogICAgZGVmIG1ldGFkYXRhKHNlbGYpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJraW5kIjogInNnZF9sb2dpc3RpYyIsCiAgICAgICAgICAgICJpbXBsZW1lbnRhdGlvbiI6ICJzY2lraXQtbGVhcm4iLAogICAgICAgICAgICAiaW1wbGVtZW50YXRpb25fbGljZW5zZSI6ICJCU0QtMy1DbGF1c2UiLAogICAgICAgICAgICAic2NhbGVkIjogVHJ1ZSwKICAgICAgICAgICAgInBhcmFtcyI6IHNlbGYucGFyYW1zLAogICAgICAgIH0KCg==', '718ac845bf1df831fa70a76bbd0e2a67a6e1c3655db37560ea427036fa1a6cd2')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/normalize.py`

~~~~python
"""Deterministic, raw-preserving entity text normalization."""

from __future__ import annotations

import unicodedata
from collections.abc import Iterable
import pandas as pd

# Legal designators are an open set: the challenge test set introduces France,
# which never appears in training. Stripping these stops identical roots from
# scoring lower purely because of a mismatched designation.
LEGAL_SUFFIXES = frozenset(
    {
        # US
        "corp", "corporation", "inc", "incorporated", "llc", "llp", "lp", "co", "company",
        # India / UK-style
        "pvt", "private", "ltd", "limited", "enterprises", "enterprise", "opc",
        # France
        "sarl", "sas", "sasu", "sci", "eurl", "sa", "snc", "gie", "scop", "scp",
    }
)


def _clean_controls(value: str) -> str:
    return "".join(" " if unicodedata.category(char).startswith("C") else char for char in value)


def _unicode_tokens(value: str) -> tuple[str, ...]:
    """Tokenize letters/numbers while retaining combining marks used by Indic scripts."""
    result: list[str] = []
    current: list[str] = []
    for char in value:
        category = unicodedata.category(char)
        if category[0] in {"L", "N", "M"}:
            current.append(char)
        elif current:
            result.append("".join(current))
            current = []
    if current:
        result.append("".join(current))
    return tuple(result)


def canonical_text(value: str, ampersand_to_and: bool = True) -> str:
    value = unicodedata.normalize("NFKC", str(value or ""))
    value = _clean_controls(value).casefold()
    if ampersand_to_and:
        value = value.replace("&", " and ")
    return " ".join(_unicode_tokens(value))


def compact_text(value: str) -> str:
    return "".join(canonical_text(value).split())


def accent_folded_text(value: str) -> str:
    decomposed = unicodedata.normalize("NFKD", canonical_text(value))
    return "".join(char for char in decomposed if not unicodedata.combining(char))


def tokens(value: str) -> tuple[str, ...]:
    normalized = canonical_text(value)
    return tuple(normalized.split()) if normalized else ()


def core_name(value: str, legal_suffixes: Iterable[str] = LEGAL_SUFFIXES) -> str:
    suffixes = frozenset(legal_suffixes)
    parts = list(tokens(value))
    while parts and parts[-1] in suffixes:
        parts.pop()
    return " ".join(parts)


def token_sorted(value: str) -> str:
    return " ".join(sorted(tokens(value)))


def numeric_tokens(value: str) -> tuple[str, ...]:
    result: list[str] = []
    current: list[str] = []
    for char in unicodedata.normalize("NFKC", str(value or "")):
        if char.isdecimal():
            current.append(char)
        elif current:
            result.append("".join(current))
            current = []
    if current:
        result.append("".join(current))
    return tuple(result)


def postal_like_tokens(value: str) -> tuple[str, ...]:
    return tuple(
        token for token in tokens(value)
        if 3 <= len(token) <= 10 and any(char.isdigit() for char in token)
    )


def normalize_country(value: str) -> str:
    return canonical_text(value, ampersand_to_and=False)


def normalize_records(frame: pd.DataFrame, legal_suffixes: Iterable[str] = LEGAL_SUFFIXES) -> pd.DataFrame:
    required = {"entity_id", "business_name", "business_address", "country"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"missing columns for normalization: {sorted(missing)}")
    out = frame.copy()
    out["business_name_raw"] = out["business_name"]
    out["business_address_raw"] = out["business_address"]
    out["country_raw"] = out["country"]
    out["country_norm"] = out["country"].map(normalize_country)
    out["name_canonical"] = out["business_name"].map(canonical_text)
    out["name_compact"] = out["business_name"].map(compact_text)
    out["name_core"] = out["business_name"].map(lambda value: core_name(value, legal_suffixes))
    out["name_token_sorted"] = out["business_name"].map(token_sorted)
    out["name_accent_folded"] = out["business_name"].map(accent_folded_text)
    out["address_canonical"] = out["business_address"].map(canonical_text)
    out["address_compact"] = out["business_address"].map(compact_text)
    out["address_token_sorted"] = out["business_address"].map(token_sorted)
    out["name_tokens"] = out["business_name"].map(tokens)
    out["address_tokens"] = out["business_address"].map(tokens)
    out["numeric_tokens"] = out["business_address"].map(numeric_tokens)
    out["postal_like_tokens"] = out["business_address"].map(postal_like_tokens)
    out["missing_address"] = out["business_address"].eq("")
    return out

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/normalize.py', 'IiIiRGV0ZXJtaW5pc3RpYywgcmF3LXByZXNlcnZpbmcgZW50aXR5IHRleHQgbm9ybWFsaXphdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB1bmljb2RlZGF0YQpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgSXRlcmFibGUKaW1wb3J0IHBhbmRhcyBhcyBwZAoKIyBMZWdhbCBkZXNpZ25hdG9ycyBhcmUgYW4gb3BlbiBzZXQ6IHRoZSBjaGFsbGVuZ2UgdGVzdCBzZXQgaW50cm9kdWNlcyBGcmFuY2UsCiMgd2hpY2ggbmV2ZXIgYXBwZWFycyBpbiB0cmFpbmluZy4gU3RyaXBwaW5nIHRoZXNlIHN0b3BzIGlkZW50aWNhbCByb290cyBmcm9tCiMgc2NvcmluZyBsb3dlciBwdXJlbHkgYmVjYXVzZSBvZiBhIG1pc21hdGNoZWQgZGVzaWduYXRpb24uCkxFR0FMX1NVRkZJWEVTID0gZnJvemVuc2V0KAogICAgewogICAgICAgICMgVVMKICAgICAgICAiY29ycCIsICJjb3Jwb3JhdGlvbiIsICJpbmMiLCAiaW5jb3Jwb3JhdGVkIiwgImxsYyIsICJsbHAiLCAibHAiLCAiY28iLCAiY29tcGFueSIsCiAgICAgICAgIyBJbmRpYSAvIFVLLXN0eWxlCiAgICAgICAgInB2dCIsICJwcml2YXRlIiwgImx0ZCIsICJsaW1pdGVkIiwgImVudGVycHJpc2VzIiwgImVudGVycHJpc2UiLCAib3BjIiwKICAgICAgICAjIEZyYW5jZQogICAgICAgICJzYXJsIiwgInNhcyIsICJzYXN1IiwgInNjaSIsICJldXJsIiwgInNhIiwgInNuYyIsICJnaWUiLCAic2NvcCIsICJzY3AiLAogICAgfQopCgoKZGVmIF9jbGVhbl9jb250cm9scyh2YWx1ZTogc3RyKSAtPiBzdHI6CiAgICByZXR1cm4gIiIuam9pbigiICIgaWYgdW5pY29kZWRhdGEuY2F0ZWdvcnkoY2hhcikuc3RhcnRzd2l0aCgiQyIpIGVsc2UgY2hhciBmb3IgY2hhciBpbiB2YWx1ZSkKCgpkZWYgX3VuaWNvZGVfdG9rZW5zKHZhbHVlOiBzdHIpIC0+IHR1cGxlW3N0ciwgLi4uXToKICAgICIiIlRva2VuaXplIGxldHRlcnMvbnVtYmVycyB3aGlsZSByZXRhaW5pbmcgY29tYmluaW5nIG1hcmtzIHVzZWQgYnkgSW5kaWMgc2NyaXB0cy4iIiIKICAgIHJlc3VsdDogbGlzdFtzdHJdID0gW10KICAgIGN1cnJlbnQ6IGxpc3Rbc3RyXSA9IFtdCiAgICBmb3IgY2hhciBpbiB2YWx1ZToKICAgICAgICBjYXRlZ29yeSA9IHVuaWNvZGVkYXRhLmNhdGVnb3J5KGNoYXIpCiAgICAgICAgaWYgY2F0ZWdvcnlbMF0gaW4geyJMIiwgIk4iLCAiTSJ9OgogICAgICAgICAgICBjdXJyZW50LmFwcGVuZChjaGFyKQogICAgICAgIGVsaWYgY3VycmVudDoKICAgICAgICAgICAgcmVzdWx0LmFwcGVuZCgiIi5qb2luKGN1cnJlbnQpKQogICAgICAgICAgICBjdXJyZW50ID0gW10KICAgIGlmIGN1cnJlbnQ6CiAgICAgICAgcmVzdWx0LmFwcGVuZCgiIi5qb2luKGN1cnJlbnQpKQogICAgcmV0dXJuIHR1cGxlKHJlc3VsdCkKCgpkZWYgY2Fub25pY2FsX3RleHQodmFsdWU6IHN0ciwgYW1wZXJzYW5kX3RvX2FuZDogYm9vbCA9IFRydWUpIC0+IHN0cjoKICAgIHZhbHVlID0gdW5pY29kZWRhdGEubm9ybWFsaXplKCJORktDIiwgc3RyKHZhbHVlIG9yICIiKSkKICAgIHZhbHVlID0gX2NsZWFuX2NvbnRyb2xzKHZhbHVlKS5jYXNlZm9sZCgpCiAgICBpZiBhbXBlcnNhbmRfdG9fYW5kOgogICAgICAgIHZhbHVlID0gdmFsdWUucmVwbGFjZSgiJiIsICIgYW5kICIpCiAgICByZXR1cm4gIiAiLmpvaW4oX3VuaWNvZGVfdG9rZW5zKHZhbHVlKSkKCgpkZWYgY29tcGFjdF90ZXh0KHZhbHVlOiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiAiIi5qb2luKGNhbm9uaWNhbF90ZXh0KHZhbHVlKS5zcGxpdCgpKQoKCmRlZiBhY2NlbnRfZm9sZGVkX3RleHQodmFsdWU6IHN0cikgLT4gc3RyOgogICAgZGVjb21wb3NlZCA9IHVuaWNvZGVkYXRhLm5vcm1hbGl6ZSgiTkZLRCIsIGNhbm9uaWNhbF90ZXh0KHZhbHVlKSkKICAgIHJldHVybiAiIi5qb2luKGNoYXIgZm9yIGNoYXIgaW4gZGVjb21wb3NlZCBpZiBub3QgdW5pY29kZWRhdGEuY29tYmluaW5nKGNoYXIpKQoKCmRlZiB0b2tlbnModmFsdWU6IHN0cikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgbm9ybWFsaXplZCA9IGNhbm9uaWNhbF90ZXh0KHZhbHVlKQogICAgcmV0dXJuIHR1cGxlKG5vcm1hbGl6ZWQuc3BsaXQoKSkgaWYgbm9ybWFsaXplZCBlbHNlICgpCgoKZGVmIGNvcmVfbmFtZSh2YWx1ZTogc3RyLCBsZWdhbF9zdWZmaXhlczogSXRlcmFibGVbc3RyXSA9IExFR0FMX1NVRkZJWEVTKSAtPiBzdHI6CiAgICBzdWZmaXhlcyA9IGZyb3plbnNldChsZWdhbF9zdWZmaXhlcykKICAgIHBhcnRzID0gbGlzdCh0b2tlbnModmFsdWUpKQogICAgd2hpbGUgcGFydHMgYW5kIHBhcnRzWy0xXSBpbiBzdWZmaXhlczoKICAgICAgICBwYXJ0cy5wb3AoKQogICAgcmV0dXJuICIgIi5qb2luKHBhcnRzKQoKCmRlZiB0b2tlbl9zb3J0ZWQodmFsdWU6IHN0cikgLT4gc3RyOgogICAgcmV0dXJuICIgIi5qb2luKHNvcnRlZCh0b2tlbnModmFsdWUpKSkKCgpkZWYgbnVtZXJpY190b2tlbnModmFsdWU6IHN0cikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgcmVzdWx0OiBsaXN0W3N0cl0gPSBbXQogICAgY3VycmVudDogbGlzdFtzdHJdID0gW10KICAgIGZvciBjaGFyIGluIHVuaWNvZGVkYXRhLm5vcm1hbGl6ZSgiTkZLQyIsIHN0cih2YWx1ZSBvciAiIikpOgogICAgICAgIGlmIGNoYXIuaXNkZWNpbWFsKCk6CiAgICAgICAgICAgIGN1cnJlbnQuYXBwZW5kKGNoYXIpCiAgICAgICAgZWxpZiBjdXJyZW50OgogICAgICAgICAgICByZXN1bHQuYXBwZW5kKCIiLmpvaW4oY3VycmVudCkpCiAgICAgICAgICAgIGN1cnJlbnQgPSBbXQogICAgaWYgY3VycmVudDoKICAgICAgICByZXN1bHQuYXBwZW5kKCIiLmpvaW4oY3VycmVudCkpCiAgICByZXR1cm4gdHVwbGUocmVzdWx0KQoKCmRlZiBwb3N0YWxfbGlrZV90b2tlbnModmFsdWU6IHN0cikgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgcmV0dXJuIHR1cGxlKAogICAgICAgIHRva2VuIGZvciB0b2tlbiBpbiB0b2tlbnModmFsdWUpCiAgICAgICAgaWYgMyA8PSBsZW4odG9rZW4pIDw9IDEwIGFuZCBhbnkoY2hhci5pc2RpZ2l0KCkgZm9yIGNoYXIgaW4gdG9rZW4pCiAgICApCgoKZGVmIG5vcm1hbGl6ZV9jb3VudHJ5KHZhbHVlOiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiBjYW5vbmljYWxfdGV4dCh2YWx1ZSwgYW1wZXJzYW5kX3RvX2FuZD1GYWxzZSkKCgpkZWYgbm9ybWFsaXplX3JlY29yZHMoZnJhbWU6IHBkLkRhdGFGcmFtZSwgbGVnYWxfc3VmZml4ZXM6IEl0ZXJhYmxlW3N0cl0gPSBMRUdBTF9TVUZGSVhFUykgLT4gcGQuRGF0YUZyYW1lOgogICAgcmVxdWlyZWQgPSB7ImVudGl0eV9pZCIsICJidXNpbmVzc19uYW1lIiwgImJ1c2luZXNzX2FkZHJlc3MiLCAiY291bnRyeSJ9CiAgICBtaXNzaW5nID0gcmVxdWlyZWQgLSBzZXQoZnJhbWUuY29sdW1ucykKICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm1pc3NpbmcgY29sdW1ucyBmb3Igbm9ybWFsaXphdGlvbjoge3NvcnRlZChtaXNzaW5nKX0iKQogICAgb3V0ID0gZnJhbWUuY29weSgpCiAgICBvdXRbImJ1c2luZXNzX25hbWVfcmF3Il0gPSBvdXRbImJ1c2luZXNzX25hbWUiXQogICAgb3V0WyJidXNpbmVzc19hZGRyZXNzX3JhdyJdID0gb3V0WyJidXNpbmVzc19hZGRyZXNzIl0KICAgIG91dFsiY291bnRyeV9yYXciXSA9IG91dFsiY291bnRyeSJdCiAgICBvdXRbImNvdW50cnlfbm9ybSJdID0gb3V0WyJjb3VudHJ5Il0ubWFwKG5vcm1hbGl6ZV9jb3VudHJ5KQogICAgb3V0WyJuYW1lX2Nhbm9uaWNhbCJdID0gb3V0WyJidXNpbmVzc19uYW1lIl0ubWFwKGNhbm9uaWNhbF90ZXh0KQogICAgb3V0WyJuYW1lX2NvbXBhY3QiXSA9IG91dFsiYnVzaW5lc3NfbmFtZSJdLm1hcChjb21wYWN0X3RleHQpCiAgICBvdXRbIm5hbWVfY29yZSJdID0gb3V0WyJidXNpbmVzc19uYW1lIl0ubWFwKGxhbWJkYSB2YWx1ZTogY29yZV9uYW1lKHZhbHVlLCBsZWdhbF9zdWZmaXhlcykpCiAgICBvdXRbIm5hbWVfdG9rZW5fc29ydGVkIl0gPSBvdXRbImJ1c2luZXNzX25hbWUiXS5tYXAodG9rZW5fc29ydGVkKQogICAgb3V0WyJuYW1lX2FjY2VudF9mb2xkZWQiXSA9IG91dFsiYnVzaW5lc3NfbmFtZSJdLm1hcChhY2NlbnRfZm9sZGVkX3RleHQpCiAgICBvdXRbImFkZHJlc3NfY2Fub25pY2FsIl0gPSBvdXRbImJ1c2luZXNzX2FkZHJlc3MiXS5tYXAoY2Fub25pY2FsX3RleHQpCiAgICBvdXRbImFkZHJlc3NfY29tcGFjdCJdID0gb3V0WyJidXNpbmVzc19hZGRyZXNzIl0ubWFwKGNvbXBhY3RfdGV4dCkKICAgIG91dFsiYWRkcmVzc190b2tlbl9zb3J0ZWQiXSA9IG91dFsiYnVzaW5lc3NfYWRkcmVzcyJdLm1hcCh0b2tlbl9zb3J0ZWQpCiAgICBvdXRbIm5hbWVfdG9rZW5zIl0gPSBvdXRbImJ1c2luZXNzX25hbWUiXS5tYXAodG9rZW5zKQogICAgb3V0WyJhZGRyZXNzX3Rva2VucyJdID0gb3V0WyJidXNpbmVzc19hZGRyZXNzIl0ubWFwKHRva2VucykKICAgIG91dFsibnVtZXJpY190b2tlbnMiXSA9IG91dFsiYnVzaW5lc3NfYWRkcmVzcyJdLm1hcChudW1lcmljX3Rva2VucykKICAgIG91dFsicG9zdGFsX2xpa2VfdG9rZW5zIl0gPSBvdXRbImJ1c2luZXNzX2FkZHJlc3MiXS5tYXAocG9zdGFsX2xpa2VfdG9rZW5zKQogICAgb3V0WyJtaXNzaW5nX2FkZHJlc3MiXSA9IG91dFsiYnVzaW5lc3NfYWRkcmVzcyJdLmVxKCIiKQogICAgcmV0dXJuIG91dAo=', 'daaee40eec9215c7cb9a7b9675da536833a48159b9d662734c28399f078b4717')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/pipeline.py`

~~~~python
"""Reusable stage functions and bounded synthetic smoke workflow."""

from __future__ import annotations

from pathlib import Path
import tempfile
import numpy as np
import pandas as pd

from .blocking import CandidateGenerator
from .features import build_pair_features, feature_matrix
from .inference import infer_partition
from .labels import force_add_training_positives, ground_truth_sets, label_candidates
from .metrics import evaluate_entity_sets
from .models import SGDPairModel
from .normalize import normalize_records
from .preflight import preflight_outputs
from .submission import write_submission_outputs


def make_smoke_fixture() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    s1 = pd.DataFrame([
        ("S1-1", "Alpha & Sons Ltd", "12 River Rd, Paris 75001", "France"),
        ("S1-2", "Beta Market", "44 Main Street", "US"),
        ("S1-3", "राम मार्केटिंग प्राइवेट लिमिटेड", "10 MG Road 560001", "India"),
        ("S1-4", "Singleton Works", "9 Quiet Lane", "US"),
    ], columns=["entity_id", "business_name", "business_address", "country"])
    s2 = pd.DataFrame([
        ("S2-10", "Alpha and Sons Limited", "12 River Road Paris 75001", "France"),
        ("S2-20", "Beta Market", "44 Main St", "US"),
        ("S2-30", "Ram Marketing Pvt Ltd", "10 MG Rd 560001", "India"),
        ("S2-40", "Other Shop", "99 Elsewhere", "US"),
    ], columns=s1.columns)
    s3 = pd.DataFrame([
        ("S3-11", "Alpha & Sons", "12 River Rd 75001", "France"),
        ("S3-21", "Beta Markets", "44 Main Street", "US"),
        ("S3-31", "राम मार्केटिंग", "10 MG Road 560001", "India"),
        ("S3-41", "Singleton Worse", "9 Quiet Lane", "US"),
    ], columns=s1.columns)
    truth = pd.DataFrame([
        ("S1-1", "S2-10,S3-11"), ("S1-2", "S2-20,S3-21"),
        ("S1-3", "S2-30,S3-31"), ("S1-4", ""),
    ], columns=["source1_entity_id", "matched_entity_ids"])
    return s1, s2, s3, truth


def run_smoke(output_dir: str | Path, seed: int = 2026) -> dict[str, object]:
    output_dir = Path(output_dir)
    s1, s2, s3, gt = make_smoke_fixture()
    truth = ground_truth_sets(gt)
    ns1, n2, n3 = normalize_records(s1), normalize_records(s2), normalize_records(s3)
    blocking = {"rare_token_max_df": 20, "token_posting_cap": 20, "tfidf_top_k": 5, "per_source_cap": 20, "batch_size": 10}
    candidate_frames = [CandidateGenerator(blocking).fit(target).transform(ns1) for target in (n2, n3)]
    candidates = pd.concat(candidate_frames, ignore_index=True)
    training_candidates = label_candidates(force_add_training_positives(candidates, truth), truth)
    features = build_pair_features(training_candidates, ns1, pd.concat([n2, n3], ignore_index=True))
    X, y = feature_matrix(features), features["label"].to_numpy(dtype=np.int8)
    model = SGDPairModel(seed=seed, max_iter=2000, tol=1e-5).fit(X, y)
    predictions, candidate_sets, _, _ = infer_partition(s1, s2, s3, model, blocking, threshold=0.5)

    test_dir = output_dir / "smoke_test_data"
    test_dir.mkdir(parents=True, exist_ok=True)
    for source, frame in ((1, s1), (2, s2), (3, s3)):
        frame.to_csv(test_dir / f"test_source{source}.tsv", sep="\t", index=False, encoding="utf-8", lineterminator="\n")
    matching, candidate = write_submission_outputs(output_dir, s1["entity_id"], predictions, candidate_sets)
    errors = preflight_outputs(matching, candidate, test_dir)
    if errors:
        raise AssertionError("smoke preflight failed: " + "; ".join(errors))
    return {
        "training_pairs": len(features), "positive_pairs": int(y.sum()),
        "candidate_pairs": int(sum(len(values) for values in candidate_sets.values())),
        "metrics": evaluate_entity_sets(truth, predictions),
        "matching_path": str(matching), "candidate_path": str(candidate),
        "preflight_errors": errors,
    }


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/pipeline.py', 'IiIiUmV1c2FibGUgc3RhZ2UgZnVuY3Rpb25zIGFuZCBib3VuZGVkIHN5bnRoZXRpYyBzbW9rZSB3b3JrZmxvdy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gLmJsb2NraW5nIGltcG9ydCBDYW5kaWRhdGVHZW5lcmF0b3IKZnJvbSAuZmVhdHVyZXMgaW1wb3J0IGJ1aWxkX3BhaXJfZmVhdHVyZXMsIGZlYXR1cmVfbWF0cml4CmZyb20gLmluZmVyZW5jZSBpbXBvcnQgaW5mZXJfcGFydGl0aW9uCmZyb20gLmxhYmVscyBpbXBvcnQgZm9yY2VfYWRkX3RyYWluaW5nX3Bvc2l0aXZlcywgZ3JvdW5kX3RydXRoX3NldHMsIGxhYmVsX2NhbmRpZGF0ZXMKZnJvbSAubWV0cmljcyBpbXBvcnQgZXZhbHVhdGVfZW50aXR5X3NldHMKZnJvbSAubW9kZWxzIGltcG9ydCBTR0RQYWlyTW9kZWwKZnJvbSAubm9ybWFsaXplIGltcG9ydCBub3JtYWxpemVfcmVjb3Jkcwpmcm9tIC5wcmVmbGlnaHQgaW1wb3J0IHByZWZsaWdodF9vdXRwdXRzCmZyb20gLnN1Ym1pc3Npb24gaW1wb3J0IHdyaXRlX3N1Ym1pc3Npb25fb3V0cHV0cwoKCmRlZiBtYWtlX3Ntb2tlX2ZpeHR1cmUoKSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIHBkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lLCBwZC5EYXRhRnJhbWVdOgogICAgczEgPSBwZC5EYXRhRnJhbWUoWwogICAgICAgICgiUzEtMSIsICJBbHBoYSAmIFNvbnMgTHRkIiwgIjEyIFJpdmVyIFJkLCBQYXJpcyA3NTAwMSIsICJGcmFuY2UiKSwKICAgICAgICAoIlMxLTIiLCAiQmV0YSBNYXJrZXQiLCAiNDQgTWFpbiBTdHJlZXQiLCAiVVMiKSwKICAgICAgICAoIlMxLTMiLCAi4KSw4KS+4KSuIOCkruCkvuCksOCljeCkleClh+Ckn+Ckv+CkguCklyDgpKrgpY3gpLDgpL7gpIfgpLXgpYfgpJ8g4KSy4KS/4KSu4KS/4KSf4KWH4KShIiwgIjEwIE1HIFJvYWQgNTYwMDAxIiwgIkluZGlhIiksCiAgICAgICAgKCJTMS00IiwgIlNpbmdsZXRvbiBXb3JrcyIsICI5IFF1aWV0IExhbmUiLCAiVVMiKSwKICAgIF0sIGNvbHVtbnM9WyJlbnRpdHlfaWQiLCAiYnVzaW5lc3NfbmFtZSIsICJidXNpbmVzc19hZGRyZXNzIiwgImNvdW50cnkiXSkKICAgIHMyID0gcGQuRGF0YUZyYW1lKFsKICAgICAgICAoIlMyLTEwIiwgIkFscGhhIGFuZCBTb25zIExpbWl0ZWQiLCAiMTIgUml2ZXIgUm9hZCBQYXJpcyA3NTAwMSIsICJGcmFuY2UiKSwKICAgICAgICAoIlMyLTIwIiwgIkJldGEgTWFya2V0IiwgIjQ0IE1haW4gU3QiLCAiVVMiKSwKICAgICAgICAoIlMyLTMwIiwgIlJhbSBNYXJrZXRpbmcgUHZ0IEx0ZCIsICIxMCBNRyBSZCA1NjAwMDEiLCAiSW5kaWEiKSwKICAgICAgICAoIlMyLTQwIiwgIk90aGVyIFNob3AiLCAiOTkgRWxzZXdoZXJlIiwgIlVTIiksCiAgICBdLCBjb2x1bW5zPXMxLmNvbHVtbnMpCiAgICBzMyA9IHBkLkRhdGFGcmFtZShbCiAgICAgICAgKCJTMy0xMSIsICJBbHBoYSAmIFNvbnMiLCAiMTIgUml2ZXIgUmQgNzUwMDEiLCAiRnJhbmNlIiksCiAgICAgICAgKCJTMy0yMSIsICJCZXRhIE1hcmtldHMiLCAiNDQgTWFpbiBTdHJlZXQiLCAiVVMiKSwKICAgICAgICAoIlMzLTMxIiwgIuCksOCkvuCkriDgpK7gpL7gpLDgpY3gpJXgpYfgpJ/gpL/gpILgpJciLCAiMTAgTUcgUm9hZCA1NjAwMDEiLCAiSW5kaWEiKSwKICAgICAgICAoIlMzLTQxIiwgIlNpbmdsZXRvbiBXb3JzZSIsICI5IFF1aWV0IExhbmUiLCAiVVMiKSwKICAgIF0sIGNvbHVtbnM9czEuY29sdW1ucykKICAgIHRydXRoID0gcGQuRGF0YUZyYW1lKFsKICAgICAgICAoIlMxLTEiLCAiUzItMTAsUzMtMTEiKSwgKCJTMS0yIiwgIlMyLTIwLFMzLTIxIiksCiAgICAgICAgKCJTMS0zIiwgIlMyLTMwLFMzLTMxIiksICgiUzEtNCIsICIiKSwKICAgIF0sIGNvbHVtbnM9WyJzb3VyY2UxX2VudGl0eV9pZCIsICJtYXRjaGVkX2VudGl0eV9pZHMiXSkKICAgIHJldHVybiBzMSwgczIsIHMzLCB0cnV0aAoKCmRlZiBydW5fc21va2Uob3V0cHV0X2Rpcjogc3RyIHwgUGF0aCwgc2VlZDogaW50ID0gMjAyNikgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICBvdXRwdXRfZGlyID0gUGF0aChvdXRwdXRfZGlyKQogICAgczEsIHMyLCBzMywgZ3QgPSBtYWtlX3Ntb2tlX2ZpeHR1cmUoKQogICAgdHJ1dGggPSBncm91bmRfdHJ1dGhfc2V0cyhndCkKICAgIG5zMSwgbjIsIG4zID0gbm9ybWFsaXplX3JlY29yZHMoczEpLCBub3JtYWxpemVfcmVjb3JkcyhzMiksIG5vcm1hbGl6ZV9yZWNvcmRzKHMzKQogICAgYmxvY2tpbmcgPSB7InJhcmVfdG9rZW5fbWF4X2RmIjogMjAsICJ0b2tlbl9wb3N0aW5nX2NhcCI6IDIwLCAidGZpZGZfdG9wX2siOiA1LCAicGVyX3NvdXJjZV9jYXAiOiAyMCwgImJhdGNoX3NpemUiOiAxMH0KICAgIGNhbmRpZGF0ZV9mcmFtZXMgPSBbQ2FuZGlkYXRlR2VuZXJhdG9yKGJsb2NraW5nKS5maXQodGFyZ2V0KS50cmFuc2Zvcm0obnMxKSBmb3IgdGFyZ2V0IGluIChuMiwgbjMpXQogICAgY2FuZGlkYXRlcyA9IHBkLmNvbmNhdChjYW5kaWRhdGVfZnJhbWVzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKICAgIHRyYWluaW5nX2NhbmRpZGF0ZXMgPSBsYWJlbF9jYW5kaWRhdGVzKGZvcmNlX2FkZF90cmFpbmluZ19wb3NpdGl2ZXMoY2FuZGlkYXRlcywgdHJ1dGgpLCB0cnV0aCkKICAgIGZlYXR1cmVzID0gYnVpbGRfcGFpcl9mZWF0dXJlcyh0cmFpbmluZ19jYW5kaWRhdGVzLCBuczEsIHBkLmNvbmNhdChbbjIsIG4zXSwgaWdub3JlX2luZGV4PVRydWUpKQogICAgWCwgeSA9IGZlYXR1cmVfbWF0cml4KGZlYXR1cmVzKSwgZmVhdHVyZXNbImxhYmVsIl0udG9fbnVtcHkoZHR5cGU9bnAuaW50OCkKICAgIG1vZGVsID0gU0dEUGFpck1vZGVsKHNlZWQ9c2VlZCwgbWF4X2l0ZXI9MjAwMCwgdG9sPTFlLTUpLmZpdChYLCB5KQogICAgcHJlZGljdGlvbnMsIGNhbmRpZGF0ZV9zZXRzLCBfLCBfID0gaW5mZXJfcGFydGl0aW9uKHMxLCBzMiwgczMsIG1vZGVsLCBibG9ja2luZywgdGhyZXNob2xkPTAuNSkKCiAgICB0ZXN0X2RpciA9IG91dHB1dF9kaXIgLyAic21va2VfdGVzdF9kYXRhIgogICAgdGVzdF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZm9yIHNvdXJjZSwgZnJhbWUgaW4gKCgxLCBzMSksICgyLCBzMiksICgzLCBzMykpOgogICAgICAgIGZyYW1lLnRvX2Nzdih0ZXN0X2RpciAvIGYidGVzdF9zb3VyY2V7c291cmNlfS50c3YiLCBzZXA9Ilx0IiwgaW5kZXg9RmFsc2UsIGVuY29kaW5nPSJ1dGYtOCIsIGxpbmV0ZXJtaW5hdG9yPSJcbiIpCiAgICBtYXRjaGluZywgY2FuZGlkYXRlID0gd3JpdGVfc3VibWlzc2lvbl9vdXRwdXRzKG91dHB1dF9kaXIsIHMxWyJlbnRpdHlfaWQiXSwgcHJlZGljdGlvbnMsIGNhbmRpZGF0ZV9zZXRzKQogICAgZXJyb3JzID0gcHJlZmxpZ2h0X291dHB1dHMobWF0Y2hpbmcsIGNhbmRpZGF0ZSwgdGVzdF9kaXIpCiAgICBpZiBlcnJvcnM6CiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoInNtb2tlIHByZWZsaWdodCBmYWlsZWQ6ICIgKyAiOyAiLmpvaW4oZXJyb3JzKSkKICAgIHJldHVybiB7CiAgICAgICAgInRyYWluaW5nX3BhaXJzIjogbGVuKGZlYXR1cmVzKSwgInBvc2l0aXZlX3BhaXJzIjogaW50KHkuc3VtKCkpLAogICAgICAgICJjYW5kaWRhdGVfcGFpcnMiOiBpbnQoc3VtKGxlbih2YWx1ZXMpIGZvciB2YWx1ZXMgaW4gY2FuZGlkYXRlX3NldHMudmFsdWVzKCkpKSwKICAgICAgICAibWV0cmljcyI6IGV2YWx1YXRlX2VudGl0eV9zZXRzKHRydXRoLCBwcmVkaWN0aW9ucyksCiAgICAgICAgIm1hdGNoaW5nX3BhdGgiOiBzdHIobWF0Y2hpbmcpLCAiY2FuZGlkYXRlX3BhdGgiOiBzdHIoY2FuZGlkYXRlKSwKICAgICAgICAicHJlZmxpZ2h0X2Vycm9ycyI6IGVycm9ycywKICAgIH0KCg==', 'a8c735773721c379f773250384f8eb779a91f4aae24f92b478c6a53600b5d55b')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/preflight.py`

~~~~python
"""Strict, competition-aware output validation."""

from __future__ import annotations

import csv
from itertools import zip_longest
from pathlib import Path
import subprocess
import sys

from .schemas import CANDIDATE_COLUMNS, MATCHING_COLUMNS


def _read_source_ids(path: Path) -> set[str]:
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle, delimiter="\t")
        next(reader, None)
        return {row[0] for row in reader if row}


def _read_output(path: Path, expected_header: tuple[str, str], valid_s1: set[str], valid_targets: set[str]) -> tuple[dict[str, set[str]], list[str]]:
    errors: list[str] = []
    mapping: dict[str, set[str]] = {}
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle, delimiter="\t")
        header = next(reader, None)
        if header != list(expected_header):
            return {}, [f"{path.name}: expected exact header {list(expected_header)}, got {header}"]
        for line_number, row in enumerate(reader, start=2):
            if len(row) != 2:
                errors.append(f"{path.name}:{line_number}: expected exactly two tab-separated columns")
                continue
            source1_id, raw = row
            if source1_id in mapping:
                errors.append(f"{path.name}:{line_number}: duplicate Source 1 row {source1_id}")
            if source1_id not in valid_s1:
                errors.append(f"{path.name}:{line_number}: unknown Source 1 ID {source1_id}")
            values = [] if raw == "" else raw.split(",")
            if raw.casefold() == "nan":
                errors.append(f"{path.name}:{line_number}: literal nan is invalid")
            if len(values) != len(set(values)):
                errors.append(f"{path.name}:{line_number}: duplicate ID in list for {source1_id}")
            for value in values:
                if value.startswith("S1-") or not value.startswith(("S2-", "S3-")):
                    errors.append(f"{path.name}:{line_number}: invalid target prefix {value}")
                elif value not in valid_targets:
                    errors.append(f"{path.name}:{line_number}: unknown target ID {value}")
            mapping[source1_id] = set(values)
    missing = valid_s1 - set(mapping)
    extra = set(mapping) - valid_s1
    if missing:
        errors.append(f"{path.name}: {len(missing)} required Source 1 rows are missing")
    if extra:
        errors.append(f"{path.name}: {len(extra)} unexpected Source 1 rows")
    return mapping, errors


def preflight_outputs(matching_path: str | Path, candidate_path: str | Path, test_dir: str | Path) -> list[str]:
    """Validate both outputs in lockstep without storing candidate mappings.

    The pipeline writes the two TSVs in identical Source 1 order. Requiring that
    order here makes exact match-subset validation streaming and prevents a
    hundreds-of-millions-of-IDs candidate dictionary from exhausting memory.
    """
    matching_path, candidate_path, test_dir = Path(matching_path), Path(candidate_path), Path(test_dir)
    for path in (matching_path, candidate_path):
        if not path.is_file():
            return [f"missing output file: {path}"]
    valid_s1 = _read_source_ids(test_dir / "test_source1.tsv")
    valid_targets = _read_source_ids(test_dir / "test_source2.tsv") | _read_source_ids(test_dir / "test_source3.tsv")
    errors: list[str] = []
    seen: set[str] = set()
    with matching_path.open(encoding="utf-8", newline="") as matching_handle, candidate_path.open(
        encoding="utf-8", newline=""
    ) as candidate_handle:
        matching_reader = csv.reader(matching_handle, delimiter="\t")
        candidate_reader = csv.reader(candidate_handle, delimiter="\t")
        matching_header = next(matching_reader, None)
        candidate_header = next(candidate_reader, None)
        if matching_header != list(MATCHING_COLUMNS):
            errors.append(f"{matching_path.name}: expected exact header {list(MATCHING_COLUMNS)}, got {matching_header}")
        if candidate_header != list(CANDIDATE_COLUMNS):
            errors.append(f"{candidate_path.name}: expected exact header {list(CANDIDATE_COLUMNS)}, got {candidate_header}")
        if errors:
            return errors
        for line_number, pair in enumerate(zip_longest(matching_reader, candidate_reader), start=2):
            matching_row, candidate_row = pair
            if matching_row is None or candidate_row is None:
                errors.append("matching and candidate files have different row counts")
                break
            if len(matching_row) != 2 or len(candidate_row) != 2:
                errors.append(f"line {line_number}: each output must contain exactly two tab-separated columns")
                continue
            source1_id, raw_matches = matching_row
            candidate_source1_id, raw_candidates = candidate_row
            if source1_id != candidate_source1_id:
                errors.append(
                    f"line {line_number}: Source 1 row order differs between outputs "
                    f"({source1_id!r} != {candidate_source1_id!r})"
                )
                continue
            if source1_id in seen:
                errors.append(f"line {line_number}: duplicate Source 1 row {source1_id}")
            seen.add(source1_id)
            if source1_id not in valid_s1:
                errors.append(f"line {line_number}: unknown Source 1 ID {source1_id}")
            matched = [] if raw_matches == "" else raw_matches.split(",")
            candidates = [] if raw_candidates == "" else raw_candidates.split(",")
            for label, raw, values in (
                ("matched_entity_ids", raw_matches, matched),
                ("candidate_entity_ids", raw_candidates, candidates),
            ):
                if raw.casefold() == "nan":
                    errors.append(f"line {line_number}: literal nan is invalid in {label}")
                if len(values) != len(set(values)):
                    errors.append(f"line {line_number}: duplicate ID in {label} for {source1_id}")
                for value in values:
                    if value.startswith("S1-") or not value.startswith(("S2-", "S3-")):
                        errors.append(f"line {line_number}: invalid target prefix {value}")
                    elif value not in valid_targets:
                        errors.append(f"line {line_number}: unknown target ID {value}")
            absent = set(matched) - set(candidates)
            if absent:
                errors.append(f"{source1_id}: {len(absent)} matches are absent from candidates")
    missing = valid_s1 - seen
    if missing:
        errors.append(f"outputs: {len(missing)} required Source 1 rows are missing")
    return errors


def run_official_validator(validator: str | Path, matching: str | Path, candidate: str | Path, test_dir: str | Path, check_ids: bool = True) -> subprocess.CompletedProcess[str]:
    command = [sys.executable, str(validator), "--matching", str(matching), "--candidate", str(candidate), "--test-dir", str(test_dir)]
    if check_ids:
        command.append("--check-ids")
    return subprocess.run(command, text=True, capture_output=True, check=False)

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/preflight.py', 'IiIiU3RyaWN0LCBjb21wZXRpdGlvbi1hd2FyZSBvdXRwdXQgdmFsaWRhdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBjc3YKZnJvbSBpdGVydG9vbHMgaW1wb3J0IHppcF9sb25nZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCgpmcm9tIC5zY2hlbWFzIGltcG9ydCBDQU5ESURBVEVfQ09MVU1OUywgTUFUQ0hJTkdfQ09MVU1OUwoKCmRlZiBfcmVhZF9zb3VyY2VfaWRzKHBhdGg6IFBhdGgpIC0+IHNldFtzdHJdOgogICAgd2l0aCBwYXRoLm9wZW4oZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgaGFuZGxlOgogICAgICAgIHJlYWRlciA9IGNzdi5yZWFkZXIoaGFuZGxlLCBkZWxpbWl0ZXI9Ilx0IikKICAgICAgICBuZXh0KHJlYWRlciwgTm9uZSkKICAgICAgICByZXR1cm4ge3Jvd1swXSBmb3Igcm93IGluIHJlYWRlciBpZiByb3d9CgoKZGVmIF9yZWFkX291dHB1dChwYXRoOiBQYXRoLCBleHBlY3RlZF9oZWFkZXI6IHR1cGxlW3N0ciwgc3RyXSwgdmFsaWRfczE6IHNldFtzdHJdLCB2YWxpZF90YXJnZXRzOiBzZXRbc3RyXSkgLT4gdHVwbGVbZGljdFtzdHIsIHNldFtzdHJdXSwgbGlzdFtzdHJdXToKICAgIGVycm9yczogbGlzdFtzdHJdID0gW10KICAgIG1hcHBpbmc6IGRpY3Rbc3RyLCBzZXRbc3RyXV0gPSB7fQogICAgd2l0aCBwYXRoLm9wZW4oZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIikgYXMgaGFuZGxlOgogICAgICAgIHJlYWRlciA9IGNzdi5yZWFkZXIoaGFuZGxlLCBkZWxpbWl0ZXI9Ilx0IikKICAgICAgICBoZWFkZXIgPSBuZXh0KHJlYWRlciwgTm9uZSkKICAgICAgICBpZiBoZWFkZXIgIT0gbGlzdChleHBlY3RlZF9oZWFkZXIpOgogICAgICAgICAgICByZXR1cm4ge30sIFtmIntwYXRoLm5hbWV9OiBleHBlY3RlZCBleGFjdCBoZWFkZXIge2xpc3QoZXhwZWN0ZWRfaGVhZGVyKX0sIGdvdCB7aGVhZGVyfSJdCiAgICAgICAgZm9yIGxpbmVfbnVtYmVyLCByb3cgaW4gZW51bWVyYXRlKHJlYWRlciwgc3RhcnQ9Mik6CiAgICAgICAgICAgIGlmIGxlbihyb3cpICE9IDI6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie3BhdGgubmFtZX06e2xpbmVfbnVtYmVyfTogZXhwZWN0ZWQgZXhhY3RseSB0d28gdGFiLXNlcGFyYXRlZCBjb2x1bW5zIikKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNvdXJjZTFfaWQsIHJhdyA9IHJvdwogICAgICAgICAgICBpZiBzb3VyY2UxX2lkIGluIG1hcHBpbmc6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie3BhdGgubmFtZX06e2xpbmVfbnVtYmVyfTogZHVwbGljYXRlIFNvdXJjZSAxIHJvdyB7c291cmNlMV9pZH0iKQogICAgICAgICAgICBpZiBzb3VyY2UxX2lkIG5vdCBpbiB2YWxpZF9zMToKICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7cGF0aC5uYW1lfTp7bGluZV9udW1iZXJ9OiB1bmtub3duIFNvdXJjZSAxIElEIHtzb3VyY2UxX2lkfSIpCiAgICAgICAgICAgIHZhbHVlcyA9IFtdIGlmIHJhdyA9PSAiIiBlbHNlIHJhdy5zcGxpdCgiLCIpCiAgICAgICAgICAgIGlmIHJhdy5jYXNlZm9sZCgpID09ICJuYW4iOgogICAgICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIntwYXRoLm5hbWV9OntsaW5lX251bWJlcn06IGxpdGVyYWwgbmFuIGlzIGludmFsaWQiKQogICAgICAgICAgICBpZiBsZW4odmFsdWVzKSAhPSBsZW4oc2V0KHZhbHVlcykpOgogICAgICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIntwYXRoLm5hbWV9OntsaW5lX251bWJlcn06IGR1cGxpY2F0ZSBJRCBpbiBsaXN0IGZvciB7c291cmNlMV9pZH0iKQogICAgICAgICAgICBmb3IgdmFsdWUgaW4gdmFsdWVzOgogICAgICAgICAgICAgICAgaWYgdmFsdWUuc3RhcnRzd2l0aCgiUzEtIikgb3Igbm90IHZhbHVlLnN0YXJ0c3dpdGgoKCJTMi0iLCAiUzMtIikpOgogICAgICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7cGF0aC5uYW1lfTp7bGluZV9udW1iZXJ9OiBpbnZhbGlkIHRhcmdldCBwcmVmaXgge3ZhbHVlfSIpCiAgICAgICAgICAgICAgICBlbGlmIHZhbHVlIG5vdCBpbiB2YWxpZF90YXJnZXRzOgogICAgICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7cGF0aC5uYW1lfTp7bGluZV9udW1iZXJ9OiB1bmtub3duIHRhcmdldCBJRCB7dmFsdWV9IikKICAgICAgICAgICAgbWFwcGluZ1tzb3VyY2UxX2lkXSA9IHNldCh2YWx1ZXMpCiAgICBtaXNzaW5nID0gdmFsaWRfczEgLSBzZXQobWFwcGluZykKICAgIGV4dHJhID0gc2V0KG1hcHBpbmcpIC0gdmFsaWRfczEKICAgIGlmIG1pc3Npbmc6CiAgICAgICAgZXJyb3JzLmFwcGVuZChmIntwYXRoLm5hbWV9OiB7bGVuKG1pc3NpbmcpfSByZXF1aXJlZCBTb3VyY2UgMSByb3dzIGFyZSBtaXNzaW5nIikKICAgIGlmIGV4dHJhOgogICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7cGF0aC5uYW1lfToge2xlbihleHRyYSl9IHVuZXhwZWN0ZWQgU291cmNlIDEgcm93cyIpCiAgICByZXR1cm4gbWFwcGluZywgZXJyb3JzCgoKZGVmIHByZWZsaWdodF9vdXRwdXRzKG1hdGNoaW5nX3BhdGg6IHN0ciB8IFBhdGgsIGNhbmRpZGF0ZV9wYXRoOiBzdHIgfCBQYXRoLCB0ZXN0X2Rpcjogc3RyIHwgUGF0aCkgLT4gbGlzdFtzdHJdOgogICAgIiIiVmFsaWRhdGUgYm90aCBvdXRwdXRzIGluIGxvY2tzdGVwIHdpdGhvdXQgc3RvcmluZyBjYW5kaWRhdGUgbWFwcGluZ3MuCgogICAgVGhlIHBpcGVsaW5lIHdyaXRlcyB0aGUgdHdvIFRTVnMgaW4gaWRlbnRpY2FsIFNvdXJjZSAxIG9yZGVyLiBSZXF1aXJpbmcgdGhhdAogICAgb3JkZXIgaGVyZSBtYWtlcyBleGFjdCBtYXRjaC1zdWJzZXQgdmFsaWRhdGlvbiBzdHJlYW1pbmcgYW5kIHByZXZlbnRzIGEKICAgIGh1bmRyZWRzLW9mLW1pbGxpb25zLW9mLUlEcyBjYW5kaWRhdGUgZGljdGlvbmFyeSBmcm9tIGV4aGF1c3RpbmcgbWVtb3J5LgogICAgIiIiCiAgICBtYXRjaGluZ19wYXRoLCBjYW5kaWRhdGVfcGF0aCwgdGVzdF9kaXIgPSBQYXRoKG1hdGNoaW5nX3BhdGgpLCBQYXRoKGNhbmRpZGF0ZV9wYXRoKSwgUGF0aCh0ZXN0X2RpcikKICAgIGZvciBwYXRoIGluIChtYXRjaGluZ19wYXRoLCBjYW5kaWRhdGVfcGF0aCk6CiAgICAgICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByZXR1cm4gW2YibWlzc2luZyBvdXRwdXQgZmlsZToge3BhdGh9Il0KICAgIHZhbGlkX3MxID0gX3JlYWRfc291cmNlX2lkcyh0ZXN0X2RpciAvICJ0ZXN0X3NvdXJjZTEudHN2IikKICAgIHZhbGlkX3RhcmdldHMgPSBfcmVhZF9zb3VyY2VfaWRzKHRlc3RfZGlyIC8gInRlc3Rfc291cmNlMi50c3YiKSB8IF9yZWFkX3NvdXJjZV9pZHModGVzdF9kaXIgLyAidGVzdF9zb3VyY2UzLnRzdiIpCiAgICBlcnJvcnM6IGxpc3Rbc3RyXSA9IFtdCiAgICBzZWVuOiBzZXRbc3RyXSA9IHNldCgpCiAgICB3aXRoIG1hdGNoaW5nX3BhdGgub3BlbihlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBtYXRjaGluZ19oYW5kbGUsIGNhbmRpZGF0ZV9wYXRoLm9wZW4oCiAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iIgogICAgKSBhcyBjYW5kaWRhdGVfaGFuZGxlOgogICAgICAgIG1hdGNoaW5nX3JlYWRlciA9IGNzdi5yZWFkZXIobWF0Y2hpbmdfaGFuZGxlLCBkZWxpbWl0ZXI9Ilx0IikKICAgICAgICBjYW5kaWRhdGVfcmVhZGVyID0gY3N2LnJlYWRlcihjYW5kaWRhdGVfaGFuZGxlLCBkZWxpbWl0ZXI9Ilx0IikKICAgICAgICBtYXRjaGluZ19oZWFkZXIgPSBuZXh0KG1hdGNoaW5nX3JlYWRlciwgTm9uZSkKICAgICAgICBjYW5kaWRhdGVfaGVhZGVyID0gbmV4dChjYW5kaWRhdGVfcmVhZGVyLCBOb25lKQogICAgICAgIGlmIG1hdGNoaW5nX2hlYWRlciAhPSBsaXN0KE1BVENISU5HX0NPTFVNTlMpOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYie21hdGNoaW5nX3BhdGgubmFtZX06IGV4cGVjdGVkIGV4YWN0IGhlYWRlciB7bGlzdChNQVRDSElOR19DT0xVTU5TKX0sIGdvdCB7bWF0Y2hpbmdfaGVhZGVyfSIpCiAgICAgICAgaWYgY2FuZGlkYXRlX2hlYWRlciAhPSBsaXN0KENBTkRJREFURV9DT0xVTU5TKToKICAgICAgICAgICAgZXJyb3JzLmFwcGVuZChmIntjYW5kaWRhdGVfcGF0aC5uYW1lfTogZXhwZWN0ZWQgZXhhY3QgaGVhZGVyIHtsaXN0KENBTkRJREFURV9DT0xVTU5TKX0sIGdvdCB7Y2FuZGlkYXRlX2hlYWRlcn0iKQogICAgICAgIGlmIGVycm9yczoKICAgICAgICAgICAgcmV0dXJuIGVycm9ycwogICAgICAgIGZvciBsaW5lX251bWJlciwgcGFpciBpbiBlbnVtZXJhdGUoemlwX2xvbmdlc3QobWF0Y2hpbmdfcmVhZGVyLCBjYW5kaWRhdGVfcmVhZGVyKSwgc3RhcnQ9Mik6CiAgICAgICAgICAgIG1hdGNoaW5nX3JvdywgY2FuZGlkYXRlX3JvdyA9IHBhaXIKICAgICAgICAgICAgaWYgbWF0Y2hpbmdfcm93IGlzIE5vbmUgb3IgY2FuZGlkYXRlX3JvdyBpcyBOb25lOgogICAgICAgICAgICAgICAgZXJyb3JzLmFwcGVuZCgibWF0Y2hpbmcgYW5kIGNhbmRpZGF0ZSBmaWxlcyBoYXZlIGRpZmZlcmVudCByb3cgY291bnRzIikKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGlmIGxlbihtYXRjaGluZ19yb3cpICE9IDIgb3IgbGVuKGNhbmRpZGF0ZV9yb3cpICE9IDI6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYibGluZSB7bGluZV9udW1iZXJ9OiBlYWNoIG91dHB1dCBtdXN0IGNvbnRhaW4gZXhhY3RseSB0d28gdGFiLXNlcGFyYXRlZCBjb2x1bW5zIikKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNvdXJjZTFfaWQsIHJhd19tYXRjaGVzID0gbWF0Y2hpbmdfcm93CiAgICAgICAgICAgIGNhbmRpZGF0ZV9zb3VyY2UxX2lkLCByYXdfY2FuZGlkYXRlcyA9IGNhbmRpZGF0ZV9yb3cKICAgICAgICAgICAgaWYgc291cmNlMV9pZCAhPSBjYW5kaWRhdGVfc291cmNlMV9pZDoKICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgZiJsaW5lIHtsaW5lX251bWJlcn06IFNvdXJjZSAxIHJvdyBvcmRlciBkaWZmZXJzIGJldHdlZW4gb3V0cHV0cyAiCiAgICAgICAgICAgICAgICAgICAgZiIoe3NvdXJjZTFfaWQhcn0gIT0ge2NhbmRpZGF0ZV9zb3VyY2UxX2lkIXJ9KSIKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIHNvdXJjZTFfaWQgaW4gc2VlbjoKICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJsaW5lIHtsaW5lX251bWJlcn06IGR1cGxpY2F0ZSBTb3VyY2UgMSByb3cge3NvdXJjZTFfaWR9IikKICAgICAgICAgICAgc2Vlbi5hZGQoc291cmNlMV9pZCkKICAgICAgICAgICAgaWYgc291cmNlMV9pZCBub3QgaW4gdmFsaWRfczE6CiAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYibGluZSB7bGluZV9udW1iZXJ9OiB1bmtub3duIFNvdXJjZSAxIElEIHtzb3VyY2UxX2lkfSIpCiAgICAgICAgICAgIG1hdGNoZWQgPSBbXSBpZiByYXdfbWF0Y2hlcyA9PSAiIiBlbHNlIHJhd19tYXRjaGVzLnNwbGl0KCIsIikKICAgICAgICAgICAgY2FuZGlkYXRlcyA9IFtdIGlmIHJhd19jYW5kaWRhdGVzID09ICIiIGVsc2UgcmF3X2NhbmRpZGF0ZXMuc3BsaXQoIiwiKQogICAgICAgICAgICBmb3IgbGFiZWwsIHJhdywgdmFsdWVzIGluICgKICAgICAgICAgICAgICAgICgibWF0Y2hlZF9lbnRpdHlfaWRzIiwgcmF3X21hdGNoZXMsIG1hdGNoZWQpLAogICAgICAgICAgICAgICAgKCJjYW5kaWRhdGVfZW50aXR5X2lkcyIsIHJhd19jYW5kaWRhdGVzLCBjYW5kaWRhdGVzKSwKICAgICAgICAgICAgKToKICAgICAgICAgICAgICAgIGlmIHJhdy5jYXNlZm9sZCgpID09ICJuYW4iOgogICAgICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJsaW5lIHtsaW5lX251bWJlcn06IGxpdGVyYWwgbmFuIGlzIGludmFsaWQgaW4ge2xhYmVsfSIpCiAgICAgICAgICAgICAgICBpZiBsZW4odmFsdWVzKSAhPSBsZW4oc2V0KHZhbHVlcykpOgogICAgICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJsaW5lIHtsaW5lX251bWJlcn06IGR1cGxpY2F0ZSBJRCBpbiB7bGFiZWx9IGZvciB7c291cmNlMV9pZH0iKQogICAgICAgICAgICAgICAgZm9yIHZhbHVlIGluIHZhbHVlczoKICAgICAgICAgICAgICAgICAgICBpZiB2YWx1ZS5zdGFydHN3aXRoKCJTMS0iKSBvciBub3QgdmFsdWUuc3RhcnRzd2l0aCgoIlMyLSIsICJTMy0iKSk6CiAgICAgICAgICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJsaW5lIHtsaW5lX251bWJlcn06IGludmFsaWQgdGFyZ2V0IHByZWZpeCB7dmFsdWV9IikKICAgICAgICAgICAgICAgICAgICBlbGlmIHZhbHVlIG5vdCBpbiB2YWxpZF90YXJnZXRzOgogICAgICAgICAgICAgICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYibGluZSB7bGluZV9udW1iZXJ9OiB1bmtub3duIHRhcmdldCBJRCB7dmFsdWV9IikKICAgICAgICAgICAgYWJzZW50ID0gc2V0KG1hdGNoZWQpIC0gc2V0KGNhbmRpZGF0ZXMpCiAgICAgICAgICAgIGlmIGFic2VudDoKICAgICAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7c291cmNlMV9pZH06IHtsZW4oYWJzZW50KX0gbWF0Y2hlcyBhcmUgYWJzZW50IGZyb20gY2FuZGlkYXRlcyIpCiAgICBtaXNzaW5nID0gdmFsaWRfczEgLSBzZWVuCiAgICBpZiBtaXNzaW5nOgogICAgICAgIGVycm9ycy5hcHBlbmQoZiJvdXRwdXRzOiB7bGVuKG1pc3NpbmcpfSByZXF1aXJlZCBTb3VyY2UgMSByb3dzIGFyZSBtaXNzaW5nIikKICAgIHJldHVybiBlcnJvcnMKCgpkZWYgcnVuX29mZmljaWFsX3ZhbGlkYXRvcih2YWxpZGF0b3I6IHN0ciB8IFBhdGgsIG1hdGNoaW5nOiBzdHIgfCBQYXRoLCBjYW5kaWRhdGU6IHN0ciB8IFBhdGgsIHRlc3RfZGlyOiBzdHIgfCBQYXRoLCBjaGVja19pZHM6IGJvb2wgPSBUcnVlKSAtPiBzdWJwcm9jZXNzLkNvbXBsZXRlZFByb2Nlc3Nbc3RyXToKICAgIGNvbW1hbmQgPSBbc3lzLmV4ZWN1dGFibGUsIHN0cih2YWxpZGF0b3IpLCAiLS1tYXRjaGluZyIsIHN0cihtYXRjaGluZyksICItLWNhbmRpZGF0ZSIsIHN0cihjYW5kaWRhdGUpLCAiLS10ZXN0LWRpciIsIHN0cih0ZXN0X2RpcildCiAgICBpZiBjaGVja19pZHM6CiAgICAgICAgY29tbWFuZC5hcHBlbmQoIi0tY2hlY2staWRzIikKICAgIHJldHVybiBzdWJwcm9jZXNzLnJ1bihjb21tYW5kLCB0ZXh0PVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUsIGNoZWNrPUZhbHNlKQo=', '34b067868517b097dc7ab2f88e90fd470258858b0ffc2909352214505fcdb448')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/resources.py`

~~~~python
"""Hardware detection and adaptive resource budgeting.

Nothing here hardcodes an absolute memory figure. Chunk sizes and batch sizes
are derived from the memory actually available on the machine at run time, and
every heavy loop is wrapped in :func:`run_with_oom_backoff` so the pipeline
survives out-of-memory conditions instead of dying.
"""

from __future__ import annotations

from dataclasses import dataclass
import os
from typing import Any, Callable, TypeVar

T = TypeVar("T")


def _available_ram_bytes() -> int:
    """Best-effort available RAM in bytes, with portable fallbacks."""
    try:
        import psutil  # type: ignore

        return int(psutil.virtual_memory().available)
    except Exception:
        pass
    try:  # POSIX
        pages = os.sysconf("SC_AVPHYS_PAGES")
        page_size = os.sysconf("SC_PAGE_SIZE")
        if pages > 0 and page_size > 0:
            return int(pages) * int(page_size)
    except Exception:
        pass
    return 8 * 1024**3  # conservative fallback only if nothing else is detectable


def _cuda_info() -> tuple[bool, int]:
    """Return (cuda_available, free_vram_bytes)."""
    try:
        import torch  # type: ignore

        if not torch.cuda.is_available():
            return False, 0
        free, _total = torch.cuda.mem_get_info()
        return True, int(free)
    except Exception:
        return False, 0


@dataclass(slots=True)
class ResourcePlan:
    device: str
    ram_budget_bytes: int
    vram_budget_bytes: int
    cpu_count: int
    n_workers: int
    chunk_pairs: int
    embed_batch_size: int
    model_batch_size: int

    def as_dict(self) -> dict[str, Any]:
        return {
            "device": self.device,
            "ram_budget_bytes": self.ram_budget_bytes,
            "vram_budget_bytes": self.vram_budget_bytes,
            "cpu_count": self.cpu_count,
            "n_workers": self.n_workers,
            "chunk_pairs": self.chunk_pairs,
            "embed_batch_size": self.embed_batch_size,
            "model_batch_size": self.model_batch_size,
        }


def plan_resources(
    *,
    mode: str = "auto",
    ram_fraction: float = 0.65,
    vram_fraction: float = 0.75,
    threads: int | None = None,
    chunk_pairs: int | None = None,
    embed_batch_size: int | None = None,
) -> ResourcePlan:
    """Derive a :class:`ResourcePlan` from the live machine.

    ``mode`` is ``auto`` | ``cpu`` | ``gpu``. Explicit ``chunk_pairs`` /
    ``embed_batch_size`` values override the derived ones.
    """
    available = _available_ram_bytes()
    ram_budget = max(256 * 1024**2, int(available * float(ram_fraction)))

    cuda_available, free_vram = _cuda_info()
    if mode == "gpu" and not cuda_available:
        mode = "cpu"
    device = "cuda" if (mode == "gpu" or (mode == "auto" and cuda_available)) else "cpu"
    vram_budget = int(free_vram * float(vram_fraction)) if device == "cuda" else 0

    cpu_count = os.cpu_count() or 1
    n_workers = max(1, min(cpu_count, int(threads))) if threads else max(1, cpu_count - 1)

    # ~4 KB per candidate-pair feature row is a deliberately generous estimate
    # (strings are not held per pair; the join is the main cost). Derived, not fixed.
    derived_chunk = max(10_000, min(2_000_000, ram_budget // 4096))
    derived_embed = max(8, min(4096, (vram_budget // (1024 * 1024 * 4)) if vram_budget else (ram_budget // (1024 * 1024 * 8))))
    resolved_chunk = int(chunk_pairs) if chunk_pairs else derived_chunk
    resolved_embed = int(embed_batch_size) if embed_batch_size else derived_embed

    return ResourcePlan(
        device=device,
        ram_budget_bytes=ram_budget,
        vram_budget_bytes=vram_budget,
        cpu_count=cpu_count,
        n_workers=n_workers,
        chunk_pairs=resolved_chunk,
        embed_batch_size=resolved_embed,
        model_batch_size=max(10_000, min(2_000_000, ram_budget // 2048)),
    )


def is_oom_error(exc: BaseException) -> bool:
    if isinstance(exc, MemoryError):
        return True
    name = type(exc).__name__
    if name == "OutOfMemoryError":
        return True
    message = str(exc).lower()
    return "out of memory" in message or "cuda out of memory" in message or "cannot allocate memory" in message


def run_with_oom_backoff(
    operation: Callable[[int], T],
    size: int,
    *,
    min_size: int = 1,
    on_retry: Callable[[int, BaseException], None] | None = None,
) -> T:
    """Run ``operation(size)``; on OOM, halve ``size`` and retry down to ``min_size``.

    The caller supplies an operation that consumes the size argument (a chunk or
    batch size). This lets constrained machines complete large stages instead of
    crashing, without the code ever fixing a memory figure up front.
    """
    current = max(int(size), int(min_size))
    while True:
        try:
            return operation(current)
        except BaseException as exc:  # noqa: BLE001 - deliberately broad to survive OOM
            if not is_oom_error(exc) or current <= int(min_size):
                raise
            new_size = max(int(min_size), current // 2)
            if on_retry is not None:
                on_retry(new_size, exc)
            current = new_size

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/resources.py', 'IiIiSGFyZHdhcmUgZGV0ZWN0aW9uIGFuZCBhZGFwdGl2ZSByZXNvdXJjZSBidWRnZXRpbmcuCgpOb3RoaW5nIGhlcmUgaGFyZGNvZGVzIGFuIGFic29sdXRlIG1lbW9yeSBmaWd1cmUuIENodW5rIHNpemVzIGFuZCBiYXRjaCBzaXplcwphcmUgZGVyaXZlZCBmcm9tIHRoZSBtZW1vcnkgYWN0dWFsbHkgYXZhaWxhYmxlIG9uIHRoZSBtYWNoaW5lIGF0IHJ1biB0aW1lLCBhbmQKZXZlcnkgaGVhdnkgbG9vcCBpcyB3cmFwcGVkIGluIDpmdW5jOmBydW5fd2l0aF9vb21fYmFja29mZmAgc28gdGhlIHBpcGVsaW5lCnN1cnZpdmVzIG91dC1vZi1tZW1vcnkgY29uZGl0aW9ucyBpbnN0ZWFkIG9mIGR5aW5nLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwppbXBvcnQgb3MKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgQ2FsbGFibGUsIFR5cGVWYXIKClQgPSBUeXBlVmFyKCJUIikKCgpkZWYgX2F2YWlsYWJsZV9yYW1fYnl0ZXMoKSAtPiBpbnQ6CiAgICAiIiJCZXN0LWVmZm9ydCBhdmFpbGFibGUgUkFNIGluIGJ5dGVzLCB3aXRoIHBvcnRhYmxlIGZhbGxiYWNrcy4iIiIKICAgIHRyeToKICAgICAgICBpbXBvcnQgcHN1dGlsICAjIHR5cGU6IGlnbm9yZQoKICAgICAgICByZXR1cm4gaW50KHBzdXRpbC52aXJ0dWFsX21lbW9yeSgpLmF2YWlsYWJsZSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwogICAgdHJ5OiAgIyBQT1NJWAogICAgICAgIHBhZ2VzID0gb3Muc3lzY29uZigiU0NfQVZQSFlTX1BBR0VTIikKICAgICAgICBwYWdlX3NpemUgPSBvcy5zeXNjb25mKCJTQ19QQUdFX1NJWkUiKQogICAgICAgIGlmIHBhZ2VzID4gMCBhbmQgcGFnZV9zaXplID4gMDoKICAgICAgICAgICAgcmV0dXJuIGludChwYWdlcykgKiBpbnQocGFnZV9zaXplKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCiAgICByZXR1cm4gOCAqIDEwMjQqKjMgICMgY29uc2VydmF0aXZlIGZhbGxiYWNrIG9ubHkgaWYgbm90aGluZyBlbHNlIGlzIGRldGVjdGFibGUKCgpkZWYgX2N1ZGFfaW5mbygpIC0+IHR1cGxlW2Jvb2wsIGludF06CiAgICAiIiJSZXR1cm4gKGN1ZGFfYXZhaWxhYmxlLCBmcmVlX3ZyYW1fYnl0ZXMpLiIiIgogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaCAgIyB0eXBlOiBpZ25vcmUKCiAgICAgICAgaWYgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIHJldHVybiBGYWxzZSwgMAogICAgICAgIGZyZWUsIF90b3RhbCA9IHRvcmNoLmN1ZGEubWVtX2dldF9pbmZvKCkKICAgICAgICByZXR1cm4gVHJ1ZSwgaW50KGZyZWUpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBGYWxzZSwgMAoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUmVzb3VyY2VQbGFuOgogICAgZGV2aWNlOiBzdHIKICAgIHJhbV9idWRnZXRfYnl0ZXM6IGludAogICAgdnJhbV9idWRnZXRfYnl0ZXM6IGludAogICAgY3B1X2NvdW50OiBpbnQKICAgIG5fd29ya2VyczogaW50CiAgICBjaHVua19wYWlyczogaW50CiAgICBlbWJlZF9iYXRjaF9zaXplOiBpbnQKICAgIG1vZGVsX2JhdGNoX3NpemU6IGludAoKICAgIGRlZiBhc19kaWN0KHNlbGYpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJkZXZpY2UiOiBzZWxmLmRldmljZSwKICAgICAgICAgICAgInJhbV9idWRnZXRfYnl0ZXMiOiBzZWxmLnJhbV9idWRnZXRfYnl0ZXMsCiAgICAgICAgICAgICJ2cmFtX2J1ZGdldF9ieXRlcyI6IHNlbGYudnJhbV9idWRnZXRfYnl0ZXMsCiAgICAgICAgICAgICJjcHVfY291bnQiOiBzZWxmLmNwdV9jb3VudCwKICAgICAgICAgICAgIm5fd29ya2VycyI6IHNlbGYubl93b3JrZXJzLAogICAgICAgICAgICAiY2h1bmtfcGFpcnMiOiBzZWxmLmNodW5rX3BhaXJzLAogICAgICAgICAgICAiZW1iZWRfYmF0Y2hfc2l6ZSI6IHNlbGYuZW1iZWRfYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgIm1vZGVsX2JhdGNoX3NpemUiOiBzZWxmLm1vZGVsX2JhdGNoX3NpemUsCiAgICAgICAgfQoKCmRlZiBwbGFuX3Jlc291cmNlcygKICAgICosCiAgICBtb2RlOiBzdHIgPSAiYXV0byIsCiAgICByYW1fZnJhY3Rpb246IGZsb2F0ID0gMC42NSwKICAgIHZyYW1fZnJhY3Rpb246IGZsb2F0ID0gMC43NSwKICAgIHRocmVhZHM6IGludCB8IE5vbmUgPSBOb25lLAogICAgY2h1bmtfcGFpcnM6IGludCB8IE5vbmUgPSBOb25lLAogICAgZW1iZWRfYmF0Y2hfc2l6ZTogaW50IHwgTm9uZSA9IE5vbmUsCikgLT4gUmVzb3VyY2VQbGFuOgogICAgIiIiRGVyaXZlIGEgOmNsYXNzOmBSZXNvdXJjZVBsYW5gIGZyb20gdGhlIGxpdmUgbWFjaGluZS4KCiAgICBgYG1vZGVgYCBpcyBgYGF1dG9gYCB8IGBgY3B1YGAgfCBgYGdwdWBgLiBFeHBsaWNpdCBgYGNodW5rX3BhaXJzYGAgLwogICAgYGBlbWJlZF9iYXRjaF9zaXplYGAgdmFsdWVzIG92ZXJyaWRlIHRoZSBkZXJpdmVkIG9uZXMuCiAgICAiIiIKICAgIGF2YWlsYWJsZSA9IF9hdmFpbGFibGVfcmFtX2J5dGVzKCkKICAgIHJhbV9idWRnZXQgPSBtYXgoMjU2ICogMTAyNCoqMiwgaW50KGF2YWlsYWJsZSAqIGZsb2F0KHJhbV9mcmFjdGlvbikpKQoKICAgIGN1ZGFfYXZhaWxhYmxlLCBmcmVlX3ZyYW0gPSBfY3VkYV9pbmZvKCkKICAgIGlmIG1vZGUgPT0gImdwdSIgYW5kIG5vdCBjdWRhX2F2YWlsYWJsZToKICAgICAgICBtb2RlID0gImNwdSIKICAgIGRldmljZSA9ICJjdWRhIiBpZiAobW9kZSA9PSAiZ3B1IiBvciAobW9kZSA9PSAiYXV0byIgYW5kIGN1ZGFfYXZhaWxhYmxlKSkgZWxzZSAiY3B1IgogICAgdnJhbV9idWRnZXQgPSBpbnQoZnJlZV92cmFtICogZmxvYXQodnJhbV9mcmFjdGlvbikpIGlmIGRldmljZSA9PSAiY3VkYSIgZWxzZSAwCgogICAgY3B1X2NvdW50ID0gb3MuY3B1X2NvdW50KCkgb3IgMQogICAgbl93b3JrZXJzID0gbWF4KDEsIG1pbihjcHVfY291bnQsIGludCh0aHJlYWRzKSkpIGlmIHRocmVhZHMgZWxzZSBtYXgoMSwgY3B1X2NvdW50IC0gMSkKCiAgICAjIH40IEtCIHBlciBjYW5kaWRhdGUtcGFpciBmZWF0dXJlIHJvdyBpcyBhIGRlbGliZXJhdGVseSBnZW5lcm91cyBlc3RpbWF0ZQogICAgIyAoc3RyaW5ncyBhcmUgbm90IGhlbGQgcGVyIHBhaXI7IHRoZSBqb2luIGlzIHRoZSBtYWluIGNvc3QpLiBEZXJpdmVkLCBub3QgZml4ZWQuCiAgICBkZXJpdmVkX2NodW5rID0gbWF4KDEwXzAwMCwgbWluKDJfMDAwXzAwMCwgcmFtX2J1ZGdldCAvLyA0MDk2KSkKICAgIGRlcml2ZWRfZW1iZWQgPSBtYXgoOCwgbWluKDQwOTYsICh2cmFtX2J1ZGdldCAvLyAoMTAyNCAqIDEwMjQgKiA0KSkgaWYgdnJhbV9idWRnZXQgZWxzZSAocmFtX2J1ZGdldCAvLyAoMTAyNCAqIDEwMjQgKiA4KSkpKQogICAgcmVzb2x2ZWRfY2h1bmsgPSBpbnQoY2h1bmtfcGFpcnMpIGlmIGNodW5rX3BhaXJzIGVsc2UgZGVyaXZlZF9jaHVuawogICAgcmVzb2x2ZWRfZW1iZWQgPSBpbnQoZW1iZWRfYmF0Y2hfc2l6ZSkgaWYgZW1iZWRfYmF0Y2hfc2l6ZSBlbHNlIGRlcml2ZWRfZW1iZWQKCiAgICByZXR1cm4gUmVzb3VyY2VQbGFuKAogICAgICAgIGRldmljZT1kZXZpY2UsCiAgICAgICAgcmFtX2J1ZGdldF9ieXRlcz1yYW1fYnVkZ2V0LAogICAgICAgIHZyYW1fYnVkZ2V0X2J5dGVzPXZyYW1fYnVkZ2V0LAogICAgICAgIGNwdV9jb3VudD1jcHVfY291bnQsCiAgICAgICAgbl93b3JrZXJzPW5fd29ya2VycywKICAgICAgICBjaHVua19wYWlycz1yZXNvbHZlZF9jaHVuaywKICAgICAgICBlbWJlZF9iYXRjaF9zaXplPXJlc29sdmVkX2VtYmVkLAogICAgICAgIG1vZGVsX2JhdGNoX3NpemU9bWF4KDEwXzAwMCwgbWluKDJfMDAwXzAwMCwgcmFtX2J1ZGdldCAvLyAyMDQ4KSksCiAgICApCgoKZGVmIGlzX29vbV9lcnJvcihleGM6IEJhc2VFeGNlcHRpb24pIC0+IGJvb2w6CiAgICBpZiBpc2luc3RhbmNlKGV4YywgTWVtb3J5RXJyb3IpOgogICAgICAgIHJldHVybiBUcnVlCiAgICBuYW1lID0gdHlwZShleGMpLl9fbmFtZV9fCiAgICBpZiBuYW1lID09ICJPdXRPZk1lbW9yeUVycm9yIjoKICAgICAgICByZXR1cm4gVHJ1ZQogICAgbWVzc2FnZSA9IHN0cihleGMpLmxvd2VyKCkKICAgIHJldHVybiAib3V0IG9mIG1lbW9yeSIgaW4gbWVzc2FnZSBvciAiY3VkYSBvdXQgb2YgbWVtb3J5IiBpbiBtZXNzYWdlIG9yICJjYW5ub3QgYWxsb2NhdGUgbWVtb3J5IiBpbiBtZXNzYWdlCgoKZGVmIHJ1bl93aXRoX29vbV9iYWNrb2ZmKAogICAgb3BlcmF0aW9uOiBDYWxsYWJsZVtbaW50XSwgVF0sCiAgICBzaXplOiBpbnQsCiAgICAqLAogICAgbWluX3NpemU6IGludCA9IDEsCiAgICBvbl9yZXRyeTogQ2FsbGFibGVbW2ludCwgQmFzZUV4Y2VwdGlvbl0sIE5vbmVdIHwgTm9uZSA9IE5vbmUsCikgLT4gVDoKICAgICIiIlJ1biBgYG9wZXJhdGlvbihzaXplKWBgOyBvbiBPT00sIGhhbHZlIGBgc2l6ZWBgIGFuZCByZXRyeSBkb3duIHRvIGBgbWluX3NpemVgYC4KCiAgICBUaGUgY2FsbGVyIHN1cHBsaWVzIGFuIG9wZXJhdGlvbiB0aGF0IGNvbnN1bWVzIHRoZSBzaXplIGFyZ3VtZW50IChhIGNodW5rIG9yCiAgICBiYXRjaCBzaXplKS4gVGhpcyBsZXRzIGNvbnN0cmFpbmVkIG1hY2hpbmVzIGNvbXBsZXRlIGxhcmdlIHN0YWdlcyBpbnN0ZWFkIG9mCiAgICBjcmFzaGluZywgd2l0aG91dCB0aGUgY29kZSBldmVyIGZpeGluZyBhIG1lbW9yeSBmaWd1cmUgdXAgZnJvbnQuCiAgICAiIiIKICAgIGN1cnJlbnQgPSBtYXgoaW50KHNpemUpLCBpbnQobWluX3NpemUpKQogICAgd2hpbGUgVHJ1ZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJldHVybiBvcGVyYXRpb24oY3VycmVudCkKICAgICAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6ICAjIG5vcWE6IEJMRTAwMSAtIGRlbGliZXJhdGVseSBicm9hZCB0byBzdXJ2aXZlIE9PTQogICAgICAgICAgICBpZiBub3QgaXNfb29tX2Vycm9yKGV4Yykgb3IgY3VycmVudCA8PSBpbnQobWluX3NpemUpOgogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgbmV3X3NpemUgPSBtYXgoaW50KG1pbl9zaXplKSwgY3VycmVudCAvLyAyKQogICAgICAgICAgICBpZiBvbl9yZXRyeSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIG9uX3JldHJ5KG5ld19zaXplLCBleGMpCiAgICAgICAgICAgIGN1cnJlbnQgPSBuZXdfc2l6ZQo=', 'ddda06193226d6e52ebce24304eb3430aeaac65c41a2e70f96a456e13c1d3a3d')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/sampling.py`

~~~~python
"""Deterministic candidate-negative sampling and entity-balanced weights."""

from __future__ import annotations

import pandas as pd

from .splits import stable_hash


def sample_candidate_negatives(frame: pd.DataFrame, max_negatives_per_entity: int = 50, seed: int = 2026) -> pd.DataFrame:
    if "label" not in frame:
        raise ValueError("labeled feature table required")
    positives = frame[frame["label"] == 1]
    negatives = frame[frame["label"] == 0].copy()
    negatives["_random"] = [
        stable_hash(f"{source1_id}|{candidate_id}", seed)
        for source1_id, candidate_id in zip(negatives["source1_entity_id"], negatives["candidate_entity_id"])
    ]
    negatives = negatives.sort_values(
        ["source1_entity_id", "retrieval_score", "_random", "candidate_entity_id"],
        ascending=[True, False, True, True],
        kind="mergesort",
    )
    negatives = negatives[negatives.groupby("source1_entity_id").cumcount() < max_negatives_per_entity].drop(columns="_random")
    sampled = pd.concat([positives, negatives], ignore_index=True).sort_values(
        ["source1_entity_id", "candidate_entity_id"], kind="mergesort"
    )
    return add_entity_balanced_weights(sampled)


def add_entity_balanced_weights(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    weights = pd.Series(0.0, index=out.index)
    for _, group in out.groupby("source1_entity_id", sort=False):
        positive = group[group["label"] == 1]
        negative = group[group["label"] == 0]
        if len(positive) and len(negative):
            weights.loc[positive.index] = 0.5 / len(positive)
            weights.loc[negative.index] = 0.5 / len(negative)
        elif len(positive):
            weights.loc[positive.index] = 1.0 / len(positive)
        elif len(negative):
            weights.loc[negative.index] = 1.0 / len(negative)
    out["sample_weight"] = weights.astype("float32")
    return out

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/sampling.py', 'IiIiRGV0ZXJtaW5pc3RpYyBjYW5kaWRhdGUtbmVnYXRpdmUgc2FtcGxpbmcgYW5kIGVudGl0eS1iYWxhbmNlZCB3ZWlnaHRzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuc3BsaXRzIGltcG9ydCBzdGFibGVfaGFzaAoKCmRlZiBzYW1wbGVfY2FuZGlkYXRlX25lZ2F0aXZlcyhmcmFtZTogcGQuRGF0YUZyYW1lLCBtYXhfbmVnYXRpdmVzX3Blcl9lbnRpdHk6IGludCA9IDUwLCBzZWVkOiBpbnQgPSAyMDI2KSAtPiBwZC5EYXRhRnJhbWU6CiAgICBpZiAibGFiZWwiIG5vdCBpbiBmcmFtZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsYWJlbGVkIGZlYXR1cmUgdGFibGUgcmVxdWlyZWQiKQogICAgcG9zaXRpdmVzID0gZnJhbWVbZnJhbWVbImxhYmVsIl0gPT0gMV0KICAgIG5lZ2F0aXZlcyA9IGZyYW1lW2ZyYW1lWyJsYWJlbCJdID09IDBdLmNvcHkoKQogICAgbmVnYXRpdmVzWyJfcmFuZG9tIl0gPSBbCiAgICAgICAgc3RhYmxlX2hhc2goZiJ7c291cmNlMV9pZH18e2NhbmRpZGF0ZV9pZH0iLCBzZWVkKQogICAgICAgIGZvciBzb3VyY2UxX2lkLCBjYW5kaWRhdGVfaWQgaW4gemlwKG5lZ2F0aXZlc1sic291cmNlMV9lbnRpdHlfaWQiXSwgbmVnYXRpdmVzWyJjYW5kaWRhdGVfZW50aXR5X2lkIl0pCiAgICBdCiAgICBuZWdhdGl2ZXMgPSBuZWdhdGl2ZXMuc29ydF92YWx1ZXMoCiAgICAgICAgWyJzb3VyY2UxX2VudGl0eV9pZCIsICJyZXRyaWV2YWxfc2NvcmUiLCAiX3JhbmRvbSIsICJjYW5kaWRhdGVfZW50aXR5X2lkIl0sCiAgICAgICAgYXNjZW5kaW5nPVtUcnVlLCBGYWxzZSwgVHJ1ZSwgVHJ1ZV0sCiAgICAgICAga2luZD0ibWVyZ2Vzb3J0IiwKICAgICkKICAgIG5lZ2F0aXZlcyA9IG5lZ2F0aXZlc1tuZWdhdGl2ZXMuZ3JvdXBieSgic291cmNlMV9lbnRpdHlfaWQiKS5jdW1jb3VudCgpIDwgbWF4X25lZ2F0aXZlc19wZXJfZW50aXR5XS5kcm9wKGNvbHVtbnM9Il9yYW5kb20iKQogICAgc2FtcGxlZCA9IHBkLmNvbmNhdChbcG9zaXRpdmVzLCBuZWdhdGl2ZXNdLCBpZ25vcmVfaW5kZXg9VHJ1ZSkuc29ydF92YWx1ZXMoCiAgICAgICAgWyJzb3VyY2UxX2VudGl0eV9pZCIsICJjYW5kaWRhdGVfZW50aXR5X2lkIl0sIGtpbmQ9Im1lcmdlc29ydCIKICAgICkKICAgIHJldHVybiBhZGRfZW50aXR5X2JhbGFuY2VkX3dlaWdodHMoc2FtcGxlZCkKCgpkZWYgYWRkX2VudGl0eV9iYWxhbmNlZF93ZWlnaHRzKGZyYW1lOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgIG91dCA9IGZyYW1lLmNvcHkoKQogICAgd2VpZ2h0cyA9IHBkLlNlcmllcygwLjAsIGluZGV4PW91dC5pbmRleCkKICAgIGZvciBfLCBncm91cCBpbiBvdXQuZ3JvdXBieSgic291cmNlMV9lbnRpdHlfaWQiLCBzb3J0PUZhbHNlKToKICAgICAgICBwb3NpdGl2ZSA9IGdyb3VwW2dyb3VwWyJsYWJlbCJdID09IDFdCiAgICAgICAgbmVnYXRpdmUgPSBncm91cFtncm91cFsibGFiZWwiXSA9PSAwXQogICAgICAgIGlmIGxlbihwb3NpdGl2ZSkgYW5kIGxlbihuZWdhdGl2ZSk6CiAgICAgICAgICAgIHdlaWdodHMubG9jW3Bvc2l0aXZlLmluZGV4XSA9IDAuNSAvIGxlbihwb3NpdGl2ZSkKICAgICAgICAgICAgd2VpZ2h0cy5sb2NbbmVnYXRpdmUuaW5kZXhdID0gMC41IC8gbGVuKG5lZ2F0aXZlKQogICAgICAgIGVsaWYgbGVuKHBvc2l0aXZlKToKICAgICAgICAgICAgd2VpZ2h0cy5sb2NbcG9zaXRpdmUuaW5kZXhdID0gMS4wIC8gbGVuKHBvc2l0aXZlKQogICAgICAgIGVsaWYgbGVuKG5lZ2F0aXZlKToKICAgICAgICAgICAgd2VpZ2h0cy5sb2NbbmVnYXRpdmUuaW5kZXhdID0gMS4wIC8gbGVuKG5lZ2F0aXZlKQogICAgb3V0WyJzYW1wbGVfd2VpZ2h0Il0gPSB3ZWlnaHRzLmFzdHlwZSgiZmxvYXQzMiIpCiAgICByZXR1cm4gb3V0Cg==', 'f041a2b8542368de32fd47400932c21d5ef78ebd9e6f9bfe5430fb62789ddb05')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/schemas.py`

~~~~python
"""Shared schemas and validation constants."""

from __future__ import annotations

SOURCE_COLUMNS = ("entity_id", "business_name", "business_address", "country")
GROUND_TRUTH_COLUMNS = ("source1_entity_id", "matched_entity_ids")
MATCHING_COLUMNS = ("source1_entity_id", "matched_entity_ids")
CANDIDATE_COLUMNS = ("source1_entity_id", "candidate_entity_ids")
CANDIDATE_LONG_COLUMNS = (
    "source1_entity_id",
    "candidate_entity_id",
    "candidate_source",
    "reason_mask",
    "retrieval_score",
    "retrieval_rank",
)
SOURCE_PREFIXES = {"source1": "S1-", "source2": "S2-", "source3": "S3-"}


class SchemaError(ValueError):
    """Raised when a challenge file violates its declared schema."""


def require_columns(actual: list[str] | tuple[str, ...], expected: tuple[str, ...], label: str) -> None:
    if tuple(actual) != expected:
        raise SchemaError(f"{label}: expected columns {list(expected)}, got {list(actual)}")


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/schemas.py', 'IiIiU2hhcmVkIHNjaGVtYXMgYW5kIHZhbGlkYXRpb24gY29uc3RhbnRzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKU09VUkNFX0NPTFVNTlMgPSAoImVudGl0eV9pZCIsICJidXNpbmVzc19uYW1lIiwgImJ1c2luZXNzX2FkZHJlc3MiLCAiY291bnRyeSIpCkdST1VORF9UUlVUSF9DT0xVTU5TID0gKCJzb3VyY2UxX2VudGl0eV9pZCIsICJtYXRjaGVkX2VudGl0eV9pZHMiKQpNQVRDSElOR19DT0xVTU5TID0gKCJzb3VyY2UxX2VudGl0eV9pZCIsICJtYXRjaGVkX2VudGl0eV9pZHMiKQpDQU5ESURBVEVfQ09MVU1OUyA9ICgic291cmNlMV9lbnRpdHlfaWQiLCAiY2FuZGlkYXRlX2VudGl0eV9pZHMiKQpDQU5ESURBVEVfTE9OR19DT0xVTU5TID0gKAogICAgInNvdXJjZTFfZW50aXR5X2lkIiwKICAgICJjYW5kaWRhdGVfZW50aXR5X2lkIiwKICAgICJjYW5kaWRhdGVfc291cmNlIiwKICAgICJyZWFzb25fbWFzayIsCiAgICAicmV0cmlldmFsX3Njb3JlIiwKICAgICJyZXRyaWV2YWxfcmFuayIsCikKU09VUkNFX1BSRUZJWEVTID0geyJzb3VyY2UxIjogIlMxLSIsICJzb3VyY2UyIjogIlMyLSIsICJzb3VyY2UzIjogIlMzLSJ9CgoKY2xhc3MgU2NoZW1hRXJyb3IoVmFsdWVFcnJvcik6CiAgICAiIiJSYWlzZWQgd2hlbiBhIGNoYWxsZW5nZSBmaWxlIHZpb2xhdGVzIGl0cyBkZWNsYXJlZCBzY2hlbWEuIiIiCgoKZGVmIHJlcXVpcmVfY29sdW1ucyhhY3R1YWw6IGxpc3Rbc3RyXSB8IHR1cGxlW3N0ciwgLi4uXSwgZXhwZWN0ZWQ6IHR1cGxlW3N0ciwgLi4uXSwgbGFiZWw6IHN0cikgLT4gTm9uZToKICAgIGlmIHR1cGxlKGFjdHVhbCkgIT0gZXhwZWN0ZWQ6CiAgICAgICAgcmFpc2UgU2NoZW1hRXJyb3IoZiJ7bGFiZWx9OiBleHBlY3RlZCBjb2x1bW5zIHtsaXN0KGV4cGVjdGVkKX0sIGdvdCB7bGlzdChhY3R1YWwpfSIpCgo=', 'a43ba01cf66c8889cbdfbd1b2d7fa0dca9e9b5535ce535abc849f4ba837ceabe')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/splits.py`

~~~~python
"""Leakage-safe deterministic Source-1 fold assignment."""

from __future__ import annotations

import hashlib
import pandas as pd

from .labels import parse_match_ids


def stable_hash(value: str, seed: int = 2026) -> int:
    payload = f"{seed}:{value}".encode("utf-8")
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "big")


def _cardinality_bin(count: int) -> str:
    if count <= 2:
        return str(count)
    return "3-4" if count <= 4 else "5+"


def assign_s1_folds(source1: pd.DataFrame, ground_truth: pd.DataFrame, n_folds: int = 10, seed: int = 2026) -> pd.DataFrame:
    if n_folds < 3:
        raise ValueError("n_folds must be at least 3")
    gt = ground_truth.copy()
    gt["matches"] = gt["matched_entity_ids"].map(parse_match_ids)
    gt["cardinality"] = gt["matches"].map(len)
    gt["has_s2"] = gt["matches"].map(lambda ids: any(value.startswith("S2-") for value in ids))
    gt["has_s3"] = gt["matches"].map(lambda ids: any(value.startswith("S3-") for value in ids))
    gt["source_pattern"] = [
        "both" if s2 and s3 else "s2" if s2 else "s3" if s3 else "neither"
        for s2, s3 in zip(gt["has_s2"], gt["has_s3"])
    ]
    merged = source1[["entity_id", "country"]].merge(
        gt[["source1_entity_id", "cardinality", "source_pattern"]],
        left_on="entity_id",
        right_on="source1_entity_id",
        how="left",
        validate="one_to_one",
    )
    if merged["cardinality"].isna().any():
        raise ValueError("ground truth does not cover every Source 1 row")
    merged["cardinality"] = merged["cardinality"].astype(int)
    merged["cardinality_bin"] = merged["cardinality"].map(_cardinality_bin)
    merged["singleton"] = merged["cardinality"].eq(0)
    merged["stratum"] = (
        merged["country"].astype(str) + "|" + merged["singleton"].astype(str) + "|"
        + merged["source_pattern"] + "|" + merged["cardinality_bin"]
    )
    merged["_hash"] = merged["entity_id"].map(lambda value: stable_hash(value, seed))
    merged = merged.sort_values(["stratum", "_hash", "entity_id"], kind="mergesort")
    merged["fold"] = merged.groupby("stratum", sort=False).cumcount() % n_folds
    return merged[["entity_id", "fold", "stratum", "cardinality", "singleton", "source_pattern"]].rename(
        columns={"entity_id": "source1_entity_id"}
    )


def select_pair_fold(
    pairs: pd.DataFrame,
    folds: pd.DataFrame,
    validation_fold: int,
    *,
    validation: bool,
) -> pd.DataFrame:
    """Select training or held-out pairs by Source 1 entity, never by pair row."""
    joined = pairs.merge(
        folds[["source1_entity_id", "fold"]],
        on="source1_entity_id",
        how="left",
        validate="many_to_one",
    )
    if joined["fold"].isna().any():
        raise ValueError("some pairs have no Source 1 fold assignment")
    mask = joined["fold"].eq(validation_fold)
    return joined.loc[mask if validation else ~mask].drop(columns="fold").reset_index(drop=True)

~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/splits.py', 'IiIiTGVha2FnZS1zYWZlIGRldGVybWluaXN0aWMgU291cmNlLTEgZm9sZCBhc3NpZ25tZW50LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAubGFiZWxzIGltcG9ydCBwYXJzZV9tYXRjaF9pZHMKCgpkZWYgc3RhYmxlX2hhc2godmFsdWU6IHN0ciwgc2VlZDogaW50ID0gMjAyNikgLT4gaW50OgogICAgcGF5bG9hZCA9IGYie3NlZWR9Ont2YWx1ZX0iLmVuY29kZSgidXRmLTgiKQogICAgcmV0dXJuIGludC5mcm9tX2J5dGVzKGhhc2hsaWIuYmxha2UyYihwYXlsb2FkLCBkaWdlc3Rfc2l6ZT04KS5kaWdlc3QoKSwgImJpZyIpCgoKZGVmIF9jYXJkaW5hbGl0eV9iaW4oY291bnQ6IGludCkgLT4gc3RyOgogICAgaWYgY291bnQgPD0gMjoKICAgICAgICByZXR1cm4gc3RyKGNvdW50KQogICAgcmV0dXJuICIzLTQiIGlmIGNvdW50IDw9IDQgZWxzZSAiNSsiCgoKZGVmIGFzc2lnbl9zMV9mb2xkcyhzb3VyY2UxOiBwZC5EYXRhRnJhbWUsIGdyb3VuZF90cnV0aDogcGQuRGF0YUZyYW1lLCBuX2ZvbGRzOiBpbnQgPSAxMCwgc2VlZDogaW50ID0gMjAyNikgLT4gcGQuRGF0YUZyYW1lOgogICAgaWYgbl9mb2xkcyA8IDM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibl9mb2xkcyBtdXN0IGJlIGF0IGxlYXN0IDMiKQogICAgZ3QgPSBncm91bmRfdHJ1dGguY29weSgpCiAgICBndFsibWF0Y2hlcyJdID0gZ3RbIm1hdGNoZWRfZW50aXR5X2lkcyJdLm1hcChwYXJzZV9tYXRjaF9pZHMpCiAgICBndFsiY2FyZGluYWxpdHkiXSA9IGd0WyJtYXRjaGVzIl0ubWFwKGxlbikKICAgIGd0WyJoYXNfczIiXSA9IGd0WyJtYXRjaGVzIl0ubWFwKGxhbWJkYSBpZHM6IGFueSh2YWx1ZS5zdGFydHN3aXRoKCJTMi0iKSBmb3IgdmFsdWUgaW4gaWRzKSkKICAgIGd0WyJoYXNfczMiXSA9IGd0WyJtYXRjaGVzIl0ubWFwKGxhbWJkYSBpZHM6IGFueSh2YWx1ZS5zdGFydHN3aXRoKCJTMy0iKSBmb3IgdmFsdWUgaW4gaWRzKSkKICAgIGd0WyJzb3VyY2VfcGF0dGVybiJdID0gWwogICAgICAgICJib3RoIiBpZiBzMiBhbmQgczMgZWxzZSAiczIiIGlmIHMyIGVsc2UgInMzIiBpZiBzMyBlbHNlICJuZWl0aGVyIgogICAgICAgIGZvciBzMiwgczMgaW4gemlwKGd0WyJoYXNfczIiXSwgZ3RbImhhc19zMyJdKQogICAgXQogICAgbWVyZ2VkID0gc291cmNlMVtbImVudGl0eV9pZCIsICJjb3VudHJ5Il1dLm1lcmdlKAogICAgICAgIGd0W1sic291cmNlMV9lbnRpdHlfaWQiLCAiY2FyZGluYWxpdHkiLCAic291cmNlX3BhdHRlcm4iXV0sCiAgICAgICAgbGVmdF9vbj0iZW50aXR5X2lkIiwKICAgICAgICByaWdodF9vbj0ic291cmNlMV9lbnRpdHlfaWQiLAogICAgICAgIGhvdz0ibGVmdCIsCiAgICAgICAgdmFsaWRhdGU9Im9uZV90b19vbmUiLAogICAgKQogICAgaWYgbWVyZ2VkWyJjYXJkaW5hbGl0eSJdLmlzbmEoKS5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncm91bmQgdHJ1dGggZG9lcyBub3QgY292ZXIgZXZlcnkgU291cmNlIDEgcm93IikKICAgIG1lcmdlZFsiY2FyZGluYWxpdHkiXSA9IG1lcmdlZFsiY2FyZGluYWxpdHkiXS5hc3R5cGUoaW50KQogICAgbWVyZ2VkWyJjYXJkaW5hbGl0eV9iaW4iXSA9IG1lcmdlZFsiY2FyZGluYWxpdHkiXS5tYXAoX2NhcmRpbmFsaXR5X2JpbikKICAgIG1lcmdlZFsic2luZ2xldG9uIl0gPSBtZXJnZWRbImNhcmRpbmFsaXR5Il0uZXEoMCkKICAgIG1lcmdlZFsic3RyYXR1bSJdID0gKAogICAgICAgIG1lcmdlZFsiY291bnRyeSJdLmFzdHlwZShzdHIpICsgInwiICsgbWVyZ2VkWyJzaW5nbGV0b24iXS5hc3R5cGUoc3RyKSArICJ8IgogICAgICAgICsgbWVyZ2VkWyJzb3VyY2VfcGF0dGVybiJdICsgInwiICsgbWVyZ2VkWyJjYXJkaW5hbGl0eV9iaW4iXQogICAgKQogICAgbWVyZ2VkWyJfaGFzaCJdID0gbWVyZ2VkWyJlbnRpdHlfaWQiXS5tYXAobGFtYmRhIHZhbHVlOiBzdGFibGVfaGFzaCh2YWx1ZSwgc2VlZCkpCiAgICBtZXJnZWQgPSBtZXJnZWQuc29ydF92YWx1ZXMoWyJzdHJhdHVtIiwgIl9oYXNoIiwgImVudGl0eV9pZCJdLCBraW5kPSJtZXJnZXNvcnQiKQogICAgbWVyZ2VkWyJmb2xkIl0gPSBtZXJnZWQuZ3JvdXBieSgic3RyYXR1bSIsIHNvcnQ9RmFsc2UpLmN1bWNvdW50KCkgJSBuX2ZvbGRzCiAgICByZXR1cm4gbWVyZ2VkW1siZW50aXR5X2lkIiwgImZvbGQiLCAic3RyYXR1bSIsICJjYXJkaW5hbGl0eSIsICJzaW5nbGV0b24iLCAic291cmNlX3BhdHRlcm4iXV0ucmVuYW1lKAogICAgICAgIGNvbHVtbnM9eyJlbnRpdHlfaWQiOiAic291cmNlMV9lbnRpdHlfaWQifQogICAgKQoKCmRlZiBzZWxlY3RfcGFpcl9mb2xkKAogICAgcGFpcnM6IHBkLkRhdGFGcmFtZSwKICAgIGZvbGRzOiBwZC5EYXRhRnJhbWUsCiAgICB2YWxpZGF0aW9uX2ZvbGQ6IGludCwKICAgICosCiAgICB2YWxpZGF0aW9uOiBib29sLAopIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlNlbGVjdCB0cmFpbmluZyBvciBoZWxkLW91dCBwYWlycyBieSBTb3VyY2UgMSBlbnRpdHksIG5ldmVyIGJ5IHBhaXIgcm93LiIiIgogICAgam9pbmVkID0gcGFpcnMubWVyZ2UoCiAgICAgICAgZm9sZHNbWyJzb3VyY2UxX2VudGl0eV9pZCIsICJmb2xkIl1dLAogICAgICAgIG9uPSJzb3VyY2UxX2VudGl0eV9pZCIsCiAgICAgICAgaG93PSJsZWZ0IiwKICAgICAgICB2YWxpZGF0ZT0ibWFueV90b19vbmUiLAogICAgKQogICAgaWYgam9pbmVkWyJmb2xkIl0uaXNuYSgpLmFueSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNvbWUgcGFpcnMgaGF2ZSBubyBTb3VyY2UgMSBmb2xkIGFzc2lnbm1lbnQiKQogICAgbWFzayA9IGpvaW5lZFsiZm9sZCJdLmVxKHZhbGlkYXRpb25fZm9sZCkKICAgIHJldHVybiBqb2luZWQubG9jW21hc2sgaWYgdmFsaWRhdGlvbiBlbHNlIH5tYXNrXS5kcm9wKGNvbHVtbnM9ImZvbGQiKS5yZXNldF9pbmRleChkcm9wPVRydWUpCg==', '95b91a41e1484c8fa2d84e616b278e0c8f8714f98f848d7989297519e88ef1a7')

## Embedded source: `code/business_entity_resolution/src/business_entity_resolution/submission.py`

~~~~python
"""Exact challenge output construction."""

from __future__ import annotations

import csv
from collections.abc import Iterable, Mapping
import os
from pathlib import Path
import pandas as pd

from .schemas import CANDIDATE_COLUMNS, MATCHING_COLUMNS


class OutputFormatError(ValueError):
    """Raised before writing when a submission file would violate the contract."""


def _joined(values: Iterable[str]) -> str:
    unique = sorted(set(str(value) for value in values), key=lambda value: (value[:2], int(value.split("-", 1)[1])))
    return ",".join(unique)


def result_frame(source1_ids: Iterable[str], mapping: Mapping[str, Iterable[str]], value_column: str) -> pd.DataFrame:
    return pd.DataFrame({
        "source1_entity_id": [str(value) for value in source1_ids],
        value_column: [_joined(mapping.get(str(value), ())) for value in source1_ids],
    })


def _assert_format(
    source1_ids: list[str],
    matching: pd.DataFrame,
    candidate: pd.DataFrame,
    valid_targets: set[str] | None,
) -> None:
    """Validate the in-memory frames against every submission rule.

    Runs *before* writing so a formatting bug fails loudly without touching the
    outputs and without requiring the model to be re-run. ``valid_targets`` is
    optional; when provided, matched/candidate IDs must exist in the test set.
    """
    required = set(source1_ids)
    for frame, value_column, expected in (
        (matching, MATCHING_COLUMNS[1], MATCHING_COLUMNS),
        (candidate, CANDIDATE_COLUMNS[1], CANDIDATE_COLUMNS),
    ):
        if list(frame.columns) != list(expected):
            raise OutputFormatError(f"unexpected columns {list(frame.columns)}; expected {list(expected)}")
        s1 = frame["source1_entity_id"].astype(str)
        if s1.duplicated().any():
            raise OutputFormatError(f"{value_column}: duplicate source1_entity_id rows")
        if set(s1) != required:
            missing = required - set(s1)
            extra = set(s1) - required
            raise OutputFormatError(f"{value_column}: S1 coverage mismatch (missing={len(missing)}, extra={len(extra)})")
        for raw in frame[value_column].astype(str):
            if raw == "":
                continue
            ids = raw.split(",")
            if any(item == "" or item != item.strip() for item in ids):
                raise OutputFormatError(f"{value_column}: whitespace/blank token in ID list {raw!r}")
            if len(ids) != len(set(ids)):
                raise OutputFormatError(f"{value_column}: duplicate ID within a list {raw!r}")
            for item in ids:
                if item.startswith("S1-") or not item.startswith(("S2-", "S3-")):
                    raise OutputFormatError(f"{value_column}: invalid target prefix {item!r}")
                if valid_targets is not None and item not in valid_targets:
                    raise OutputFormatError(f"{value_column}: unknown target ID {item!r}")
    # Every final match must have been a candidate.
    match_map = {row.source1_entity_id: set(str(row.matched_entity_ids).split(",")) - {""} for row in matching.itertuples(index=False)}
    cand_map = {row.source1_entity_id: set(str(row.candidate_entity_ids).split(",")) - {""} for row in candidate.itertuples(index=False)}
    for s1_id, matched in match_map.items():
        absent = matched - cand_map.get(s1_id, set())
        if absent:
            raise OutputFormatError(f"matching: {s1_id} has matches absent from candidates: {sorted(absent)[:3]}")


def write_submission_outputs(
    output_dir: str | Path,
    source1_ids: Iterable[str],
    predictions: Mapping[str, Iterable[str]],
    candidates: Mapping[str, Iterable[str]],
    valid_targets: set[str] | None = None,
) -> tuple[Path, Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    ids = [str(value) for value in source1_ids]
    matching = result_frame(ids, predictions, MATCHING_COLUMNS[1])
    candidate = result_frame(ids, candidates, CANDIDATE_COLUMNS[1])
    _assert_format(ids, matching, candidate, valid_targets)
    matching_path, candidate_path = output_dir / "matching_results.tsv", output_dir / "candidate_pairs.tsv"
    matching.to_csv(matching_path, sep="\t", index=False, encoding="utf-8", lineterminator="\n")
    candidate.to_csv(candidate_path, sep="\t", index=False, encoding="utf-8", lineterminator="\n")
    return matching_path, candidate_path


def write_submission_outputs_sharded(
    output_dir: str | Path,
    shards: Iterable[
        tuple[Iterable[str], Mapping[str, Iterable[str]], Mapping[str, Iterable[str]]]
    ],
) -> tuple[Path, Path]:
    """Write exact submission TSVs one Source 1 shard at a time.

    This enforces the row/list/subset contract without materializing hundreds
    of millions of candidate IDs in a process-wide dictionary.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    matching_path = output_dir / "matching_results.tsv"
    candidate_path = output_dir / "candidate_pairs.tsv"
    matching_tmp = matching_path.with_suffix(".tsv.tmp")
    candidate_tmp = candidate_path.with_suffix(".tsv.tmp")
    seen: set[str] = set()
    try:
        with matching_tmp.open("w", encoding="utf-8", newline="") as matching_handle, candidate_tmp.open(
            "w", encoding="utf-8", newline=""
        ) as candidate_handle:
            matching_writer = csv.writer(matching_handle, delimiter="\t", lineterminator="\n")
            candidate_writer = csv.writer(candidate_handle, delimiter="\t", lineterminator="\n")
            matching_writer.writerow(MATCHING_COLUMNS)
            candidate_writer.writerow(CANDIDATE_COLUMNS)
            for source1_ids, predictions, candidates in shards:
                for raw_source1_id in source1_ids:
                    source1_id = str(raw_source1_id)
                    if source1_id in seen:
                        raise OutputFormatError(f"duplicate Source 1 row across shards: {source1_id}")
                    seen.add(source1_id)
                    matched = set(str(value) for value in predictions.get(source1_id, ()))
                    candidate_set = set(str(value) for value in candidates.get(source1_id, ()))
                    absent = matched - candidate_set
                    if absent:
                        raise OutputFormatError(
                            f"matching: {source1_id} has matches absent from candidates: {sorted(absent)[:3]}"
                        )
                    for value in matched | candidate_set:
                        if value.startswith("S1-") or not value.startswith(("S2-", "S3-")):
                            raise OutputFormatError(f"invalid target prefix {value!r}")
                    matching_writer.writerow((source1_id, _joined(matched)))
                    candidate_writer.writerow((source1_id, _joined(candidate_set)))
        os.replace(matching_tmp, matching_path)
        os.replace(candidate_tmp, candidate_path)
    except Exception:
        matching_tmp.unlink(missing_ok=True)
        candidate_tmp.unlink(missing_ok=True)
        raise
    return matching_path, candidate_path


~~~~

In [ ]:
materialize('code/business_entity_resolution/src/business_entity_resolution/submission.py', 'IiIiRXhhY3QgY2hhbGxlbmdlIG91dHB1dCBjb25zdHJ1Y3Rpb24uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY3N2CmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBJdGVyYWJsZSwgTWFwcGluZwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gLnNjaGVtYXMgaW1wb3J0IENBTkRJREFURV9DT0xVTU5TLCBNQVRDSElOR19DT0xVTU5TCgoKY2xhc3MgT3V0cHV0Rm9ybWF0RXJyb3IoVmFsdWVFcnJvcik6CiAgICAiIiJSYWlzZWQgYmVmb3JlIHdyaXRpbmcgd2hlbiBhIHN1Ym1pc3Npb24gZmlsZSB3b3VsZCB2aW9sYXRlIHRoZSBjb250cmFjdC4iIiIKCgpkZWYgX2pvaW5lZCh2YWx1ZXM6IEl0ZXJhYmxlW3N0cl0pIC0+IHN0cjoKICAgIHVuaXF1ZSA9IHNvcnRlZChzZXQoc3RyKHZhbHVlKSBmb3IgdmFsdWUgaW4gdmFsdWVzKSwga2V5PWxhbWJkYSB2YWx1ZTogKHZhbHVlWzoyXSwgaW50KHZhbHVlLnNwbGl0KCItIiwgMSlbMV0pKSkKICAgIHJldHVybiAiLCIuam9pbih1bmlxdWUpCgoKZGVmIHJlc3VsdF9mcmFtZShzb3VyY2UxX2lkczogSXRlcmFibGVbc3RyXSwgbWFwcGluZzogTWFwcGluZ1tzdHIsIEl0ZXJhYmxlW3N0cl1dLCB2YWx1ZV9jb2x1bW46IHN0cikgLT4gcGQuRGF0YUZyYW1lOgogICAgcmV0dXJuIHBkLkRhdGFGcmFtZSh7CiAgICAgICAgInNvdXJjZTFfZW50aXR5X2lkIjogW3N0cih2YWx1ZSkgZm9yIHZhbHVlIGluIHNvdXJjZTFfaWRzXSwKICAgICAgICB2YWx1ZV9jb2x1bW46IFtfam9pbmVkKG1hcHBpbmcuZ2V0KHN0cih2YWx1ZSksICgpKSkgZm9yIHZhbHVlIGluIHNvdXJjZTFfaWRzXSwKICAgIH0pCgoKZGVmIF9hc3NlcnRfZm9ybWF0KAogICAgc291cmNlMV9pZHM6IGxpc3Rbc3RyXSwKICAgIG1hdGNoaW5nOiBwZC5EYXRhRnJhbWUsCiAgICBjYW5kaWRhdGU6IHBkLkRhdGFGcmFtZSwKICAgIHZhbGlkX3RhcmdldHM6IHNldFtzdHJdIHwgTm9uZSwKKSAtPiBOb25lOgogICAgIiIiVmFsaWRhdGUgdGhlIGluLW1lbW9yeSBmcmFtZXMgYWdhaW5zdCBldmVyeSBzdWJtaXNzaW9uIHJ1bGUuCgogICAgUnVucyAqYmVmb3JlKiB3cml0aW5nIHNvIGEgZm9ybWF0dGluZyBidWcgZmFpbHMgbG91ZGx5IHdpdGhvdXQgdG91Y2hpbmcgdGhlCiAgICBvdXRwdXRzIGFuZCB3aXRob3V0IHJlcXVpcmluZyB0aGUgbW9kZWwgdG8gYmUgcmUtcnVuLiBgYHZhbGlkX3RhcmdldHNgYCBpcwogICAgb3B0aW9uYWw7IHdoZW4gcHJvdmlkZWQsIG1hdGNoZWQvY2FuZGlkYXRlIElEcyBtdXN0IGV4aXN0IGluIHRoZSB0ZXN0IHNldC4KICAgICIiIgogICAgcmVxdWlyZWQgPSBzZXQoc291cmNlMV9pZHMpCiAgICBmb3IgZnJhbWUsIHZhbHVlX2NvbHVtbiwgZXhwZWN0ZWQgaW4gKAogICAgICAgIChtYXRjaGluZywgTUFUQ0hJTkdfQ09MVU1OU1sxXSwgTUFUQ0hJTkdfQ09MVU1OUyksCiAgICAgICAgKGNhbmRpZGF0ZSwgQ0FORElEQVRFX0NPTFVNTlNbMV0sIENBTkRJREFURV9DT0xVTU5TKSwKICAgICk6CiAgICAgICAgaWYgbGlzdChmcmFtZS5jb2x1bW5zKSAhPSBsaXN0KGV4cGVjdGVkKToKICAgICAgICAgICAgcmFpc2UgT3V0cHV0Rm9ybWF0RXJyb3IoZiJ1bmV4cGVjdGVkIGNvbHVtbnMge2xpc3QoZnJhbWUuY29sdW1ucyl9OyBleHBlY3RlZCB7bGlzdChleHBlY3RlZCl9IikKICAgICAgICBzMSA9IGZyYW1lWyJzb3VyY2UxX2VudGl0eV9pZCJdLmFzdHlwZShzdHIpCiAgICAgICAgaWYgczEuZHVwbGljYXRlZCgpLmFueSgpOgogICAgICAgICAgICByYWlzZSBPdXRwdXRGb3JtYXRFcnJvcihmInt2YWx1ZV9jb2x1bW59OiBkdXBsaWNhdGUgc291cmNlMV9lbnRpdHlfaWQgcm93cyIpCiAgICAgICAgaWYgc2V0KHMxKSAhPSByZXF1aXJlZDoKICAgICAgICAgICAgbWlzc2luZyA9IHJlcXVpcmVkIC0gc2V0KHMxKQogICAgICAgICAgICBleHRyYSA9IHNldChzMSkgLSByZXF1aXJlZAogICAgICAgICAgICByYWlzZSBPdXRwdXRGb3JtYXRFcnJvcihmInt2YWx1ZV9jb2x1bW59OiBTMSBjb3ZlcmFnZSBtaXNtYXRjaCAobWlzc2luZz17bGVuKG1pc3NpbmcpfSwgZXh0cmE9e2xlbihleHRyYSl9KSIpCiAgICAgICAgZm9yIHJhdyBpbiBmcmFtZVt2YWx1ZV9jb2x1bW5dLmFzdHlwZShzdHIpOgogICAgICAgICAgICBpZiByYXcgPT0gIiI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZHMgPSByYXcuc3BsaXQoIiwiKQogICAgICAgICAgICBpZiBhbnkoaXRlbSA9PSAiIiBvciBpdGVtICE9IGl0ZW0uc3RyaXAoKSBmb3IgaXRlbSBpbiBpZHMpOgogICAgICAgICAgICAgICAgcmFpc2UgT3V0cHV0Rm9ybWF0RXJyb3IoZiJ7dmFsdWVfY29sdW1ufTogd2hpdGVzcGFjZS9ibGFuayB0b2tlbiBpbiBJRCBsaXN0IHtyYXchcn0iKQogICAgICAgICAgICBpZiBsZW4oaWRzKSAhPSBsZW4oc2V0KGlkcykpOgogICAgICAgICAgICAgICAgcmFpc2UgT3V0cHV0Rm9ybWF0RXJyb3IoZiJ7dmFsdWVfY29sdW1ufTogZHVwbGljYXRlIElEIHdpdGhpbiBhIGxpc3Qge3JhdyFyfSIpCiAgICAgICAgICAgIGZvciBpdGVtIGluIGlkczoKICAgICAgICAgICAgICAgIGlmIGl0ZW0uc3RhcnRzd2l0aCgiUzEtIikgb3Igbm90IGl0ZW0uc3RhcnRzd2l0aCgoIlMyLSIsICJTMy0iKSk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgT3V0cHV0Rm9ybWF0RXJyb3IoZiJ7dmFsdWVfY29sdW1ufTogaW52YWxpZCB0YXJnZXQgcHJlZml4IHtpdGVtIXJ9IikKICAgICAgICAgICAgICAgIGlmIHZhbGlkX3RhcmdldHMgaXMgbm90IE5vbmUgYW5kIGl0ZW0gbm90IGluIHZhbGlkX3RhcmdldHM6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgT3V0cHV0Rm9ybWF0RXJyb3IoZiJ7dmFsdWVfY29sdW1ufTogdW5rbm93biB0YXJnZXQgSUQge2l0ZW0hcn0iKQogICAgIyBFdmVyeSBmaW5hbCBtYXRjaCBtdXN0IGhhdmUgYmVlbiBhIGNhbmRpZGF0ZS4KICAgIG1hdGNoX21hcCA9IHtyb3cuc291cmNlMV9lbnRpdHlfaWQ6IHNldChzdHIocm93Lm1hdGNoZWRfZW50aXR5X2lkcykuc3BsaXQoIiwiKSkgLSB7IiJ9IGZvciByb3cgaW4gbWF0Y2hpbmcuaXRlcnR1cGxlcyhpbmRleD1GYWxzZSl9CiAgICBjYW5kX21hcCA9IHtyb3cuc291cmNlMV9lbnRpdHlfaWQ6IHNldChzdHIocm93LmNhbmRpZGF0ZV9lbnRpdHlfaWRzKS5zcGxpdCgiLCIpKSAtIHsiIn0gZm9yIHJvdyBpbiBjYW5kaWRhdGUuaXRlcnR1cGxlcyhpbmRleD1GYWxzZSl9CiAgICBmb3IgczFfaWQsIG1hdGNoZWQgaW4gbWF0Y2hfbWFwLml0ZW1zKCk6CiAgICAgICAgYWJzZW50ID0gbWF0Y2hlZCAtIGNhbmRfbWFwLmdldChzMV9pZCwgc2V0KCkpCiAgICAgICAgaWYgYWJzZW50OgogICAgICAgICAgICByYWlzZSBPdXRwdXRGb3JtYXRFcnJvcihmIm1hdGNoaW5nOiB7czFfaWR9IGhhcyBtYXRjaGVzIGFic2VudCBmcm9tIGNhbmRpZGF0ZXM6IHtzb3J0ZWQoYWJzZW50KVs6M119IikKCgpkZWYgd3JpdGVfc3VibWlzc2lvbl9vdXRwdXRzKAogICAgb3V0cHV0X2Rpcjogc3RyIHwgUGF0aCwKICAgIHNvdXJjZTFfaWRzOiBJdGVyYWJsZVtzdHJdLAogICAgcHJlZGljdGlvbnM6IE1hcHBpbmdbc3RyLCBJdGVyYWJsZVtzdHJdXSwKICAgIGNhbmRpZGF0ZXM6IE1hcHBpbmdbc3RyLCBJdGVyYWJsZVtzdHJdXSwKICAgIHZhbGlkX3RhcmdldHM6IHNldFtzdHJdIHwgTm9uZSA9IE5vbmUsCikgLT4gdHVwbGVbUGF0aCwgUGF0aF06CiAgICBvdXRwdXRfZGlyID0gUGF0aChvdXRwdXRfZGlyKQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBpZHMgPSBbc3RyKHZhbHVlKSBmb3IgdmFsdWUgaW4gc291cmNlMV9pZHNdCiAgICBtYXRjaGluZyA9IHJlc3VsdF9mcmFtZShpZHMsIHByZWRpY3Rpb25zLCBNQVRDSElOR19DT0xVTU5TWzFdKQogICAgY2FuZGlkYXRlID0gcmVzdWx0X2ZyYW1lKGlkcywgY2FuZGlkYXRlcywgQ0FORElEQVRFX0NPTFVNTlNbMV0pCiAgICBfYXNzZXJ0X2Zvcm1hdChpZHMsIG1hdGNoaW5nLCBjYW5kaWRhdGUsIHZhbGlkX3RhcmdldHMpCiAgICBtYXRjaGluZ19wYXRoLCBjYW5kaWRhdGVfcGF0aCA9IG91dHB1dF9kaXIgLyAibWF0Y2hpbmdfcmVzdWx0cy50c3YiLCBvdXRwdXRfZGlyIC8gImNhbmRpZGF0ZV9wYWlycy50c3YiCiAgICBtYXRjaGluZy50b19jc3YobWF0Y2hpbmdfcGF0aCwgc2VwPSJcdCIsIGluZGV4PUZhbHNlLCBlbmNvZGluZz0idXRmLTgiLCBsaW5ldGVybWluYXRvcj0iXG4iKQogICAgY2FuZGlkYXRlLnRvX2NzdihjYW5kaWRhdGVfcGF0aCwgc2VwPSJcdCIsIGluZGV4PUZhbHNlLCBlbmNvZGluZz0idXRmLTgiLCBsaW5ldGVybWluYXRvcj0iXG4iKQogICAgcmV0dXJuIG1hdGNoaW5nX3BhdGgsIGNhbmRpZGF0ZV9wYXRoCgoKZGVmIHdyaXRlX3N1Ym1pc3Npb25fb3V0cHV0c19zaGFyZGVkKAogICAgb3V0cHV0X2Rpcjogc3RyIHwgUGF0aCwKICAgIHNoYXJkczogSXRlcmFibGVbCiAgICAgICAgdHVwbGVbSXRlcmFibGVbc3RyXSwgTWFwcGluZ1tzdHIsIEl0ZXJhYmxlW3N0cl1dLCBNYXBwaW5nW3N0ciwgSXRlcmFibGVbc3RyXV1dCiAgICBdLAopIC0+IHR1cGxlW1BhdGgsIFBhdGhdOgogICAgIiIiV3JpdGUgZXhhY3Qgc3VibWlzc2lvbiBUU1ZzIG9uZSBTb3VyY2UgMSBzaGFyZCBhdCBhIHRpbWUuCgogICAgVGhpcyBlbmZvcmNlcyB0aGUgcm93L2xpc3Qvc3Vic2V0IGNvbnRyYWN0IHdpdGhvdXQgbWF0ZXJpYWxpemluZyBodW5kcmVkcwogICAgb2YgbWlsbGlvbnMgb2YgY2FuZGlkYXRlIElEcyBpbiBhIHByb2Nlc3Mtd2lkZSBkaWN0aW9uYXJ5LgogICAgIiIiCiAgICBvdXRwdXRfZGlyID0gUGF0aChvdXRwdXRfZGlyKQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBtYXRjaGluZ19wYXRoID0gb3V0cHV0X2RpciAvICJtYXRjaGluZ19yZXN1bHRzLnRzdiIKICAgIGNhbmRpZGF0ZV9wYXRoID0gb3V0cHV0X2RpciAvICJjYW5kaWRhdGVfcGFpcnMudHN2IgogICAgbWF0Y2hpbmdfdG1wID0gbWF0Y2hpbmdfcGF0aC53aXRoX3N1ZmZpeCgiLnRzdi50bXAiKQogICAgY2FuZGlkYXRlX3RtcCA9IGNhbmRpZGF0ZV9wYXRoLndpdGhfc3VmZml4KCIudHN2LnRtcCIpCiAgICBzZWVuOiBzZXRbc3RyXSA9IHNldCgpCiAgICB0cnk6CiAgICAgICAgd2l0aCBtYXRjaGluZ190bXAub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIpIGFzIG1hdGNoaW5nX2hhbmRsZSwgY2FuZGlkYXRlX3RtcC5vcGVuKAogICAgICAgICAgICAidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IiIKICAgICAgICApIGFzIGNhbmRpZGF0ZV9oYW5kbGU6CiAgICAgICAgICAgIG1hdGNoaW5nX3dyaXRlciA9IGNzdi53cml0ZXIobWF0Y2hpbmdfaGFuZGxlLCBkZWxpbWl0ZXI9Ilx0IiwgbGluZXRlcm1pbmF0b3I9IlxuIikKICAgICAgICAgICAgY2FuZGlkYXRlX3dyaXRlciA9IGNzdi53cml0ZXIoY2FuZGlkYXRlX2hhbmRsZSwgZGVsaW1pdGVyPSJcdCIsIGxpbmV0ZXJtaW5hdG9yPSJcbiIpCiAgICAgICAgICAgIG1hdGNoaW5nX3dyaXRlci53cml0ZXJvdyhNQVRDSElOR19DT0xVTU5TKQogICAgICAgICAgICBjYW5kaWRhdGVfd3JpdGVyLndyaXRlcm93KENBTkRJREFURV9DT0xVTU5TKQogICAgICAgICAgICBmb3Igc291cmNlMV9pZHMsIHByZWRpY3Rpb25zLCBjYW5kaWRhdGVzIGluIHNoYXJkczoKICAgICAgICAgICAgICAgIGZvciByYXdfc291cmNlMV9pZCBpbiBzb3VyY2UxX2lkczoKICAgICAgICAgICAgICAgICAgICBzb3VyY2UxX2lkID0gc3RyKHJhd19zb3VyY2UxX2lkKQogICAgICAgICAgICAgICAgICAgIGlmIHNvdXJjZTFfaWQgaW4gc2VlbjoKICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgT3V0cHV0Rm9ybWF0RXJyb3IoZiJkdXBsaWNhdGUgU291cmNlIDEgcm93IGFjcm9zcyBzaGFyZHM6IHtzb3VyY2UxX2lkfSIpCiAgICAgICAgICAgICAgICAgICAgc2Vlbi5hZGQoc291cmNlMV9pZCkKICAgICAgICAgICAgICAgICAgICBtYXRjaGVkID0gc2V0KHN0cih2YWx1ZSkgZm9yIHZhbHVlIGluIHByZWRpY3Rpb25zLmdldChzb3VyY2UxX2lkLCAoKSkpCiAgICAgICAgICAgICAgICAgICAgY2FuZGlkYXRlX3NldCA9IHNldChzdHIodmFsdWUpIGZvciB2YWx1ZSBpbiBjYW5kaWRhdGVzLmdldChzb3VyY2UxX2lkLCAoKSkpCiAgICAgICAgICAgICAgICAgICAgYWJzZW50ID0gbWF0Y2hlZCAtIGNhbmRpZGF0ZV9zZXQKICAgICAgICAgICAgICAgICAgICBpZiBhYnNlbnQ6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIE91dHB1dEZvcm1hdEVycm9yKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZiJtYXRjaGluZzoge3NvdXJjZTFfaWR9IGhhcyBtYXRjaGVzIGFic2VudCBmcm9tIGNhbmRpZGF0ZXM6IHtzb3J0ZWQoYWJzZW50KVs6M119IgogICAgICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgZm9yIHZhbHVlIGluIG1hdGNoZWQgfCBjYW5kaWRhdGVfc2V0OgogICAgICAgICAgICAgICAgICAgICAgICBpZiB2YWx1ZS5zdGFydHN3aXRoKCJTMS0iKSBvciBub3QgdmFsdWUuc3RhcnRzd2l0aCgoIlMyLSIsICJTMy0iKSk6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICByYWlzZSBPdXRwdXRGb3JtYXRFcnJvcihmImludmFsaWQgdGFyZ2V0IHByZWZpeCB7dmFsdWUhcn0iKQogICAgICAgICAgICAgICAgICAgIG1hdGNoaW5nX3dyaXRlci53cml0ZXJvdygoc291cmNlMV9pZCwgX2pvaW5lZChtYXRjaGVkKSkpCiAgICAgICAgICAgICAgICAgICAgY2FuZGlkYXRlX3dyaXRlci53cml0ZXJvdygoc291cmNlMV9pZCwgX2pvaW5lZChjYW5kaWRhdGVfc2V0KSkpCiAgICAgICAgb3MucmVwbGFjZShtYXRjaGluZ190bXAsIG1hdGNoaW5nX3BhdGgpCiAgICAgICAgb3MucmVwbGFjZShjYW5kaWRhdGVfdG1wLCBjYW5kaWRhdGVfcGF0aCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgbWF0Y2hpbmdfdG1wLnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICAgICAgY2FuZGlkYXRlX3RtcC51bmxpbmsobWlzc2luZ19vaz1UcnVlKQogICAgICAgIHJhaXNlCiAgICByZXR1cm4gbWF0Y2hpbmdfcGF0aCwgY2FuZGlkYXRlX3BhdGgKCg==', '3c71eeb256d73569492dcdcfb37133de941b22c80fd7426bcb9f40f22162dbca')

## Embedded source: `code/business_entity_resolution/tests/README.md`

~~~~text
# Tests

Planned coverage includes TSV schema validation, ID-prefix integrity, open-set country handling, label parsing, candidate containment, singleton output rows, deterministic behavior, and compatibility with the supplied submission validator.


~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/README.md', 'IyBUZXN0cwoKUGxhbm5lZCBjb3ZlcmFnZSBpbmNsdWRlcyBUU1Ygc2NoZW1hIHZhbGlkYXRpb24sIElELXByZWZpeCBpbnRlZ3JpdHksIG9wZW4tc2V0IGNvdW50cnkgaGFuZGxpbmcsIGxhYmVsIHBhcnNpbmcsIGNhbmRpZGF0ZSBjb250YWlubWVudCwgc2luZ2xldG9uIG91dHB1dCByb3dzLCBkZXRlcm1pbmlzdGljIGJlaGF2aW9yLCBhbmQgY29tcGF0aWJpbGl0eSB3aXRoIHRoZSBzdXBwbGllZCBzdWJtaXNzaW9uIHZhbGlkYXRvci4KCg==', 'c81d103f9980f09f0d3d2b3aaf196742ec95ece05cb05528014b2f5a7923372e')

## Embedded source: `code/business_entity_resolution/tests/conftest.py`

~~~~python
from __future__ import annotations

import pandas as pd
import pytest

from business_entity_resolution.pipeline import make_smoke_fixture


@pytest.fixture
def smoke_frames():
    return make_smoke_fixture()


@pytest.fixture
def tiny_source():
    return pd.DataFrame([
        ("S1-1", "Acme & Sons Ltd", "12 Main Rd, Paris 75001", "France"),
        ("S1-2", "राम मार्केटिंग", "10 MG Road 560001", "India"),
    ], columns=["entity_id", "business_name", "business_address", "country"])


~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/conftest.py', 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgcHl0ZXN0Cgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLnBpcGVsaW5lIGltcG9ydCBtYWtlX3Ntb2tlX2ZpeHR1cmUKCgpAcHl0ZXN0LmZpeHR1cmUKZGVmIHNtb2tlX2ZyYW1lcygpOgogICAgcmV0dXJuIG1ha2Vfc21va2VfZml4dHVyZSgpCgoKQHB5dGVzdC5maXh0dXJlCmRlZiB0aW55X3NvdXJjZSgpOgogICAgcmV0dXJuIHBkLkRhdGFGcmFtZShbCiAgICAgICAgKCJTMS0xIiwgIkFjbWUgJiBTb25zIEx0ZCIsICIxMiBNYWluIFJkLCBQYXJpcyA3NTAwMSIsICJGcmFuY2UiKSwKICAgICAgICAoIlMxLTIiLCAi4KSw4KS+4KSuIOCkruCkvuCksOCljeCkleClh+Ckn+Ckv+CkguCklyIsICIxMCBNRyBSb2FkIDU2MDAwMSIsICJJbmRpYSIpLAogICAgXSwgY29sdW1ucz1bImVudGl0eV9pZCIsICJidXNpbmVzc19uYW1lIiwgImJ1c2luZXNzX2FkZHJlc3MiLCAiY291bnRyeSJdKQoK', '7709ad9fb4df7c0c85699e6250b765a7af1fb47374810c990bf048ba4f3d0872')

## Embedded source: `code/business_entity_resolution/tests/test_blocking.py`

~~~~python
import pandas as pd
import numpy as np

from business_entity_resolution.blocking import CandidateGenerator
from business_entity_resolution.blocking.base import CandidateReason
from business_entity_resolution.blocking.tokens import RareTokenBlocker
from business_entity_resolution.blocking.tfidf import TfidfTopKBlocker
from business_entity_resolution.normalize import normalize_records


def test_blocking_union_is_deterministic_and_open_set(smoke_frames):
    s1, s2, _, _ = smoke_frames
    ns1, n2 = normalize_records(s1), normalize_records(s2)
    config = {"rare_token_max_df": 20, "token_posting_cap": 20, "tfidf_top_k": 3, "per_source_cap": 10, "batch_size": 2}
    first = CandidateGenerator(config).fit(n2).transform(ns1)
    second = CandidateGenerator(config).fit(n2).transform(ns1)
    pd.testing.assert_frame_equal(first, second)
    alpha = first[first.source1_entity_id == "S1-1"]
    assert "S2-10" in set(alpha.candidate_entity_id)
    assert len(first[first.source1_entity_id == "S1-1"]) <= 10


def test_duplicate_content_ids_are_preserved():
    s1 = normalize_records(pd.DataFrame([("S1-1", "Acme", "1 Road", "US")], columns=["entity_id", "business_name", "business_address", "country"]))
    targets = normalize_records(pd.DataFrame([
        ("S2-1", "Acme", "1 Road", "US"), ("S2-2", "Acme", "1 Road", "US")
    ], columns=["entity_id", "business_name", "business_address", "country"]))
    candidates = CandidateGenerator({"token_posting_cap": 10, "per_source_cap": 10}, include_tfidf=False).fit(targets).transform(s1)
    assert set(candidates.candidate_entity_id) == {"S2-1", "S2-2"}


def test_blockers_accept_arrow_roundtrip_array_tokens(smoke_frames):
    s1, s2, _, _ = smoke_frames
    ns1, n2 = normalize_records(s1), normalize_records(s2)
    for frame in (ns1, n2):
        for column in ("name_tokens", "address_tokens", "numeric_tokens"):
            frame[column] = frame[column].map(np.asarray)
    candidates = CandidateGenerator({"token_posting_cap": 10, "per_source_cap": 10}, include_tfidf=False).fit(n2).transform(ns1)
    assert not candidates.empty


def test_token_posting_cap_is_independent_of_target_row_order():
    columns = ["entity_id", "country_norm", "name_tokens"]
    targets = pd.DataFrame([
        (f"S2-{index}", "us", ("shared",)) for index in range(1, 301)
    ], columns=columns)
    query = pd.DataFrame([("S1-1", "us", ("shared",))], columns=columns)
    first = RareTokenBlocker("name_tokens", CandidateReason.RARE_NAME_TOKEN, 500, 25).fit(targets).transform(query)
    second = RareTokenBlocker("name_tokens", CandidateReason.RARE_NAME_TOKEN, 500, 25).fit(targets.iloc[::-1]).transform(query)
    assert first["candidate_entity_id"].tolist() == second["candidate_entity_id"].tolist()


def test_country_fallback_reason_is_candidate_specific():
    columns = ["entity_id", "country_norm", "name_tokens"]
    targets = pd.DataFrame([
        ("S2-1", "france", ("local",)),
        ("S2-2", "us", ("global",)),
    ], columns=columns)
    query = pd.DataFrame([
        ("S1-1", "france", ("local", "global")),
    ], columns=columns)
    result = RareTokenBlocker("name_tokens", CandidateReason.RARE_NAME_TOKEN, 10, 10).fit(targets).transform(query)
    bits = dict(zip(result["candidate_entity_id"], result["reason_bits"]))
    assert not bits["S2-1"] & int(CandidateReason.COUNTRY_FALLBACK)
    assert bits["S2-2"] & int(CandidateReason.COUNTRY_FALLBACK)


def test_chunked_tfidf_matches_single_target_block(smoke_frames):
    s1, s2, _, _ = smoke_frames
    queries, targets = normalize_records(s1), normalize_records(s2)
    chunked = TfidfTopKBlocker(top_k=3, min_score=0.0, batch_size=2, target_batch_size=2).fit(targets).transform(queries)
    single = TfidfTopKBlocker(top_k=3, min_score=0.0, batch_size=2, target_batch_size=100).fit(targets).transform(queries)
    pd.testing.assert_frame_equal(chunked, single)

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_blocking.py', 'aW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24uYmxvY2tpbmcgaW1wb3J0IENhbmRpZGF0ZUdlbmVyYXRvcgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLmJsb2NraW5nLmJhc2UgaW1wb3J0IENhbmRpZGF0ZVJlYXNvbgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLmJsb2NraW5nLnRva2VucyBpbXBvcnQgUmFyZVRva2VuQmxvY2tlcgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLmJsb2NraW5nLnRmaWRmIGltcG9ydCBUZmlkZlRvcEtCbG9ja2VyCmZyb20gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24ubm9ybWFsaXplIGltcG9ydCBub3JtYWxpemVfcmVjb3JkcwoKCmRlZiB0ZXN0X2Jsb2NraW5nX3VuaW9uX2lzX2RldGVybWluaXN0aWNfYW5kX29wZW5fc2V0KHNtb2tlX2ZyYW1lcyk6CiAgICBzMSwgczIsIF8sIF8gPSBzbW9rZV9mcmFtZXMKICAgIG5zMSwgbjIgPSBub3JtYWxpemVfcmVjb3JkcyhzMSksIG5vcm1hbGl6ZV9yZWNvcmRzKHMyKQogICAgY29uZmlnID0geyJyYXJlX3Rva2VuX21heF9kZiI6IDIwLCAidG9rZW5fcG9zdGluZ19jYXAiOiAyMCwgInRmaWRmX3RvcF9rIjogMywgInBlcl9zb3VyY2VfY2FwIjogMTAsICJiYXRjaF9zaXplIjogMn0KICAgIGZpcnN0ID0gQ2FuZGlkYXRlR2VuZXJhdG9yKGNvbmZpZykuZml0KG4yKS50cmFuc2Zvcm0obnMxKQogICAgc2Vjb25kID0gQ2FuZGlkYXRlR2VuZXJhdG9yKGNvbmZpZykuZml0KG4yKS50cmFuc2Zvcm0obnMxKQogICAgcGQudGVzdGluZy5hc3NlcnRfZnJhbWVfZXF1YWwoZmlyc3QsIHNlY29uZCkKICAgIGFscGhhID0gZmlyc3RbZmlyc3Quc291cmNlMV9lbnRpdHlfaWQgPT0gIlMxLTEiXQogICAgYXNzZXJ0ICJTMi0xMCIgaW4gc2V0KGFscGhhLmNhbmRpZGF0ZV9lbnRpdHlfaWQpCiAgICBhc3NlcnQgbGVuKGZpcnN0W2ZpcnN0LnNvdXJjZTFfZW50aXR5X2lkID09ICJTMS0xIl0pIDw9IDEwCgoKZGVmIHRlc3RfZHVwbGljYXRlX2NvbnRlbnRfaWRzX2FyZV9wcmVzZXJ2ZWQoKToKICAgIHMxID0gbm9ybWFsaXplX3JlY29yZHMocGQuRGF0YUZyYW1lKFsoIlMxLTEiLCAiQWNtZSIsICIxIFJvYWQiLCAiVVMiKV0sIGNvbHVtbnM9WyJlbnRpdHlfaWQiLCAiYnVzaW5lc3NfbmFtZSIsICJidXNpbmVzc19hZGRyZXNzIiwgImNvdW50cnkiXSkpCiAgICB0YXJnZXRzID0gbm9ybWFsaXplX3JlY29yZHMocGQuRGF0YUZyYW1lKFsKICAgICAgICAoIlMyLTEiLCAiQWNtZSIsICIxIFJvYWQiLCAiVVMiKSwgKCJTMi0yIiwgIkFjbWUiLCAiMSBSb2FkIiwgIlVTIikKICAgIF0sIGNvbHVtbnM9WyJlbnRpdHlfaWQiLCAiYnVzaW5lc3NfbmFtZSIsICJidXNpbmVzc19hZGRyZXNzIiwgImNvdW50cnkiXSkpCiAgICBjYW5kaWRhdGVzID0gQ2FuZGlkYXRlR2VuZXJhdG9yKHsidG9rZW5fcG9zdGluZ19jYXAiOiAxMCwgInBlcl9zb3VyY2VfY2FwIjogMTB9LCBpbmNsdWRlX3RmaWRmPUZhbHNlKS5maXQodGFyZ2V0cykudHJhbnNmb3JtKHMxKQogICAgYXNzZXJ0IHNldChjYW5kaWRhdGVzLmNhbmRpZGF0ZV9lbnRpdHlfaWQpID09IHsiUzItMSIsICJTMi0yIn0KCgpkZWYgdGVzdF9ibG9ja2Vyc19hY2NlcHRfYXJyb3dfcm91bmR0cmlwX2FycmF5X3Rva2VucyhzbW9rZV9mcmFtZXMpOgogICAgczEsIHMyLCBfLCBfID0gc21va2VfZnJhbWVzCiAgICBuczEsIG4yID0gbm9ybWFsaXplX3JlY29yZHMoczEpLCBub3JtYWxpemVfcmVjb3JkcyhzMikKICAgIGZvciBmcmFtZSBpbiAobnMxLCBuMik6CiAgICAgICAgZm9yIGNvbHVtbiBpbiAoIm5hbWVfdG9rZW5zIiwgImFkZHJlc3NfdG9rZW5zIiwgIm51bWVyaWNfdG9rZW5zIik6CiAgICAgICAgICAgIGZyYW1lW2NvbHVtbl0gPSBmcmFtZVtjb2x1bW5dLm1hcChucC5hc2FycmF5KQogICAgY2FuZGlkYXRlcyA9IENhbmRpZGF0ZUdlbmVyYXRvcih7InRva2VuX3Bvc3RpbmdfY2FwIjogMTAsICJwZXJfc291cmNlX2NhcCI6IDEwfSwgaW5jbHVkZV90ZmlkZj1GYWxzZSkuZml0KG4yKS50cmFuc2Zvcm0obnMxKQogICAgYXNzZXJ0IG5vdCBjYW5kaWRhdGVzLmVtcHR5CgoKZGVmIHRlc3RfdG9rZW5fcG9zdGluZ19jYXBfaXNfaW5kZXBlbmRlbnRfb2ZfdGFyZ2V0X3Jvd19vcmRlcigpOgogICAgY29sdW1ucyA9IFsiZW50aXR5X2lkIiwgImNvdW50cnlfbm9ybSIsICJuYW1lX3Rva2VucyJdCiAgICB0YXJnZXRzID0gcGQuRGF0YUZyYW1lKFsKICAgICAgICAoZiJTMi17aW5kZXh9IiwgInVzIiwgKCJzaGFyZWQiLCkpIGZvciBpbmRleCBpbiByYW5nZSgxLCAzMDEpCiAgICBdLCBjb2x1bW5zPWNvbHVtbnMpCiAgICBxdWVyeSA9IHBkLkRhdGFGcmFtZShbKCJTMS0xIiwgInVzIiwgKCJzaGFyZWQiLCkpXSwgY29sdW1ucz1jb2x1bW5zKQogICAgZmlyc3QgPSBSYXJlVG9rZW5CbG9ja2VyKCJuYW1lX3Rva2VucyIsIENhbmRpZGF0ZVJlYXNvbi5SQVJFX05BTUVfVE9LRU4sIDUwMCwgMjUpLmZpdCh0YXJnZXRzKS50cmFuc2Zvcm0ocXVlcnkpCiAgICBzZWNvbmQgPSBSYXJlVG9rZW5CbG9ja2VyKCJuYW1lX3Rva2VucyIsIENhbmRpZGF0ZVJlYXNvbi5SQVJFX05BTUVfVE9LRU4sIDUwMCwgMjUpLmZpdCh0YXJnZXRzLmlsb2NbOjotMV0pLnRyYW5zZm9ybShxdWVyeSkKICAgIGFzc2VydCBmaXJzdFsiY2FuZGlkYXRlX2VudGl0eV9pZCJdLnRvbGlzdCgpID09IHNlY29uZFsiY2FuZGlkYXRlX2VudGl0eV9pZCJdLnRvbGlzdCgpCgoKZGVmIHRlc3RfY291bnRyeV9mYWxsYmFja19yZWFzb25faXNfY2FuZGlkYXRlX3NwZWNpZmljKCk6CiAgICBjb2x1bW5zID0gWyJlbnRpdHlfaWQiLCAiY291bnRyeV9ub3JtIiwgIm5hbWVfdG9rZW5zIl0KICAgIHRhcmdldHMgPSBwZC5EYXRhRnJhbWUoWwogICAgICAgICgiUzItMSIsICJmcmFuY2UiLCAoImxvY2FsIiwpKSwKICAgICAgICAoIlMyLTIiLCAidXMiLCAoImdsb2JhbCIsKSksCiAgICBdLCBjb2x1bW5zPWNvbHVtbnMpCiAgICBxdWVyeSA9IHBkLkRhdGFGcmFtZShbCiAgICAgICAgKCJTMS0xIiwgImZyYW5jZSIsICgibG9jYWwiLCAiZ2xvYmFsIikpLAogICAgXSwgY29sdW1ucz1jb2x1bW5zKQogICAgcmVzdWx0ID0gUmFyZVRva2VuQmxvY2tlcigibmFtZV90b2tlbnMiLCBDYW5kaWRhdGVSZWFzb24uUkFSRV9OQU1FX1RPS0VOLCAxMCwgMTApLmZpdCh0YXJnZXRzKS50cmFuc2Zvcm0ocXVlcnkpCiAgICBiaXRzID0gZGljdCh6aXAocmVzdWx0WyJjYW5kaWRhdGVfZW50aXR5X2lkIl0sIHJlc3VsdFsicmVhc29uX2JpdHMiXSkpCiAgICBhc3NlcnQgbm90IGJpdHNbIlMyLTEiXSAmIGludChDYW5kaWRhdGVSZWFzb24uQ09VTlRSWV9GQUxMQkFDSykKICAgIGFzc2VydCBiaXRzWyJTMi0yIl0gJiBpbnQoQ2FuZGlkYXRlUmVhc29uLkNPVU5UUllfRkFMTEJBQ0spCgoKZGVmIHRlc3RfY2h1bmtlZF90ZmlkZl9tYXRjaGVzX3NpbmdsZV90YXJnZXRfYmxvY2soc21va2VfZnJhbWVzKToKICAgIHMxLCBzMiwgXywgXyA9IHNtb2tlX2ZyYW1lcwogICAgcXVlcmllcywgdGFyZ2V0cyA9IG5vcm1hbGl6ZV9yZWNvcmRzKHMxKSwgbm9ybWFsaXplX3JlY29yZHMoczIpCiAgICBjaHVua2VkID0gVGZpZGZUb3BLQmxvY2tlcih0b3Bfaz0zLCBtaW5fc2NvcmU9MC4wLCBiYXRjaF9zaXplPTIsIHRhcmdldF9iYXRjaF9zaXplPTIpLmZpdCh0YXJnZXRzKS50cmFuc2Zvcm0ocXVlcmllcykKICAgIHNpbmdsZSA9IFRmaWRmVG9wS0Jsb2NrZXIodG9wX2s9MywgbWluX3Njb3JlPTAuMCwgYmF0Y2hfc2l6ZT0yLCB0YXJnZXRfYmF0Y2hfc2l6ZT0xMDApLmZpdCh0YXJnZXRzKS50cmFuc2Zvcm0ocXVlcmllcykKICAgIHBkLnRlc3RpbmcuYXNzZXJ0X2ZyYW1lX2VxdWFsKGNodW5rZWQsIHNpbmdsZSkK', 'dac92055cdfed10b4b3876180b9fcfce5449ebac0f6b9e743299171577a2ae63')

## Embedded source: `code/business_entity_resolution/tests/test_cli_regressions.py`

~~~~python
from argparse import Namespace
import json

import pandas as pd

from business_entity_resolution.cli import _new_model, command_tune_decision
from business_entity_resolution.config import ProjectConfig


def _write_config(tmp_path):
    data_root = tmp_path / "dataset"
    artifact_root = tmp_path / "artifacts"
    output_root = tmp_path / "output"
    (data_root / "train").mkdir(parents=True)
    config_path = tmp_path / "config.json"
    config_path.write_text(json.dumps({
        "data_root": str(data_root),
        "artifact_root": str(artifact_root),
        "output_root": str(output_root),
        "run_id": "test",
        "model": {
            "kind": "lightgbm",
            "params": {"n_estimators": 20, "num_leaves": 7},
        },
    }), encoding="utf-8")
    return config_path, data_root, artifact_root / "test"


def test_model_override_does_not_receive_other_backend_params(tmp_path):
    config_path, _, _ = _write_config(tmp_path)
    config = ProjectConfig.load(config_path)
    model = _new_model(config, "sgd")
    assert "n_estimators" not in model.params
    assert "num_leaves" not in model.params


def test_tuning_includes_zero_candidate_validation_entities(tmp_path):
    config_path, data_root, run_root = _write_config(tmp_path)
    truth = pd.DataFrame([
        ("S1-1", "S2-1"),
        ("S1-2", ""),
    ], columns=["source1_entity_id", "matched_entity_ids"])
    truth.to_csv(data_root / "train" / "train_ground_truth.tsv", sep="\t", index=False)

    (run_root / "splits").mkdir(parents=True)
    pd.DataFrame([
        ("S1-1", 0), ("S1-2", 0),
    ], columns=["source1_entity_id", "fold"]).to_parquet(run_root / "splits" / "s1_folds.parquet", index=False)
    (run_root / "scores").mkdir(parents=True)
    pd.DataFrame([
        ("S1-1", "S2-1", "S2", 0.9, 1),
    ], columns=["source1_entity_id", "candidate_entity_id", "candidate_source", "score", "label"]).to_parquet(
        run_root / "scores" / "train.parquet", index=False
    )

    args = Namespace(
        config=str(config_path), max_rows=None, force=False,
        score_file="train.parquet", validation_fold=0, threshold_count=5,
    )
    assert command_tune_decision(args) == 0
    report = pd.read_parquet(run_root / "decisions" / "threshold_sweep.parquet")
    assert set(report["entity_count"]) == {2}
    assert (run_root / "decisions" / "selected_threshold.json").is_file()

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_cli_regressions.py', 'ZnJvbSBhcmdwYXJzZSBpbXBvcnQgTmFtZXNwYWNlCmltcG9ydCBqc29uCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLmNsaSBpbXBvcnQgX25ld19tb2RlbCwgY29tbWFuZF90dW5lX2RlY2lzaW9uCmZyb20gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24uY29uZmlnIGltcG9ydCBQcm9qZWN0Q29uZmlnCgoKZGVmIF93cml0ZV9jb25maWcodG1wX3BhdGgpOgogICAgZGF0YV9yb290ID0gdG1wX3BhdGggLyAiZGF0YXNldCIKICAgIGFydGlmYWN0X3Jvb3QgPSB0bXBfcGF0aCAvICJhcnRpZmFjdHMiCiAgICBvdXRwdXRfcm9vdCA9IHRtcF9wYXRoIC8gIm91dHB1dCIKICAgIChkYXRhX3Jvb3QgLyAidHJhaW4iKS5ta2RpcihwYXJlbnRzPVRydWUpCiAgICBjb25maWdfcGF0aCA9IHRtcF9wYXRoIC8gImNvbmZpZy5qc29uIgogICAgY29uZmlnX3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKHsKICAgICAgICAiZGF0YV9yb290Ijogc3RyKGRhdGFfcm9vdCksCiAgICAgICAgImFydGlmYWN0X3Jvb3QiOiBzdHIoYXJ0aWZhY3Rfcm9vdCksCiAgICAgICAgIm91dHB1dF9yb290Ijogc3RyKG91dHB1dF9yb290KSwKICAgICAgICAicnVuX2lkIjogInRlc3QiLAogICAgICAgICJtb2RlbCI6IHsKICAgICAgICAgICAgImtpbmQiOiAibGlnaHRnYm0iLAogICAgICAgICAgICAicGFyYW1zIjogeyJuX2VzdGltYXRvcnMiOiAyMCwgIm51bV9sZWF2ZXMiOiA3fSwKICAgICAgICB9LAogICAgfSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gY29uZmlnX3BhdGgsIGRhdGFfcm9vdCwgYXJ0aWZhY3Rfcm9vdCAvICJ0ZXN0IgoKCmRlZiB0ZXN0X21vZGVsX292ZXJyaWRlX2RvZXNfbm90X3JlY2VpdmVfb3RoZXJfYmFja2VuZF9wYXJhbXModG1wX3BhdGgpOgogICAgY29uZmlnX3BhdGgsIF8sIF8gPSBfd3JpdGVfY29uZmlnKHRtcF9wYXRoKQogICAgY29uZmlnID0gUHJvamVjdENvbmZpZy5sb2FkKGNvbmZpZ19wYXRoKQogICAgbW9kZWwgPSBfbmV3X21vZGVsKGNvbmZpZywgInNnZCIpCiAgICBhc3NlcnQgIm5fZXN0aW1hdG9ycyIgbm90IGluIG1vZGVsLnBhcmFtcwogICAgYXNzZXJ0ICJudW1fbGVhdmVzIiBub3QgaW4gbW9kZWwucGFyYW1zCgoKZGVmIHRlc3RfdHVuaW5nX2luY2x1ZGVzX3plcm9fY2FuZGlkYXRlX3ZhbGlkYXRpb25fZW50aXRpZXModG1wX3BhdGgpOgogICAgY29uZmlnX3BhdGgsIGRhdGFfcm9vdCwgcnVuX3Jvb3QgPSBfd3JpdGVfY29uZmlnKHRtcF9wYXRoKQogICAgdHJ1dGggPSBwZC5EYXRhRnJhbWUoWwogICAgICAgICgiUzEtMSIsICJTMi0xIiksCiAgICAgICAgKCJTMS0yIiwgIiIpLAogICAgXSwgY29sdW1ucz1bInNvdXJjZTFfZW50aXR5X2lkIiwgIm1hdGNoZWRfZW50aXR5X2lkcyJdKQogICAgdHJ1dGgudG9fY3N2KGRhdGFfcm9vdCAvICJ0cmFpbiIgLyAidHJhaW5fZ3JvdW5kX3RydXRoLnRzdiIsIHNlcD0iXHQiLCBpbmRleD1GYWxzZSkKCiAgICAocnVuX3Jvb3QgLyAic3BsaXRzIikubWtkaXIocGFyZW50cz1UcnVlKQogICAgcGQuRGF0YUZyYW1lKFsKICAgICAgICAoIlMxLTEiLCAwKSwgKCJTMS0yIiwgMCksCiAgICBdLCBjb2x1bW5zPVsic291cmNlMV9lbnRpdHlfaWQiLCAiZm9sZCJdKS50b19wYXJxdWV0KHJ1bl9yb290IC8gInNwbGl0cyIgLyAiczFfZm9sZHMucGFycXVldCIsIGluZGV4PUZhbHNlKQogICAgKHJ1bl9yb290IC8gInNjb3JlcyIpLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIHBkLkRhdGFGcmFtZShbCiAgICAgICAgKCJTMS0xIiwgIlMyLTEiLCAiUzIiLCAwLjksIDEpLAogICAgXSwgY29sdW1ucz1bInNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiLCAiY2FuZGlkYXRlX3NvdXJjZSIsICJzY29yZSIsICJsYWJlbCJdKS50b19wYXJxdWV0KAogICAgICAgIHJ1bl9yb290IC8gInNjb3JlcyIgLyAidHJhaW4ucGFycXVldCIsIGluZGV4PUZhbHNlCiAgICApCgogICAgYXJncyA9IE5hbWVzcGFjZSgKICAgICAgICBjb25maWc9c3RyKGNvbmZpZ19wYXRoKSwgbWF4X3Jvd3M9Tm9uZSwgZm9yY2U9RmFsc2UsCiAgICAgICAgc2NvcmVfZmlsZT0idHJhaW4ucGFycXVldCIsIHZhbGlkYXRpb25fZm9sZD0wLCB0aHJlc2hvbGRfY291bnQ9NSwKICAgICkKICAgIGFzc2VydCBjb21tYW5kX3R1bmVfZGVjaXNpb24oYXJncykgPT0gMAogICAgcmVwb3J0ID0gcGQucmVhZF9wYXJxdWV0KHJ1bl9yb290IC8gImRlY2lzaW9ucyIgLyAidGhyZXNob2xkX3N3ZWVwLnBhcnF1ZXQiKQogICAgYXNzZXJ0IHNldChyZXBvcnRbImVudGl0eV9jb3VudCJdKSA9PSB7Mn0KICAgIGFzc2VydCAocnVuX3Jvb3QgLyAiZGVjaXNpb25zIiAvICJzZWxlY3RlZF90aHJlc2hvbGQuanNvbiIpLmlzX2ZpbGUoKQo=', 'c1e5b2766cce865e1a32fd13e58015add6b02a2bae4c119ae338c0e862a7227a')

## Embedded source: `code/business_entity_resolution/tests/test_config.py`

~~~~python
import json

from business_entity_resolution.config import ProjectConfig


def test_config_loads_relative_paths_and_env(tmp_path, monkeypatch):
    config_path = tmp_path / "config.json"
    config_path.write_text(json.dumps({"data_root": "data", "artifact_root": "artifacts", "output_root": "out"}), encoding="utf-8")
    monkeypatch.chdir(tmp_path)
    monkeypatch.setenv("BER_RUN_ID", "test-run")
    config = ProjectConfig.load(config_path)
    assert config.data_root == tmp_path / "data"
    assert config.run_id == "test-run"


~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_config.py', 'aW1wb3J0IGpzb24KCmZyb20gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24uY29uZmlnIGltcG9ydCBQcm9qZWN0Q29uZmlnCgoKZGVmIHRlc3RfY29uZmlnX2xvYWRzX3JlbGF0aXZlX3BhdGhzX2FuZF9lbnYodG1wX3BhdGgsIG1vbmtleXBhdGNoKToKICAgIGNvbmZpZ19wYXRoID0gdG1wX3BhdGggLyAiY29uZmlnLmpzb24iCiAgICBjb25maWdfcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyJkYXRhX3Jvb3QiOiAiZGF0YSIsICJhcnRpZmFjdF9yb290IjogImFydGlmYWN0cyIsICJvdXRwdXRfcm9vdCI6ICJvdXQifSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBtb25rZXlwYXRjaC5jaGRpcih0bXBfcGF0aCkKICAgIG1vbmtleXBhdGNoLnNldGVudigiQkVSX1JVTl9JRCIsICJ0ZXN0LXJ1biIpCiAgICBjb25maWcgPSBQcm9qZWN0Q29uZmlnLmxvYWQoY29uZmlnX3BhdGgpCiAgICBhc3NlcnQgY29uZmlnLmRhdGFfcm9vdCA9PSB0bXBfcGF0aCAvICJkYXRhIgogICAgYXNzZXJ0IGNvbmZpZy5ydW5faWQgPT0gInRlc3QtcnVuIgoK', 'dd9f4f7db25b67f7eba5d6b42f99f3829858a4d8b2d0c48f564cd21ff2835cd5')

## Embedded source: `code/business_entity_resolution/tests/test_data.py`

~~~~python
import pandas as pd
import pytest

from business_entity_resolution.data import load_ground_truth_for_ids, read_tsv, validate_entity_frame
from business_entity_resolution.schemas import SOURCE_COLUMNS, SchemaError


def test_tsv_loader_preserves_commas_and_empty_address(tmp_path):
    path = tmp_path / "source.tsv"
    path.write_text("entity_id\tbusiness_name\tbusiness_address\tcountry\nS2-1\tCafe\tParis, France\tFrance\nS2-2\tEmpty\t\tAtlantis\n", encoding="utf-8")
    frame = read_tsv(path, SOURCE_COLUMNS)
    assert frame.loc[0, "business_address"] == "Paris, France"
    assert frame.loc[1, "business_address"] == ""
    validate_entity_frame(frame, 2)


def test_wrong_schema_and_duplicate_ids_fail(tmp_path):
    path = tmp_path / "bad.tsv"
    path.write_text("entity_id,business_name,business_address,country\n", encoding="utf-8")
    with pytest.raises(SchemaError):
        read_tsv(path, SOURCE_COLUMNS)
    frame = pd.DataFrame([
        ("S1-1", "A", "X", "US"), ("S1-1", "B", "Y", "US")
    ], columns=SOURCE_COLUMNS)
    with pytest.raises(SchemaError):
        validate_entity_frame(frame, 1)


def test_ground_truth_subset_is_selected_by_id_not_row_position(tmp_path):
    train = tmp_path / "train"
    train.mkdir()
    (train / "train_ground_truth.tsv").write_text(
        "source1_entity_id\tmatched_entity_ids\n"
        "S1-99\tS2-99\n"
        "S1-2\t\n"
        "S1-1\tS3-1\n",
        encoding="utf-8",
    )
    selected = load_ground_truth_for_ids(tmp_path, ["S1-1", "S1-2"], batch_size=1)
    assert set(selected["source1_entity_id"]) == {"S1-1", "S1-2"}

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_data.py', 'aW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgcHl0ZXN0Cgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLmRhdGEgaW1wb3J0IGxvYWRfZ3JvdW5kX3RydXRoX2Zvcl9pZHMsIHJlYWRfdHN2LCB2YWxpZGF0ZV9lbnRpdHlfZnJhbWUKZnJvbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5zY2hlbWFzIGltcG9ydCBTT1VSQ0VfQ09MVU1OUywgU2NoZW1hRXJyb3IKCgpkZWYgdGVzdF90c3ZfbG9hZGVyX3ByZXNlcnZlc19jb21tYXNfYW5kX2VtcHR5X2FkZHJlc3ModG1wX3BhdGgpOgogICAgcGF0aCA9IHRtcF9wYXRoIC8gInNvdXJjZS50c3YiCiAgICBwYXRoLndyaXRlX3RleHQoImVudGl0eV9pZFx0YnVzaW5lc3NfbmFtZVx0YnVzaW5lc3NfYWRkcmVzc1x0Y291bnRyeVxuUzItMVx0Q2FmZVx0UGFyaXMsIEZyYW5jZVx0RnJhbmNlXG5TMi0yXHRFbXB0eVx0XHRBdGxhbnRpc1xuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIGZyYW1lID0gcmVhZF90c3YocGF0aCwgU09VUkNFX0NPTFVNTlMpCiAgICBhc3NlcnQgZnJhbWUubG9jWzAsICJidXNpbmVzc19hZGRyZXNzIl0gPT0gIlBhcmlzLCBGcmFuY2UiCiAgICBhc3NlcnQgZnJhbWUubG9jWzEsICJidXNpbmVzc19hZGRyZXNzIl0gPT0gIiIKICAgIHZhbGlkYXRlX2VudGl0eV9mcmFtZShmcmFtZSwgMikKCgpkZWYgdGVzdF93cm9uZ19zY2hlbWFfYW5kX2R1cGxpY2F0ZV9pZHNfZmFpbCh0bXBfcGF0aCk6CiAgICBwYXRoID0gdG1wX3BhdGggLyAiYmFkLnRzdiIKICAgIHBhdGgud3JpdGVfdGV4dCgiZW50aXR5X2lkLGJ1c2luZXNzX25hbWUsYnVzaW5lc3NfYWRkcmVzcyxjb3VudHJ5XG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFNjaGVtYUVycm9yKToKICAgICAgICByZWFkX3RzdihwYXRoLCBTT1VSQ0VfQ09MVU1OUykKICAgIGZyYW1lID0gcGQuRGF0YUZyYW1lKFsKICAgICAgICAoIlMxLTEiLCAiQSIsICJYIiwgIlVTIiksICgiUzEtMSIsICJCIiwgIlkiLCAiVVMiKQogICAgXSwgY29sdW1ucz1TT1VSQ0VfQ09MVU1OUykKICAgIHdpdGggcHl0ZXN0LnJhaXNlcyhTY2hlbWFFcnJvcik6CiAgICAgICAgdmFsaWRhdGVfZW50aXR5X2ZyYW1lKGZyYW1lLCAxKQoKCmRlZiB0ZXN0X2dyb3VuZF90cnV0aF9zdWJzZXRfaXNfc2VsZWN0ZWRfYnlfaWRfbm90X3Jvd19wb3NpdGlvbih0bXBfcGF0aCk6CiAgICB0cmFpbiA9IHRtcF9wYXRoIC8gInRyYWluIgogICAgdHJhaW4ubWtkaXIoKQogICAgKHRyYWluIC8gInRyYWluX2dyb3VuZF90cnV0aC50c3YiKS53cml0ZV90ZXh0KAogICAgICAgICJzb3VyY2UxX2VudGl0eV9pZFx0bWF0Y2hlZF9lbnRpdHlfaWRzXG4iCiAgICAgICAgIlMxLTk5XHRTMi05OVxuIgogICAgICAgICJTMS0yXHRcbiIKICAgICAgICAiUzEtMVx0UzMtMVxuIiwKICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgKQogICAgc2VsZWN0ZWQgPSBsb2FkX2dyb3VuZF90cnV0aF9mb3JfaWRzKHRtcF9wYXRoLCBbIlMxLTEiLCAiUzEtMiJdLCBiYXRjaF9zaXplPTEpCiAgICBhc3NlcnQgc2V0KHNlbGVjdGVkWyJzb3VyY2UxX2VudGl0eV9pZCJdKSA9PSB7IlMxLTEiLCAiUzEtMiJ9Cg==', '7c0c7a3db4a507f3596f1d4c6832f416dc9e2d7254686b2a78159de725796222')

## Embedded source: `code/business_entity_resolution/tests/test_features.py`

~~~~python
import numpy as np
import pandas as pd

from business_entity_resolution.features import FEATURE_COLUMNS, build_pair_features, feature_matrix
from business_entity_resolution.normalize import normalize_records


def test_pair_features_use_only_candidates(smoke_frames):
    s1, s2, _, _ = smoke_frames
    candidates = pd.DataFrame([{
        "source1_entity_id": "S1-1", "candidate_entity_id": "S2-10", "candidate_source": "S2",
        "reason_mask": "exact_name", "retrieval_score": 1.0, "retrieval_rank": 1,
    }])
    features = build_pair_features(candidates, normalize_records(s1), normalize_records(s2))
    assert len(features) == 1
    assert set(FEATURE_COLUMNS).issubset(features.columns)
    assert feature_matrix(features).dtype == np.float32
    assert features.loc[0, "country_equal"] == 1.0


def test_missing_fields_are_neutral_not_perfect():
    columns = ["entity_id", "business_name", "business_address", "country"]
    s1 = normalize_records(pd.DataFrame([["S1-1", "Cafe Ecole", "", "France"]], columns=columns))
    targets = normalize_records(pd.DataFrame([["S2-1", "Café École", "", "France"]], columns=columns))
    candidates = pd.DataFrame([{
        "source1_entity_id": "S1-1", "candidate_entity_id": "S2-1", "candidate_source": "S2",
        "reason_mask": "exact_accent_folded", "retrieval_score": 1.0, "retrieval_rank": 1,
    }])
    features = build_pair_features(candidates, s1, targets).iloc[0]
    assert features["name_accent_folded_exact"] == 1.0
    assert features["address_ratio"] == 0.0
    assert features["address_token_jaccard"] == 0.0
    assert features["numeric_jaccard"] == 0.0
    assert features["postal_jaccard"] == 0.0
    assert features["name_address_interaction"] == 0.0

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_features.py', 'aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24uZmVhdHVyZXMgaW1wb3J0IEZFQVRVUkVfQ09MVU1OUywgYnVpbGRfcGFpcl9mZWF0dXJlcywgZmVhdHVyZV9tYXRyaXgKZnJvbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5ub3JtYWxpemUgaW1wb3J0IG5vcm1hbGl6ZV9yZWNvcmRzCgoKZGVmIHRlc3RfcGFpcl9mZWF0dXJlc191c2Vfb25seV9jYW5kaWRhdGVzKHNtb2tlX2ZyYW1lcyk6CiAgICBzMSwgczIsIF8sIF8gPSBzbW9rZV9mcmFtZXMKICAgIGNhbmRpZGF0ZXMgPSBwZC5EYXRhRnJhbWUoW3sKICAgICAgICAic291cmNlMV9lbnRpdHlfaWQiOiAiUzEtMSIsICJjYW5kaWRhdGVfZW50aXR5X2lkIjogIlMyLTEwIiwgImNhbmRpZGF0ZV9zb3VyY2UiOiAiUzIiLAogICAgICAgICJyZWFzb25fbWFzayI6ICJleGFjdF9uYW1lIiwgInJldHJpZXZhbF9zY29yZSI6IDEuMCwgInJldHJpZXZhbF9yYW5rIjogMSwKICAgIH1dKQogICAgZmVhdHVyZXMgPSBidWlsZF9wYWlyX2ZlYXR1cmVzKGNhbmRpZGF0ZXMsIG5vcm1hbGl6ZV9yZWNvcmRzKHMxKSwgbm9ybWFsaXplX3JlY29yZHMoczIpKQogICAgYXNzZXJ0IGxlbihmZWF0dXJlcykgPT0gMQogICAgYXNzZXJ0IHNldChGRUFUVVJFX0NPTFVNTlMpLmlzc3Vic2V0KGZlYXR1cmVzLmNvbHVtbnMpCiAgICBhc3NlcnQgZmVhdHVyZV9tYXRyaXgoZmVhdHVyZXMpLmR0eXBlID09IG5wLmZsb2F0MzIKICAgIGFzc2VydCBmZWF0dXJlcy5sb2NbMCwgImNvdW50cnlfZXF1YWwiXSA9PSAxLjAKCgpkZWYgdGVzdF9taXNzaW5nX2ZpZWxkc19hcmVfbmV1dHJhbF9ub3RfcGVyZmVjdCgpOgogICAgY29sdW1ucyA9IFsiZW50aXR5X2lkIiwgImJ1c2luZXNzX25hbWUiLCAiYnVzaW5lc3NfYWRkcmVzcyIsICJjb3VudHJ5Il0KICAgIHMxID0gbm9ybWFsaXplX3JlY29yZHMocGQuRGF0YUZyYW1lKFtbIlMxLTEiLCAiQ2FmZSBFY29sZSIsICIiLCAiRnJhbmNlIl1dLCBjb2x1bW5zPWNvbHVtbnMpKQogICAgdGFyZ2V0cyA9IG5vcm1hbGl6ZV9yZWNvcmRzKHBkLkRhdGFGcmFtZShbWyJTMi0xIiwgIkNhZsOpIMOJY29sZSIsICIiLCAiRnJhbmNlIl1dLCBjb2x1bW5zPWNvbHVtbnMpKQogICAgY2FuZGlkYXRlcyA9IHBkLkRhdGFGcmFtZShbewogICAgICAgICJzb3VyY2UxX2VudGl0eV9pZCI6ICJTMS0xIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiOiAiUzItMSIsICJjYW5kaWRhdGVfc291cmNlIjogIlMyIiwKICAgICAgICAicmVhc29uX21hc2siOiAiZXhhY3RfYWNjZW50X2ZvbGRlZCIsICJyZXRyaWV2YWxfc2NvcmUiOiAxLjAsICJyZXRyaWV2YWxfcmFuayI6IDEsCiAgICB9XSkKICAgIGZlYXR1cmVzID0gYnVpbGRfcGFpcl9mZWF0dXJlcyhjYW5kaWRhdGVzLCBzMSwgdGFyZ2V0cykuaWxvY1swXQogICAgYXNzZXJ0IGZlYXR1cmVzWyJuYW1lX2FjY2VudF9mb2xkZWRfZXhhY3QiXSA9PSAxLjAKICAgIGFzc2VydCBmZWF0dXJlc1siYWRkcmVzc19yYXRpbyJdID09IDAuMAogICAgYXNzZXJ0IGZlYXR1cmVzWyJhZGRyZXNzX3Rva2VuX2phY2NhcmQiXSA9PSAwLjAKICAgIGFzc2VydCBmZWF0dXJlc1sibnVtZXJpY19qYWNjYXJkIl0gPT0gMC4wCiAgICBhc3NlcnQgZmVhdHVyZXNbInBvc3RhbF9qYWNjYXJkIl0gPT0gMC4wCiAgICBhc3NlcnQgZmVhdHVyZXNbIm5hbWVfYWRkcmVzc19pbnRlcmFjdGlvbiJdID09IDAuMAo=', '85d45002a8b4bfc932bb9744f5f40bfca00177f39b60f0ee770427fc7afce2fa')

## Embedded source: `code/business_entity_resolution/tests/test_labels_splits.py`

~~~~python
import pandas as pd
import pytest

from business_entity_resolution.labels import ground_truth_sets, parse_match_ids
from business_entity_resolution.splits import assign_s1_folds, select_pair_fold


def test_ground_truth_parsing_and_empty_singleton(smoke_frames):
    _, _, _, gt = smoke_frames
    mapping = ground_truth_sets(gt)
    assert mapping["S1-1"] == {"S2-10", "S3-11"}
    assert mapping["S1-4"] == frozenset()
    with pytest.raises(ValueError):
        parse_match_ids("S2-1,S2-1")


def test_split_is_deterministic(smoke_frames):
    s1, _, _, gt = smoke_frames
    first = assign_s1_folds(s1, gt, n_folds=3, seed=7)
    second = assign_s1_folds(s1, gt, n_folds=3, seed=7)
    pd.testing.assert_frame_equal(first, second)
    assert set(first.source1_entity_id) == set(s1.entity_id)


def test_pair_fold_selection_keeps_entities_disjoint():
    pairs = pd.DataFrame({
        "source1_entity_id": ["S1-1", "S1-1", "S1-2", "S1-3"],
        "candidate_entity_id": ["S2-1", "S3-1", "S2-2", "S2-3"],
    })
    folds = pd.DataFrame({"source1_entity_id": ["S1-1", "S1-2", "S1-3"], "fold": [0, 1, 0]})
    training = select_pair_fold(pairs, folds, 0, validation=False)
    held_out = select_pair_fold(pairs, folds, 0, validation=True)
    assert set(training["source1_entity_id"]) == {"S1-2"}
    assert set(held_out["source1_entity_id"]) == {"S1-1", "S1-3"}
    assert set(training["source1_entity_id"]).isdisjoint(held_out["source1_entity_id"])

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_labels_splits.py', 'aW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgcHl0ZXN0Cgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLmxhYmVscyBpbXBvcnQgZ3JvdW5kX3RydXRoX3NldHMsIHBhcnNlX21hdGNoX2lkcwpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLnNwbGl0cyBpbXBvcnQgYXNzaWduX3MxX2ZvbGRzLCBzZWxlY3RfcGFpcl9mb2xkCgoKZGVmIHRlc3RfZ3JvdW5kX3RydXRoX3BhcnNpbmdfYW5kX2VtcHR5X3NpbmdsZXRvbihzbW9rZV9mcmFtZXMpOgogICAgXywgXywgXywgZ3QgPSBzbW9rZV9mcmFtZXMKICAgIG1hcHBpbmcgPSBncm91bmRfdHJ1dGhfc2V0cyhndCkKICAgIGFzc2VydCBtYXBwaW5nWyJTMS0xIl0gPT0geyJTMi0xMCIsICJTMy0xMSJ9CiAgICBhc3NlcnQgbWFwcGluZ1siUzEtNCJdID09IGZyb3plbnNldCgpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgcGFyc2VfbWF0Y2hfaWRzKCJTMi0xLFMyLTEiKQoKCmRlZiB0ZXN0X3NwbGl0X2lzX2RldGVybWluaXN0aWMoc21va2VfZnJhbWVzKToKICAgIHMxLCBfLCBfLCBndCA9IHNtb2tlX2ZyYW1lcwogICAgZmlyc3QgPSBhc3NpZ25fczFfZm9sZHMoczEsIGd0LCBuX2ZvbGRzPTMsIHNlZWQ9NykKICAgIHNlY29uZCA9IGFzc2lnbl9zMV9mb2xkcyhzMSwgZ3QsIG5fZm9sZHM9Mywgc2VlZD03KQogICAgcGQudGVzdGluZy5hc3NlcnRfZnJhbWVfZXF1YWwoZmlyc3QsIHNlY29uZCkKICAgIGFzc2VydCBzZXQoZmlyc3Quc291cmNlMV9lbnRpdHlfaWQpID09IHNldChzMS5lbnRpdHlfaWQpCgoKZGVmIHRlc3RfcGFpcl9mb2xkX3NlbGVjdGlvbl9rZWVwc19lbnRpdGllc19kaXNqb2ludCgpOgogICAgcGFpcnMgPSBwZC5EYXRhRnJhbWUoewogICAgICAgICJzb3VyY2UxX2VudGl0eV9pZCI6IFsiUzEtMSIsICJTMS0xIiwgIlMxLTIiLCAiUzEtMyJdLAogICAgICAgICJjYW5kaWRhdGVfZW50aXR5X2lkIjogWyJTMi0xIiwgIlMzLTEiLCAiUzItMiIsICJTMi0zIl0sCiAgICB9KQogICAgZm9sZHMgPSBwZC5EYXRhRnJhbWUoeyJzb3VyY2UxX2VudGl0eV9pZCI6IFsiUzEtMSIsICJTMS0yIiwgIlMxLTMiXSwgImZvbGQiOiBbMCwgMSwgMF19KQogICAgdHJhaW5pbmcgPSBzZWxlY3RfcGFpcl9mb2xkKHBhaXJzLCBmb2xkcywgMCwgdmFsaWRhdGlvbj1GYWxzZSkKICAgIGhlbGRfb3V0ID0gc2VsZWN0X3BhaXJfZm9sZChwYWlycywgZm9sZHMsIDAsIHZhbGlkYXRpb249VHJ1ZSkKICAgIGFzc2VydCBzZXQodHJhaW5pbmdbInNvdXJjZTFfZW50aXR5X2lkIl0pID09IHsiUzEtMiJ9CiAgICBhc3NlcnQgc2V0KGhlbGRfb3V0WyJzb3VyY2UxX2VudGl0eV9pZCJdKSA9PSB7IlMxLTEiLCAiUzEtMyJ9CiAgICBhc3NlcnQgc2V0KHRyYWluaW5nWyJzb3VyY2UxX2VudGl0eV9pZCJdKS5pc2Rpc2pvaW50KGhlbGRfb3V0WyJzb3VyY2UxX2VudGl0eV9pZCJdKQo=', 'fe9f0f666dfecaf678be50c7b6aae0ed34b720697abc7c0dfa8574bd57f6e84f')

## Embedded source: `code/business_entity_resolution/tests/test_metrics.py`

~~~~python
import pytest
import pandas as pd

from business_entity_resolution.metrics import (
    candidate_metrics,
    candidate_metrics_from_frames,
    entity_scores,
    evaluate_entity_sets,
)


def test_singleton_edge_cases():
    assert entity_scores(set(), set()) == (1.0, 1.0, 1.0)
    assert entity_scores(set(), {"S2-1"}) == (0.0, 0.0, 0.0)
    assert entity_scores({"S2-1"}, set()) == (0.0, 0.0, 0.0)


def test_macro_f05_and_candidate_metrics():
    truth = {"S1-1": {"S2-1", "S3-1"}, "S1-2": set()}
    pred = {"S1-1": {"S2-1", "S2-9"}, "S1-2": set()}
    report = evaluate_entity_sets(truth, pred)
    assert report["macro_f0_5"] == pytest.approx((0.5 + 1.0) / 2)
    candidates = {"S1-1": {"S2-1", "S3-1", "S2-9"}, "S1-2": set()}
    candidate_report = candidate_metrics(truth, candidates, target_universe_size=10)
    assert candidate_report["candidate_recall"] == 1.0
    assert candidate_report["complete_entity_recall"] == 1.0

    frame = pd.DataFrame([
        ("S1-1", "S2-1"), ("S1-1", "S3-1"), ("S1-1", "S2-9"),
    ], columns=["source1_entity_id", "candidate_entity_id"])
    streamed = candidate_metrics_from_frames(truth, [frame], target_universe_size=10)
    assert streamed == candidate_report

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_metrics.py', 'aW1wb3J0IHB5dGVzdAppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLm1ldHJpY3MgaW1wb3J0ICgKICAgIGNhbmRpZGF0ZV9tZXRyaWNzLAogICAgY2FuZGlkYXRlX21ldHJpY3NfZnJvbV9mcmFtZXMsCiAgICBlbnRpdHlfc2NvcmVzLAogICAgZXZhbHVhdGVfZW50aXR5X3NldHMsCikKCgpkZWYgdGVzdF9zaW5nbGV0b25fZWRnZV9jYXNlcygpOgogICAgYXNzZXJ0IGVudGl0eV9zY29yZXMoc2V0KCksIHNldCgpKSA9PSAoMS4wLCAxLjAsIDEuMCkKICAgIGFzc2VydCBlbnRpdHlfc2NvcmVzKHNldCgpLCB7IlMyLTEifSkgPT0gKDAuMCwgMC4wLCAwLjApCiAgICBhc3NlcnQgZW50aXR5X3Njb3Jlcyh7IlMyLTEifSwgc2V0KCkpID09ICgwLjAsIDAuMCwgMC4wKQoKCmRlZiB0ZXN0X21hY3JvX2YwNV9hbmRfY2FuZGlkYXRlX21ldHJpY3MoKToKICAgIHRydXRoID0geyJTMS0xIjogeyJTMi0xIiwgIlMzLTEifSwgIlMxLTIiOiBzZXQoKX0KICAgIHByZWQgPSB7IlMxLTEiOiB7IlMyLTEiLCAiUzItOSJ9LCAiUzEtMiI6IHNldCgpfQogICAgcmVwb3J0ID0gZXZhbHVhdGVfZW50aXR5X3NldHModHJ1dGgsIHByZWQpCiAgICBhc3NlcnQgcmVwb3J0WyJtYWNyb19mMF81Il0gPT0gcHl0ZXN0LmFwcHJveCgoMC41ICsgMS4wKSAvIDIpCiAgICBjYW5kaWRhdGVzID0geyJTMS0xIjogeyJTMi0xIiwgIlMzLTEiLCAiUzItOSJ9LCAiUzEtMiI6IHNldCgpfQogICAgY2FuZGlkYXRlX3JlcG9ydCA9IGNhbmRpZGF0ZV9tZXRyaWNzKHRydXRoLCBjYW5kaWRhdGVzLCB0YXJnZXRfdW5pdmVyc2Vfc2l6ZT0xMCkKICAgIGFzc2VydCBjYW5kaWRhdGVfcmVwb3J0WyJjYW5kaWRhdGVfcmVjYWxsIl0gPT0gMS4wCiAgICBhc3NlcnQgY2FuZGlkYXRlX3JlcG9ydFsiY29tcGxldGVfZW50aXR5X3JlY2FsbCJdID09IDEuMAoKICAgIGZyYW1lID0gcGQuRGF0YUZyYW1lKFsKICAgICAgICAoIlMxLTEiLCAiUzItMSIpLCAoIlMxLTEiLCAiUzMtMSIpLCAoIlMxLTEiLCAiUzItOSIpLAogICAgXSwgY29sdW1ucz1bInNvdXJjZTFfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9lbnRpdHlfaWQiXSkKICAgIHN0cmVhbWVkID0gY2FuZGlkYXRlX21ldHJpY3NfZnJvbV9mcmFtZXModHJ1dGgsIFtmcmFtZV0sIHRhcmdldF91bml2ZXJzZV9zaXplPTEwKQogICAgYXNzZXJ0IHN0cmVhbWVkID09IGNhbmRpZGF0ZV9yZXBvcnQK', 'c62f25f4da3dff5cfc1c75d091254cf5f9ec3253a69a892bfc22e9312c948598')

## Embedded source: `code/business_entity_resolution/tests/test_models_decisions.py`

~~~~python
import numpy as np
import pandas as pd

from business_entity_resolution.decisions import apply_thresholds, score_quantile_thresholds, sweep_thresholds
from business_entity_resolution.models import SGDPairModel


def test_sgd_save_load_and_thresholding(tmp_path):
    X = np.asarray([[0, 0], [1, 1], [0.1, 0], [0.9, 1]], dtype=np.float32)
    y = np.asarray([0, 1, 0, 1], dtype=np.int8)
    model = SGDPairModel(seed=3, max_iter=2000, tol=1e-5).fit(X, y)
    path = tmp_path / "model.joblib"
    model.save(path)
    loaded = SGDPairModel.load(path)
    np.testing.assert_allclose(model.predict_scores(X), loaded.predict_scores(X))

    scored = pd.DataFrame([
        ("S1-1", "S2-1", "S2", 0.9), ("S1-1", "S3-1", "S3", 0.4),
    ], columns=["source1_entity_id", "candidate_entity_id", "candidate_source", "score"])
    predictions = apply_thresholds(scored, ["S1-1", "S1-2"], 0.5)
    assert predictions["S1-1"] == {"S2-1"}
    assert predictions["S1-2"] == frozenset()
    report = sweep_thresholds(scored, {"S1-1": {"S2-1"}, "S1-2": set()}, [0.3, 0.5])
    assert report.iloc[0].threshold == 0.5


def test_threshold_grid_includes_all_empty_decision():
    scored = pd.DataFrame([
        ("S1-1", "S2-1", "S2", 0.9),
    ], columns=["source1_entity_id", "candidate_entity_id", "candidate_source", "score"])
    thresholds = score_quantile_thresholds(scored, count=3)
    assert max(thresholds) > 0.9
    predictions = apply_thresholds(scored, ["S1-1"], max(thresholds))
    assert predictions["S1-1"] == frozenset()

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_models_decisions.py', 'aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24uZGVjaXNpb25zIGltcG9ydCBhcHBseV90aHJlc2hvbGRzLCBzY29yZV9xdWFudGlsZV90aHJlc2hvbGRzLCBzd2VlcF90aHJlc2hvbGRzCmZyb20gYnVzaW5lc3NfZW50aXR5X3Jlc29sdXRpb24ubW9kZWxzIGltcG9ydCBTR0RQYWlyTW9kZWwKCgpkZWYgdGVzdF9zZ2Rfc2F2ZV9sb2FkX2FuZF90aHJlc2hvbGRpbmcodG1wX3BhdGgpOgogICAgWCA9IG5wLmFzYXJyYXkoW1swLCAwXSwgWzEsIDFdLCBbMC4xLCAwXSwgWzAuOSwgMV1dLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgeSA9IG5wLmFzYXJyYXkoWzAsIDEsIDAsIDFdLCBkdHlwZT1ucC5pbnQ4KQogICAgbW9kZWwgPSBTR0RQYWlyTW9kZWwoc2VlZD0zLCBtYXhfaXRlcj0yMDAwLCB0b2w9MWUtNSkuZml0KFgsIHkpCiAgICBwYXRoID0gdG1wX3BhdGggLyAibW9kZWwuam9ibGliIgogICAgbW9kZWwuc2F2ZShwYXRoKQogICAgbG9hZGVkID0gU0dEUGFpck1vZGVsLmxvYWQocGF0aCkKICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FsbGNsb3NlKG1vZGVsLnByZWRpY3Rfc2NvcmVzKFgpLCBsb2FkZWQucHJlZGljdF9zY29yZXMoWCkpCgogICAgc2NvcmVkID0gcGQuRGF0YUZyYW1lKFsKICAgICAgICAoIlMxLTEiLCAiUzItMSIsICJTMiIsIDAuOSksICgiUzEtMSIsICJTMy0xIiwgIlMzIiwgMC40KSwKICAgIF0sIGNvbHVtbnM9WyJzb3VyY2UxX2VudGl0eV9pZCIsICJjYW5kaWRhdGVfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9zb3VyY2UiLCAic2NvcmUiXSkKICAgIHByZWRpY3Rpb25zID0gYXBwbHlfdGhyZXNob2xkcyhzY29yZWQsIFsiUzEtMSIsICJTMS0yIl0sIDAuNSkKICAgIGFzc2VydCBwcmVkaWN0aW9uc1siUzEtMSJdID09IHsiUzItMSJ9CiAgICBhc3NlcnQgcHJlZGljdGlvbnNbIlMxLTIiXSA9PSBmcm96ZW5zZXQoKQogICAgcmVwb3J0ID0gc3dlZXBfdGhyZXNob2xkcyhzY29yZWQsIHsiUzEtMSI6IHsiUzItMSJ9LCAiUzEtMiI6IHNldCgpfSwgWzAuMywgMC41XSkKICAgIGFzc2VydCByZXBvcnQuaWxvY1swXS50aHJlc2hvbGQgPT0gMC41CgoKZGVmIHRlc3RfdGhyZXNob2xkX2dyaWRfaW5jbHVkZXNfYWxsX2VtcHR5X2RlY2lzaW9uKCk6CiAgICBzY29yZWQgPSBwZC5EYXRhRnJhbWUoWwogICAgICAgICgiUzEtMSIsICJTMi0xIiwgIlMyIiwgMC45KSwKICAgIF0sIGNvbHVtbnM9WyJzb3VyY2UxX2VudGl0eV9pZCIsICJjYW5kaWRhdGVfZW50aXR5X2lkIiwgImNhbmRpZGF0ZV9zb3VyY2UiLCAic2NvcmUiXSkKICAgIHRocmVzaG9sZHMgPSBzY29yZV9xdWFudGlsZV90aHJlc2hvbGRzKHNjb3JlZCwgY291bnQ9MykKICAgIGFzc2VydCBtYXgodGhyZXNob2xkcykgPiAwLjkKICAgIHByZWRpY3Rpb25zID0gYXBwbHlfdGhyZXNob2xkcyhzY29yZWQsIFsiUzEtMSJdLCBtYXgodGhyZXNob2xkcykpCiAgICBhc3NlcnQgcHJlZGljdGlvbnNbIlMxLTEiXSA9PSBmcm96ZW5zZXQoKQo=', '90dae8a361e06770cbf567ceaf18cd1392b9642f2cbbda8afe46218dc38675e4')

## Embedded source: `code/business_entity_resolution/tests/test_normalize.py`

~~~~python
from business_entity_resolution.normalize import canonical_text, core_name, normalize_records, numeric_tokens


def test_normalization_is_unicode_safe_and_deterministic(tiny_source):
    first = normalize_records(tiny_source)
    second = normalize_records(tiny_source)
    assert first.to_dict("records") == second.to_dict("records")
    assert first.loc[0, "country_norm"] == "france"
    assert "राम" in first.loc[1, "name_canonical"]
    assert tiny_source.loc[0, "business_name"] == "Acme & Sons Ltd"


def test_name_and_numeric_variants():
    assert canonical_text(" ACME & Sons, Ltd. ") == "acme and sons ltd"
    assert core_name("Acme Corporation") == "acme"
    assert numeric_tokens("12-A Road 75001") == ("12", "75001")


~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_normalize.py', 'ZnJvbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5ub3JtYWxpemUgaW1wb3J0IGNhbm9uaWNhbF90ZXh0LCBjb3JlX25hbWUsIG5vcm1hbGl6ZV9yZWNvcmRzLCBudW1lcmljX3Rva2VucwoKCmRlZiB0ZXN0X25vcm1hbGl6YXRpb25faXNfdW5pY29kZV9zYWZlX2FuZF9kZXRlcm1pbmlzdGljKHRpbnlfc291cmNlKToKICAgIGZpcnN0ID0gbm9ybWFsaXplX3JlY29yZHModGlueV9zb3VyY2UpCiAgICBzZWNvbmQgPSBub3JtYWxpemVfcmVjb3Jkcyh0aW55X3NvdXJjZSkKICAgIGFzc2VydCBmaXJzdC50b19kaWN0KCJyZWNvcmRzIikgPT0gc2Vjb25kLnRvX2RpY3QoInJlY29yZHMiKQogICAgYXNzZXJ0IGZpcnN0LmxvY1swLCAiY291bnRyeV9ub3JtIl0gPT0gImZyYW5jZSIKICAgIGFzc2VydCAi4KSw4KS+4KSuIiBpbiBmaXJzdC5sb2NbMSwgIm5hbWVfY2Fub25pY2FsIl0KICAgIGFzc2VydCB0aW55X3NvdXJjZS5sb2NbMCwgImJ1c2luZXNzX25hbWUiXSA9PSAiQWNtZSAmIFNvbnMgTHRkIgoKCmRlZiB0ZXN0X25hbWVfYW5kX251bWVyaWNfdmFyaWFudHMoKToKICAgIGFzc2VydCBjYW5vbmljYWxfdGV4dCgiIEFDTUUgJiBTb25zLCBMdGQuICIpID09ICJhY21lIGFuZCBzb25zIGx0ZCIKICAgIGFzc2VydCBjb3JlX25hbWUoIkFjbWUgQ29ycG9yYXRpb24iKSA9PSAiYWNtZSIKICAgIGFzc2VydCBudW1lcmljX3Rva2VucygiMTItQSBSb2FkIDc1MDAxIikgPT0gKCIxMiIsICI3NTAwMSIpCgo=', '27aebedfce48972bb9fea89866eb2604bc72f4ffdcc722f5b4da6e70f49cf528')

## Embedded source: `code/business_entity_resolution/tests/test_pipeline_smoke.py`

~~~~python
from business_entity_resolution.pipeline import run_smoke


def test_complete_smoke_pipeline(tmp_path):
    report = run_smoke(tmp_path)
    assert report["training_pairs"] > 0
    assert report["positive_pairs"] == 6
    assert report["preflight_errors"] == []


~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_pipeline_smoke.py', 'ZnJvbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5waXBlbGluZSBpbXBvcnQgcnVuX3Ntb2tlCgoKZGVmIHRlc3RfY29tcGxldGVfc21va2VfcGlwZWxpbmUodG1wX3BhdGgpOgogICAgcmVwb3J0ID0gcnVuX3Ntb2tlKHRtcF9wYXRoKQogICAgYXNzZXJ0IHJlcG9ydFsidHJhaW5pbmdfcGFpcnMiXSA+IDAKICAgIGFzc2VydCByZXBvcnRbInBvc2l0aXZlX3BhaXJzIl0gPT0gNgogICAgYXNzZXJ0IHJlcG9ydFsicHJlZmxpZ2h0X2Vycm9ycyJdID09IFtdCgo=', '457228efd291bcef679408539f71bace3f56c6883d8474f86bf8deb242fc59eb')

## Embedded source: `code/business_entity_resolution/tests/test_prepare_shards.py`

~~~~python
from argparse import Namespace
import json

import pandas as pd

from business_entity_resolution.cli import (
    _candidate_store,
    _feature_store,
    _load_config,
    _prepared_shard_store,
    command_build_features,
    command_generate_candidates,
    command_prepare,
    command_score,
    command_train,
)


def test_prepare_never_overwrites_earlier_chunk_rows(tmp_path):
    data_root = tmp_path / "dataset"
    train = data_root / "train"
    train.mkdir(parents=True)
    columns = ["entity_id", "business_name", "business_address", "country"]
    for source in (1, 2, 3):
        frame = pd.DataFrame([
            (f"S{source}-{index}", f"Business {index}", f"{index} Main Rd", "US")
            for index in range(1, 8)
        ], columns=columns)
        frame.to_csv(train / f"train_source{source}.tsv", sep="\t", index=False)
    pd.DataFrame([
        (f"S1-{index}", f"S2-{index},S3-{index}") for index in range(1, 8)
    ], columns=["source1_entity_id", "matched_entity_ids"]).to_csv(
        train / "train_ground_truth.tsv", sep="\t", index=False
    )

    config_path = tmp_path / "config.json"
    config_path.write_text(json.dumps({
        "data_root": str(data_root),
        "artifact_root": str(tmp_path / "artifacts"),
        "output_root": str(tmp_path / "output"),
        "run_id": "prepare-test",
        "batch_size": 2,
        "n_shards": 2,
        "resources": {"mode": "manual", "chunk_pairs": 2},
    }), encoding="utf-8")
    args = Namespace(config=str(config_path), max_rows=None, force=False, split="train")
    assert command_prepare(args) == 0

    config = _load_config(args)
    for source in (1, 2, 3):
        store = _prepared_shard_store(config, "train", source)
        assert store.is_complete()
        prepared = store.read_all()
        assert len(prepared) == 7
        assert prepared["entity_id"].nunique() == 7

    candidate_args = Namespace(
        config=str(config_path), max_rows=None, force=False, split="train", no_tfidf=False,
    )
    assert command_generate_candidates(candidate_args) == 0
    candidates = _candidate_store(config, "train").read_all()
    candidate_pairs = set(zip(candidates["source1_entity_id"], candidates["candidate_entity_id"]))
    expected_pairs = {
        (f"S1-{index}", f"S{source}-{index}")
        for index in range(1, 8)
        for source in (2, 3)
    }
    assert expected_pairs.issubset(candidate_pairs)

    feature_args = Namespace(
        config=str(config_path), max_rows=None, force=False, split="train",
        no_tfidf=False, no_embeddings=True,
    )
    assert command_build_features(feature_args) == 0
    features = _feature_store(config, "train").read_all()
    assert len(features) == len(candidates)
    assert int(features["label"].sum()) == 14

    split_dir = config.artifact_dir("splits")
    split_dir.mkdir(parents=True)
    pd.DataFrame([
        (f"S1-{index}", index % 2) for index in range(1, 8)
    ], columns=["source1_entity_id", "fold"]).to_parquet(
        split_dir / "s1_folds.parquet", index=False
    )
    train_args = Namespace(
        config=str(config_path), max_rows=None, force=False, model="sgd",
        validation_fold=0, exclude_folds=None, all_training_data=False,
        output_name="sgd-fold0",
    )
    assert command_train(train_args) == 0
    model_path = config.artifact_dir("models") / "sgd-fold0.joblib"
    assert model_path.is_file()

    score_args = Namespace(
        config=str(config_path), max_rows=None, force=False, split="train",
        model="sgd", model_path=str(model_path), validation_fold=0,
        all_training_data=False, output_name="fold0.parquet",
    )
    assert command_score(score_args) == 0
    scored = pd.read_parquet(config.artifact_dir("scores") / "fold0.parquet")
    assert not scored.empty
    assert set(scored["source1_entity_id"]).issubset({"S1-2", "S1-4", "S1-6"})

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_prepare_shards.py', 'ZnJvbSBhcmdwYXJzZSBpbXBvcnQgTmFtZXNwYWNlCmltcG9ydCBqc29uCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLmNsaSBpbXBvcnQgKAogICAgX2NhbmRpZGF0ZV9zdG9yZSwKICAgIF9mZWF0dXJlX3N0b3JlLAogICAgX2xvYWRfY29uZmlnLAogICAgX3ByZXBhcmVkX3NoYXJkX3N0b3JlLAogICAgY29tbWFuZF9idWlsZF9mZWF0dXJlcywKICAgIGNvbW1hbmRfZ2VuZXJhdGVfY2FuZGlkYXRlcywKICAgIGNvbW1hbmRfcHJlcGFyZSwKICAgIGNvbW1hbmRfc2NvcmUsCiAgICBjb21tYW5kX3RyYWluLAopCgoKZGVmIHRlc3RfcHJlcGFyZV9uZXZlcl9vdmVyd3JpdGVzX2VhcmxpZXJfY2h1bmtfcm93cyh0bXBfcGF0aCk6CiAgICBkYXRhX3Jvb3QgPSB0bXBfcGF0aCAvICJkYXRhc2V0IgogICAgdHJhaW4gPSBkYXRhX3Jvb3QgLyAidHJhaW4iCiAgICB0cmFpbi5ta2RpcihwYXJlbnRzPVRydWUpCiAgICBjb2x1bW5zID0gWyJlbnRpdHlfaWQiLCAiYnVzaW5lc3NfbmFtZSIsICJidXNpbmVzc19hZGRyZXNzIiwgImNvdW50cnkiXQogICAgZm9yIHNvdXJjZSBpbiAoMSwgMiwgMyk6CiAgICAgICAgZnJhbWUgPSBwZC5EYXRhRnJhbWUoWwogICAgICAgICAgICAoZiJTe3NvdXJjZX0te2luZGV4fSIsIGYiQnVzaW5lc3Mge2luZGV4fSIsIGYie2luZGV4fSBNYWluIFJkIiwgIlVTIikKICAgICAgICAgICAgZm9yIGluZGV4IGluIHJhbmdlKDEsIDgpCiAgICAgICAgXSwgY29sdW1ucz1jb2x1bW5zKQogICAgICAgIGZyYW1lLnRvX2Nzdih0cmFpbiAvIGYidHJhaW5fc291cmNle3NvdXJjZX0udHN2Iiwgc2VwPSJcdCIsIGluZGV4PUZhbHNlKQogICAgcGQuRGF0YUZyYW1lKFsKICAgICAgICAoZiJTMS17aW5kZXh9IiwgZiJTMi17aW5kZXh9LFMzLXtpbmRleH0iKSBmb3IgaW5kZXggaW4gcmFuZ2UoMSwgOCkKICAgIF0sIGNvbHVtbnM9WyJzb3VyY2UxX2VudGl0eV9pZCIsICJtYXRjaGVkX2VudGl0eV9pZHMiXSkudG9fY3N2KAogICAgICAgIHRyYWluIC8gInRyYWluX2dyb3VuZF90cnV0aC50c3YiLCBzZXA9Ilx0IiwgaW5kZXg9RmFsc2UKICAgICkKCiAgICBjb25maWdfcGF0aCA9IHRtcF9wYXRoIC8gImNvbmZpZy5qc29uIgogICAgY29uZmlnX3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKHsKICAgICAgICAiZGF0YV9yb290Ijogc3RyKGRhdGFfcm9vdCksCiAgICAgICAgImFydGlmYWN0X3Jvb3QiOiBzdHIodG1wX3BhdGggLyAiYXJ0aWZhY3RzIiksCiAgICAgICAgIm91dHB1dF9yb290Ijogc3RyKHRtcF9wYXRoIC8gIm91dHB1dCIpLAogICAgICAgICJydW5faWQiOiAicHJlcGFyZS10ZXN0IiwKICAgICAgICAiYmF0Y2hfc2l6ZSI6IDIsCiAgICAgICAgIm5fc2hhcmRzIjogMiwKICAgICAgICAicmVzb3VyY2VzIjogeyJtb2RlIjogIm1hbnVhbCIsICJjaHVua19wYWlycyI6IDJ9LAogICAgfSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBhcmdzID0gTmFtZXNwYWNlKGNvbmZpZz1zdHIoY29uZmlnX3BhdGgpLCBtYXhfcm93cz1Ob25lLCBmb3JjZT1GYWxzZSwgc3BsaXQ9InRyYWluIikKICAgIGFzc2VydCBjb21tYW5kX3ByZXBhcmUoYXJncykgPT0gMAoKICAgIGNvbmZpZyA9IF9sb2FkX2NvbmZpZyhhcmdzKQogICAgZm9yIHNvdXJjZSBpbiAoMSwgMiwgMyk6CiAgICAgICAgc3RvcmUgPSBfcHJlcGFyZWRfc2hhcmRfc3RvcmUoY29uZmlnLCAidHJhaW4iLCBzb3VyY2UpCiAgICAgICAgYXNzZXJ0IHN0b3JlLmlzX2NvbXBsZXRlKCkKICAgICAgICBwcmVwYXJlZCA9IHN0b3JlLnJlYWRfYWxsKCkKICAgICAgICBhc3NlcnQgbGVuKHByZXBhcmVkKSA9PSA3CiAgICAgICAgYXNzZXJ0IHByZXBhcmVkWyJlbnRpdHlfaWQiXS5udW5pcXVlKCkgPT0gNwoKICAgIGNhbmRpZGF0ZV9hcmdzID0gTmFtZXNwYWNlKAogICAgICAgIGNvbmZpZz1zdHIoY29uZmlnX3BhdGgpLCBtYXhfcm93cz1Ob25lLCBmb3JjZT1GYWxzZSwgc3BsaXQ9InRyYWluIiwgbm9fdGZpZGY9RmFsc2UsCiAgICApCiAgICBhc3NlcnQgY29tbWFuZF9nZW5lcmF0ZV9jYW5kaWRhdGVzKGNhbmRpZGF0ZV9hcmdzKSA9PSAwCiAgICBjYW5kaWRhdGVzID0gX2NhbmRpZGF0ZV9zdG9yZShjb25maWcsICJ0cmFpbiIpLnJlYWRfYWxsKCkKICAgIGNhbmRpZGF0ZV9wYWlycyA9IHNldCh6aXAoY2FuZGlkYXRlc1sic291cmNlMV9lbnRpdHlfaWQiXSwgY2FuZGlkYXRlc1siY2FuZGlkYXRlX2VudGl0eV9pZCJdKSkKICAgIGV4cGVjdGVkX3BhaXJzID0gewogICAgICAgIChmIlMxLXtpbmRleH0iLCBmIlN7c291cmNlfS17aW5kZXh9IikKICAgICAgICBmb3IgaW5kZXggaW4gcmFuZ2UoMSwgOCkKICAgICAgICBmb3Igc291cmNlIGluICgyLCAzKQogICAgfQogICAgYXNzZXJ0IGV4cGVjdGVkX3BhaXJzLmlzc3Vic2V0KGNhbmRpZGF0ZV9wYWlycykKCiAgICBmZWF0dXJlX2FyZ3MgPSBOYW1lc3BhY2UoCiAgICAgICAgY29uZmlnPXN0cihjb25maWdfcGF0aCksIG1heF9yb3dzPU5vbmUsIGZvcmNlPUZhbHNlLCBzcGxpdD0idHJhaW4iLAogICAgICAgIG5vX3RmaWRmPUZhbHNlLCBub19lbWJlZGRpbmdzPVRydWUsCiAgICApCiAgICBhc3NlcnQgY29tbWFuZF9idWlsZF9mZWF0dXJlcyhmZWF0dXJlX2FyZ3MpID09IDAKICAgIGZlYXR1cmVzID0gX2ZlYXR1cmVfc3RvcmUoY29uZmlnLCAidHJhaW4iKS5yZWFkX2FsbCgpCiAgICBhc3NlcnQgbGVuKGZlYXR1cmVzKSA9PSBsZW4oY2FuZGlkYXRlcykKICAgIGFzc2VydCBpbnQoZmVhdHVyZXNbImxhYmVsIl0uc3VtKCkpID09IDE0CgogICAgc3BsaXRfZGlyID0gY29uZmlnLmFydGlmYWN0X2Rpcigic3BsaXRzIikKICAgIHNwbGl0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUpCiAgICBwZC5EYXRhRnJhbWUoWwogICAgICAgIChmIlMxLXtpbmRleH0iLCBpbmRleCAlIDIpIGZvciBpbmRleCBpbiByYW5nZSgxLCA4KQogICAgXSwgY29sdW1ucz1bInNvdXJjZTFfZW50aXR5X2lkIiwgImZvbGQiXSkudG9fcGFycXVldCgKICAgICAgICBzcGxpdF9kaXIgLyAiczFfZm9sZHMucGFycXVldCIsIGluZGV4PUZhbHNlCiAgICApCiAgICB0cmFpbl9hcmdzID0gTmFtZXNwYWNlKAogICAgICAgIGNvbmZpZz1zdHIoY29uZmlnX3BhdGgpLCBtYXhfcm93cz1Ob25lLCBmb3JjZT1GYWxzZSwgbW9kZWw9InNnZCIsCiAgICAgICAgdmFsaWRhdGlvbl9mb2xkPTAsIGV4Y2x1ZGVfZm9sZHM9Tm9uZSwgYWxsX3RyYWluaW5nX2RhdGE9RmFsc2UsCiAgICAgICAgb3V0cHV0X25hbWU9InNnZC1mb2xkMCIsCiAgICApCiAgICBhc3NlcnQgY29tbWFuZF90cmFpbih0cmFpbl9hcmdzKSA9PSAwCiAgICBtb2RlbF9wYXRoID0gY29uZmlnLmFydGlmYWN0X2RpcigibW9kZWxzIikgLyAic2dkLWZvbGQwLmpvYmxpYiIKICAgIGFzc2VydCBtb2RlbF9wYXRoLmlzX2ZpbGUoKQoKICAgIHNjb3JlX2FyZ3MgPSBOYW1lc3BhY2UoCiAgICAgICAgY29uZmlnPXN0cihjb25maWdfcGF0aCksIG1heF9yb3dzPU5vbmUsIGZvcmNlPUZhbHNlLCBzcGxpdD0idHJhaW4iLAogICAgICAgIG1vZGVsPSJzZ2QiLCBtb2RlbF9wYXRoPXN0cihtb2RlbF9wYXRoKSwgdmFsaWRhdGlvbl9mb2xkPTAsCiAgICAgICAgYWxsX3RyYWluaW5nX2RhdGE9RmFsc2UsIG91dHB1dF9uYW1lPSJmb2xkMC5wYXJxdWV0IiwKICAgICkKICAgIGFzc2VydCBjb21tYW5kX3Njb3JlKHNjb3JlX2FyZ3MpID09IDAKICAgIHNjb3JlZCA9IHBkLnJlYWRfcGFycXVldChjb25maWcuYXJ0aWZhY3RfZGlyKCJzY29yZXMiKSAvICJmb2xkMC5wYXJxdWV0IikKICAgIGFzc2VydCBub3Qgc2NvcmVkLmVtcHR5CiAgICBhc3NlcnQgc2V0KHNjb3JlZFsic291cmNlMV9lbnRpdHlfaWQiXSkuaXNzdWJzZXQoeyJTMS0yIiwgIlMxLTQiLCAiUzEtNiJ9KQo=', 'b6911df5f081b4bd48ad4bb1fae185b64bddcd343da5e63b1b3d4281a0424f7e')

## Embedded source: `code/business_entity_resolution/tests/test_submission.py`

~~~~python
import pytest
import pandas as pd

from business_entity_resolution.preflight import preflight_outputs
from business_entity_resolution.submission import (
    OutputFormatError,
    write_submission_outputs,
    write_submission_outputs_sharded,
)


def test_submission_writer_and_preflight(tmp_path, smoke_frames):
    s1, s2, s3, _ = smoke_frames
    test_dir = tmp_path / "test"
    test_dir.mkdir()
    for source, frame in ((1, s1), (2, s2), (3, s3)):
        frame.to_csv(test_dir / f"test_source{source}.tsv", sep="\t", index=False)
    predictions = {"S1-1": {"S2-10"}, "S1-2": set(), "S1-3": set(), "S1-4": set()}
    candidates = {"S1-1": {"S2-10", "S3-11"}, "S1-2": set(), "S1-3": set(), "S1-4": set()}
    matching, candidate = write_submission_outputs(tmp_path / "out", s1.entity_id, predictions, candidates)
    assert preflight_outputs(matching, candidate, test_dir) == []


def test_writer_refuses_match_not_in_candidates(tmp_path, smoke_frames):
    s1, s2, s3, _ = smoke_frames
    # A matched ID that never appeared as a candidate must fail before writing,
    # so a pipeline bug cannot silently produce an invalid submission.
    with pytest.raises(OutputFormatError):
        write_submission_outputs(tmp_path / "out", s1.entity_id, {"S1-1": {"S2-10"}}, {})


def test_sharded_writer_preserves_empty_rows_and_candidate_subset(tmp_path):
    shards = [
        (["S1-1"], {"S1-1": {"S2-1"}}, {"S1-1": {"S2-1", "S3-1"}}),
        (["S1-2"], {"S1-2": set()}, {"S1-2": set()}),
    ]
    matching, candidates = write_submission_outputs_sharded(tmp_path, shards)
    matching_frame = pd.read_csv(matching, sep="\t", dtype=str, keep_default_na=False)
    candidate_frame = pd.read_csv(candidates, sep="\t", dtype=str, keep_default_na=False)
    assert matching_frame.to_dict("records") == [
        {"source1_entity_id": "S1-1", "matched_entity_ids": "S2-1"},
        {"source1_entity_id": "S1-2", "matched_entity_ids": ""},
    ]
    assert candidate_frame.loc[1, "candidate_entity_ids"] == ""

~~~~

In [ ]:
materialize('code/business_entity_resolution/tests/test_submission.py', 'aW1wb3J0IHB5dGVzdAppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIGJ1c2luZXNzX2VudGl0eV9yZXNvbHV0aW9uLnByZWZsaWdodCBpbXBvcnQgcHJlZmxpZ2h0X291dHB1dHMKZnJvbSBidXNpbmVzc19lbnRpdHlfcmVzb2x1dGlvbi5zdWJtaXNzaW9uIGltcG9ydCAoCiAgICBPdXRwdXRGb3JtYXRFcnJvciwKICAgIHdyaXRlX3N1Ym1pc3Npb25fb3V0cHV0cywKICAgIHdyaXRlX3N1Ym1pc3Npb25fb3V0cHV0c19zaGFyZGVkLAopCgoKZGVmIHRlc3Rfc3VibWlzc2lvbl93cml0ZXJfYW5kX3ByZWZsaWdodCh0bXBfcGF0aCwgc21va2VfZnJhbWVzKToKICAgIHMxLCBzMiwgczMsIF8gPSBzbW9rZV9mcmFtZXMKICAgIHRlc3RfZGlyID0gdG1wX3BhdGggLyAidGVzdCIKICAgIHRlc3RfZGlyLm1rZGlyKCkKICAgIGZvciBzb3VyY2UsIGZyYW1lIGluICgoMSwgczEpLCAoMiwgczIpLCAoMywgczMpKToKICAgICAgICBmcmFtZS50b19jc3YodGVzdF9kaXIgLyBmInRlc3Rfc291cmNle3NvdXJjZX0udHN2Iiwgc2VwPSJcdCIsIGluZGV4PUZhbHNlKQogICAgcHJlZGljdGlvbnMgPSB7IlMxLTEiOiB7IlMyLTEwIn0sICJTMS0yIjogc2V0KCksICJTMS0zIjogc2V0KCksICJTMS00Ijogc2V0KCl9CiAgICBjYW5kaWRhdGVzID0geyJTMS0xIjogeyJTMi0xMCIsICJTMy0xMSJ9LCAiUzEtMiI6IHNldCgpLCAiUzEtMyI6IHNldCgpLCAiUzEtNCI6IHNldCgpfQogICAgbWF0Y2hpbmcsIGNhbmRpZGF0ZSA9IHdyaXRlX3N1Ym1pc3Npb25fb3V0cHV0cyh0bXBfcGF0aCAvICJvdXQiLCBzMS5lbnRpdHlfaWQsIHByZWRpY3Rpb25zLCBjYW5kaWRhdGVzKQogICAgYXNzZXJ0IHByZWZsaWdodF9vdXRwdXRzKG1hdGNoaW5nLCBjYW5kaWRhdGUsIHRlc3RfZGlyKSA9PSBbXQoKCmRlZiB0ZXN0X3dyaXRlcl9yZWZ1c2VzX21hdGNoX25vdF9pbl9jYW5kaWRhdGVzKHRtcF9wYXRoLCBzbW9rZV9mcmFtZXMpOgogICAgczEsIHMyLCBzMywgXyA9IHNtb2tlX2ZyYW1lcwogICAgIyBBIG1hdGNoZWQgSUQgdGhhdCBuZXZlciBhcHBlYXJlZCBhcyBhIGNhbmRpZGF0ZSBtdXN0IGZhaWwgYmVmb3JlIHdyaXRpbmcsCiAgICAjIHNvIGEgcGlwZWxpbmUgYnVnIGNhbm5vdCBzaWxlbnRseSBwcm9kdWNlIGFuIGludmFsaWQgc3VibWlzc2lvbi4KICAgIHdpdGggcHl0ZXN0LnJhaXNlcyhPdXRwdXRGb3JtYXRFcnJvcik6CiAgICAgICAgd3JpdGVfc3VibWlzc2lvbl9vdXRwdXRzKHRtcF9wYXRoIC8gIm91dCIsIHMxLmVudGl0eV9pZCwgeyJTMS0xIjogeyJTMi0xMCJ9fSwge30pCgoKZGVmIHRlc3Rfc2hhcmRlZF93cml0ZXJfcHJlc2VydmVzX2VtcHR5X3Jvd3NfYW5kX2NhbmRpZGF0ZV9zdWJzZXQodG1wX3BhdGgpOgogICAgc2hhcmRzID0gWwogICAgICAgIChbIlMxLTEiXSwgeyJTMS0xIjogeyJTMi0xIn19LCB7IlMxLTEiOiB7IlMyLTEiLCAiUzMtMSJ9fSksCiAgICAgICAgKFsiUzEtMiJdLCB7IlMxLTIiOiBzZXQoKX0sIHsiUzEtMiI6IHNldCgpfSksCiAgICBdCiAgICBtYXRjaGluZywgY2FuZGlkYXRlcyA9IHdyaXRlX3N1Ym1pc3Npb25fb3V0cHV0c19zaGFyZGVkKHRtcF9wYXRoLCBzaGFyZHMpCiAgICBtYXRjaGluZ19mcmFtZSA9IHBkLnJlYWRfY3N2KG1hdGNoaW5nLCBzZXA9Ilx0IiwgZHR5cGU9c3RyLCBrZWVwX2RlZmF1bHRfbmE9RmFsc2UpCiAgICBjYW5kaWRhdGVfZnJhbWUgPSBwZC5yZWFkX2NzdihjYW5kaWRhdGVzLCBzZXA9Ilx0IiwgZHR5cGU9c3RyLCBrZWVwX2RlZmF1bHRfbmE9RmFsc2UpCiAgICBhc3NlcnQgbWF0Y2hpbmdfZnJhbWUudG9fZGljdCgicmVjb3JkcyIpID09IFsKICAgICAgICB7InNvdXJjZTFfZW50aXR5X2lkIjogIlMxLTEiLCAibWF0Y2hlZF9lbnRpdHlfaWRzIjogIlMyLTEifSwKICAgICAgICB7InNvdXJjZTFfZW50aXR5X2lkIjogIlMxLTIiLCAibWF0Y2hlZF9lbnRpdHlfaWRzIjogIiJ9LAogICAgXQogICAgYXNzZXJ0IGNhbmRpZGF0ZV9mcmFtZS5sb2NbMSwgImNhbmRpZGF0ZV9lbnRpdHlfaWRzIl0gPT0gIiIK', '67f4759b8d5ed8df62752f467134fe7e8d75f1d30794675f40427ae0958c4c91')

## Dependency installation and environment record

In [ ]:
requirements = SOURCE_ROOT / 'requirements.txt'
remote_requirements = SOURCE_ROOT / 'requirements-remote.txt'
if INSTALL_DEPS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(remote_requirements)])
environment = {'python': sys.version, 'platform': platform.platform(), 'branch': BRANCH_NAME, 'sha': BRANCH_SHA}
RUN_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
environment

## Import and source-integrity gate

In [ ]:
sys.path.insert(0, str(SOURCE_ROOT / 'src'))
from business_entity_resolution.cli import main as cli_main
from business_entity_resolution.config import ProjectConfig
from business_entity_resolution.data import load_source, load_ground_truth
assert DATA_ROOT.joinpath('train', 'train_source1.tsv').is_file(), DATA_ROOT
train_preview = load_source(DATA_ROOT, 'train', 1, nrows=5)
print(train_preview.to_string(index=False))
print('embedded package import and real training-data read: PASS')

## Runtime configuration

In [ ]:
base = json.loads((SOURCE_ROOT / 'configs' / 'base.json').read_text(encoding='utf-8'))
base.update({'data_root': str(DATA_ROOT), 'artifact_root': str(RUN_ROOT / 'artifacts'), 'output_root': str(OUTPUT_ROOT), 'run_id': os.environ.get('BER_RUN_ID', 'notebook'), 'seed': SEED})
base.setdefault('embeddings', {})['enable'] = 'false'
base.setdefault('resources', {})['enable_embeddings'] = 'false'
runtime_config = RUN_ROOT / 'runtime_config.json'
runtime_config.write_text(json.dumps(base, indent=2), encoding='utf-8')
runtime_config

## Branch-native stage runner

In [ ]:
def run_cli(*arguments: str) -> None:
    print('CLI:', ' '.join(arguments), flush=True)
    result = int(cli_main(list(arguments)))
    if result:
        raise RuntimeError(f'branch CLI failed with exit code {result}: {arguments}')

def run_functional_smoke() -> None:
    # Native synthetic smoke covers normalization, blocking, features, SGD training, inference, and TSV formatting.
    run_cli('smoke', '--config', str(runtime_config))

def run_native_full() -> None:
    if not RUN_TEST:
        raise RuntimeError('Full native pipeline includes test inference; set BER_RUN_TEST=1 only after freeze')
    env = os.environ.copy()
    env.update({'BER_DATA_ROOT': str(DATA_ROOT), 'BER_ARTIFACT_ROOT': str(RUN_ROOT / 'artifacts'), 'BER_OUTPUT_ROOT': str(OUTPUT_ROOT), 'BER_RUN_ID': 'native-full'})
    subprocess.check_call([sys.executable, str(SOURCE_ROOT / 'run_pipeline.py'), '--profile', 'full', '--skip-embeddings', '--validator', str(VALIDATOR_PATH)], env=env)

if 'functional' in RUN_STAGES:
    run_functional_smoke()
if 'native-full' in RUN_STAGES:
    run_native_full()

## Common evaluation contract

Authoritative comparison is executed by the shared, branch-isolated runner using the same fold manifest. The runner calls only this embedded branch package for normalization, blocking, features, models, and decisions. It never creates candidates. Experiment 0 output remains **REPRODUCTION ONLY — NOT A COMPARATIVE SCORE**.

## Frozen test inference and submission validation

In [ ]:
def require_test_unlock() -> dict:
    unlock = RUN_ROOT / 'FROZEN_TEST_UNLOCK.json'
    if not RUN_TEST or not unlock.is_file():
        raise RuntimeError('test inference remains locked until RUN_TEST=1 and FROZEN_TEST_UNLOCK.json exists')
    payload = json.loads(unlock.read_text(encoding='utf-8'))
    if payload.get('branch_sha') != BRANCH_SHA:
        raise RuntimeError('test unlock SHA does not match embedded branch')
    return payload

if 'test' in RUN_STAGES:
    require_test_unlock()
    run_native_full()

## Experiment manifest

In [ ]:
embedded_manifest = '[{"bytes": 9383, "path": "code/business_entity_resolution/README.md", "sha256": "dc1198aace37b4f450d9a9d42341298a6f538f43c422f93e09834bc860760dbd"}, {"bytes": 843, "path": "code/business_entity_resolution/THIRD_PARTY_LICENSES.md", "sha256": "45db09ffe6f8f0bdc6e0f3ff2a2303eae47a270496271790883b012429c976c6"}, {"bytes": 924, "path": "code/business_entity_resolution/configs/base.json", "sha256": "f5c0404c41e0a2d19a2cdd76dc6f537d020c19a4d43c896876a51f5dca616181"}, {"bytes": 964, "path": "code/business_entity_resolution/configs/local_smoke.json", "sha256": "e1334683ac088988f28cea66831e4f10401b4102b31a233cccb2970755545570"}, {"bytes": 1045, "path": "code/business_entity_resolution/configs/mini.json", "sha256": "9e1b9f702667851e58e1baa5790c483dd9543ec24f8c544868ee079d825b0d86"}, {"bytes": 1111, "path": "code/business_entity_resolution/configs/remote_full.json", "sha256": "ba2827dfb1ef16b08f9b226c9f817d37df4e732c595253c0f4ade8762cc38ca6"}, {"bytes": 692, "path": "code/business_entity_resolution/pyproject.toml", "sha256": "6f9bc6c00dc3c490ddedd82e05fbf2b7d611221da8d793531378a4baa41da91a"}, {"bytes": 194, "path": "code/business_entity_resolution/requirements-embeddings.txt", "sha256": "74742ed024654a5d9318d6ecbe82dd01b4152835796f6e68da64d5f7f1d7842f"}, {"bytes": 109, "path": "code/business_entity_resolution/requirements-remote.txt", "sha256": "77f9a1de8c7fc618f3ad5fb5731f922c0518189111d0fd0bdf086116eb11dfc7"}, {"bytes": 123, "path": "code/business_entity_resolution/requirements.txt", "sha256": "b0f54f1f482faf0fb2931c4e4da99153e20ef48eeab14d9f043ed37345952093"}, {"bytes": 8854, "path": "code/business_entity_resolution/run_pipeline.py", "sha256": "f145e61f5ad0baca8aaba18680698d4ecbdb8d4278f542c78111375eee84e8a6"}, {"bytes": 82, "path": "code/business_entity_resolution/src/business_entity_resolution/__init__.py", "sha256": "4f8c899ddf314d9cc236a94c622a6b761421a648cd33b1518b382e0db4c1e7aa"}, {"bytes": 49, "path": "code/business_entity_resolution/src/business_entity_resolution/__main__.py", "sha256": "76fa916e93e0ae1a6c1bb51dcc0bcf289c014e9f760a47165f8a84401ccf41d7"}, {"bytes": 2104, "path": "code/business_entity_resolution/src/business_entity_resolution/artifacts.py", "sha256": "2c5a65e0886c9580999218e331a475c7b2f661e7d4f7eeca9e519bd22b9eaf76"}, {"bytes": 2946, "path": "code/business_entity_resolution/src/business_entity_resolution/audit.py", "sha256": "a24328e56d3bd8d2e9dc557d98402e63e3ebc8d70c2d21fbba28db8bdc694124"}, {"bytes": 371, "path": "code/business_entity_resolution/src/business_entity_resolution/blocking/__init__.py", "sha256": "d36dbf3a5bac0ab841b0c148fc674118eaef4f8959f6baaf3d633404760ae988"}, {"bytes": 2745, "path": "code/business_entity_resolution/src/business_entity_resolution/blocking/base.py", "sha256": "ade39457e4f7005cbe90f9a9949c355de192435ca79e3ef9bb17226470ac554b"}, {"bytes": 3512, "path": "code/business_entity_resolution/src/business_entity_resolution/blocking/exact.py", "sha256": "351ae3bd3e8cbcdcf47ef854329e6a35c2381a25da478bbc08309fd82d4141b0"}, {"bytes": 5208, "path": "code/business_entity_resolution/src/business_entity_resolution/blocking/tfidf.py", "sha256": "b56e0c79910cee715455dfef47977e015aec8e364cb3078ec2b25b95a5cff1c7"}, {"bytes": 3798, "path": "code/business_entity_resolution/src/business_entity_resolution/blocking/tokens.py", "sha256": "9f908901fe768215457daad1a7fc675518a280cebea74bc3f45adb3d730d0eaa"}, {"bytes": 2576, "path": "code/business_entity_resolution/src/business_entity_resolution/blocking/union.py", "sha256": "5c5371a5e900440e33c8d8a225af965c5d31c1613d0fe88670dd1c2893a59e16"}, {"bytes": 6999, "path": "code/business_entity_resolution/src/business_entity_resolution/checkpoint.py", "sha256": "e9b916c832b56a7a35eedbfb3d5fb12a7b11d8f9da48bbd5e94f874b537b7a10"}, {"bytes": 48431, "path": "code/business_entity_resolution/src/business_entity_resolution/cli.py", "sha256": "2e35bd0b08a5f7ec6e0fcab8d93c625bb8d060fcfc5c815875cd938ef4732fc1"}, {"bytes": 5091, "path": "code/business_entity_resolution/src/business_entity_resolution/config.py", "sha256": "422bb6957b275d5ac35a4cbf3a35a10aeadfb920e2dc9173ca5a5da0499c8333"}, {"bytes": 4245, "path": "code/business_entity_resolution/src/business_entity_resolution/data.py", "sha256": "9bf747141ea31df111fc13e1b0df99ba2f5a70f3d70948d2499fa5e1eb6f3394"}, {"bytes": 2247, "path": "code/business_entity_resolution/src/business_entity_resolution/decisions.py", "sha256": "a77a755384aef20a00e9862fc496c0a5ac61c75638cbc9b02ce0a73b3e6f2675"}, {"bytes": 8152, "path": "code/business_entity_resolution/src/business_entity_resolution/embeddings.py", "sha256": "2fac3c61edb6f97a700109c44464bb26f88f2faba787042c567f276765338e5d"}, {"bytes": 2264, "path": "code/business_entity_resolution/src/business_entity_resolution/error_analysis.py", "sha256": "c6bde6f001882b3f04e2a02705f440fb48de02196aa2318f6aeb29dcf24dffbc"}, {"bytes": 11101, "path": "code/business_entity_resolution/src/business_entity_resolution/features.py", "sha256": "19631d8ce73f9cc1ae51f83079dce2301487facb90ecf784b9b645652aee2aac"}, {"bytes": 2580, "path": "code/business_entity_resolution/src/business_entity_resolution/inference.py", "sha256": "7e4e9b93277e4f6096caefcdcfec786316a26e7b8e2e688eee93114fb6bdee0f"}, {"bytes": 3040, "path": "code/business_entity_resolution/src/business_entity_resolution/labels.py", "sha256": "f61e48606892201cf5ef70ea22c24bcf28aaa63a5574177a01d66ec2b09b1caa"}, {"bytes": 5124, "path": "code/business_entity_resolution/src/business_entity_resolution/logging.py", "sha256": "4e384d2fc1af6504bb4e6f1be5a4783b55e8bc6b4287ba5b49dd0b5ea8e489d3"}, {"bytes": 6485, "path": "code/business_entity_resolution/src/business_entity_resolution/metrics.py", "sha256": "a4140f3f0f7eb6190fb1500ec455c6bc50693b1f42e9f751ad187aa1e750cc73"}, {"bytes": 9500, "path": "code/business_entity_resolution/src/business_entity_resolution/mini.py", "sha256": "bd2dc5042c8452e0eae73f3d9a985cbc4debfcb8536bce331f4c4a5d0f4661f9"}, {"bytes": 879, "path": "code/business_entity_resolution/src/business_entity_resolution/mining.py", "sha256": "7426776c2c7b1024091147ed71f3fc1d7bc8b1cc0788ac63a2bfa254ee31e2a4"}, {"bytes": 182, "path": "code/business_entity_resolution/src/business_entity_resolution/models/__init__.py", "sha256": "c1bb37397bbc4d455b2560ebd960c00db35d610d37f236db8ce661e17bd1a513"}, {"bytes": 463, "path": "code/business_entity_resolution/src/business_entity_resolution/models/base.py", "sha256": "2d34c84ae027ebdfc727d8afbd5f54e5ec86cc94729dbe936bc9ab5d310bde39"}, {"bytes": 1810, "path": "code/business_entity_resolution/src/business_entity_resolution/models/deterministic.py", "sha256": "f4fa9de801fd4ca1b286c19c052bbc85ecf5adf451ae8037dc8fb5fdd45035e0"}, {"bytes": 2917, "path": "code/business_entity_resolution/src/business_entity_resolution/models/lightgbm_model.py", "sha256": "b020472c24ec64d6ddbd2af8f45f36b3faed75b30cc12221b34932ac38ea17c6"}, {"bytes": 3199, "path": "code/business_entity_resolution/src/business_entity_resolution/models/sklearn_model.py", "sha256": "718ac845bf1df831fa70a76bbd0e2a67a6e1c3655db37560ea427036fa1a6cd2"}, {"bytes": 4670, "path": "code/business_entity_resolution/src/business_entity_resolution/normalize.py", "sha256": "daaee40eec9215c7cb9a7b9675da536833a48159b9d662734c28399f078b4717"}, {"bytes": 3913, "path": "code/business_entity_resolution/src/business_entity_resolution/pipeline.py", "sha256": "a8c735773721c379f773250384f8eb779a91f4aae24f92b478c6a53600b5d55b"}, {"bytes": 7232, "path": "code/business_entity_resolution/src/business_entity_resolution/preflight.py", "sha256": "34b067868517b097dc7ab2f88e90fd470258858b0ffc2909352214505fcdb448"}, {"bytes": 5183, "path": "code/business_entity_resolution/src/business_entity_resolution/resources.py", "sha256": "ddda06193226d6e52ebce24304eb3430aeaac65c41a2e70f96a456e13c1d3a3d"}, {"bytes": 1924, "path": "code/business_entity_resolution/src/business_entity_resolution/sampling.py", "sha256": "f041a2b8542368de32fd47400932c21d5ef78ebd9e6f9bfe5430fb62789ddb05"}, {"bytes": 938, "path": "code/business_entity_resolution/src/business_entity_resolution/schemas.py", "sha256": "a43ba01cf66c8889cbdfbd1b2d7fa0dca9e9b5535ce535abc849f4ba837ceabe"}, {"bytes": 2983, "path": "code/business_entity_resolution/src/business_entity_resolution/splits.py", "sha256": "95b91a41e1484c8fa2d84e616b278e0c8f8714f98f848d7989297519e88ef1a7"}, {"bytes": 7147, "path": "code/business_entity_resolution/src/business_entity_resolution/submission.py", "sha256": "3c71eeb256d73569492dcdcfb37133de941b22c80fd7426bcb9f40f22162dbca"}, {"bytes": 250, "path": "code/business_entity_resolution/tests/README.md", "sha256": "c81d103f9980f09f0d3d2b3aaf196742ec95ece05cb05528014b2f5a7923372e"}, {"bytes": 516, "path": "code/business_entity_resolution/tests/conftest.py", "sha256": "7709ad9fb4df7c0c85699e6250b765a7af1fb47374810c990bf048ba4f3d0872"}, {"bytes": 3888, "path": "code/business_entity_resolution/tests/test_blocking.py", "sha256": "dac92055cdfed10b4b3876180b9fcfce5449ebac0f6b9e743299171577a2ae63"}, {"bytes": 2393, "path": "code/business_entity_resolution/tests/test_cli_regressions.py", "sha256": "c1e5b2766cce865e1a32fd13e58015add6b02a2bae4c119ae338c0e862a7227a"}, {"bytes": 534, "path": "code/business_entity_resolution/tests/test_config.py", "sha256": "dd9f4f7db25b67f7eba5d6b42f99f3829858a4d8b2d0c48f564cd21ff2835cd5"}, {"bytes": 1591, "path": "code/business_entity_resolution/tests/test_data.py", "sha256": "7c0c7a3db4a507f3596f1d4c6832f416dc9e2d7254686b2a78159de725796222"}, {"bytes": 1730, "path": "code/business_entity_resolution/tests/test_features.py", "sha256": "85d45002a8b4bfc932bb9744f5f40bfca00177f39b60f0ee770427fc7afce2fa"}, {"bytes": 1454, "path": "code/business_entity_resolution/tests/test_labels_splits.py", "sha256": "fe9f0f666dfecaf678be50c7b6aae0ed34b720697abc7c0dfa8574bd57f6e84f"}, {"bytes": 1215, "path": "code/business_entity_resolution/tests/test_metrics.py", "sha256": "c62f25f4da3dff5cfc1c75d091254cf5f9ec3253a69a892bfc22e9312c948598"}, {"bytes": 1520, "path": "code/business_entity_resolution/tests/test_models_decisions.py", "sha256": "90dae8a361e06770cbf567ceaf18cd1392b9642f2cbbda8afe46218dc38675e4"}, {"bytes": 740, "path": "code/business_entity_resolution/tests/test_normalize.py", "sha256": "27aebedfce48972bb9fea89866eb2604bc72f4ffdcc722f5b4da6e70f49cf528"}, {"bytes": 263, "path": "code/business_entity_resolution/tests/test_pipeline_smoke.py", "sha256": "457228efd291bcef679408539f71bace3f56c6883d8474f86bf8deb242fc59eb"}, {"bytes": 3878, "path": "code/business_entity_resolution/tests/test_prepare_shards.py", "sha256": "b6911df5f081b4bd48ad4bb1fae185b64bddcd343da5e63b1b3d4281a0424f7e"}, {"bytes": 2010, "path": "code/business_entity_resolution/tests/test_submission.py", "sha256": "67f4759b8d5ed8df62752f467134fe7e8d75f1d30794675f40427ae0958c4c91"}]'
embedded_manifest = json.loads(embedded_manifest)
manifest = {'branch': BRANCH_NAME, 'branch_sha': BRANCH_SHA, 'seed': SEED, 'data_root': str(DATA_ROOT), 'run_root': str(RUN_ROOT), 'test_unlocked': RUN_TEST, 'source_files': embedded_manifest, 'experiment_0_label': 'REPRODUCTION ONLY — NOT A COMPARATIVE SCORE'}
manifest_path = RUN_ROOT / 'notebook_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
print(manifest_path)